In [63]:
# ============================================================
# OIP v1.0.39 — Waymo Safety Claim Audit
# C01 — Source Inventory & SHA-256 Provenance Lock
# ============================================================

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone
from collections import defaultdict

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C01")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


# ------------------------------------------------------------
# 1. Inventory
# ------------------------------------------------------------
files = sorted([p for p in INPUT_ROOT.rglob("*") if p.is_file()])

records = []
errors = []

for path in files:
    try:
        stat = path.stat()
        sha = sha256_file(path)

        records.append({
            "relative_path": str(path.relative_to(INPUT_ROOT)),
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": stat.st_size,
            "sha256": sha,
        })

    except Exception as e:
        errors.append({
            "path": str(path),
            "error_type": type(e).__name__,
            "error_message": str(e),
        })


# ------------------------------------------------------------
# 2. Duplicate diagnostic
# ------------------------------------------------------------
hash_groups = defaultdict(list)

for r in records:
    hash_groups[r["sha256"]].append(r["relative_path"])

duplicate_groups = {
    sha: paths
    for sha, paths in hash_groups.items()
    if len(paths) > 1
}


# ------------------------------------------------------------
# 3. File-type summary
# ------------------------------------------------------------
extension_counts = defaultdict(int)

for r in records:
    ext = r["extension"] if r["extension"] else "[NO_EXTENSION]"
    extension_counts[ext] += 1


# ------------------------------------------------------------
# 4. Waymo package structural check
# ------------------------------------------------------------
# We deliberately DO NOT invent or hard-code filenames.
# Exact filenames are established by this inventory.

csv_files = [
    r["file_name"]
    for r in records
    if r["extension"] == ".csv"
]

pdf_files = [
    r["file_name"]
    for r in records
    if r["extension"] == ".pdf"
]

# Expected package structure from the current audit plan:
# 4 CSV + supporting documentation.
# Exact corpus completeness remains unassessed until C04.

package_structure = {
    "csv_count": len(csv_files),
    "pdf_count": len(pdf_files),
    "csv_files": sorted(csv_files),
    "pdf_files": sorted(pdf_files),
    "expected_core_csv_count": 4,
    "core_csv_count_satisfied": len(csv_files) >= 4,
}


# ------------------------------------------------------------
# 5. Status
# ------------------------------------------------------------
if len(files) == 0:
    status = "FAIL_CLOSED"

elif len(errors) > 0:
    status = "FAIL_CLOSED"

elif len(csv_files) < 4:
    status = "PENDING_EVIDENCE"

else:
    status = "PASS_SOURCE_INVENTORY"


# Important boundary:
# C01 does NOT establish that these files constitute the
# complete Waymo evidence corpus.
evidence_corpus_completeness = "NOT_ASSESSED"


# ------------------------------------------------------------
# 6. C01 artifact
# ------------------------------------------------------------
artifact = {
    "protocol_version": "OIP v1.0.39",
    "audit_type": "Corporate Safety Claim Audit",
    "dataset_target": "Waymo Safety Claims",

    "cell": "C01",
    "cell_name": "Source Inventory & SHA-256 Provenance Lock",

    "status": status,

    "boundary": {
        "construct_audit": "NOT_APPLICABLE",
        "road_safety_verdict": "NOT_ISSUED",
        "waymo_safe_or_unsafe": "NOT_CLAIMED",
        "claims_authorized": 0,
        "empirical_validity_claim": False,
    },

    "authorization_state": {
        "source_inventory_established": len(files) > 0 and len(errors) == 0,
        "claim_authorization": False,
        "evidence_sufficiency": "NOT_ASSESSED",
        "downstream_execution": "GATED_BY_C02",
    },

    "evidence_corpus_completeness": evidence_corpus_completeness,

    "inventory_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "input_root": str(INPUT_ROOT),

    "total_files_detected": len(files),
    "successfully_hashed": len(records),
    "hash_errors": len(errors),

    "unique_sha256": len(hash_groups),
    "duplicate_hash_groups": len(duplicate_groups),

    "extension_counts":
        dict(sorted(extension_counts.items())),

    "package_structure":
        package_structure,

    "files": records,
    "errors": errors,
    "duplicate_hash_groups_detail": duplicate_groups,
}


# ------------------------------------------------------------
# 7. JSON lock
# ------------------------------------------------------------
json_path = OUTPUT_DIR / "C01_SOURCE_INVENTORY_SHA256_LOCK.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 8. CSV manifest
# ------------------------------------------------------------
def csv_safe(value):
    return (
        str(value)
        .replace(",", " ")
        .replace("\n", " ")
        .replace("\r", " ")
    )


manifest_path = OUTPUT_DIR / "C01_SOURCE_MANIFEST.csv"

with open(manifest_path, "w", encoding="utf-8") as f:

    headers = [
        "relative_path",
        "file_name",
        "extension",
        "size_bytes",
        "sha256",
    ]

    f.write(",".join(headers) + "\n")

    for r in records:

        row = [
            csv_safe(r["relative_path"]),
            csv_safe(r["file_name"]),
            csv_safe(r["extension"]),
            str(r["size_bytes"]),
            r["sha256"],
        ]

        f.write(",".join(row) + "\n")


# ------------------------------------------------------------
# 9. Console audit output
# ------------------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C01 SOURCE INVENTORY & SHA-256 LOCK")
print("=" * 72)

print(f"Audit Type              : Corporate Safety Claim Audit")
print(f"Target                  : Waymo Safety Claims")
print(f"GFL/IDS/AML             : NOT APPLICABLE")
print(f"Road Safety Verdict     : NOT_ISSUED")
print(f"Waymo Safe/Unsafe       : NOT_CLAIMED")

print()
print(f"Files detected          : {len(files)}")
print(f"Successfully hashed     : {len(records)}")
print(f"Hash errors             : {len(errors)}")
print(f"Unique SHA-256          : {len(hash_groups)}")
print(f"Duplicate hash groups   : {len(duplicate_groups)}")

print()
print(f"CSV files detected      : {len(csv_files)}")
print(f"PDF files detected      : {len(pdf_files)}")

print()
print("CSV FILES:")
for name in sorted(csv_files):
    print(f" - {name}")

print()
print("PDF FILES:")
for name in sorted(pdf_files):
    print(f" - {name}")

print()
print(f"Evidence completeness   : {evidence_corpus_completeness}")
print(f"Claim authorization     : FALSE")
print(f"C01 STATUS              : {status}")

if errors:
    print("\nERRORS:")
    for e in errors:
        print(e)

if duplicate_groups:
    print("\nDUPLICATE SHA-256 GROUPS:")
    for sha, paths in duplicate_groups.items():
        print(f"SHA256: {sha}")
        for p in paths:
            print(f"   - {p}")

print()
print("Artifacts:")
print(f"JSON : {json_path}")
print(f"CSV  : {manifest_path}")

print("=" * 72)

OIP v1.0.39 — C01 SOURCE INVENTORY & SHA-256 LOCK
Audit Type              : Corporate Safety Claim Audit
Target                  : Waymo Safety Claims
GFL/IDS/AML             : NOT APPLICABLE
Road Safety Verdict     : NOT_ISSUED
Waymo Safe/Unsafe       : NOT_CLAIMED

Files detected          : 7
Successfully hashed     : 7
Hash errors             : 0
Unique SHA-256          : 7
Duplicate hash groups   : 0

CSV files detected      : 4
PDF files detected      : 3

CSV FILES:
 - CSV1 - RO Miles per Location 202009-202606-2022benchmark.csv
 - CSV2 - Crashes with SGO ID and Group Membership 202009-202606-2022benchmark.csv
 - CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv
 - CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

PDF FILES:
 - 2312.12675v3.pdf
 - 2312.13228v2.pdf
 - Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf

Evidence completeness   : NOT_ASSESSED
Claim authorization     : FALSE
C01 STATUS   

In [64]:
# ============================================================
# OIP v1.0.39 — Waymo Safety Claim Audit
# C02 — File Integrity & Reader Check
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

import pandas as pd
from pypdf import PdfReader


INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C02")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Discover files from C01-style inventory
# ------------------------------------------------------------

files = sorted([
    p for p in INPUT_ROOT.rglob("*")
    if p.is_file()
])


# ------------------------------------------------------------
# 2. Reader test
# ------------------------------------------------------------

results = []
errors = []

for path in files:

    ext = path.suffix.lower()

    record = {
        "file_name": path.name,
        "relative_path": str(path.relative_to(INPUT_ROOT)),
        "extension": ext,
        "size_bytes": path.stat().st_size,
        "reader": None,
        "read_status": None,
        "rows_sampled": None,
        "columns_detected": None,
        "pages_detected": None,
        "error_type": None,
        "error_message": None,
    }

    try:

        # ----------------------------------------------------
        # CSV reader test
        # ----------------------------------------------------
        if ext == ".csv":

            record["reader"] = "pandas"

            df_sample = pd.read_csv(
                path,
                nrows=5
            )

            record["read_status"] = "READ_OK"
            record["rows_sampled"] = len(df_sample)
            record["columns_detected"] = len(df_sample.columns)

        # ----------------------------------------------------
        # PDF reader test
        # ----------------------------------------------------
        elif ext == ".pdf":

            record["reader"] = "pypdf"

            reader = PdfReader(str(path))

            page_count = len(reader.pages)

            record["read_status"] = "READ_OK"
            record["pages_detected"] = page_count

        else:

            record["reader"] = "UNSUPPORTED_FOR_C02"
            record["read_status"] = "PENDING_EVIDENCE"

        results.append(record)

    except Exception as e:

        record["read_status"] = "READ_FAIL"
        record["error_type"] = type(e).__name__
        record["error_message"] = str(e)

        results.append(record)

        errors.append(record)


# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

total_files = len(results)
read_ok = sum(
    1 for r in results
    if r["read_status"] == "READ_OK"
)

read_fail = sum(
    1 for r in results
    if r["read_status"] == "READ_FAIL"
)

pending = sum(
    1 for r in results
    if r["read_status"] == "PENDING_EVIDENCE"
)


# ------------------------------------------------------------
# 4. Fail-closed decision
# ------------------------------------------------------------

if total_files == 0:

    status = "FAIL_CLOSED"

elif read_fail > 0:

    status = "FAIL_CLOSED"

elif pending > 0:

    status = "PENDING_EVIDENCE"

elif read_ok == total_files:

    status = "PASS_READER_CHECK"

else:

    status = "FAIL_CLOSED"


# ------------------------------------------------------------
# 5. Boundary
# ------------------------------------------------------------

artifact = {

    "protocol_version": "OIP v1.0.39",

    "audit_type": "Corporate Safety Claim Audit",

    "target": "Waymo Safety Claims",

    "cell": "C02",

    "cell_name": "File Integrity & Reader Check",

    "status": status,

    "boundary": {
        "schema_authorization": False,
        "claim_authorization": False,
        "benchmark_authorization": False,
        "statistical_authorization": False,
        "road_safety_verdict": "NOT_ISSUED",
    },

    "reader_contract": {
        "csv_reader": "pandas",
        "pdf_reader": "pypdf",
    },

    "execution_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "total_files": total_files,

    "read_ok": read_ok,

    "read_fail": read_fail,

    "pending": pending,

    "results": results,

    "errors": errors,
}


# ------------------------------------------------------------
# 6. Save artifact
# ------------------------------------------------------------

json_path = (
    OUTPUT_DIR /
    "C02_FILE_INTEGRITY_READER_CHECK.json"
)

with open(json_path, "w", encoding="utf-8") as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 7. Console output
# ------------------------------------------------------------

print("=" * 72)
print("OIP v1.0.39 — C02 FILE INTEGRITY & READER CHECK")
print("=" * 72)

print(f"Audit Type          : Corporate Safety Claim Audit")
print(f"Target              : Waymo Safety Claims")

print()
print(f"Total files         : {total_files}")
print(f"Read OK             : {read_ok}")
print(f"Read FAIL           : {read_fail}")
print(f"Pending             : {pending}")

print()
print("FILE RESULTS:")

for r in results:

    print(
        f" - {r['file_name']} | "
        f"{r['extension']} | "
        f"{r['read_status']} | "
        f"reader={r['reader']}"
    )

    if r["rows_sampled"] is not None:
        print(
            f"   CSV sample rows={r['rows_sampled']} | "
            f"columns={r['columns_detected']}"
        )

    if r["pages_detected"] is not None:
        print(
            f"   PDF pages={r['pages_detected']}"
        )

    if r["error_message"]:
        print(
            f"   ERROR: {r['error_type']} — "
            f"{r['error_message']}"
        )

print()
print(f"C02 STATUS          : {status}")

print()
print(f"Artifact            : {json_path}")

print("=" * 72)

OIP v1.0.39 — C02 FILE INTEGRITY & READER CHECK
Audit Type          : Corporate Safety Claim Audit
Target              : Waymo Safety Claims

Total files         : 7
Read OK             : 7
Read FAIL           : 0
Pending             : 0

FILE RESULTS:
 - 2312.12675v3.pdf | .pdf | READ_OK | reader=pypdf
   PDF pages=23
 - 2312.13228v2.pdf | .pdf | READ_OK | reader=pypdf
   PDF pages=29
 - CSV1 - RO Miles per Location 202009-202606-2022benchmark.csv | .csv | READ_OK | reader=pandas
   CSV sample rows=5 | columns=2
 - CSV2 - Crashes with SGO ID and Group Membership 202009-202606-2022benchmark.csv | .csv | READ_OK | reader=pandas
   CSV sample rows=5 | columns=19
 - CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv | .csv | READ_OK | reader=pandas
   CSV sample rows=5 | columns=21
 - CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv | .csv | READ_OK | reader=pandas
   CSV sample rows=5 | columns=7
 - Waymo_Safety_Im

In [65]:
# ============================================================
# OIP v1.0.39 — Waymo Safety Claim Audit
# C03 — Schema / Column / Unit / Time-Window Audit
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone
from collections import defaultdict

import pandas as pd


INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C03")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Discover CSV files
# ------------------------------------------------------------

csv_files = sorted(
    [
        p for p in INPUT_ROOT.rglob("*.csv")
        if p.is_file()
    ]
)


# ------------------------------------------------------------
# 2. Schema audit
# ------------------------------------------------------------

csv_results = []
errors = []

for path in csv_files:

    record = {
        "file_name": path.name,
        "relative_path": str(path.relative_to(INPUT_ROOT)),
        "size_bytes": path.stat().st_size,
        "read_status": None,
        "row_count": None,
        "column_count": None,
        "columns": [],
        "duplicate_columns": [],
        "blank_column_names": [],
        "column_details": [],
        "suspected_time_fields": [],
        "suspected_location_fields": [],
        "suspected_exposure_fields": [],
        "suspected_outcome_fields": [],
        "semantic_authorization": "NOT_AUTHORIZED",
        "unit_authorization": "NOT_AUTHORIZED",
    }

    try:

        # ----------------------------------------------------
        # Read header first
        # ----------------------------------------------------
        header_df = pd.read_csv(
            path,
            nrows=0
        )

        columns = list(header_df.columns)

        record["columns"] = columns
        record["column_count"] = len(columns)

        # Duplicate / blank column diagnostics
        seen = defaultdict(int)

        for col in columns:
            seen[str(col)] += 1

        record["duplicate_columns"] = [
            col for col, count in seen.items()
            if count > 1
        ]

        record["blank_column_names"] = [
            str(col)
            for col in columns
            if str(col).strip() == ""
            or str(col).lower() == "nan"
        ]

        # ----------------------------------------------------
        # Full CSV scan in chunks
        # ----------------------------------------------------
        total_rows = 0
        missing_counts = defaultdict(int)

        dtype_observations = defaultdict(set)

        CHUNK_SIZE = 10000

        for chunk in pd.read_csv(
            path,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ):

            total_rows += len(chunk)

            for col in columns:

                missing_counts[col] += int(
                    chunk[col].isna().sum()
                )

                dtype_observations[col].add(
                    str(chunk[col].dtype)
                )

        record["row_count"] = total_rows

        # ----------------------------------------------------
        # Column details
        # ----------------------------------------------------
        for col in columns:

            missing = missing_counts[col]

            if total_rows > 0:
                missing_pct = (
                    missing / total_rows
                ) * 100
            else:
                missing_pct = None

            details = {
                "column_name": str(col),
                "observed_dtypes":
                    sorted(dtype_observations[col]),
                "missing_count": missing,
                "missing_percent": missing_pct,
            }

            record["column_details"].append(details)

        # ----------------------------------------------------
        # Structural field discovery
        # IMPORTANT:
        # These are ONLY candidate names.
        # No semantic authorization is made.
        # ----------------------------------------------------

        for col in columns:

            c = str(col).lower()

            if any(
                token in c
                for token in [
                    "date",
                    "time",
                    "year",
                    "month",
                    "period",
                    "2020",
                    "2021",
                    "2022",
                    "2023",
                    "2024",
                    "2025",
                    "2026",
                ]
            ):
                record["suspected_time_fields"].append(
                    str(col)
                )

            if any(
                token in c
                for token in [
                    "geo",
                    "location",
                    "city",
                    "state",
                    "region",
                    "area",
                    "s2",
                    "latitude",
                    "longitude",
                ]
            ):
                record["suspected_location_fields"].append(
                    str(col)
                )

            if any(
                token in c
                for token in [
                    "mile",
                    "miles",
                    "vmt",
                    "exposure",
                    "vehicle",
                ]
            ):
                record["suspected_exposure_fields"].append(
                    str(col)
                )

            if any(
                token in c
                for token in [
                    "crash",
                    "collision",
                    "injury",
                    "airbag",
                    "severity",
                    "outcome",
                    "sgo",
                ]
            ):
                record["suspected_outcome_fields"].append(
                    str(col)
                )

        record["read_status"] = "SCHEMA_READ_OK"
        csv_results.append(record)

    except Exception as e:

        record["read_status"] = "SCHEMA_READ_FAIL"
        record["error_type"] = type(e).__name__
        record["error_message"] = str(e)

        csv_results.append(record)
        errors.append(record)


# ------------------------------------------------------------
# 3. Structural summary
# ------------------------------------------------------------

total_csv = len(csv_results)

schema_ok = sum(
    1 for r in csv_results
    if r["read_status"] == "SCHEMA_READ_OK"
)

schema_fail = sum(
    1 for r in csv_results
    if r["read_status"] == "SCHEMA_READ_FAIL"
)

duplicate_column_files = sum(
    1 for r in csv_results
    if r["duplicate_columns"]
)

blank_column_files = sum(
    1 for r in csv_results
    if r["blank_column_names"]
)


# ------------------------------------------------------------
# 4. C03 decision
# ------------------------------------------------------------

if total_csv == 0:

    status = "FAIL_CLOSED"

elif schema_fail > 0:

    status = "FAIL_CLOSED"

elif duplicate_column_files > 0:

    status = "FAIL_CLOSED"

elif blank_column_files > 0:

    status = "FAIL_CLOSED"

elif schema_ok == total_csv:

    status = "PASS_SCHEMA_INVENTORY"

else:

    status = "FAIL_CLOSED"


# ------------------------------------------------------------
# 5. Important evidence boundary
# ------------------------------------------------------------

boundary = {
    "column_existence": "OBSERVED",
    "column_semantic_meaning": "NOT_AUTHORIZED",
    "unit_meaning": "NOT_AUTHORIZED",
    "time_window_meaning": "NOT_AUTHORIZED",
    "location_meaning": "NOT_AUTHORIZED",
    "outcome_meaning": "NOT_AUTHORIZED",
    "claim_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}


# ------------------------------------------------------------
# 6. Artifact
# ------------------------------------------------------------

artifact = {

    "protocol_version": "OIP v1.0.39",

    "audit_type":
        "Corporate Safety Claim Audit",

    "target":
        "Waymo Safety Claims",

    "cell":
        "C03",

    "cell_name":
        "Schema / Column / Unit / Time-Window Audit",

    "status":
        status,

    "execution_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "boundary":
        boundary,

    "summary": {
        "total_csv": total_csv,
        "schema_ok": schema_ok,
        "schema_fail": schema_fail,
        "duplicate_column_files":
            duplicate_column_files,
        "blank_column_files":
            blank_column_files,
    },

    "csv_results":
        csv_results,

    "errors":
        errors,
}


# ------------------------------------------------------------
# 7. Save JSON
# ------------------------------------------------------------

json_path = (
    OUTPUT_DIR /
    "C03_SCHEMA_COLUMN_UNIT_TIME_AUDIT.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 8. Console output
# ------------------------------------------------------------

print("=" * 72)
print("OIP v1.0.39 — C03 SCHEMA / COLUMN / UNIT / TIME AUDIT")
print("=" * 72)

print("Audit Type          : Corporate Safety Claim Audit")
print("Target              : Waymo Safety Claims")

print()
print(f"CSV files           : {total_csv}")
print(f"Schema read OK      : {schema_ok}")
print(f"Schema read FAIL    : {schema_fail}")
print(f"Duplicate columns   : {duplicate_column_files}")
print(f"Blank column files  : {blank_column_files}")

print()
print("BOUNDARY:")
print("Column meaning      : NOT_AUTHORIZED")
print("Unit meaning        : NOT_AUTHORIZED")
print("Time-window meaning : NOT_AUTHORIZED")
print("Outcome meaning     : NOT_AUTHORIZED")
print("Claim authorization : FALSE")
print("Road safety verdict : NOT_ISSUED")

print()
print("CSV SCHEMA RESULTS:")

for r in csv_results:

    print()
    print(f"FILE: {r['file_name']}")
    print(f"Status              : {r['read_status']}")
    print(f"Rows                : {r['row_count']}")
    print(f"Columns             : {r['column_count']}")

    print("Column names:")
    for col in r["columns"]:
        print(f"  - {col}")

    print("Candidate time fields:")
    print(
        "  ",
        r["suspected_time_fields"]
    )

    print("Candidate location fields:")
    print(
        "  ",
        r["suspected_location_fields"]
    )

    print("Candidate exposure fields:")
    print(
        "  ",
        r["suspected_exposure_fields"]
    )

    print("Candidate outcome fields:")
    print(
        "  ",
        r["suspected_outcome_fields"]
    )

    if r["duplicate_columns"]:
        print(
            "DUPLICATE COLUMNS:",
            r["duplicate_columns"]
        )

    if r["blank_column_names"]:
        print(
            "BLANK COLUMN NAMES:",
            r["blank_column_names"]
        )

print()
print(f"C03 STATUS          : {status}")
print()
print(f"Artifact            : {json_path}")
print("=" * 72)

OIP v1.0.39 — C03 SCHEMA / COLUMN / UNIT / TIME AUDIT
Audit Type          : Corporate Safety Claim Audit
Target              : Waymo Safety Claims

CSV files           : 4
Schema read OK      : 4
Schema read FAIL    : 0
Duplicate columns   : 0
Blank column files  : 0

BOUNDARY:
Column meaning      : NOT_AUTHORIZED
Unit meaning        : NOT_AUTHORIZED
Time-window meaning : NOT_AUTHORIZED
Outcome meaning     : NOT_AUTHORIZED
Claim authorization : FALSE
Road safety verdict : NOT_ISSUED

CSV SCHEMA RESULTS:

FILE: CSV1 - RO Miles per Location 202009-202606-2022benchmark.csv
Status              : SCHEMA_READ_OK
Rows                : 11
Columns             : 2
Column names:
  - County Name
  - Waymo RO Miles (Millions)
Candidate time fields:
   []
Candidate location fields:
   []
Candidate exposure fields:
   ['Waymo RO Miles (Millions)']
Candidate outcome fields:
   []

FILE: CSV2 - Crashes with SGO ID and Group Membership 202009-202606-2022benchmark.csv
Status              : SCHEMA_READ_OK

In [66]:
# ============================================================
# OIP v1.0.39 — C04
# RELEASE NOTES / DATA DICTIONARY EVIDENCE GATE
# ============================================================

import os
import re
import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

from pypdf import PdfReader


# ============================================================
# 1. AUDIT IDENTITY & BOUNDARY
# ============================================================

AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C04"

AUDIT_TYPE = "Corporate Safety Claim Audit"

BOUNDARY = {
    "audit_type": AUDIT_TYPE,

    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "road_safety_verdict": "NOT_ISSUED",
    "waymo_safe_unsafe_claim": "NOT_CLAIMED",

    "claim_authorization": False,
    "semantic_authorization_at_C04": False,

    "candidate_classifications_are_authoritative": False
}


# ============================================================
# 2. FIND WAYMO RELEASE NOTES
# ============================================================

INPUT_ROOT = Path("/kaggle/input")

pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

release_candidates = [
    p for p in pdf_files
    if (
        "release" in p.name.lower()
        and "notes" in p.name.lower()
    )
]

if len(release_candidates) == 0:
    raise RuntimeError(
        "C04 FAIL_CLOSED: Waymo Release Notes PDF was not found."
    )

if len(release_candidates) > 1:
    print("WARNING: Multiple Release Notes candidates found:")
    for p in release_candidates:
        print(" -", p)

RELEASE_NOTES = release_candidates[0]


# ============================================================
# 3. READ RELEASE NOTES
# ============================================================

try:

    reader = PdfReader(str(RELEASE_NOTES))

    page_count = len(reader.pages)

    extracted_pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text() or ""

        extracted_pages.append({
            "page": page_number,
            "text": page_text
        })

except Exception as e:

    raise RuntimeError(
        "C04 FAIL_CLOSED: Release Notes extraction failed: "
        + repr(e)
    )


FULL_TEXT = "\n".join(
    page["text"]
    for page in extracted_pages
)

if not FULL_TEXT.strip():

    raise RuntimeError(
        "C04 FAIL_CLOSED: Release Notes PDF contains "
        "no extractable text."
    )


# ============================================================
# 4. TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def search_document(term, text, context_chars=350):

    lower_text = text.lower()
    lower_term = term.lower()

    matches = []

    start = 0

    while True:

        position = lower_text.find(
            lower_term,
            start
        )

        if position == -1:
            break

        left = max(
            0,
            position - context_chars
        )

        right = min(
            len(text),
            position + len(term) + context_chars
        )

        context = normalize_text(
            text[left:right]
        )

        matches.append({
            "position": position,
            "context": context
        })

        start = position + len(lower_term)

    return matches


# ============================================================
# 5. CSV FILE DOCUMENTARY OCCURRENCE TARGETS
# ============================================================

CSV_TARGETS = [

    "CSV1 - RO Miles per Location 202009-202606-2022benchmark.csv",

    "CSV2 - Crashes with SGO ID and Group Membership 202009-202606-2022benchmark.csv",

    "CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv",

    "CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv"
]


# ============================================================
# 6. C03 CLASSIFICATION VERIFICATION TARGETS
# ============================================================

CLASSIFICATION_TARGETS = [

    "Waymo RO Miles",

    "SGO Report ID",

    "SGO Report Version",

    "SGO Amendment",

    "Is Any Vehicle Airbag Deployment",

    "Is Ego Vehicle Airbag Deployment",

    "HPMS",

    "Vehicle Miles Traveled",

    "Outcome",

    "Benchmark Crash Count",

    "IPMM",

    "Rate Ratio",

    "Percent Reduction",

    "Delta V"
]


# ============================================================
# 7. DOCUMENTATION / METHODOLOGY TARGETS
# ============================================================

DOCUMENTATION_TARGETS = [

    "data dictionary",

    "methodology",

    "benchmark",

    "crash",

    "outcome",

    "exposure",

    "miles",

    "SGO",

    "airbag",

    "HPMS"
]


# ============================================================
# 8. DOCUMENTARY OCCURRENCE AUDIT
# ============================================================

CSV_EVIDENCE = {}

for term in CSV_TARGETS:

    matches = search_document(
        term,
        FULL_TEXT
    )

    CSV_EVIDENCE[term] = {

        "documentary_occurrence":
            len(matches) > 0,

        "match_count":
            len(matches),

        "contexts":
            matches[:5]
    }


CLASSIFICATION_EVIDENCE = {}

for term in CLASSIFICATION_TARGETS:

    matches = search_document(
        term,
        FULL_TEXT
    )

    CLASSIFICATION_EVIDENCE[term] = {

        "documentary_occurrence":
            len(matches) > 0,

        "match_count":
            len(matches),

        "contexts":
            matches[:5]
    }


DOCUMENTATION_EVIDENCE = {}

for term in DOCUMENTATION_TARGETS:

    matches = search_document(
        term,
        FULL_TEXT
    )

    DOCUMENTATION_EVIDENCE[term] = {

        "documentary_occurrence":
            len(matches) > 0,

        "match_count":
            len(matches),

        "contexts":
            matches[:3]
    }


# ============================================================
# 9. C03 KNOWN CLASSIFICATION ISSUES
# ============================================================

C03_KNOWN_CLASSIFICATION_ISSUES = {

    "CSV2_airbag_fields": {

        "fields": [
            "Is Any Vehicle Airbag Deployment",
            "Is Ego Vehicle Airbag Deployment"
        ],

        "issue":
            "Previously candidate-classified as exposure and outcome.",

        "status":
            "REQUIRES_DOCUMENTARY_VERIFICATION"
    },


    "CSV4_HPMS_field": {

        "field":
            "HPMS Yearly Vehicle Miles Traveled",

        "issue":
            "Previously candidate-classified as time field.",

        "status":
            "REQUIRES_DOCUMENTARY_VERIFICATION"
    },


    "CSV2_SGO_fields": {

        "fields": [
            "SGO Report ID",
            "SGO Report Version",
            "SGO Amendment"
        ],

        "issue":
            "Previously candidate-classified as outcome fields.",

        "status":
            "REQUIRES_DOCUMENTARY_VERIFICATION"
    },


    "CSV3_aggregate_metrics": {

        "fields": [
            "Waymo IPMM",
            "Benchmark IPMM",
            "Waymo Percent Reduction",
            "Waymo Rate Ratio"
        ],

        "issue":
            "Derived/aggregate metrics require documentary verification.",

        "status":
            "REQUIRES_DOCUMENTARY_VERIFICATION"
    }
}


# ============================================================
# 10. EXPLICIT OIP DISCOVERY DISCLAIMER
# ============================================================

DISCOVERY_DISCLAIMER = (

    "C03 candidate classifications are discovery-level "
    "heuristics only. Column-name similarity does not "
    "establish semantic alignment, construct meaning, "
    "variable role, outcome status, exposure status, "
    "or documentary validity. All candidate classifications "
    "must be independently verified against authoritative "
    "Waymo documentation before downstream authorization."
)


# ============================================================
# 11. DOCUMENT HASH
# ============================================================

release_sha256 = hashlib.sha256(
    RELEASE_NOTES.read_bytes()
).hexdigest()


# ============================================================
# 12. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/OIP_v1_0_39_C04"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 13. SAVE COMPLETE RELEASE-NOTE TEXT
# ============================================================

TEXT_OUTPUT = (
    OUTPUT_DIR /
    "C04_RELEASE_NOTES_EXTRACTED.txt"
)

with open(
    TEXT_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    f.write(FULL_TEXT)


# ============================================================
# 14. BUILD C04 AUDIT ARTIFACT
# ============================================================

ARTIFACT = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "source": {

        "release_notes_filename":
            RELEASE_NOTES.name,

        "release_notes_path":
            str(RELEASE_NOTES),

        "sha256":
            release_sha256,

        "page_count":
            page_count,

        "text_extraction_success":
            True,

        "extracted_character_count":
            len(FULL_TEXT)
    },

    "csv_documentary_occurrence":
        CSV_EVIDENCE,

    "classification_documentary_occurrence":
        CLASSIFICATION_EVIDENCE,

    "documentation_terms":
        DOCUMENTATION_EVIDENCE,

    "c03_known_classification_issues":
        C03_KNOWN_CLASSIFICATION_ISSUES,

    "discovery_disclaimer":
        DISCOVERY_DISCLAIMER,

    "semantic_authorization":
        "NOT_AUTHORIZED",

    "claim_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED",

    "status":
        "PASS_DOCUMENTATION_EXTRACTION"
}


JSON_OUTPUT = (
    OUTPUT_DIR /
    "C04_RELEASE_NOTES_EVIDENCE_GATE.json"
)

with open(
    JSON_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ARTIFACT,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 15. FINAL CONSOLE OUTPUT
# ============================================================

print("=" * 72)

print(
    "OIP v1.0.39 — C04 "
    "RELEASE NOTES / DATA DICTIONARY EVIDENCE GATE"
)

print("=" * 72)

print()

print("SOURCE")
print(
    "Release Notes file :",
    RELEASE_NOTES.name
)

print(
    "PDF pages          :",
    page_count
)

print(
    "Extracted chars    :",
    len(FULL_TEXT)
)

print(
    "SHA-256            :",
    release_sha256
)

print()

print("-" * 72)
print("CSV DOCUMENTARY OCCURRENCE")
print("-" * 72)

for term, result in CSV_EVIDENCE.items():

    print(
        f"{result['documentary_occurrence']!s:5s} | "
        f"matches={result['match_count']:3d} | "
        f"{term}"
    )


print()

print("-" * 72)
print("C03 CLASSIFICATION VERIFICATION TARGETS")
print("-" * 72)

for term, result in CLASSIFICATION_EVIDENCE.items():

    print(
        f"{result['documentary_occurrence']!s:5s} | "
        f"matches={result['match_count']:3d} | "
        f"{term}"
    )


print()

print("-" * 72)
print("DOCUMENTATION / METHODOLOGY TERMS")
print("-" * 72)

for term, result in DOCUMENTATION_EVIDENCE.items():

    print(
        f"{result['documentary_occurrence']!s:5s} | "
        f"matches={result['match_count']:3d} | "
        f"{term}"
    )


print()

print("-" * 72)
print("C03 KNOWN CLASSIFICATION ISSUES")
print("-" * 72)

print(
    "CSV2 airbag fields         : "
    "REQUIRES_DOCUMENTARY_VERIFICATION"
)

print(
    "CSV4 HPMS field            : "
    "REQUIRES_DOCUMENTARY_VERIFICATION"
)

print(
    "CSV2 SGO identifier fields : "
    "REQUIRES_DOCUMENTARY_VERIFICATION"
)

print(
    "CSV3 aggregate metrics     : "
    "REQUIRES_DOCUMENTARY_VERIFICATION"
)


print()

print("-" * 72)
print("OIP BOUNDARY")
print("-" * 72)

print(
    "Semantic authorization :",
    "NOT_AUTHORIZED"
)

print(
    "Claim authorization    :",
    False
)

print(
    "Road safety verdict    :",
    "NOT_ISSUED"
)

print(
    "Candidate classifications authoritative :",
    False
)


print()

print("-" * 72)
print("C04 STATUS")
print("-" * 72)

print(
    "C04 STATUS :",
    "PASS_DOCUMENTATION_EXTRACTION"
)


print()

print("-" * 72)
print("ARTIFACTS")
print("-" * 72)

print(
    "JSON :",
    JSON_OUTPUT
)

print(
    "TEXT :",
    TEXT_OUTPUT
)

print()

print("=" * 72)
print("C04 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C04 RELEASE NOTES / DATA DICTIONARY EVIDENCE GATE

SOURCE
Release Notes file : Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf
PDF pages          : 18
Extracted chars    : 46984
SHA-256            : d5e8ee1da8d0c51cccc4f11e5fea45507fe142cbfa403ad8bd35007283da703b

------------------------------------------------------------------------
CSV DOCUMENTARY OCCURRENCE
------------------------------------------------------------------------
False | matches=  0 | CSV1 - RO Miles per Location 202009-202606-2022benchmark.csv
False | matches=  0 | CSV2 - Crashes with SGO ID and Group Membership 202009-202606-2022benchmark.csv
False | matches=  0 | CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv
False | matches=  0 | CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

------------------------------------------------------------------------
C03 CLASSIFICATION VERIFICATION TARGETS
-----------------------

In [67]:
# ============================================================
# OIP v1.0.39 — C05R3
# EXPANDED CLAIM EVIDENCE GATE
# Release Notes + Waymo CSVs + Research PDFs
# ============================================================

import re
import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd
from pypdf import PdfReader


# ============================================================
# 1. IDENTITY / BOUNDARY
# ============================================================

AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C05R3"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C05R3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "claim_truth_evaluation": "NOT_PERFORMED"
}


# ============================================================
# 2. FIND ALL SOURCE FILES
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))
pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C05R3 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

if len(pdf_files) != 3:
    raise RuntimeError(
        f"C05R3 FAIL_CLOSED: Expected 3 PDF files, found {len(pdf_files)}."
    )


# ============================================================
# 3. LOAD CSV CONTENT AS DOCUMENTARY EVIDENCE
# ============================================================

csv_documents = {}

for path in csv_files:

    try:
        df = pd.read_csv(path)

        csv_documents[path.name] = {
            "rows": int(df.shape[0]),
            "columns": int(df.shape[1]),
            "columns_list": list(df.columns),
            "text": df.to_csv(index=False)
        }

    except Exception as e:

        raise RuntimeError(
            f"C05R3 FAIL_CLOSED: CSV read failed: "
            f"{path.name} | {repr(e)}"
        )


# ============================================================
# 4. EXTRACT PDF TEXT
# ============================================================

pdf_documents = {}

for path in pdf_files:

    try:

        reader = PdfReader(str(path))

        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

        pdf_documents[path.name] = {
            "pages": len(reader.pages),
            "text": text
        }

    except Exception as e:

        raise RuntimeError(
            f"C05R3 FAIL_CLOSED: PDF read failed: "
            f"{path.name} | {repr(e)}"
        )


# ============================================================
# 5. NORMALIZATION
# ============================================================

def normalize(text):

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def search_text(
    phrase,
    text,
    context_chars=450
):

    text = str(text)

    lower_text = text.lower()
    lower_phrase = phrase.lower()

    matches = []
    start = 0

    while True:

        idx = lower_text.find(
            lower_phrase,
            start
        )

        if idx == -1:
            break

        left = max(
            0,
            idx - context_chars
        )

        right = min(
            len(text),
            idx + len(phrase) + context_chars
        )

        matches.append({
            "position": idx,
            "context": normalize(
                text[left:right]
            )
        })

        start = idx + len(lower_phrase)

    return matches


# ============================================================
# 6. CLAIM TARGETS
# ============================================================

CLAIMS = {

    "CLAIM_01_SERIOUS_INJURY": {
        "phrases": [
            "95% fewer",
            "serious injury",
            "serious injury or worse"
        ]
    },

    "CLAIM_02_AIRBAG_ANY_VEHICLE": {
        "phrases": [
            "82% fewer",
            "airbag",
            "airbag deployment"
        ]
    },

    "CLAIM_03_INJURY_CAUSING": {
        "phrases": [
            "injury-causing",
            "injury causing"
        ]
    },

    "CLAIM_04_PEDESTRIAN": {
        "phrases": [
            "93% fewer",
            "pedestrian injury",
            "pedestrian"
        ]
    },

    "CLAIM_05_CYCLIST": {
        "phrases": [
            "86% fewer",
            "cyclist injury",
            "cyclist"
        ]
    },

    "CLAIM_06_MOTORCYCLE": {
        "phrases": [
            "82% fewer",
            "motorcycle injury",
            "motorcycle"
        ]
    },

    "CLAIM_07_HUMAN_COMPARISON": {
        "phrases": [
            "better than humans",
            "human benchmark",
            "human benchmarks"
        ]
    },

    "CLAIM_08_RO_MILES": {
        "phrases": [
            "rider-only",
            "RO miles",
            "Waymo RO Miles"
        ]
    },

    "CLAIM_09_SAFER_ROADS": {
        "phrases": [
            "making roads safer",
            "safer roads"
        ]
    }
}


# ============================================================
# 7. SEARCH EACH CLAIM ACROSS ALL 7 FILES
# ============================================================

claim_records = {}

for claim_id, claim_info in CLAIMS.items():

    evidence = []

    for phrase in claim_info["phrases"]:

        # ---------- PDFs ----------
        for filename, doc in pdf_documents.items():

            matches = search_text(
                phrase,
                doc["text"]
            )

            if matches:

                evidence.append({
                    "source_type": "PDF",
                    "filename": filename,
                    "phrase": phrase,
                    "match_count": len(matches),
                    "contexts": matches[:3]
                })


        # ---------- CSVs ----------
        for filename, doc in csv_documents.items():

            matches = search_text(
                phrase,
                doc["text"]
            )

            if matches:

                evidence.append({
                    "source_type": "CSV",
                    "filename": filename,
                    "phrase": phrase,
                    "match_count": len(matches),
                    "contexts": matches[:3]
                })


    # Remove exact duplicate evidence records
    unique = []

    seen = set()

    for item in evidence:

        key = (
            item["source_type"],
            item["filename"],
            item["phrase"]
        )

        if key not in seen:

            seen.add(key)
            unique.append(item)


    if len(unique) == 0:

        documentary_status = (
            "NO_DOCUMENTARY_OCCURRENCE_IN_7_FILE_CORPUS"
        )

    else:

        documentary_status = (
            "DOCUMENTARY_OCCURRENCE_IN_CORPUS"
        )


    claim_records[claim_id] = {

        "claim_id": claim_id,

        "documentary_status":
            documentary_status,

        "evidence_source_count":
            len(unique),

        "evidence":
            unique,

        # IMPORTANT:
        # No claim type is authorized here.
        "claim_type":
            "PENDING_EVIDENCE",

        "scientific_authorization":
            False,

        "final_claim_decision":
            "NOT_YET_EVALUATED"
    }


# ============================================================
# 8. SUMMARY
# ============================================================

documented_claims = [
    k for k, v in claim_records.items()
    if v["documentary_status"]
    == "DOCUMENTARY_OCCURRENCE_IN_CORPUS"
]

undocumented_claims = [
    k for k, v in claim_records.items()
    if v["documentary_status"]
    == "NO_DOCUMENTARY_OCCURRENCE_IN_7_FILE_CORPUS"
]


# ============================================================
# 9. FAIL-CLOSED DECISION
# ============================================================

if len(undocumented_claims) > 0:

    overall_status = "PENDING_EVIDENCE"

else:

    # Even if every claim occurs somewhere,
    # claim-type authorization remains deferred.
    overall_status = "PENDING_EVIDENCE"


# ============================================================
# 10. HASH ALL SOURCES
# ============================================================

source_hashes = {}

for path in csv_files + pdf_files:

    source_hashes[path.name] = (
        hashlib.sha256(
            path.read_bytes()
        ).hexdigest()
    )


# ============================================================
# 11. ARTIFACT
# ============================================================

artifact = {

    "audit_version": AUDIT_VERSION,

    "cell": CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary": BOUNDARY,

    "source_inventory": {

        "csv_count": len(csv_files),

        "pdf_count": len(pdf_files),

        "total_files":
            len(csv_files) + len(pdf_files),

        "sha256":
            source_hashes
    },

    "claims":
        claim_records,

    "summary": {

        "total_claim_targets":
            len(CLAIMS),

        "documented_in_corpus":
            len(documented_claims),

        "not_documented_in_corpus":
            len(undocumented_claims),

        "undocumented_claim_ids":
            undocumented_claims
    },

    "rules": {

        "documentary_occurrence_is_not_claim_validation":
            True,

        "documentary_occurrence_is_not_claim_authorization":
            True,

        "claim_type_authorization":
            "DEFERRED",

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        overall_status
}


# ============================================================
# 12. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C05R3_EXPANDED_CLAIM_EVIDENCE_GATE.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 13. CONSOLE OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C05R3 "
    "EXPANDED CLAIM EVIDENCE GATE"
)
print("=" * 72)

print()

print("SOURCE CORPUS")
print(
    "CSV files :",
    len(csv_files)
)

print(
    "PDF files :",
    len(pdf_files)
)

print(
    "Total     :",
    len(csv_files) + len(pdf_files)
)

print()

print("-" * 72)
print("CLAIM EVIDENCE STATUS")
print("-" * 72)

for claim_id, record in claim_records.items():

    print(
        f"{claim_id:32s} | "
        f"{record['documentary_status']} | "
        f"sources={record['evidence_source_count']}"
    )


print()

print("-" * 72)
print("SUMMARY")
print("-" * 72)

print(
    "Total claim targets       :",
    len(CLAIMS)
)

print(
    "Documented in 7-file corpus:",
    len(documented_claims)
)

print(
    "Not documented            :",
    len(undocumented_claims)
)

if undocumented_claims:

    print()
    print("Undocumented claims:")

    for claim in undocumented_claims:

        print(
            " -",
            claim
        )


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Claim authorization    :",
    False
)

print(
    "Scientific authorization:",
    False
)

print(
    "Claim type authorization:",
    "DEFERRED"
)

print(
    "Road safety verdict    :",
    "NOT_ISSUED"
)

print(
    "Final claim decisions  :",
    "NOT_YET_EVALUATED"
)

print()

print("-" * 72)
print("C05R3 STATUS")
print("-" * 72)

print(
    "C05R3 STATUS :",
    overall_status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C05R3 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C05R3 EXPANDED CLAIM EVIDENCE GATE

SOURCE CORPUS
CSV files : 4
PDF files : 3
Total     : 7

------------------------------------------------------------------------
CLAIM EVIDENCE STATUS
------------------------------------------------------------------------
CLAIM_01_SERIOUS_INJURY          | DOCUMENTARY_OCCURRENCE_IN_CORPUS | sources=6
CLAIM_02_AIRBAG_ANY_VEHICLE      | DOCUMENTARY_OCCURRENCE_IN_CORPUS | sources=10
CLAIM_03_INJURY_CAUSING          | NO_DOCUMENTARY_OCCURRENCE_IN_7_FILE_CORPUS | sources=0
CLAIM_04_PEDESTRIAN              | DOCUMENTARY_OCCURRENCE_IN_CORPUS | sources=5
CLAIM_05_CYCLIST                 | DOCUMENTARY_OCCURRENCE_IN_CORPUS | sources=5
CLAIM_06_MOTORCYCLE              | DOCUMENTARY_OCCURRENCE_IN_CORPUS | sources=4
CLAIM_07_HUMAN_COMPARISON        | DOCUMENTARY_OCCURRENCE_IN_CORPUS | sources=6
CLAIM_08_RO_MILES                | DOCUMENTARY_OCCURRENCE_IN_CORPUS | sources=10
CLAIM_09_SAFER_ROADS             | NO_DOCUMENTARY_OCCURRENCE_IN_7_FILE_CO

In [68]:
# ============================================================
# OIP v1.0.39 — C05R4
# CLAIM EVIDENCE RESOLUTION / SEARCH-VARIANT GATE
# ============================================================

import re
import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd
from pypdf import PdfReader


# ============================================================
# 1. IDENTITY
# ============================================================

AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C05R4"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C05R4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. BOUNDARY
# ============================================================

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "claim_truth_evaluation": "NOT_PERFORMED",
    "claim_type_authorization": "DEFERRED"
}


# ============================================================
# 3. LOAD ALL 7 FILES
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))
pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C05R4 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

if len(pdf_files) != 3:
    raise RuntimeError(
        f"C05R4 FAIL_CLOSED: Expected 3 PDF files, found {len(pdf_files)}."
    )


documents = {}

for path in pdf_files:

    try:

        reader = PdfReader(str(path))

        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

        documents[path.name] = {
            "source_type": "PDF",
            "text": text
        }

    except Exception as e:

        raise RuntimeError(
            f"C05R4 FAIL_CLOSED: PDF read failed: "
            f"{path.name} | {repr(e)}"
        )


for path in csv_files:

    try:

        df = pd.read_csv(path)

        documents[path.name] = {
            "source_type": "CSV",
            "text": df.to_csv(index=False),
            "columns": list(df.columns),
            "rows": int(df.shape[0])
        }

    except Exception as e:

        raise RuntimeError(
            f"C05R4 FAIL_CLOSED: CSV read failed: "
            f"{path.name} | {repr(e)}"
        )


# ============================================================
# 4. SEARCH FUNCTION
# ============================================================

def normalize(text):

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def search_term(term, text, context_chars=500):

    lower_text = text.lower()
    lower_term = term.lower()

    results = []
    start = 0

    while True:

        idx = lower_text.find(
            lower_term,
            start
        )

        if idx == -1:
            break

        left = max(
            0,
            idx - context_chars
        )

        right = min(
            len(text),
            idx + len(term) + context_chars
        )

        results.append({
            "position": idx,
            "context": normalize(
                text[left:right]
            )
        })

        start = idx + len(lower_term)

    return results


# ============================================================
# 5. RESOLUTION TARGETS
#
# These are SEARCH VARIANTS only.
# They do NOT authorize claim meaning.
# ============================================================

TARGETS = {

    "CLAIM_03_INJURY_CAUSING": [

        "82%",

        "841",

        "injury",

        "injury-causing",

        "injury causing",

        "injury crashes",

        "any injury",

        "Any-Injury",

        "Any Injury"
    ],

    "CLAIM_09_SAFER_ROADS": [

        "safer",

        "making roads safer",

        "roads safer",

        "safety",

        "making roads",

        "road safety"
    ]
}


# ============================================================
# 6. SEARCH ACROSS EVERY SOURCE
# ============================================================

resolution = {}

for claim_id, terms in TARGETS.items():

    claim_sources = []

    for term in terms:

        for filename, document in documents.items():

            matches = search_term(
                term,
                document["text"]
            )

            if matches:

                claim_sources.append({

                    "term":
                        term,

                    "source_type":
                        document["source_type"],

                    "filename":
                        filename,

                    "match_count":
                        len(matches),

                    "contexts":
                        matches[:5]
                })


    resolution[claim_id] = claim_sources


# ============================================================
# 7. DEDUPLICATE
# ============================================================

for claim_id in resolution:

    unique = []
    seen = set()

    for item in resolution[claim_id]:

        key = (
            item["term"],
            item["source_type"],
            item["filename"]
        )

        if key not in seen:

            seen.add(key)
            unique.append(item)

    resolution[claim_id] = unique


# ============================================================
# 8. STATUS
# ============================================================

claim_status = {}

for claim_id, evidence in resolution.items():

    if len(evidence) == 0:

        status = (
            "NO_MATCH_AFTER_SEARCH_VARIANTS"
        )

    else:

        status = (
            "SEARCH_VARIANT_EVIDENCE_FOUND"
        )

    claim_status[claim_id] = {

        "status":
            status,

        "evidence_record_count":
            len(evidence),

        "evidence":
            evidence,

        "semantic_authorization":
            False,

        "claim_type":
            "PENDING_EVIDENCE",

        "final_claim_decision":
            "NOT_YET_EVALUATED"
    }


# ============================================================
# 9. FINAL C05R4 STATE
# ============================================================

unresolved = [
    claim_id
    for claim_id, record in claim_status.items()
    if record["status"]
    == "NO_MATCH_AFTER_SEARCH_VARIANTS"
]


if len(unresolved) > 0:

    overall_status = "PENDING_EVIDENCE"

else:

    # Search evidence exists, but semantic/type authorization
    # remains deferred.
    overall_status = "PENDING_EVIDENCE"


# ============================================================
# 10. HASH SOURCES
# ============================================================

source_hashes = {}

for path in csv_files + pdf_files:

    source_hashes[path.name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


# ============================================================
# 11. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "source_count": {

        "csv":
            len(csv_files),

        "pdf":
            len(pdf_files),

        "total":
            len(documents)
    },

    "source_hashes":
        source_hashes,

    "resolution":
        claim_status,

    "unresolved_claims":
        unresolved,

    "rules": {

        "search_variant_evidence_is_not_semantic_authorization":
            True,

        "search_variant_evidence_is_not_claim_validation":
            True,

        "claim_type_authorization":
            "DEFERRED",

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        overall_status
}


# ============================================================
# 12. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C05R4_CLAIM_EVIDENCE_RESOLUTION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 13. CONSOLE
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C05R4 "
    "CLAIM EVIDENCE RESOLUTION"
)
print("=" * 72)

print()

print("SOURCE CORPUS")
print(
    "CSV files :",
    len(csv_files)
)

print(
    "PDF files :",
    len(pdf_files)
)

print(
    "Total     :",
    len(documents)
)

print()

print("-" * 72)
print("CLAIM RESOLUTION")
print("-" * 72)

for claim_id, record in claim_status.items():

    print()

    print(
        claim_id
    )

    print(
        "  Status       :",
        record["status"]
    )

    print(
        "  Evidence rows:",
        record["evidence_record_count"]
    )

    print(
        "  Claim type   :",
        record["claim_type"]
    )

    print(
        "  Authorization:",
        record["semantic_authorization"]
    )


print()

print("-" * 72)
print("UNRESOLVED")
print("-" * 72)

if unresolved:

    for claim in unresolved:
        print("-", claim)

else:

    print("None")


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Scientific authorization :",
    False
)

print(
    "Claim authorization      :",
    False
)

print(
    "Claim type authorization :",
    "DEFERRED"
)

print(
    "Road safety verdict      :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C05R4 STATUS")
print("-" * 72)

print(
    "C05R4 STATUS :",
    overall_status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C05R4 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C05R4 CLAIM EVIDENCE RESOLUTION

SOURCE CORPUS
CSV files : 4
PDF files : 3
Total     : 7

------------------------------------------------------------------------
CLAIM RESOLUTION
------------------------------------------------------------------------

CLAIM_03_INJURY_CAUSING
  Status       : SEARCH_VARIANT_EVIDENCE_FOUND
  Evidence rows: 26
  Claim type   : PENDING_EVIDENCE
  Authorization: False

CLAIM_09_SAFER_ROADS
  Status       : SEARCH_VARIANT_EVIDENCE_FOUND
  Evidence rows: 3
  Claim type   : PENDING_EVIDENCE
  Authorization: False

------------------------------------------------------------------------
UNRESOLVED
------------------------------------------------------------------------
None

------------------------------------------------------------------------
BOUNDARY
------------------------------------------------------------------------
Scientific authorization : False
Claim authorization      : False
Claim type authorization : DEFERRED
Road safety verdic

In [69]:
# ============================================================
# OIP v1.0.39 — C05R5
# DOCUMENTARY CONTEXT RESOLUTION
# ============================================================

import json
import re
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd
from pypdf import PdfReader


# ============================================================
# 1. IDENTITY / BOUNDARY
# ============================================================

AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C05R5"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C05R5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "claim_type_authorization": "DEFERRED"
}


# ============================================================
# 2. LOAD 7-FILE CORPUS
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))
pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C05R5 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

if len(pdf_files) != 3:
    raise RuntimeError(
        f"C05R5 FAIL_CLOSED: Expected 3 PDF files, found {len(pdf_files)}."
    )


documents = {}

for path in pdf_files:

    reader = PdfReader(str(path))

    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    documents[path.name] = {
        "source_type": "PDF",
        "text": text
    }


for path in csv_files:

    df = pd.read_csv(path)

    documents[path.name] = {
        "source_type": "CSV",
        "text": df.to_csv(index=False)
    }


# ============================================================
# 3. TARGET CLAIMS
# ============================================================

TARGETS = {

    "CLAIM_03_INJURY_CAUSING": [
        "82%",
        "841",
        "injury",
        "injury-causing",
        "injury causing",
        "injury crashes",
        "any injury",
        "Any-Injury",
        "Any Injury"
    ],

    "CLAIM_09_SAFER_ROADS": [
        "safer",
        "making roads safer",
        "roads safer",
        "safety",
        "making roads",
        "road safety"
    ]
}


# ============================================================
# 4. SEARCH WITH PAGE / SOURCE CONTEXT
# ============================================================

def normalize(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def search_in_text(
    term,
    text,
    context_chars=700
):

    lower = text.lower()
    target = term.lower()

    results = []
    start = 0

    while True:

        idx = lower.find(target, start)

        if idx == -1:
            break

        left = max(
            0,
            idx - context_chars
        )

        right = min(
            len(text),
            idx + len(term) + context_chars
        )

        results.append(
            normalize(
                text[left:right]
            )
        )

        start = idx + len(target)

    return results


# ============================================================
# 5. COLLECT CONTEXT
# ============================================================

claim_evidence = {}

for claim_id, terms in TARGETS.items():

    records = []

    seen = set()

    for term in terms:

        for filename, document in documents.items():

            contexts = search_in_text(
                term,
                document["text"]
            )

            for context in contexts[:10]:

                key = (
                    filename,
                    term,
                    context
                )

                if key in seen:
                    continue

                seen.add(key)

                records.append({

                    "source_type":
                        document["source_type"],

                    "filename":
                        filename,

                    "search_term":
                        term,

                    "context":
                        context
                })


    claim_evidence[claim_id] = records


# ============================================================
# 6. LIMIT OUTPUT TO REPRESENTATIVE EVIDENCE
# ============================================================

# Maximum 12 contexts per claim in console output.
# Full evidence is preserved in JSON.

representative = {}

for claim_id, records in claim_evidence.items():

    representative[claim_id] = records[:12]


# ============================================================
# 7. STATUS
# ============================================================

status = {

    claim_id: {
        "evidence_found":
            len(records) > 0,

        "evidence_count":
            len(records),

        "claim_type":
            "PENDING_EVIDENCE",

        "authorization":
            False,

        "final_decision":
            "NOT_YET_EVALUATED"
    }

    for claim_id, records
    in claim_evidence.items()
}


# ============================================================
# 8. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "claims":
        status,

    "representative_evidence":
        representative,

    "full_evidence":
        claim_evidence,

    "rules": {

        "context_evidence_is_not_claim_validation":
            True,

        "context_evidence_is_not_claim_authorization":
            True,

        "claim_type":
            "PENDING_EVIDENCE",

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        "PENDING_EVIDENCE"
}


artifact_path = (
    OUTPUT_DIR /
    "C05R5_DOCUMENTARY_CONTEXT_RESOLUTION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 9. CONSOLE OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C05R5 "
    "DOCUMENTARY CONTEXT RESOLUTION"
)
print("=" * 72)

print()

for claim_id, records in representative.items():

    print("-" * 72)
    print(claim_id)
    print("-" * 72)

    print(
        "Total evidence records:",
        len(claim_evidence[claim_id])
    )

    print()

    for i, record in enumerate(records, 1):

        print(
            f"[{i}] SOURCE TYPE : "
            f"{record['source_type']}"
        )

        print(
            f"    FILE        : "
            f"{record['filename']}"
        )

        print(
            f"    SEARCH TERM : "
            f"{record['search_term']}"
        )

        print(
            f"    CONTEXT     : "
            f"{record['context']}"
        )

        print()


print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Claim authorization    :",
    False
)

print(
    "Scientific authorization:",
    False
)

print(
    "Claim type             :",
    "PENDING_EVIDENCE"
)

print(
    "Road safety verdict    :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C05R5 STATUS")
print("-" * 72)

print(
    "C05R5 STATUS :",
    "PENDING_EVIDENCE"
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C05R5 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C05R5 DOCUMENTARY CONTEXT RESOLUTION

------------------------------------------------------------------------
CLAIM_03_INJURY_CAUSING
------------------------------------------------------------------------
Total evidence records: 163

[1] SOURCE TYPE : PDF
    FILE        : 2312.12675v3.pdf
    SEARCH TERM : 82%
    CONTEXT     : velopers usemultiple,complementarysafetyassurancemethodstomitigateriskofhighseverityoutcomesacrossarchitectural, behavioral, and operational layers (e.g., platform verification and validation, hazard analysis, see Webb et al. 2020, Favarò et al. 2023a). Traditionally, automotive safety systems have used prospective safety benefit analyses that use test track data and simulations as risk assessment tools and to determine potential high severity benefits. These types ofprospectivestudieshavealsobeenperformedforADS.Forexample,onestudythatusedreconstructionsofallfatal crashes in Chandler, AZ found that a level 4 ADS was able to prevent 100% of cras

In [70]:
# ============================================================
# OIP v1.0.39 — C06
# CRASH / OUTCOME DEFINITION GATE
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


# ============================================================
# 1. IDENTITY / BOUNDARY
# ============================================================

AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C06"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C06")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "road_safety_verdict": "NOT_ISSUED",
    "final_claim_decision": "NOT_YET_EVALUATED"
}


# ============================================================
# 2. LOCATE EXACT CSV FILES
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C06 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

CSV2 = next(
    (
        p for p in csv_files
        if "Crashes with SGO ID and Group Membership"
        in p.name
    ),
    None
)

CSV3 = next(
    (
        p for p in csv_files
        if "Collision Counts and Comparison to Benchmarks"
        in p.name
    ),
    None
)

CSV4 = next(
    (
        p for p in csv_files
        if "Miles and Benchmark Crashes for Dynamic Benchmark"
        in p.name
    ),
    None
)

if CSV2 is None:
    raise RuntimeError(
        "C06 FAIL_CLOSED: CSV2 crash file not found."
    )

if CSV3 is None:
    raise RuntimeError(
        "C06 FAIL_CLOSED: CSV3 aggregate comparison file not found."
    )

if CSV4 is None:
    raise RuntimeError(
        "C06 FAIL_CLOSED: CSV4 dynamic benchmark file not found."
    )


# ============================================================
# 3. READ DATA
# ============================================================

try:

    df2 = pd.read_csv(CSV2)
    df3 = pd.read_csv(CSV3)
    df4 = pd.read_csv(CSV4)

except Exception as e:

    raise RuntimeError(
        "C06 FAIL_CLOSED: CSV reading failed: "
        + repr(e)
    )


# ============================================================
# 4. REQUIRED COLUMN CHECK
# ============================================================

required_csv2 = [
    "SGO Report ID",
    "Crash Type",
    "Is Any-Injury-Reported",
    "Is Any Vehicle Airbag Deployment",
    "Is Ego Vehicle Airbag Deployment",
    "Is Suspected Serious Injury+",
    "Is Any Fatal Injury",
    "Incident Date"
]

required_csv3 = [
    "Outcome",
    "Waymo Count",
    "Waymo IPMM",
    "Benchmark IPMM",
    "Waymo Percent Reduction",
    "Waymo Rate Ratio"
]

required_csv4 = [
    "Outcome",
    "Benchmark Crash Count",
    "Waymo RO Miles"
]

missing_csv2 = [
    c for c in required_csv2
    if c not in df2.columns
]

missing_csv3 = [
    c for c in required_csv3
    if c not in df3.columns
]

missing_csv4 = [
    c for c in required_csv4
    if c not in df4.columns
]

if missing_csv2 or missing_csv3 or missing_csv4:

    raise RuntimeError(
        "C06 FAIL_CLOSED: Required columns missing.\n"
        f"CSV2 missing: {missing_csv2}\n"
        f"CSV3 missing: {missing_csv3}\n"
        f"CSV4 missing: {missing_csv4}"
    )


# ============================================================
# 5. CSV2 OUTCOME FIELD INSPECTION
# ============================================================

csv2_fields = [
    "Crash Type",
    "Is Any-Injury-Reported",
    "Is Any Vehicle Airbag Deployment",
    "Is Ego Vehicle Airbag Deployment",
    "Is Suspected Serious Injury+",
    "Is Any Fatal Injury",
    "Is NHTSA Reportable In-Transport",
    "Is NHTSA Reportable In-Transport Delta-V Less than 1 MPH",
    "Is Police-Reported"
]

csv2_value_inventory = {}

for field in csv2_fields:

    series = df2[field]

    value_counts = (
        series
        .fillna("<NA>")
        .astype(str)
        .value_counts(dropna=False)
        .to_dict()
    )

    csv2_value_inventory[field] = {
        "non_null_count":
            int(series.notna().sum()),

        "null_count":
            int(series.isna().sum()),

        "unique_count":
            int(series.nunique(dropna=True)),

        "values":
            value_counts
    }


# ============================================================
# 6. CSV3 OUTCOME INVENTORY
# ============================================================

csv3_outcomes = (
    df3["Outcome"]
    .fillna("<NA>")
    .astype(str)
    .value_counts(dropna=False)
    .to_dict()
)


# ============================================================
# 7. CSV3 OUTCOME + BENCHMARK RELATION
# ============================================================

csv3_outcome_summary = []

for outcome, group in df3.groupby(
    "Outcome",
    dropna=False
):

    csv3_outcome_summary.append({

        "Outcome":
            str(outcome),

        "rows":
            int(len(group)),

        "non_null_Waymo_Count":
            int(group["Waymo Count"].notna().sum()),

        "non_null_Waymo_IPMM":
            int(group["Waymo IPMM"].notna().sum()),

        "non_null_Benchmark_IPMM":
            int(group["Benchmark IPMM"].notna().sum()),

        "non_null_Percent_Reduction":
            int(group["Waymo Percent Reduction"].notna().sum()),

        "non_null_Rate_Ratio":
            int(group["Waymo Rate Ratio"].notna().sum())
    })


# ============================================================
# 8. CSV4 OUTCOME INVENTORY
# ============================================================

csv4_outcomes = (
    df4["Outcome"]
    .fillna("<NA>")
    .astype(str)
    .value_counts(dropna=False)
    .to_dict()
)


# ============================================================
# 9. DIRECT TARGET OUTCOME SEARCH
# ============================================================

target_terms = [
    "Any Injury",
    "Any-injury",
    "Any Fatal",
    "Serious",
    "Airbag",
    "Police",
    "Delta",
    "Pedestrian",
    "Cyclist",
    "Motorcycle"
]

target_outcome_matches = {}

for term in target_terms:

    matches = []

    for value in csv3_outcomes.keys():

        if term.lower() in str(value).lower():

            matches.append(str(value))

    target_outcome_matches[term] = matches


# ============================================================
# 10. OUTCOME DEFINITION STATUS
# ============================================================

# Important:
# This cell inventories actual observed outcome labels,
# but does NOT yet authorize their semantic interpretation.
#
# Documentary definitions will be reconciled in the next
# evidence stage if required.

outcome_definition_status = {

    "CSV2_field_inventory":
        "OBSERVED_NOT_SEMANTICALLY_AUTHORIZED",

    "CSV3_outcome_inventory":
        "OBSERVED_NOT_SEMANTICALLY_AUTHORIZED",

    "CSV4_outcome_inventory":
        "OBSERVED_NOT_SEMANTICALLY_AUTHORIZED",

    "claim_outcome_mapping":
        "NOT_AUTHORIZED",

    "scientific_validity":
        "NOT_EVALUATED"
}


# ============================================================
# 11. SOURCE HASHES
# ============================================================

source_hashes = {}

for path in [CSV2, CSV3, CSV4]:

    source_hashes[path.name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


# ============================================================
# 12. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "sources":
        source_hashes,

    "csv2": {

        "rows":
            int(df2.shape[0]),

        "columns":
            int(df2.shape[1]),

        "outcome_field_inventory":
            csv2_value_inventory
    },

    "csv3": {

        "rows":
            int(df3.shape[0]),

        "columns":
            int(df3.shape[1]),

        "outcome_inventory":
            csv3_outcomes,

        "outcome_metric_summary":
            csv3_outcome_summary,

        "target_outcome_matches":
            target_outcome_matches
    },

    "csv4": {

        "rows":
            int(df4.shape[0]),

        "columns":
            int(df4.shape[1]),

        "outcome_inventory":
            csv4_outcomes
    },

    "outcome_definition_status":
        outcome_definition_status,

    "claim_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED",

    "status":
        "PASS_OUTCOME_INVENTORY"
}


# ============================================================
# 13. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C06_CRASH_OUTCOME_DEFINITION_GATE.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 14. CONSOLE OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C06 "
    "CRASH / OUTCOME DEFINITION GATE"
)
print("=" * 72)

print()

print("SOURCE FILES")
print(
    "CSV2 :",
    CSV2.name
)

print(
    "CSV3 :",
    CSV3.name
)

print(
    "CSV4 :",
    CSV4.name
)

print()

print("-" * 72)
print("CSV2 OUTCOME FIELD INVENTORY")
print("-" * 72)

for field, info in csv2_value_inventory.items():

    print(
        f"{field:55s} | "
        f"non_null={info['non_null_count']} | "
        f"null={info['null_count']} | "
        f"unique={info['unique_count']}"
    )


print()

print("-" * 72)
print("CSV3 OUTCOME VALUES")
print("-" * 72)

for outcome, count in csv3_outcomes.items():

    print(
        f"{outcome} : {count}"
    )


print()

print("-" * 72)
print("TARGET OUTCOME MATCHES")
print("-" * 72)

for term, matches in target_outcome_matches.items():

    print(
        f"{term:20s} : {matches}"
    )


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Outcome semantic authorization :",
    "NOT_AUTHORIZED"
)

print(
    "Claim authorization            :",
    False
)

print(
    "Scientific authorization       :",
    False
)

print(
    "Road safety verdict            :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C06 STATUS")
print("-" * 72)

print(
    "C06 STATUS :",
    "PASS_OUTCOME_INVENTORY"
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C06 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C06 CRASH / OUTCOME DEFINITION GATE

SOURCE FILES
CSV2 : CSV2 - Crashes with SGO ID and Group Membership 202009-202606-2022benchmark.csv
CSV3 : CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv
CSV4 : CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

------------------------------------------------------------------------
CSV2 OUTCOME FIELD INVENTORY
------------------------------------------------------------------------
Crash Type                                              | non_null=1977 | null=0 | unique=11
Is Any-Injury-Reported                                  | non_null=1977 | null=0 | unique=2
Is Any Vehicle Airbag Deployment                        | non_null=1977 | null=0 | unique=2
Is Ego Vehicle Airbag Deployment                        | non_null=1977 | null=0 | unique=2
Is Suspected Serious Injury+                            | non_null=1977 | null=0 | unique=2
Is Any Fatal Injury     

In [71]:
# ============================================================
# OIP v1.0.39 — C07
# HUMAN BENCHMARK DEFINITION & COMPARABILITY AUDIT
# ============================================================

import json
import hashlib
import re
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd
from pypdf import PdfReader


# ============================================================
# 1. IDENTITY / BOUNDARY
# ============================================================

AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C07"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C07")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "benchmark_semantic_authorization": False,
    "comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 2. FIND SOURCE FILES
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))
pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C07 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

if len(pdf_files) != 3:
    raise RuntimeError(
        f"C07 FAIL_CLOSED: Expected 3 PDF files, found {len(pdf_files)}."
    )


CSV1 = next(
    p for p in csv_files
    if "RO Miles per Location" in p.name
)

CSV2 = next(
    p for p in csv_files
    if "Crashes with SGO ID and Group Membership" in p.name
)

CSV3 = next(
    p for p in csv_files
    if "Collision Counts and Comparison to Benchmarks" in p.name
)

CSV4 = next(
    p for p in csv_files
    if "Miles and Benchmark Crashes for Dynamic Benchmark" in p.name
)


# ============================================================
# 3. READ CSV FILES
# ============================================================

try:

    df1 = pd.read_csv(CSV1)
    df2 = pd.read_csv(CSV2)
    df3 = pd.read_csv(CSV3)
    df4 = pd.read_csv(CSV4)

except Exception as e:

    raise RuntimeError(
        "C07 FAIL_CLOSED: CSV read failed: "
        + repr(e)
    )


# ============================================================
# 4. REQUIRED BENCHMARK FIELDS
# ============================================================

required_csv3 = [
    "Outcome",
    "County Name",
    "Waymo Count",
    "Waymo IPMM",
    "Waymo IPMM CI Lower",
    "Waymo IPMM CI Upper",
    "Benchmark IPMM",
    "Benchmark IPMM CI Lower",
    "Benchmark IPMM CI Upper",
    "Predicted Benchmark Count",
    "Waymo Count Reduction",
    "Waymo Percent Reduction",
    "Waymo Percent Reduction CI Lower",
    "Waymo Percent Reduction CI Upper",
    "Waymo Rate Ratio",
    "Waymo Rate Ratio CI Lower",
    "Waymo Rate Ratio CI Upper"
]

missing_csv3 = [
    c for c in required_csv3
    if c not in df3.columns
]

if missing_csv3:

    raise RuntimeError(
        "C07 FAIL_CLOSED: Required benchmark fields missing: "
        + repr(missing_csv3)
    )


required_csv4 = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing_csv4 = [
    c for c in required_csv4
    if c not in df4.columns
]

if missing_csv4:

    raise RuntimeError(
        "C07 FAIL_CLOSED: Required dynamic benchmark fields missing: "
        + repr(missing_csv4)
    )


# ============================================================
# 5. CSV3 BENCHMARK INVENTORY
# ============================================================

benchmark_summary = {

    "rows":
        int(len(df3)),

    "benchmark_ipmm_non_null":
        int(df3["Benchmark IPMM"].notna().sum()),

    "benchmark_ipmm_null":
        int(df3["Benchmark IPMM"].isna().sum()),

    "benchmark_ci_lower_non_null":
        int(df3["Benchmark IPMM CI Lower"].notna().sum()),

    "benchmark_ci_upper_non_null":
        int(df3["Benchmark IPMM CI Upper"].notna().sum()),

    "predicted_benchmark_count_non_null":
        int(df3["Predicted Benchmark Count"].notna().sum()),

    "waymo_ipmm_non_null":
        int(df3["Waymo IPMM"].notna().sum()),

    "waymo_percent_reduction_non_null":
        int(df3["Waymo Percent Reduction"].notna().sum()),

    "waymo_rate_ratio_non_null":
        int(df3["Waymo Rate Ratio"].notna().sum())
}


# ============================================================
# 6. OUTCOME × BENCHMARK STRUCTURE
# ============================================================

outcome_benchmark_summary = []

for outcome, group in df3.groupby(
    "Outcome",
    dropna=False
):

    outcome_benchmark_summary.append({

        "Outcome":
            str(outcome),

        "rows":
            int(len(group)),

        "benchmark_ipmm_non_null":
            int(group["Benchmark IPMM"].notna().sum()),

        "waymo_ipmm_non_null":
            int(group["Waymo IPMM"].notna().sum()),

        "percent_reduction_non_null":
            int(
                group["Waymo Percent Reduction"]
                .notna()
                .sum()
            ),

        "rate_ratio_non_null":
            int(
                group["Waymo Rate Ratio"]
                .notna()
                .sum()
            )
    })


# ============================================================
# 7. GEOGRAPHIC BENCHMARK STRUCTURE
# ============================================================

geo_summary = {

    "CSV1_location_rows":
        int(len(df1)),

    "CSV1_total_RO_miles_millions":
        float(
            df1["Waymo RO Miles (Millions)"]
            .sum()
        ),

    "CSV4_unique_states":
        int(df4["state_name"].nunique()),

    "CSV4_unique_counties":
        int(df4["county_name"].nunique()),

    "CSV4_unique_S2_cells":
        int(df4["S2 Cell"].nunique()),

    "CSV4_benchmark_crash_count_non_null":
        int(
            df4["Benchmark Crash Count"]
            .notna()
            .sum()
        ),

    "CSV4_HPMS_VMT_non_null":
        int(
            df4["HPMS Yearly Vehicle Miles Traveled"]
            .notna()
            .sum()
        ),

    "CSV4_Waymo_RO_miles_non_null":
        int(
            df4["Waymo RO Miles"]
            .notna()
            .sum()
        )
}


# ============================================================
# 8. PDF DOCUMENTARY SEARCH
# ============================================================

def normalize(text):
    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def search_term(
    term,
    text,
    context_chars=450
):

    lower_text = text.lower()
    lower_term = term.lower()

    results = []
    start = 0

    while True:

        idx = lower_text.find(
            lower_term,
            start
        )

        if idx == -1:
            break

        left = max(
            0,
            idx - context_chars
        )

        right = min(
            len(text),
            idx + len(term) + context_chars
        )

        results.append(
            normalize(
                text[left:right]
            )
        )

        start = idx + len(lower_term)

    return results


pdf_documents = {}

for path in pdf_files:

    reader = PdfReader(str(path))

    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    pdf_documents[path.name] = text


# ============================================================
# 9. BENCHMARK DOCUMENTARY TARGETS
# ============================================================

benchmark_terms = [

    "human benchmark",
    "human benchmarks",
    "benchmark crash rate",
    "benchmark",
    "IPMM",
    "incidents per million miles",
    "million miles",
    "vehicle miles traveled",
    "VMT",
    "police-reported",
    "Rider-Only",
    "rider-only",
    "crash rate",
    "benchmark methodology",
    "dynamic benchmark",
    "geographic",
    "S2 Cell"
]


benchmark_documentary_evidence = {}

for term in benchmark_terms:

    term_results = []

    for filename, text in pdf_documents.items():

        matches = search_term(
            term,
            text
        )

        if matches:

            term_results.append({

                "filename":
                    filename,

                "match_count":
                    len(matches),

                "contexts":
                    matches[:3]
            })

    benchmark_documentary_evidence[term] = {
        "documentary_occurrence":
            len(term_results) > 0,

        "sources":
            term_results
    }


# ============================================================
# 10. STRUCTURAL COMPARABILITY CHECK
# ============================================================

comparability_components = {

    "waymo_exposure_present":
        bool(
            df3["Waymo IPMM"].notna().any()
        ),

    "human_benchmark_rate_present":
        bool(
            df3["Benchmark IPMM"].notna().any()
        ),

    "benchmark_uncertainty_present":
        bool(
            df3["Benchmark IPMM CI Lower"]
            .notna()
            .any()
            and
            df3["Benchmark IPMM CI Upper"]
            .notna()
            .any()
        ),

    "waymo_uncertainty_present":
        bool(
            df3["Waymo IPMM CI Lower"]
            .notna()
            .any()
            and
            df3["Waymo IPMM CI Upper"]
            .notna()
            .any()
        ),

    "predicted_benchmark_count_present":
        bool(
            df3["Predicted Benchmark Count"]
            .notna()
            .any()
        ),

    "geographic_benchmark_structure_present":
        bool(
            df4["S2 Cell"].notna().any()
            and
            df4["Benchmark Crash Count"]
            .notna()
            .any()
        ),

    "waymo_ro_exposure_structure_present":
        bool(
            df4["Waymo RO Miles"]
            .notna()
            .any()
        )
}


# ============================================================
# 11. IMPORTANT OIP BOUNDARY
# ============================================================

# Structural presence does NOT prove comparability.
#
# We therefore do not authorize comparative validity here.
#
# C07 only establishes whether the required benchmark
# components are observable and whether documentary
# evidence exists for further comparability review.

structural_components_missing = [
    key
    for key, value in comparability_components.items()
    if value is False
]

if len(structural_components_missing) > 0:

    structural_status = "PENDING_EVIDENCE"

else:

    structural_status = "COMPUTABLE_BUT_UNVALIDATED"


# ============================================================
# 12. SOURCE HASHES
# ============================================================

source_hashes = {}

for path in csv_files + pdf_files:

    source_hashes[path.name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


# ============================================================
# 13. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "source_hashes":
        source_hashes,

    "benchmark_summary":
        benchmark_summary,

    "outcome_benchmark_summary":
        outcome_benchmark_summary,

    "geographic_summary":
        geo_summary,

    "benchmark_documentary_evidence":
        benchmark_documentary_evidence,

    "comparability_components":
        comparability_components,

    "structural_components_missing":
        structural_components_missing,

    "comparability_status":
        structural_status,

    "rules": {

        "structural_presence_equals_comparability":
            False,

        "benchmark_presence_equals_validity":
            False,

        "comparative_claim_authorization":
            False,

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        structural_status
}


# ============================================================
# 14. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C07_HUMAN_BENCHMARK_COMPARABILITY_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 15. CONSOLE OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C07 "
    "HUMAN BENCHMARK DEFINITION & COMPARABILITY AUDIT"
)
print("=" * 72)

print()

print("-" * 72)
print("BENCHMARK STRUCTURE")
print("-" * 72)

for key, value in benchmark_summary.items():

    print(
        f"{key:42s} : {value}"
    )


print()

print("-" * 72)
print("OUTCOME × BENCHMARK STRUCTURE")
print("-" * 72)

for row in outcome_benchmark_summary:

    print(
        row
    )


print()

print("-" * 72)
print("GEOGRAPHIC / EXPOSURE STRUCTURE")
print("-" * 72)

for key, value in geo_summary.items():

    print(
        f"{key:42s} : {value}"
    )


print()

print("-" * 72)
print("COMPARABILITY COMPONENTS")
print("-" * 72)

for key, value in comparability_components.items():

    print(
        f"{key:45s} : {value}"
    )


print()

print("-" * 72)
print("DOCUMENTARY BENCHMARK EVIDENCE")
print("-" * 72)

for term, result in benchmark_documentary_evidence.items():

    print(
        f"{term:32s} : "
        f"{result['documentary_occurrence']}"
    )


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Benchmark semantic authorization :",
    False
)

print(
    "Comparability authorization      :",
    False
)

print(
    "Claim authorization              :",
    False
)

print(
    "Scientific authorization         :",
    False
)

print(
    "Road safety verdict              :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C07 STATUS")
print("-" * 72)

print(
    "C07 STATUS :",
    structural_status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C07 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C07 HUMAN BENCHMARK DEFINITION & COMPARABILITY AUDIT

------------------------------------------------------------------------
BENCHMARK STRUCTURE
------------------------------------------------------------------------
rows                                       : 1804
benchmark_ipmm_non_null                    : 1804
benchmark_ipmm_null                        : 0
benchmark_ci_lower_non_null                : 1804
benchmark_ci_upper_non_null                : 1804
predicted_benchmark_count_non_null         : 1804
waymo_ipmm_non_null                        : 1804
waymo_percent_reduction_non_null           : 1694
waymo_rate_ratio_non_null                  : 1694

------------------------------------------------------------------------
OUTCOME × BENCHMARK STRUCTURE
------------------------------------------------------------------------
{'Outcome': 'Any Airbag Deployment', 'rows': 260, 'benchmark_ipmm_non_null': 260, 'waymo_ipmm_non_null': 260, 'percent_reduction_non_null': 25

In [72]:
# ============================================================
# OIP v1.0.39 — C08
# MISSINGNESS / ENCODING / COVERAGE AUDIT
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. BOUNDARY
# ============================================================

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",

    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 2. SOURCE DISCOVERY
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C08 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )


CSV1 = next(
    (
        p for p in csv_files
        if "RO Miles per Location" in p.name
    ),
    None
)

CSV2 = next(
    (
        p for p in csv_files
        if "Crashes with SGO ID and Group Membership" in p.name
    ),
    None
)

CSV3 = next(
    (
        p for p in csv_files
        if "Collision Counts and Comparison to Benchmarks" in p.name
    ),
    None
)

CSV4 = next(
    (
        p for p in csv_files
        if "Miles and Benchmark Crashes for Dynamic Benchmark" in p.name
    ),
    None
)


if any(
    x is None
    for x in [CSV1, CSV2, CSV3, CSV4]
):
    raise RuntimeError(
        "C08 FAIL_CLOSED: One or more required CSV files were not found."
    )


# ============================================================
# 3. READ DATA
# ============================================================

try:

    df1 = pd.read_csv(CSV1)
    df2 = pd.read_csv(CSV2)
    df3 = pd.read_csv(CSV3)
    df4 = pd.read_csv(CSV4)

except Exception as e:

    raise RuntimeError(
        "C08 FAIL_CLOSED: CSV reading failed: "
        + repr(e)
    )


# ============================================================
# 4. GENERAL MISSINGNESS INVENTORY
# ============================================================

def missingness_inventory(df):

    result = {}

    for column in df.columns:

        series = df[column]

        result[column] = {

            "rows":
                int(len(series)),

            "non_null":
                int(series.notna().sum()),

            "null":
                int(series.isna().sum()),

            "missing_percent":
                float(
                    series.isna().mean() * 100
                ),

            "unique_non_null":
                int(
                    series.nunique(
                        dropna=True
                    )
                )
        }

    return result


missingness = {

    "CSV1": missingness_inventory(df1),
    "CSV2": missingness_inventory(df2),
    "CSV3": missingness_inventory(df3),
    "CSV4": missingness_inventory(df4)
}


# ============================================================
# 5. BOOLEAN / ENCODING INVENTORY — CSV2
# ============================================================

binary_fields = [

    "Is Any-Injury-Reported",
    "Is Any Vehicle Airbag Deployment",
    "Is Ego Vehicle Airbag Deployment",
    "Is Suspected Serious Injury+",
    "Is Any Fatal Injury",
    "Is NHTSA Reportable In-Transport",
    "Is NHTSA Reportable In-Transport Delta-V Less than 1 MPH",
    "Is Police-Reported"
]


binary_encoding = {}

for field in binary_fields:

    if field not in df2.columns:
        raise RuntimeError(
            f"C08 FAIL_CLOSED: Missing binary field: {field}"
        )

    series = df2[field]

    values = (
        series
        .fillna("<NA>")
        .astype(str)
        .value_counts(dropna=False)
        .to_dict()
    )

    binary_encoding[field] = {

        "dtype":
            str(series.dtype),

        "unique_non_null":
            int(series.nunique(dropna=True)),

        "values":
            values,

        "null_count":
            int(series.isna().sum())
    }


# ============================================================
# 6. CRITICAL IDENTIFIER / DATE COVERAGE
# ============================================================

critical_fields = {

    "CSV2_SGO_Report_ID":
        "SGO Report ID",

    "CSV2_Incident_Date":
        "Incident Date",

    "CSV3_Outcome":
        "Outcome",

    "CSV3_Waymo_IPMM":
        "Waymo IPMM",

    "CSV3_Benchmark_IPMM":
        "Benchmark IPMM",

    "CSV3_Predicted_Benchmark_Count":
        "Predicted Benchmark Count",

    "CSV4_Benchmark_Crash_Count":
        "Benchmark Crash Count",

    "CSV4_HPMS_VMT":
        "HPMS Yearly Vehicle Miles Traveled",

    "CSV4_Waymo_RO_Miles":
        "Waymo RO Miles"
}


critical_coverage = {}


for label, field in critical_fields.items():

    if label.startswith("CSV2_"):
        df = df2

    elif label.startswith("CSV3_"):
        df = df3

    else:
        df = df4

    series = df[field]

    critical_coverage[label] = {

        "field":
            field,

        "rows":
            int(len(series)),

        "non_null":
            int(series.notna().sum()),

        "null":
            int(series.isna().sum()),

        "coverage_percent":
            float(
                series.notna().mean() * 100
            )
    }


# ============================================================
# 7. CSV3 OUTCOME COVERAGE
# ============================================================

csv3_outcome_coverage = []

for outcome, group in df3.groupby(
    "Outcome",
    dropna=False
):

    row = {

        "Outcome":
            str(outcome),

        "rows":
            int(len(group)),

        "Waymo_IPMM_non_null":
            int(group["Waymo IPMM"].notna().sum()),

        "Benchmark_IPMM_non_null":
            int(group["Benchmark IPMM"].notna().sum()),

        "Predicted_Benchmark_Count_non_null":
            int(
                group["Predicted Benchmark Count"]
                .notna()
                .sum()
            ),

        "Percent_Reduction_non_null":
            int(
                group["Waymo Percent Reduction"]
                .notna()
                .sum()
            ),

        "Rate_Ratio_non_null":
            int(
                group["Waymo Rate Ratio"]
                .notna()
                .sum()
            )
    }

    csv3_outcome_coverage.append(row)


# ============================================================
# 8. CSV4 EXPOSURE / BENCHMARK COVERAGE
# ============================================================

csv4_coverage = {

    "rows":
        int(len(df4)),

    "Benchmark_Crash_Count_non_null":
        int(
            df4["Benchmark Crash Count"]
            .notna()
            .sum()
        ),

    "HPMS_VMT_non_null":
        int(
            df4["HPMS Yearly Vehicle Miles Traveled"]
            .notna()
            .sum()
        ),

    "Waymo_RO_Miles_non_null":
        int(
            df4["Waymo RO Miles"]
            .notna()
            .sum()
        ),

    "S2_Cell_non_null":
        int(
            df4["S2 Cell"]
            .notna()
            .sum()
        ),

    "Outcome_non_null":
        int(
            df4["Outcome"]
            .notna()
            .sum()
        )
}


# ============================================================
# 9. TEMPORAL COVERAGE INVENTORY
# ============================================================

temporal_coverage = {

    "CSV2_Incident_Date_dtype":
        str(df2["Incident Date"].dtype),

    "CSV2_Incident_Date_non_null":
        int(df2["Incident Date"].notna().sum()),

    "CSV2_Incident_Date_null":
        int(df2["Incident Date"].isna().sum()),

    "CSV2_Year_Month_non_null":
        int(df2["Year Month"].notna().sum()),

    "CSV2_Year_Month_null":
        int(df2["Year Month"].isna().sum())
}


# ============================================================
# 10. DUPLICATE INVENTORY
# ============================================================

duplicate_inventory = {

    "CSV1_duplicate_rows":
        int(df1.duplicated().sum()),

    "CSV2_duplicate_rows":
        int(df2.duplicated().sum()),

    "CSV3_duplicate_rows":
        int(df3.duplicated().sum()),

    "CSV4_duplicate_rows":
        int(df4.duplicated().sum())
}


# ============================================================
# 11. NUMERIC NEGATIVE / ZERO EXPOSURE INVENTORY
# ============================================================

numeric_exposure_inventory = {

    "CSV1_zero_RO_miles":
        int(
            (
                df1["Waymo RO Miles (Millions)"] == 0
            ).sum()
        ),

    "CSV1_negative_RO_miles":
        int(
            (
                df1["Waymo RO Miles (Millions)"] < 0
            ).sum()
        ),

    "CSV4_zero_HPMS_VMT":
        int(
            (
                df4["HPMS Yearly Vehicle Miles Traveled"] == 0
            ).sum()
        ),

    "CSV4_negative_HPMS_VMT":
        int(
            (
                df4["HPMS Yearly Vehicle Miles Traveled"] < 0
            ).sum()
        ),

    "CSV4_zero_Waymo_RO_miles":
        int(
            (
                df4["Waymo RO Miles"] == 0
            ).sum()
        ),

    "CSV4_negative_Waymo_RO_miles":
        int(
            (
                df4["Waymo RO Miles"] < 0
            ).sum()
        )
}


# ============================================================
# 12. STRUCTURAL STATUS
# ============================================================

critical_nulls = {
    key: value["null"]
    for key, value in critical_coverage.items()
    if value["null"] > 0
}

unexpected_binary_cardinality = {

    field: info["unique_non_null"]

    for field, info in binary_encoding.items()

    if info["unique_non_null"] != 2
}


if len(unexpected_binary_cardinality) > 0:

    structural_status = "PENDING_EVIDENCE"

elif len(critical_nulls) > 0:

    structural_status = "PENDING_EVIDENCE"

else:

    structural_status = "PASS_MISSINGNESS_ENCODING_COVERAGE"


# ============================================================
# 13. SOURCE HASHES
# ============================================================

source_hashes = {}

for path in csv_files:

    source_hashes[path.name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


# ============================================================
# 14. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "source_hashes":
        source_hashes,

    "missingness":
        missingness,

    "binary_encoding":
        binary_encoding,

    "critical_coverage":
        critical_coverage,

    "csv3_outcome_coverage":
        csv3_outcome_coverage,

    "csv4_coverage":
        csv4_coverage,

    "temporal_coverage":
        temporal_coverage,

    "duplicate_inventory":
        duplicate_inventory,

    "numeric_exposure_inventory":
        numeric_exposure_inventory,

    "critical_nulls":
        critical_nulls,

    "unexpected_binary_cardinality":
        unexpected_binary_cardinality,

    "rules": {

        "missingness_equals_zero":
            False,

        "binary_encoding_equals_semantic_validity":
            False,

        "coverage_equals_estimability":
            False,

        "semantic_authorization":
            False,

        "benchmark_comparability_authorization":
            False,

        "claim_authorization":
            False,

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        structural_status
}


# ============================================================
# 15. SAVE ARTIFACT
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08_MISSINGNESS_ENCODING_COVERAGE_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 16. CONSOLE OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08 "
    "MISSINGNESS / ENCODING / COVERAGE AUDIT"
)
print("=" * 72)

print()

print("-" * 72)
print("CRITICAL FIELD COVERAGE")
print("-" * 72)

for key, info in critical_coverage.items():

    print(
        f"{key:42s} | "
        f"non_null={info['non_null']} | "
        f"null={info['null']} | "
        f"coverage={info['coverage_percent']:.2f}%"
    )


print()

print("-" * 72)
print("BINARY ENCODING")
print("-" * 72)

for field, info in binary_encoding.items():

    print(
        f"{field:55s} | "
        f"unique={info['unique_non_null']} | "
        f"null={info['null_count']} | "
        f"values={info['values']}"
    )


print()

print("-" * 72)
print("CSV3 OUTCOME COVERAGE")
print("-" * 72)

for row in csv3_outcome_coverage:

    print(row)


print()

print("-" * 72)
print("CSV4 EXPOSURE / BENCHMARK COVERAGE")
print("-" * 72)

for key, value in csv4_coverage.items():

    print(
        f"{key:42s} : {value}"
    )


print()

print("-" * 72)
print("TEMPORAL COVERAGE")
print("-" * 72)

for key, value in temporal_coverage.items():

    print(
        f"{key:42s} : {value}"
    )


print()

print("-" * 72)
print("DUPLICATE INVENTORY")
print("-" * 72)

for key, value in duplicate_inventory.items():

    print(
        f"{key:32s} : {value}"
    )


print()

print("-" * 72)
print("EXPOSURE ZERO / NEGATIVE INVENTORY")
print("-" * 72)

for key, value in numeric_exposure_inventory.items():

    print(
        f"{key:32s} : {value}"
    )


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Semantic authorization              :",
    False
)

print(
    "Benchmark comparability             :",
    False
)

print(
    "Claim authorization                 :",
    False
)

print(
    "Scientific authorization            :",
    False
)

print(
    "Road safety verdict                 :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C08 STATUS")
print("-" * 72)

print(
    "C08 STATUS :",
    structural_status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08 MISSINGNESS / ENCODING / COVERAGE AUDIT

------------------------------------------------------------------------
CRITICAL FIELD COVERAGE
------------------------------------------------------------------------
CSV2_SGO_Report_ID                         | non_null=1975 | null=2 | coverage=99.90%
CSV2_Incident_Date                         | non_null=1977 | null=0 | coverage=100.00%
CSV3_Outcome                               | non_null=1804 | null=0 | coverage=100.00%
CSV3_Waymo_IPMM                            | non_null=1804 | null=0 | coverage=100.00%
CSV3_Benchmark_IPMM                        | non_null=1804 | null=0 | coverage=100.00%
CSV3_Predicted_Benchmark_Count             | non_null=1804 | null=0 | coverage=100.00%
CSV4_Benchmark_Crash_Count                 | non_null=18352 | null=0 | coverage=100.00%
CSV4_HPMS_VMT                              | non_null=18352 | null=0 | coverage=100.00%
CSV4_Waymo_RO_Miles                        | non_null=18352 | null=0 | cov

In [73]:
# ============================================================
# OIP v1.0.39 — C08R1
# MISSINGNESS & DUPLICATE EXCEPTION RESOLUTION
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R1"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. BOUNDARY
# ============================================================

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",

    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_value_repair_authorized": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 2. SOURCE DISCOVERY
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C08R1 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

CSV2 = next(
    (
        p for p in csv_files
        if "Crashes with SGO ID and Group Membership" in p.name
    ),
    None
)

CSV3 = next(
    (
        p for p in csv_files
        if "Collision Counts and Comparison to Benchmarks" in p.name
    ),
    None
)

CSV4 = next(
    (
        p for p in csv_files
        if "Miles and Benchmark Crashes for Dynamic Benchmark" in p.name
    ),
    None
)

if any(x is None for x in [CSV2, CSV3, CSV4]):
    raise RuntimeError(
        "C08R1 FAIL_CLOSED: Required source file not found."
    )


# ============================================================
# 3. READ
# ============================================================

try:
    df2 = pd.read_csv(CSV2)
    df3 = pd.read_csv(CSV3)
    df4 = pd.read_csv(CSV4)
except Exception as e:
    raise RuntimeError(
        "C08R1 FAIL_CLOSED: CSV read failed: " + repr(e)
    )


# ============================================================
# 4. MISSING SGO REPORT ID
# ============================================================

missing_id_rows = df2[df2["SGO Report ID"].isna()].copy()

missing_id_summary = {
    "count": int(len(missing_id_rows)),
    "columns_present": list(df2.columns),
    "rows": []
}

for idx, row in missing_id_rows.iterrows():

    missing_id_summary["rows"].append({
        "dataframe_row_index": int(idx),

        "Incident Date":
            str(row.get("Incident Date")),

        "Year Month":
            str(row.get("Year Month")),

        "State":
            str(row.get("State")),

        "County":
            str(row.get("County")),

        "Crash Type":
            str(row.get("Crash Type")),

        "SGO Report Version":
            str(row.get("SGO Report Version")),

        "SGO Amendment":
            str(row.get("SGO Amendment")),

        "Is Police-Reported":
            str(row.get("Is Police-Reported")),

        "Is Any-Injury-Reported":
            str(row.get("Is Any-Injury-Reported")),

        "Is Any Vehicle Airbag Deployment":
            str(row.get("Is Any Vehicle Airbag Deployment")),

        "Is Ego Vehicle Airbag Deployment":
            str(row.get("Is Ego Vehicle Airbag Deployment")),

        "Is Suspected Serious Injury+":
            str(row.get("Is Suspected Serious Injury+")),

        "Is Any Fatal Injury":
            str(row.get("Is Any Fatal Injury"))
    })


# ============================================================
# 5. EXACT DUPLICATES — CSV2
# ============================================================

csv2_duplicate_mask = df2.duplicated(
    keep=False
)

csv2_duplicate_rows = df2[
    csv2_duplicate_mask
].copy()

csv2_duplicate_groups = (
    csv2_duplicate_rows
    .value_counts(dropna=False)
    .reset_index(name="multiplicity")
)

csv2_duplicate_summary = {
    "duplicate_row_count_including_all_copies":
        int(len(csv2_duplicate_rows)),

    "duplicate_excess_rows":
        int(df2.duplicated().sum()),

    "duplicate_groups":
        int(len(csv2_duplicate_groups)),

    "multiplicity_distribution":
        csv2_duplicate_groups["multiplicity"]
        .value_counts()
        .sort_index()
        .astype(int)
        .to_dict()
}


# ============================================================
# 6. EXACT DUPLICATES — CSV3
# ============================================================

csv3_duplicate_summary = {
    "duplicate_excess_rows":
        int(df3.duplicated().sum()),

    "duplicate_rows_including_all_copies":
        int(
            df3[df3.duplicated(keep=False)].shape[0]
        )
}


# ============================================================
# 7. EXACT DUPLICATES — CSV4
# ============================================================

csv4_duplicate_mask = df4.duplicated(
    keep=False
)

csv4_duplicate_rows = df4[
    csv4_duplicate_mask
].copy()

csv4_duplicate_groups = (
    csv4_duplicate_rows
    .value_counts(dropna=False)
    .reset_index(name="multiplicity")
)

csv4_duplicate_summary = {

    "duplicate_row_count_including_all_copies":
        int(len(csv4_duplicate_rows)),

    "duplicate_excess_rows":
        int(df4.duplicated().sum()),

    "duplicate_groups":
        int(len(csv4_duplicate_groups)),

    "multiplicity_distribution":
        csv4_duplicate_groups["multiplicity"]
        .value_counts()
        .sort_index()
        .astype(int)
        .to_dict()
}


# ============================================================
# 8. CSV4 DUPLICATE GROUP PREVIEW
# ============================================================

csv4_duplicate_preview = []

for _, row in csv4_duplicate_groups.head(20).iterrows():

    csv4_duplicate_preview.append({
        "state_name":
            str(row["state_name"]),

        "county_name":
            str(row["county_name"]),

        "S2 Cell":
            str(row["S2 Cell"]),

        "Outcome":
            str(row["Outcome"]),

        "Benchmark Crash Count":
            row["Benchmark Crash Count"],

        "HPMS Yearly Vehicle Miles Traveled":
            row["HPMS Yearly Vehicle Miles Traveled"],

        "Waymo RO Miles":
            row["Waymo RO Miles"],

        "multiplicity":
            int(row["multiplicity"])
    })


# ============================================================
# 9. DUPLICATE IMPACT — NO DELETION
# ============================================================

impact_inventory = {

    "CSV2_rows":
        int(len(df2)),

    "CSV2_duplicate_excess":
        int(df2.duplicated().sum()),

    "CSV4_rows":
        int(len(df4)),

    "CSV4_duplicate_excess":
        int(df4.duplicated().sum()),

    "CSV4_duplicate_excess_percent":
        float(
            df4.duplicated().sum()
            / len(df4)
            * 100
        )
}


# ============================================================
# 10. SOURCE HASHES
# ============================================================

source_hashes = {}

for path in [CSV2, CSV3, CSV4]:

    source_hashes[path.name] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


# ============================================================
# 11. STATUS
# ============================================================

exceptions_present = (
    missing_id_summary["count"] > 0
    or csv2_duplicate_summary["duplicate_excess_rows"] > 0
    or csv4_duplicate_summary["duplicate_excess_rows"] > 0
)

if exceptions_present:
    status = "PENDING_EVIDENCE"
else:
    status = "PASS_EXCEPTION_RESOLUTION"


# ============================================================
# 12. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "source_hashes":
        source_hashes,

    "missing_sgo_report_id":
        missing_id_summary,

    "csv2_duplicate_summary":
        csv2_duplicate_summary,

    "csv3_duplicate_summary":
        csv3_duplicate_summary,

    "csv4_duplicate_summary":
        csv4_duplicate_summary,

    "csv4_duplicate_preview":
        csv4_duplicate_preview,

    "impact_inventory":
        impact_inventory,

    "rules": {

        "drop_rows":
            False,

        "deduplicate":
            False,

        "fill_missing_id":
            False,

        "missing_equals_zero":
            False,

        "duplicate_equals_data_error":
            False,

        "semantic_authorization":
            False,

        "benchmark_comparability_authorization":
            False,

        "claim_authorization":
            False,

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        status
}


# ============================================================
# 13. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R1_MISSINGNESS_DUPLICATE_EXCEPTION_RESOLUTION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 14. OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R1 "
    "MISSINGNESS & DUPLICATE EXCEPTION RESOLUTION"
)
print("=" * 72)

print()

print("-" * 72)
print("MISSING SGO REPORT ID")
print("-" * 72)

print(
    "Missing SGO Report ID count :",
    missing_id_summary["count"]
)

for row in missing_id_summary["rows"]:
    print(row)


print()

print("-" * 72)
print("CSV2 EXACT DUPLICATES")
print("-" * 72)

for key, value in csv2_duplicate_summary.items():
    print(f"{key:45s} : {value}")


print()

print("-" * 72)
print("CSV3 EXACT DUPLICATES")
print("-" * 72)

for key, value in csv3_duplicate_summary.items():
    print(f"{key:45s} : {value}")


print()

print("-" * 72)
print("CSV4 EXACT DUPLICATES")
print("-" * 72)

for key, value in csv4_duplicate_summary.items():
    print(f"{key:45s} : {value}")


print()

print("-" * 72)
print("CSV4 DUPLICATE GROUP PREVIEW")
print("-" * 72)

for row in csv4_duplicate_preview:
    print(row)


print()

print("-" * 72)
print("DUPLICATE / MISSINGNESS IMPACT")
print("-" * 72)

for key, value in impact_inventory.items():
    print(f"{key:45s} : {value}")


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Row deletion authorized       :",
    False
)

print(
    "Deduplication authorized     :",
    False
)

print(
    "Missing ID repair authorized :",
    False
)

print(
    "Semantic authorization       :",
    False
)

print(
    "Benchmark comparability      :",
    False
)

print(
    "Claim authorization          :",
    False
)

print(
    "Scientific authorization     :",
    False
)

print(
    "Road safety verdict           :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C08R1 STATUS")
print("-" * 72)

print(
    "C08R1 STATUS :",
    status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R1 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R1 MISSINGNESS & DUPLICATE EXCEPTION RESOLUTION

------------------------------------------------------------------------
MISSING SGO REPORT ID
------------------------------------------------------------------------
Missing SGO Report ID count : 2
{'dataframe_row_index': 1975, 'Incident Date': '05/28/2021', 'Year Month': '202105', 'State': 'Arizona', 'County': 'Maricopa', 'Crash Type': 'V2V F2R', 'SGO Report Version': 'nan', 'SGO Amendment': 'Prior to SGO', 'Is Police-Reported': 'False', 'Is Any-Injury-Reported': 'False', 'Is Any Vehicle Airbag Deployment': 'False', 'Is Ego Vehicle Airbag Deployment': 'False', 'Is Suspected Serious Injury+': 'False', 'Is Any Fatal Injury': 'False'}
{'dataframe_row_index': 1976, 'Incident Date': '9/15/2020', 'Year Month': '202009', 'State': 'Arizona', 'County': 'Maricopa', 'Crash Type': 'V2V F2R', 'SGO Report Version': 'nan', 'SGO Amendment': 'Prior to SGO', 'Is Police-Reported': 'True', 'Is Any-Injury-Reported': 'False', 'Is Any Vehic

In [74]:
# ============================================================
# OIP v1.0.39 — C08R2
# DUPLICATE STRUCTURE & MISSING-ID CONTEXT AUDIT
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R2"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 1. SOURCE DISCOVERY
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C08R2 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

CSV2 = next(
    p for p in csv_files
    if "Crashes with SGO ID and Group Membership" in p.name
)

CSV4 = next(
    p for p in csv_files
    if "Miles and Benchmark Crashes for Dynamic Benchmark" in p.name
)

CSV2_HASH = hashlib.sha256(CSV2.read_bytes()).hexdigest()
CSV4_HASH = hashlib.sha256(CSV4.read_bytes()).hexdigest()


# ============================================================
# 2. READ
# ============================================================

try:
    df2 = pd.read_csv(CSV2)
    df4 = pd.read_csv(CSV4)
except Exception as e:
    raise RuntimeError(
        "C08R2 FAIL_CLOSED: CSV read failed: " + repr(e)
    )


# ============================================================
# 3. CSV2 MISSING-ID CONTEXT
# ============================================================

missing_id = df2[
    df2["SGO Report ID"].isna()
].copy()

missing_id_context = []

for idx, row in missing_id.iterrows():

    missing_id_context.append({
        "row_index": int(idx),
        "Incident Date": str(row["Incident Date"]),
        "Year Month": str(row["Year Month"]),
        "State": str(row["State"]),
        "County": str(row["County"]),
        "Road Type": str(row["Road Type"]),
        "Crash Type": str(row["Crash Type"]),
        "SGO Report Version": str(row["SGO Report Version"]),
        "SGO Amendment": str(row["SGO Amendment"]),
        "Is Police-Reported":
            str(row["Is Police-Reported"]),
        "Is Any-Injury-Reported":
            str(row["Is Any-Injury-Reported"]),
        "Is Any Vehicle Airbag Deployment":
            str(row["Is Any Vehicle Airbag Deployment"]),
        "Is Ego Vehicle Airbag Deployment":
            str(row["Is Ego Vehicle Airbag Deployment"]),
        "Is Suspected Serious Injury+":
            str(row["Is Suspected Serious Injury+"]),
        "Is Any Fatal Injury":
            str(row["Is Any Fatal Injury"])
    })


# ============================================================
# 4. CSV2 EXACT DUPLICATE CONTEXT
# ============================================================

dup2_mask = df2.duplicated(
    keep=False
)

dup2 = df2[dup2_mask].copy()

dup2_context = []

for idx, row in dup2.iterrows():

    dup2_context.append({
        "row_index": int(idx),
        "SGO Report ID":
            str(row["SGO Report ID"]),
        "Incident Date":
            str(row["Incident Date"]),
        "Year Month":
            str(row["Year Month"]),
        "State":
            str(row["State"]),
        "County":
            str(row["County"]),
        "Crash Type":
            str(row["Crash Type"]),
        "SGO Report Version":
            str(row["SGO Report Version"]),
        "SGO Amendment":
            str(row["SGO Amendment"])
    })


# ============================================================
# 5. CSV4 DUPLICATE GROUPS
# ============================================================

dup4_mask = df4.duplicated(
    keep=False
)

dup4 = df4[dup4_mask].copy()

dup4_groups = (
    dup4
    .value_counts(dropna=False)
    .reset_index(name="multiplicity")
)

# Every exact duplicate group should have multiplicity 2
multiplicity_distribution = (
    dup4_groups["multiplicity"]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)


# ============================================================
# 6. CSV4 DUPLICATE STRUCTURE BY OUTCOME
# ============================================================

dup4_by_outcome = (
    dup4_groups
    .groupby("Outcome", dropna=False)
    .agg(
        duplicate_groups=("multiplicity", "size"),
        duplicate_excess_rows=("multiplicity",
                                lambda x: int((x - 1).sum())),
        total_copies=("multiplicity", "sum")
    )
    .reset_index()
)

dup4_by_outcome_records = []

for _, row in dup4_by_outcome.iterrows():

    dup4_by_outcome_records.append({
        "Outcome":
            str(row["Outcome"]),
        "duplicate_groups":
            int(row["duplicate_groups"]),
        "duplicate_excess_rows":
            int(row["duplicate_excess_rows"]),
        "total_copies":
            int(row["total_copies"])
    })


# ============================================================
# 7. CSV4 DUPLICATE STRUCTURE BY COUNTY
# ============================================================

dup4_by_county = (
    dup4_groups
    .groupby(
        ["state_name", "county_name"],
        dropna=False
    )
    .agg(
        duplicate_groups=("multiplicity", "size"),
        duplicate_excess_rows=("multiplicity",
                                lambda x: int((x - 1).sum()))
    )
    .reset_index()
)

dup4_by_county_records = []

for _, row in dup4_by_county.iterrows():

    dup4_by_county_records.append({
        "state_name":
            str(row["state_name"]),
        "county_name":
            str(row["county_name"]),
        "duplicate_groups":
            int(row["duplicate_groups"]),
        "duplicate_excess_rows":
            int(row["duplicate_excess_rows"])
    })


# ============================================================
# 8. CSV4 DUPLICATE STRUCTURE BY OUTCOME + CELL
# ============================================================

dup4_outcome_cell = (
    dup4_groups
    .groupby(
        ["S2 Cell", "Outcome"],
        dropna=False
    )
    .size()
    .reset_index(name="duplicate_groups")
)

outcome_cell_summary = {

    "unique_S2_cells_with_exact_duplicates":
        int(
            dup4_outcome_cell["S2 Cell"].nunique()
        ),

    "unique_outcome_types_with_exact_duplicates":
        int(
            dup4_outcome_cell["Outcome"].nunique()
        ),

    "outcome_cell_duplicate_groups":
        int(
            len(dup4_outcome_cell)
        )
}


# ============================================================
# 9. DUPLICATE EXPOSURE VALUE CHECK
# ============================================================

duplicate_exposure_summary = {

    "duplicate_groups_with_zero_benchmark_crash_count":
        int(
            (
                dup4_groups["Benchmark Crash Count"] == 0
            ).sum()
        ),

    "duplicate_groups_with_positive_benchmark_crash_count":
        int(
            (
                dup4_groups["Benchmark Crash Count"] > 0
            ).sum()
        ),

    "duplicate_groups_with_zero_waymo_ro_miles":
        int(
            (
                dup4_groups["Waymo RO Miles"] == 0
            ).sum()
        ),

    "duplicate_groups_with_positive_waymo_ro_miles":
        int(
            (
                dup4_groups["Waymo RO Miles"] > 0
            ).sum()
        )
}


# ============================================================
# 10. STRUCTURAL INTERPRETATION — NO AUTHORIZATION
# ============================================================

structural_findings = {

    "CSV2_missing_ID_records":
        int(len(missing_id)),

    "CSV2_exact_duplicate_excess":
        int(df2.duplicated().sum()),

    "CSV4_exact_duplicate_excess":
        int(df4.duplicated().sum()),

    "CSV4_duplicate_groups":
        int(len(dup4_groups)),

    "CSV4_all_duplicate_groups_multiplicity_2":
        bool(
            set(
                dup4_groups["multiplicity"]
            ) == {2}
        ),

    "CSV4_duplicates_span_multiple_outcomes":
        bool(
            dup4_groups["Outcome"].nunique() > 1
        ),

    "CSV4_duplicates_span_multiple_cells":
        bool(
            dup4_groups["S2 Cell"].nunique() > 1
        )
}


# ============================================================
# 11. STATUS
# ============================================================

# The cell deliberately does NOT decide whether duplicates
# are errors or intentional source structure.
#
# Therefore the evidence gate remains pending.

status = "PENDING_EVIDENCE"


# ============================================================
# 12. ARTIFACT
# ============================================================

artifact = {

    "audit_version": AUDIT_VERSION,
    "cell": CELL_ID,
    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary": BOUNDARY,

    "source_hashes": {
        CSV2.name: CSV2_HASH,
        CSV4.name: CSV4_HASH
    },

    "missing_id_context":
        missing_id_context,

    "csv2_duplicate_context":
        dup2_context,

    "csv4_duplicate_multiplicity_distribution":
        multiplicity_distribution,

    "csv4_duplicate_by_outcome":
        dup4_by_outcome_records,

    "csv4_duplicate_by_county":
        dup4_by_county_records,

    "csv4_outcome_cell_summary":
        outcome_cell_summary,

    "duplicate_exposure_summary":
        duplicate_exposure_summary,

    "structural_findings":
        structural_findings,

    "rules": {

        "duplicate_equals_error":
            False,

        "duplicate_equals_valid_record":
            False,

        "missing_id_equals_invalid_record":
            False,

        "row_deletion_authorized":
            False,

        "deduplication_authorized":
            False,

        "missing_id_repair_authorized":
            False,

        "semantic_authorization":
            False,

        "benchmark_comparability_authorization":
            False,

        "claim_authorization":
            False,

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status": status
}


# ============================================================
# 13. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R2_DUPLICATE_STRUCTURE_MISSING_ID_CONTEXT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 14. CONSOLE
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R2 "
    "DUPLICATE STRUCTURE & MISSING-ID CONTEXT AUDIT"
)
print("=" * 72)

print()

print("-" * 72)
print("MISSING SGO ID CONTEXT")
print("-" * 72)

for row in missing_id_context:
    print(row)


print()

print("-" * 72)
print("CSV2 DUPLICATE CONTEXT")
print("-" * 72)

for row in dup2_context:
    print(row)


print()

print("-" * 72)
print("CSV4 DUPLICATE MULTIPLICITY")
print("-" * 72)

print(
    multiplicity_distribution
)


print()

print("-" * 72)
print("CSV4 DUPLICATES BY OUTCOME")
print("-" * 72)

for row in dup4_by_outcome_records:
    print(row)


print()

print("-" * 72)
print("CSV4 DUPLICATES BY COUNTY")
print("-" * 72)

for row in dup4_by_county_records:
    print(row)


print()

print("-" * 72)
print("CSV4 OUTCOME / CELL SUMMARY")
print("-" * 72)

for key, value in outcome_cell_summary.items():
    print(
        f"{key:50s} : {value}"
    )


print()

print("-" * 72)
print("DUPLICATE EXPOSURE SUMMARY")
print("-" * 72)

for key, value in duplicate_exposure_summary.items():
    print(
        f"{key:55s} : {value}"
    )


print()

print("-" * 72)
print("STRUCTURAL FINDINGS")
print("-" * 72)

for key, value in structural_findings.items():
    print(
        f"{key:55s} : {value}"
    )


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Duplicate = error                  :",
    False
)

print(
    "Duplicate = valid record           :",
    False
)

print(
    "Missing ID = invalid record        :",
    False
)

print(
    "Row deletion authorized            :",
    False
)

print(
    "Deduplication authorized           :",
    False
)

print(
    "Missing ID repair authorized       :",
    False
)

print(
    "Semantic authorization             :",
    False
)

print(
    "Benchmark comparability            :",
    False
)

print(
    "Claim authorization                :",
    False
)

print(
    "Scientific authorization           :",
    False
)

print(
    "Road safety verdict                :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C08R2 STATUS")
print("-" * 72)

print(
    "C08R2 STATUS :",
    status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R2 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R2 DUPLICATE STRUCTURE & MISSING-ID CONTEXT AUDIT

------------------------------------------------------------------------
MISSING SGO ID CONTEXT
------------------------------------------------------------------------
{'row_index': 1975, 'Incident Date': '05/28/2021', 'Year Month': '202105', 'State': 'Arizona', 'County': 'Maricopa', 'Road Type': 'Surface Street', 'Crash Type': 'V2V F2R', 'SGO Report Version': 'nan', 'SGO Amendment': 'Prior to SGO', 'Is Police-Reported': 'False', 'Is Any-Injury-Reported': 'False', 'Is Any Vehicle Airbag Deployment': 'False', 'Is Ego Vehicle Airbag Deployment': 'False', 'Is Suspected Serious Injury+': 'False', 'Is Any Fatal Injury': 'False'}
{'row_index': 1976, 'Incident Date': '9/15/2020', 'Year Month': '202009', 'State': 'Arizona', 'County': 'Maricopa', 'Road Type': 'Surface Street', 'Crash Type': 'V2V F2R', 'SGO Report Version': 'nan', 'SGO Amendment': 'Prior to SGO', 'Is Police-Reported': 'True', 'Is Any-Injury-Reported': 'False', 

In [75]:
# ============================================================
# OIP v1.0.39 — C08R3
# SOURCE DOCUMENTATION CROSS-CHECK FOR EXCEPTIONS
# ============================================================

import json
import hashlib
import re
from pathlib import Path
from datetime import datetime, UTC

from pypdf import PdfReader


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R3"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 1. FIND RELEASE NOTES
# ============================================================

pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

release_notes = next(
    (
        p for p in pdf_files
        if "Release_Notes" in p.name
    ),
    None
)

if release_notes is None:
    raise RuntimeError(
        "C08R3 FAIL_CLOSED: Release Notes PDF not found."
    )


# ============================================================
# 2. EXTRACT TEXT
# ============================================================

try:

    reader = PdfReader(str(release_notes))

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text() or ""

        pages.append({
            "page": page_number,
            "text": text
        })

except Exception as e:

    raise RuntimeError(
        "C08R3 FAIL_CLOSED: Release Notes extraction failed: "
        + repr(e)
    )


full_text = "\n".join(
    x["text"]
    for x in pages
)


# ============================================================
# 3. SEARCH FUNCTION
# ============================================================

def normalize(text):

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def search_term(term, context_chars=600):

    lower_text = full_text.lower()
    lower_term = term.lower()

    results = []
    start = 0

    while True:

        idx = lower_text.find(
            lower_term,
            start
        )

        if idx == -1:
            break

        left = max(
            0,
            idx - context_chars
        )

        right = min(
            len(full_text),
            idx + len(term) + context_chars
        )

        results.append({
            "term": term,
            "context":
                normalize(
                    full_text[left:right]
                )
        })

        start = idx + len(lower_term)

    return results


# ============================================================
# 4. EXCEPTION-RELATED SEARCH TERMS
# ============================================================

search_terms = [

    # SGO / historical records
    "Prior to SGO",
    "prior to SGO",
    "SGO Report ID",
    "SGO Report Version",
    "SGO Amendment",
    "historical",
    "amendment",

    # duplicate / record structure
    "duplicate",
    "duplicates",
    "duplicate rows",
    "records",
    "row",

    # dynamic benchmark structure
    "dynamic benchmark",
    "S2 Cell",
    "geographic distribution",
    "geographic",
    "benchmark crash count",
    "HPMS",
    "Vehicle Miles Traveled",
    "Waymo RO Miles",

    # Georgia / location
    "Georgia",
    "DeKalb",
    "Fulton",

    # outcome structures found in CSV4
    "fatal",
    "airbag",
    "ego_airbag",
    "blincoe_any_injury",
    "observed_any_injury",
    "ka",

    # methodology / generation
    "methodology",
    "calculation",
    "calculated",
    "derived",
    "aggregation",
    "aggregate",
    "cell"
]


documentary_evidence = {}

for term in search_terms:

    matches = search_term(term)

    documentary_evidence[term] = {

        "occurrence":
            len(matches) > 0,

        "match_count":
            len(matches),

        "contexts":
            matches[:5]
    }


# ============================================================
# 5. TARGETED EVIDENCE GROUPS
# ============================================================

evidence_groups = {

    "pre_sgo_records": [
        documentary_evidence[x]
        for x in [
            "Prior to SGO",
            "prior to SGO",
            "historical",
            "SGO Amendment"
        ]
    ],

    "duplicate_structure": [
        documentary_evidence[x]
        for x in [
            "duplicate",
            "duplicates",
            "duplicate rows",
            "records",
            "row"
        ]
    ],

    "dynamic_benchmark": [
        documentary_evidence[x]
        for x in [
            "dynamic benchmark",
            "S2 Cell",
            "geographic distribution",
            "benchmark crash count",
            "HPMS",
            "Vehicle Miles Traveled",
            "Waymo RO Miles"
        ]
    ],

    "georgia_location": [
        documentary_evidence[x]
        for x in [
            "Georgia",
            "DeKalb",
            "Fulton"
        ]
    ],

    "csv4_outcome_terms": [
        documentary_evidence[x]
        for x in [
            "fatal",
            "airbag",
            "ego_airbag",
            "blincoe_any_injury",
            "observed_any_injury",
            "ka"
        ]
    ],

    "methodology_terms": [
        documentary_evidence[x]
        for x in [
            "methodology",
            "calculation",
            "calculated",
            "derived",
            "aggregation",
            "aggregate",
            "cell"
        ]
    ]
}


# ============================================================
# 6. EVIDENCE FLAGS
# ============================================================

def any_occurrence(group):

    for item in group:

        if item.get("occurrence"):

            return True

    return False


evidence_flags = {

    "pre_sgo_documented":
        any_occurrence(
            evidence_groups["pre_sgo_records"]
        ),

    "duplicate_handling_documented":
        any_occurrence(
            evidence_groups["duplicate_structure"]
        ),

    "dynamic_benchmark_documented":
        any_occurrence(
            evidence_groups["dynamic_benchmark"]
        ),

    "georgia_documented":
        any_occurrence(
            evidence_groups["georgia_location"]
        ),

    "csv4_outcome_terms_documented":
        any_occurrence(
            evidence_groups["csv4_outcome_terms"]
        ),

    "methodology_documented":
        any_occurrence(
            evidence_groups["methodology_terms"]
        )
}


# ============================================================
# 7. IMPORTANT OIP INTERPRETATION
# ============================================================

# Documentary occurrence is NOT treated as proof that the
# observed duplicate/missing-ID pattern is authorized.
#
# The purpose of this cell is only to determine whether the
# Release Notes contain evidence capable of explaining the
# exceptions.
#
# No rows are deleted.
# No IDs are generated.
# No semantic outcome meaning is authorized.


if (
    evidence_flags["pre_sgo_documented"]
    and evidence_flags["dynamic_benchmark_documented"]
):

    documentation_status = "DOCUMENTED_REVIEW_REQUIRED"

else:

    documentation_status = "PENDING_EVIDENCE"


# Duplicate handling requires explicit documentary evidence.
# Merely finding the word "duplicate" somewhere is not enough.

duplicate_specific_evidence = any(
    item.get("occurrence")
    and item.get("match_count", 0) > 0
    for item in evidence_groups["duplicate_structure"]
)


if duplicate_specific_evidence:

    duplicate_status = "DOCUMENTARY_REFERENCE_FOUND"

else:

    duplicate_status = "NO_DUPLICATE_HANDLING_REFERENCE_FOUND"


# ============================================================
# 8. SOURCE HASH
# ============================================================

release_notes_hash = hashlib.sha256(
    release_notes.read_bytes()
).hexdigest()


# ============================================================
# 9. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "release_notes":
        release_notes.name,

    "release_notes_sha256":
        release_notes_hash,

    "page_count":
        len(reader.pages),

    "documentary_evidence":
        documentary_evidence,

    "evidence_groups":
        evidence_groups,

    "evidence_flags":
        evidence_flags,

    "documentation_status":
        documentation_status,

    "duplicate_status":
        duplicate_status,

    "rules": {

        "documentary_occurrence_equals_authorization":
            False,

        "duplicate_equals_error":
            False,

        "duplicate_equals_valid_record":
            False,

        "missing_id_equals_invalid_record":
            False,

        "row_deletion_authorized":
            False,

        "deduplication_authorized":
            False,

        "missing_id_repair_authorized":
            False,

        "semantic_authorization":
            False,

        "benchmark_comparability_authorization":
            False,

        "claim_authorization":
            False,

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        "PENDING_EVIDENCE"
}


# ============================================================
# 10. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R3_SOURCE_DOCUMENTATION_EXCEPTION_CROSSCHECK.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 11. OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R3 "
    "SOURCE DOCUMENTATION EXCEPTION CROSS-CHECK"
)
print("=" * 72)

print()

print("-" * 72)
print("DOCUMENTARY EVIDENCE FLAGS")
print("-" * 72)

for key, value in evidence_flags.items():

    print(
        f"{key:45s} : {value}"
    )


print()

print("-" * 72)
print("EXCEPTION DOCUMENTATION STATUS")
print("-" * 72)

print(
    "Documentation status :",
    documentation_status
)

print(
    "Duplicate status     :",
    duplicate_status
)


print()

print("-" * 72)
print("TARGETED SEARCH COUNTS")
print("-" * 72)

for term, result in documentary_evidence.items():

    print(
        f"{term:35s} : "
        f"{result['match_count']}"
    )


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Document occurrence = authorization :",
    False
)

print(
    "Row deletion authorized             :",
    False
)

print(
    "Deduplication authorized            :",
    False
)

print(
    "Missing ID repair authorized        :",
    False
)

print(
    "Semantic authorization              :",
    False
)

print(
    "Benchmark comparability             :",
    False
)

print(
    "Claim authorization                 :",
    False
)

print(
    "Scientific authorization            :",
    False
)

print(
    "Road safety verdict                 :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C08R3 STATUS")
print("-" * 72)

print(
    "C08R3 STATUS :",
    "PENDING_EVIDENCE"
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R3 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R3 SOURCE DOCUMENTATION EXCEPTION CROSS-CHECK

------------------------------------------------------------------------
DOCUMENTARY EVIDENCE FLAGS
------------------------------------------------------------------------
pre_sgo_documented                            : True
duplicate_handling_documented                 : True
dynamic_benchmark_documented                  : True
georgia_documented                            : True
csv4_outcome_terms_documented                 : True
methodology_documented                        : True

------------------------------------------------------------------------
EXCEPTION DOCUMENTATION STATUS
------------------------------------------------------------------------
Documentation status : DOCUMENTED_REVIEW_REQUIRED
Duplicate status     : DOCUMENTARY_REFERENCE_FOUND

------------------------------------------------------------------------
TARGETED SEARCH COUNTS
---------------------------------------------------------------------

In [76]:
# ============================================================
# OIP v1.0.39 — C08R4
# DOCUMENTARY CONTEXT RESOLUTION
# ============================================================

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, UTC
from pypdf import PdfReader


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R4"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 1. FIND RELEASE NOTES
# ============================================================

pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

release_notes = next(
    (
        p for p in pdf_files
        if "Release_Notes" in p.name
    ),
    None
)

if release_notes is None:
    raise RuntimeError(
        "C08R4 FAIL_CLOSED: Release Notes PDF not found."
    )


# ============================================================
# 2. EXTRACT PAGE-BY-PAGE TEXT
# ============================================================

try:

    reader = PdfReader(str(release_notes))

    page_texts = []

    for page_no, page in enumerate(reader.pages, start=1):

        page_texts.append({
            "page": page_no,
            "text": page.extract_text() or ""
        })

except Exception as e:

    raise RuntimeError(
        "C08R4 FAIL_CLOSED: PDF extraction failed: "
        + repr(e)
    )


# ============================================================
# 3. CONTEXT SEARCH
# ============================================================

def normalize(text):

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def search_context(term, context_chars=900):

    results = []

    for item in page_texts:

        text = item["text"]
        lower = text.lower()
        target = term.lower()

        start = 0

        while True:

            idx = lower.find(
                target,
                start
            )

            if idx == -1:
                break

            left = max(
                0,
                idx - context_chars
            )

            right = min(
                len(text),
                idx + len(term) + context_chars
            )

            results.append({

                "page":
                    item["page"],

                "term":
                    term,

                "context":
                    normalize(
                        text[left:right]
                    )
            })

            start = idx + len(target)

    return results


# ============================================================
# 4. HIGH-VALUE TERMS
# ============================================================

terms = [

    # SGO / historical
    "SGO Report ID",
    "SGO Report Version",
    "SGO Amendment",
    "historical",
    "amendment",

    # benchmark methodology
    "dynamic benchmark",
    "geographic distribution",
    "S2 Cell",
    "HPMS",
    "Vehicle Miles Traveled",
    "Waymo RO Miles",

    # data handling / construction
    "derived",
    "calculated",
    "calculation",
    "aggregation",
    "aggregate",
    "records",
    "row",

    # outcome-specific structures
    "blincoe_any_injury",
    "observed_any_injury",
    "ka"
]


# ============================================================
# 5. COLLECT CONTEXT
# ============================================================

contexts = {}

for term in terms:

    matches = search_context(term)

    contexts[term] = {

        "count":
            len(matches),

        "matches":
            matches[:10]
    }


# ============================================================
# 6. TARGETED INTERPRETATION FLAGS
# ============================================================

flags = {

    "SGO_metadata_context_found":
        (
            contexts["SGO Report ID"]["count"] > 0
            or
            contexts["SGO Report Version"]["count"] > 0
            or
            contexts["SGO Amendment"]["count"] > 0
        ),

    "historical_context_found":
        (
            contexts["historical"]["count"] > 0
        ),

    "dynamic_benchmark_context_found":
        (
            contexts["dynamic benchmark"]["count"] > 0
        ),

    "geographic_benchmark_context_found":
        (
            contexts["geographic distribution"]["count"] > 0
            or
            contexts["S2 Cell"]["count"] > 0
        ),

    "exposure_context_found":
        (
            contexts["HPMS"]["count"] > 0
            or
            contexts["Vehicle Miles Traveled"]["count"] > 0
            or
            contexts["Waymo RO Miles"]["count"] > 0
        ),

    "record_handling_context_found":
        (
            contexts["records"]["count"] > 0
            or
            contexts["row"]["count"] > 0
        ),

    "derivation_context_found":
        (
            contexts["derived"]["count"] > 0
            or
            contexts["calculated"]["count"] > 0
            or
            contexts["calculation"]["count"] > 0
            or
            contexts["aggregation"]["count"] > 0
            or
            contexts["aggregate"]["count"] > 0
        ),

    "exact_duplicate_handling_rule_found":
        False,

    "exact_prior_sgo_rule_found":
        False,

    "explicit_missing_id_rule_found":
        False
}


# ============================================================
# 7. FAIL-CLOSED INTERPRETATION
# ============================================================

# Context presence does not authorize an interpretation.
#
# In particular:
#
# - SGO metadata ≠ explanation for missing IDs
# - historical ≠ permission to exclude records
# - dynamic benchmark ≠ explanation for duplicate rows
# - derived/aggregate ≠ proof of duplicate legitimacy
#
# Therefore the exception gate remains pending unless
# explicit documentary rules are found.


if (
    flags["exact_duplicate_handling_rule_found"]
    or flags["exact_prior_sgo_rule_found"]
    or flags["explicit_missing_id_rule_found"]
):

    status = "DOCUMENTARY_RULE_FOUND_REVIEW_REQUIRED"

else:

    status = "PENDING_EVIDENCE"


# ============================================================
# 8. HASH
# ============================================================

release_notes_hash = hashlib.sha256(
    release_notes.read_bytes()
).hexdigest()


# ============================================================
# 9. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "release_notes":
        release_notes.name,

    "release_notes_sha256":
        release_notes_hash,

    "page_count":
        len(reader.pages),

    "contexts":
        contexts,

    "flags":
        flags,

    "rules": {

        "context_occurrence_equals_authorization":
            False,

        "historical_equals_exclusion":
            False,

        "SGO_metadata_equals_missing_ID_explanation":
            False,

        "dynamic_benchmark_equals_duplicate_explanation":
            False,

        "derived_equals_valid_duplicate":
            False,

        "row_deletion_authorized":
            False,

        "deduplication_authorized":
            False,

        "missing_id_repair_authorized":
            False,

        "semantic_authorization":
            False,

        "benchmark_comparability_authorization":
            False,

        "claim_authorization":
            False,

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        status
}


# ============================================================
# 10. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R4_DOCUMENTARY_CONTEXT_RESOLUTION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 11. OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R4 "
    "DOCUMENTARY CONTEXT RESOLUTION"
)
print("=" * 72)

print()

print("-" * 72)
print("DOCUMENTARY FLAGS")
print("-" * 72)

for key, value in flags.items():

    print(
        f"{key:55s} : {value}"
    )


print()

print("-" * 72)
print("CONTEXT COUNTS")
print("-" * 72)

for term, result in contexts.items():

    print(
        f"{term:35s} : {result['count']}"
    )


print()

print("-" * 72)
print("KEY DOCUMENTARY CONTEXT")
print("-" * 72)

for term in [
    "SGO Report ID",
    "SGO Report Version",
    "SGO Amendment",
    "historical",
    "dynamic benchmark",
    "geographic distribution",
    "S2 Cell",
    "HPMS",
    "Vehicle Miles Traveled",
    "Waymo RO Miles",
    "derived",
    "calculated",
    "aggregation"
]:

    print()
    print(f"[{term}]")

    for match in contexts[term]["matches"][:2]:

        print(
            f"Page {match['page']}: "
            f"{match['context']}"
        )


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Exact duplicate-handling rule found :",
    flags["exact_duplicate_handling_rule_found"]
)

print(
    "Exact prior-SGO rule found          :",
    flags["exact_prior_sgo_rule_found"]
)

print(
    "Explicit missing-ID rule found      :",
    flags["explicit_missing_id_rule_found"]
)

print(
    "Row deletion authorized              :",
    False
)

print(
    "Deduplication authorized             :",
    False
)

print(
    "Missing ID repair authorized         :",
    False
)

print(
    "Semantic authorization               :",
    False
)

print(
    "Benchmark comparability              :",
    False
)

print(
    "Claim authorization                  :",
    False
)

print(
    "Scientific authorization             :",
    False
)

print(
    "Road safety verdict                  :",
    "NOT_ISSUED"
)

print()

print("-" * 72)
print("C08R4 STATUS")
print("-" * 72)

print(
    "C08R4 STATUS :",
    status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R4 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R4 DOCUMENTARY CONTEXT RESOLUTION

------------------------------------------------------------------------
DOCUMENTARY FLAGS
------------------------------------------------------------------------
SGO_metadata_context_found                              : True
historical_context_found                                : True
dynamic_benchmark_context_found                         : True
geographic_benchmark_context_found                      : True
exposure_context_found                                  : True
record_handling_context_found                           : True
derivation_context_found                                : True
exact_duplicate_handling_rule_found                     : False
exact_prior_sgo_rule_found                              : False
explicit_missing_id_rule_found                          : False

------------------------------------------------------------------------
CONTEXT COUNTS
--------------------------------------------------------------

In [77]:
# ============================================================
# OIP v1.0.39 — C08R5
# CSV4 EXACT-DUPLICATE STRUCTURE RESOLUTION
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R5"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",

    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 1. SOURCE
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

if len(csv_files) != 4:
    raise RuntimeError(
        f"C08R5 FAIL_CLOSED: Expected 4 CSV files, found {len(csv_files)}."
    )

CSV4 = next(
    (
        p for p in csv_files
        if "Miles and Benchmark Crashes for Dynamic Benchmark"
        in p.name
    ),
    None
)

if CSV4 is None:
    raise RuntimeError(
        "C08R5 FAIL_CLOSED: CSV4 not found."
    )


# ============================================================
# 2. READ
# ============================================================

try:
    df4 = pd.read_csv(CSV4)
except Exception as e:
    raise RuntimeError(
        "C08R5 FAIL_CLOSED: CSV4 read failed: " + repr(e)
    )


required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [
    c for c in required
    if c not in df4.columns
]

if missing:
    raise RuntimeError(
        "C08R5 FAIL_CLOSED: Missing columns: "
        + repr(missing)
    )


# ============================================================
# 3. EXACT DUPLICATE GROUPS
# ============================================================

duplicate_groups = (
    df4[
        df4.duplicated(
            keep=False
        )
    ]
    .value_counts(
        dropna=False
    )
    .reset_index(
        name="multiplicity"
    )
)

if len(duplicate_groups) == 0:

    status = "PASS_EXCEPTION_RESOLUTION"

    duplicate_records = []

else:

    duplicate_records = []


# ============================================================
# 4. CELL + OUTCOME STRUCTURE
# ============================================================

for _, row in duplicate_groups.iterrows():

    cell = row["S2 Cell"]
    outcome = row["Outcome"]

    subset = df4[
        (df4["S2 Cell"] == cell)
        &
        (df4["Outcome"] == outcome)
    ].copy()

    all_outcomes_for_cell = sorted(
        subset["Outcome"]
        .astype(str)
        .unique()
        .tolist()
    )

    same_cell = df4[
        df4["S2 Cell"] == cell
    ]

    cell_outcomes = sorted(
        same_cell["Outcome"]
        .astype(str)
        .unique()
        .tolist()
    )

    duplicate_records.append({

        "state_name":
            str(row["state_name"]),

        "county_name":
            str(row["county_name"]),

        "S2_Cell":
            str(cell),

        "Outcome":
            str(outcome),

        "multiplicity":
            int(row["multiplicity"]),

        "same_cell_same_outcome_rows":
            int(len(subset)),

        "same_cell_unique_outcomes":
            int(len(cell_outcomes)),

        "same_cell_outcomes":
            cell_outcomes,

        "benchmark_crash_count":
            float(row["Benchmark Crash Count"]),

        "HPMS_VMT":
            float(
                row["HPMS Yearly Vehicle Miles Traveled"]
            ),

        "Waymo_RO_Miles":
            float(
                row["Waymo RO Miles"]
            )
    })


# ============================================================
# 5. EXPECTED OUTCOME-LEVEL REPLICATION CHECK
# ============================================================

outcome_counts_by_cell = (
    df4
    .groupby(
        "S2 Cell",
        dropna=False
    )["Outcome"]
    .nunique()
)

outcome_level_distribution = (
    outcome_counts_by_cell
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)


# ============================================================
# 6. EXACT DUPLICATE IMPACT BY COUNTY
# ============================================================

duplicate_df = pd.DataFrame(
    duplicate_records
)

if len(duplicate_df) > 0:

    county_summary = (
        duplicate_df
        .groupby(
            ["state_name", "county_name"],
            dropna=False
        )
        .agg(
            duplicate_groups=(
                "S2_Cell",
                "size"
            ),
            unique_cells=(
                "S2_Cell",
                "nunique"
            ),
            unique_outcomes=(
                "Outcome",
                "nunique"
            )
        )
        .reset_index()
    )

else:

    county_summary = pd.DataFrame(
        columns=[
            "state_name",
            "county_name",
            "duplicate_groups",
            "unique_cells",
            "unique_outcomes"
        ]
    )


# ============================================================
# 7. SAME-CELL STRUCTURE
# ============================================================

same_cell_outcome_counts = []

for cell in sorted(
    duplicate_df["S2_Cell"].unique()
    if len(duplicate_df) > 0
    else []
):

    cell_rows = df4[
        df4["S2 Cell"] == cell
    ]

    outcome_counts = (
        cell_rows["Outcome"]
        .value_counts()
        .to_dict()
    )

    same_cell_outcome_counts.append({

        "S2_Cell":
            str(cell),

        "total_rows":
            int(len(cell_rows)),

        "outcome_counts":
            {
                str(k): int(v)
                for k, v in outcome_counts.items()
            }
    })


# ============================================================
# 8. CLASSIFY STRUCTURAL OBSERVATION
# ============================================================

if len(duplicate_records) == 0:

    structural_observation = (
        "NO_EXACT_DUPLICATE_GROUPS_OBSERVED"
    )

else:

    all_multiplicity_two = all(
        x["multiplicity"] == 2
        for x in duplicate_records
    )

    duplicate_cells_multiple_outcomes = all(
        x["same_cell_unique_outcomes"] > 1
        for x in duplicate_records
    )

    if (
        all_multiplicity_two
        and duplicate_cells_multiple_outcomes
    ):

        structural_observation = (
            "EXACT_DUPLICATES_EXIST_WITHIN_MULTI_OUTCOME_CELLS"
        )

    else:

        structural_observation = (
            "EXACT_DUPLICATE_PATTERN_REQUIRES_REVIEW"
        )


# ============================================================
# 9. FAIL-CLOSED DECISION
# ============================================================

# This cell does not authorize deletion.
# Even if the pattern is structurally consistent with a
# multi-outcome CSV4 design, exact duplicate rows remain
# unresolved unless an explicit source rule authorizes them.

if structural_observation == "NO_EXACT_DUPLICATE_GROUPS_OBSERVED":

    status = "PASS_EXCEPTION_RESOLUTION"

else:

    status = "PENDING_EVIDENCE"


# ============================================================
# 10. HASH
# ============================================================

csv4_hash = hashlib.sha256(
    CSV4.read_bytes()
).hexdigest()


# ============================================================
# 11. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "boundary":
        BOUNDARY,

    "source":
        CSV4.name,

    "source_sha256":
        csv4_hash,

    "total_rows":
        int(len(df4)),

    "exact_duplicate_groups":
        int(len(duplicate_records)),

    "exact_duplicate_excess_rows":
        int(
            df4.duplicated().sum()
        ),

    "duplicate_records":
        duplicate_records,

    "outcome_level_distribution":
        outcome_level_distribution,

    "county_summary":
        county_summary.to_dict(
            orient="records"
        ),

    "same_cell_outcome_counts":
        same_cell_outcome_counts,

    "structural_observation":
        structural_observation,

    "rules": {

        "replicated_VMT_equals_duplicate_error":
            False,

        "replicated_RO_miles_equals_duplicate_error":
            False,

        "exact_duplicate_equals_valid_record":
            False,

        "exact_duplicate_equals_invalid_record":
            False,

        "row_deletion_authorized":
            False,

        "deduplication_authorized":
            False,

        "missing_id_repair_authorized":
            False,

        "semantic_authorization":
            False,

        "benchmark_comparability_authorization":
            False,

        "claim_authorization":
            False,

        "scientific_authorization":
            False,

        "road_safety_verdict":
            "NOT_ISSUED"
    },

    "status":
        status
}


# ============================================================
# 12. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R5_CSV4_EXACT_DUPLICATE_STRUCTURE.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 13. OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R5 "
    "CSV4 EXACT-DUPLICATE STRUCTURE RESOLUTION"
)
print("=" * 72)

print()

print("-" * 72)
print("BASIC STRUCTURE")
print("-" * 72)

print(
    "Total CSV4 rows                    :",
    len(df4)
)

print(
    "Exact duplicate groups             :",
    len(duplicate_records)
)

print(
    "Exact duplicate excess rows        :",
    int(df4.duplicated().sum())
)


print()

print("-" * 72)
print("OUTCOME-LEVEL DISTRIBUTION PER S2 CELL")
print("-" * 72)

print(
    outcome_level_distribution
)


print()

print("-" * 72)
print("STRUCTURAL OBSERVATION")
print("-" * 72)

print(
    structural_observation
)


print()

print("-" * 72)
print("DUPLICATE GROUP PREVIEW")
print("-" * 72)

for row in duplicate_records[:25]:
    print(row)


print()

print("-" * 72)
print("COUNTY SUMMARY")
print("-" * 72)

for row in county_summary.to_dict(
    orient="records"
):
    print(row)


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Replicated VMT = duplicate error :",
    False
)

print(
    "Replicated RO Miles = error     :",
    False
)

print(
    "Exact duplicate = valid         :",
    False
)

print(
    "Exact duplicate = invalid      :",
    False
)

print(
    "Row deletion authorized         :",
    False
)

print(
    "Deduplication authorized        :",
    False
)

print(
    "Missing ID repair authorized    :",
    False
)

print(
    "Semantic authorization          :",
    False
)

print(
    "Benchmark comparability         :",
    False
)

print(
    "Claim authorization             :",
    False
)

print(
    "Scientific authorization        :",
    False
)

print(
    "Road safety verdict             :",
    "NOT_ISSUED"
)


print()

print("-" * 72)
print("C08R5 STATUS")
print("-" * 72)

print(
    "C08R5 STATUS :",
    status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R5 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R5 CSV4 EXACT-DUPLICATE STRUCTURE RESOLUTION

------------------------------------------------------------------------
BASIC STRUCTURE
------------------------------------------------------------------------
Total CSV4 rows                    : 18352
Exact duplicate groups             : 79
Exact duplicate excess rows        : 79

------------------------------------------------------------------------
OUTCOME-LEVEL DISTRIBUTION PER S2 CELL
------------------------------------------------------------------------
{8: 2184}

------------------------------------------------------------------------
STRUCTURAL OBSERVATION
------------------------------------------------------------------------
EXACT_DUPLICATES_EXIST_WITHIN_MULTI_OUTCOME_CELLS

------------------------------------------------------------------------
DUPLICATE GROUP PREVIEW
------------------------------------------------------------------------
{'state_name': 'Georgia', 'county_name': 'DeKalb', 'S2_Cell': '98

In [78]:
# ============================================================
# OIP v1.0.39 — C08R6
# GEORGIA / ATLANTA DUPLICATE DOCUMENTARY COMPARISON
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R6"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",

    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 1. LOCATE CSV4 + RELEASE NOTES
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))
pdf_files = sorted(INPUT_ROOT.rglob("*.pdf"))

CSV4 = next(
    (
        p for p in csv_files
        if "Miles and Benchmark Crashes for Dynamic Benchmark"
        in p.name
    ),
    None
)

RELEASE = next(
    (
        p for p in pdf_files
        if "Release_Notes" in p.name
        or "Release Notes" in p.name
    ),
    None
)

if CSV4 is None:
    raise RuntimeError(
        "C08R6 FAIL_CLOSED: CSV4 not found."
    )

if RELEASE is None:
    raise RuntimeError(
        "C08R6 FAIL_CLOSED: Release Notes PDF not found."
    )


# ============================================================
# 2. READ CSV4
# ============================================================

try:
    df = pd.read_csv(CSV4)
except Exception as e:
    raise RuntimeError(
        "C08R6 FAIL_CLOSED: CSV4 read failed: " + repr(e)
    )


required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        "C08R6 FAIL_CLOSED: Missing CSV4 columns: "
        + repr(missing)
    )


# ============================================================
# 3. IDENTIFY EXACT DUPLICATES
# ============================================================

dup_mask = df.duplicated(
    keep=False
)

dup_df = df.loc[
    dup_mask
].copy()

duplicate_excess = int(
    df.duplicated().sum()
)

duplicate_groups = (
    df[
        dup_mask
    ]
    .value_counts(
        dropna=False
    )
    .reset_index(
        name="multiplicity"
    )
)


# ============================================================
# 4. GEORGIA / ATLANTA SCOPE
# ============================================================

georgia = df[
    df["state_name"]
    .astype(str)
    .str.lower()
    .eq("georgia")
].copy()

georgia_dup = dup_df[
    dup_df["state_name"]
    .astype(str)
    .str.lower()
    .eq("georgia")
].copy()


# ============================================================
# 5. DUPLICATE SUMMARY
# ============================================================

duplicate_outcomes = sorted(
    georgia_dup["Outcome"]
    .astype(str)
    .unique()
    .tolist()
)

duplicate_counties = sorted(
    georgia_dup["county_name"]
    .astype(str)
    .unique()
    .tolist()
)

duplicate_cells = sorted(
    georgia_dup["S2 Cell"]
    .astype(str)
    .unique()
    .tolist()
)


# ============================================================
# 6. GEORGIA CELL STRUCTURE
# ============================================================

georgia_cell_summary = (
    georgia
    .groupby(
        ["county_name", "S2 Cell"],
        dropna=False
    )
    .agg(
        row_count=("Outcome", "size"),
        unique_outcomes=("Outcome", "nunique"),
        outcomes=("Outcome", lambda x:
            sorted(
                x.astype(str).unique().tolist()
            )
        )
    )
    .reset_index()
)


# ============================================================
# 7. DUPLICATE CELLS: ROW COUNTS
# ============================================================

duplicate_cell_summary = (
    georgia_cell_summary[
        georgia_cell_summary["S2 Cell"]
        .astype(str)
        .isin(duplicate_cells)
    ]
    .copy()
)

duplicate_cell_summary["has_exact_duplicate"] = (
    duplicate_cell_summary["row_count"] > 
    duplicate_cell_summary["unique_outcomes"]
)


# ============================================================
# 8. OUTCOME DISTRIBUTION
# ============================================================

georgia_outcome_counts = (
    georgia["Outcome"]
    .value_counts()
    .sort_index()
    .to_dict()
)

georgia_duplicate_outcome_counts = (
    georgia_dup["Outcome"]
    .value_counts()
    .sort_index()
    .to_dict()
)


# ============================================================
# 9. RELEASE-NOTES TEXT EXTRACTION
# ============================================================

try:
    from pypdf import PdfReader
except Exception as e:
    raise RuntimeError(
        "C08R6 FAIL_CLOSED: pypdf unavailable: " + repr(e)
    )

reader = PdfReader(
    str(RELEASE)
)

pages = []

for i, page in enumerate(reader.pages, start=1):

    try:
        text = page.extract_text() or ""
    except Exception:
        text = ""

    pages.append({
        "page": i,
        "text": text
    })


# ============================================================
# 10. TARGETED DOCUMENTARY TERMS
# ============================================================

terms = [
    "Atlanta",
    "Georgia",
    "Fulton",
    "DeKalb",
    "dynamic benchmark",
    "S2 Cell",
    "replicated across Outcome levels",
    "duplicat",
    "duplicate",
    "dedup",
    "deduplicate",
    "duplicate rows",
    "duplicate records",
    "repeated",
    "same row",
    "identical",
    "aggregation",
    "combined aggregation",
    "June 24, 2026",
    "September 24, 2026"
]


context_hits = []

for item in pages:

    page_no = item["page"]
    text = item["text"]

    lower = text.lower()

    for term in terms:

        if term.lower() in lower:

            context_hits.append({
                "page": page_no,
                "term": term,
                "context": text
            })


# ============================================================
# 11. EXPLICIT RULE SEARCH
# ============================================================

rule_terms = [
    "duplicate rows",
    "duplicate records",
    "deduplicate",
    "deduplication",
    "remove duplicate",
    "duplicates are",
    "duplicate entries",
    "identical rows",
    "repeated rows"
]

explicit_duplicate_rule_hits = []

for item in pages:

    lower = item["text"].lower()

    for term in rule_terms:

        if term in lower:

            explicit_duplicate_rule_hits.append({
                "page": item["page"],
                "term": term,
                "context": item["text"]
            })


# ============================================================
# 12. EXPLICIT GEORGIA DOCUMENTATION
# ============================================================

georgia_documentation = []

for item in pages:

    lower = item["text"].lower()

    if (
        "georgia" in lower
        or "atlanta" in lower
        or "fulton" in lower
        or "dekalb" in lower
    ):

        georgia_documentation.append({
            "page": item["page"],
            "context": item["text"]
        })


# ============================================================
# 13. DOCUMENTARY FINDINGS
# ============================================================

exact_duplicate_rule_found = (
    len(explicit_duplicate_rule_hits) > 0
)

georgia_context_found = (
    len(georgia_documentation) > 0
)

s2_replication_documented = any(
    "replicated across outcome levels"
    in x["context"].lower()
    for x in context_hits
)

dynamic_benchmark_documented = any(
    "dynamic benchmark"
    in x["context"].lower()
    for x in context_hits
)


# ============================================================
# 14. CRITICAL DISTINCTION
# ============================================================

# Even if Georgia/Atlanta and S2 replication are documented,
# that does NOT authorize treating exact duplicate rows as
# valid or invalid.

if exact_duplicate_rule_found:

    exception_resolution = (
        "DOCUMENTARY_DUPLICATE_RULE_FOUND_REVIEW_REQUIRED"
    )

else:

    exception_resolution = (
        "NO_EXPLICIT_DUPLICATE_HANDLING_RULE_FOUND"
    )


# ============================================================
# 15. FAIL-CLOSED STATUS
# ============================================================

if exact_duplicate_rule_found:

    status = "PENDING_EVIDENCE"

else:

    status = "PENDING_EVIDENCE"


# ============================================================
# 16. HASHES
# ============================================================

csv4_hash = hashlib.sha256(
    CSV4.read_bytes()
).hexdigest()

release_hash = hashlib.sha256(
    RELEASE.read_bytes()
).hexdigest()


# ============================================================
# 17. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "source_files": {

        "csv4":
            CSV4.name,

        "csv4_sha256":
            csv4_hash,

        "release_notes":
            RELEASE.name,

        "release_notes_sha256":
            release_hash
    },

    "duplicate_structure": {

        "total_csv4_rows":
            int(len(df)),

        "exact_duplicate_groups":
            int(len(duplicate_groups)),

        "exact_duplicate_excess_rows":
            duplicate_excess,

        "georgia_rows":
            int(len(georgia)),

        "georgia_duplicate_rows":
            int(len(georgia_dup)),

        "duplicate_counties":
            duplicate_counties,

        "duplicate_outcomes":
            duplicate_outcomes,

        "duplicate_unique_s2_cells":
            int(len(duplicate_cells))
    },

    "georgia_outcome_counts":
        {
            str(k): int(v)
            for k, v in georgia_outcome_counts.items()
        },

    "georgia_duplicate_outcome_counts":
        {
            str(k): int(v)
            for k, v in
            georgia_duplicate_outcome_counts.items()
        },

    "georgia_duplicate_cell_summary":
        duplicate_cell_summary.to_dict(
            orient="records"
        ),

    "documentary_flags": {

        "georgia_context_found":
            georgia_context_found,

        "s2_replication_documented":
            s2_replication_documented,

        "dynamic_benchmark_documented":
            dynamic_benchmark_documented,

        "exact_duplicate_handling_rule_found":
            exact_duplicate_rule_found
    },

    "explicit_duplicate_rule_hits":
        explicit_duplicate_rule_hits,

    "georgia_documentation_pages":
        [
            x["page"]
            for x in georgia_documentation
        ],

    "exception_resolution":
        exception_resolution,

    "boundary":
        BOUNDARY,

    "status":
        status
}


# ============================================================
# 18. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R6_GEORGIA_ATLANTA_DUPLICATE_DOCUMENTARY_COMPARISON.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 19. OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R6 "
    "GEORGIA / ATLANTA DUPLICATE DOCUMENTARY COMPARISON"
)
print("=" * 72)

print()

print("-" * 72)
print("DUPLICATE STRUCTURE")
print("-" * 72)

print(
    "Total CSV4 rows              :",
    len(df)
)

print(
    "Exact duplicate groups       :",
    len(duplicate_groups)
)

print(
    "Exact duplicate excess rows  :",
    duplicate_excess
)

print(
    "Georgia rows                 :",
    len(georgia)
)

print(
    "Georgia duplicate rows       :",
    len(georgia_dup)
)

print(
    "Duplicate counties           :",
    duplicate_counties
)

print(
    "Duplicate outcomes           :",
    duplicate_outcomes
)

print(
    "Duplicate unique S2 cells    :",
    len(duplicate_cells)
)


print()

print("-" * 72)
print("DOCUMENTARY FLAGS")
print("-" * 72)

print(
    "Georgia context found                 :",
    georgia_context_found
)

print(
    "S2 replication documented             :",
    s2_replication_documented
)

print(
    "Dynamic benchmark documented         :",
    dynamic_benchmark_documented
)

print(
    "Exact duplicate handling rule found  :",
    exact_duplicate_rule_found
)


print()

print("-" * 72)
print("EXPLICIT DUPLICATE RULE HITS")
print("-" * 72)

if explicit_duplicate_rule_hits:

    for x in explicit_duplicate_rule_hits:
        print(
            "Page:",
            x["page"],
            "| Term:",
            x["term"]
        )
        print(
            x["context"][:1500]
        )
        print("-" * 40)

else:

    print("NONE")


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Row deletion authorized         :",
    False
)

print(
    "Deduplication authorized        :",
    False
)

print(
    "Missing ID repair authorized    :",
    False
)

print(
    "Semantic authorization          :",
    False
)

print(
    "Benchmark comparability         :",
    False
)

print(
    "Claim authorization             :",
    False
)

print(
    "Scientific authorization        :",
    False
)

print(
    "Road safety verdict             :",
    "NOT_ISSUED"
)


print()

print("-" * 72)
print("C08R6 STATUS")
print("-" * 72)

print(
    "C08R6 STATUS :",
    status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R6 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R6 GEORGIA / ATLANTA DUPLICATE DOCUMENTARY COMPARISON

------------------------------------------------------------------------
DUPLICATE STRUCTURE
------------------------------------------------------------------------
Total CSV4 rows              : 18352
Exact duplicate groups       : 79
Exact duplicate excess rows  : 79
Georgia rows                 : 2472
Georgia duplicate rows       : 158
Duplicate counties           : ['DeKalb', 'Fulton']
Duplicate outcomes           : ['airbag', 'blincoe_any_injury', 'ego_airbag', 'fatal', 'ka', 'observed_any_injury']
Duplicate unique S2 cells    : 59

------------------------------------------------------------------------
DOCUMENTARY FLAGS
------------------------------------------------------------------------
Georgia context found                 : True
S2 replication documented             : False
Dynamic benchmark documented         : True
Exact duplicate handling rule found  : False

--------------------------------------

In [79]:
# ============================================================
# OIP v1.0.39 — C08R7
# CSV4 GRAIN / UNIQUENESS AUTHORIZATION TEST
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R7"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R7")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# BOUNDARY
# ============================================================

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",

    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",

    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,

    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,

    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 1. LOCATE CSV4
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

CSV4 = next(
    (
        p for p in csv_files
        if "Miles and Benchmark Crashes for Dynamic Benchmark"
        in p.name
    ),
    None
)

if CSV4 is None:
    raise RuntimeError(
        "C08R7 FAIL_CLOSED: CSV4 not found."
    )


# ============================================================
# 2. READ
# ============================================================

try:
    df = pd.read_csv(CSV4)
except Exception as e:
    raise RuntimeError(
        "C08R7 FAIL_CLOSED: CSV4 read failed: " + repr(e)
    )


required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        "C08R7 FAIL_CLOSED: Missing columns: "
        + repr(missing)
    )


# ============================================================
# 3. DATA DICTIONARY-BASED DOCUMENTARY EXPECTATION
# ============================================================

# Release Notes data dictionary states that CSV4 contains:
# - S2 Cell
# - Outcome
# - Benchmark Crash Count
# - HPMS Yearly Vehicle Miles Traveled
# - Waymo RO Miles
#
# It also states that HPMS VMT and Waymo RO Miles are
# replicated across Outcome levels, meaning those fields
# are not expected to uniquely identify a row.
#
# This cell therefore tests the structural grain:
#
#       S2 Cell × Outcome
#
# WITHOUT authorizing any correction.

grain_cols = [
    "S2 Cell",
    "Outcome"
]


# ============================================================
# 4. GRAIN COUNTS
# ============================================================

grain_counts = (
    df
    .groupby(
        grain_cols,
        dropna=False
    )
    .size()
    .reset_index(
        name="row_count"
    )
)

duplicate_grain = grain_counts[
    grain_counts["row_count"] > 1
].copy()

unique_grain = grain_counts[
    grain_counts["row_count"] == 1
].copy()


# ============================================================
# 5. GRAIN SUMMARY
# ============================================================

total_grain_combinations = int(
    len(grain_counts)
)

unique_grain_combinations = int(
    len(unique_grain)
)

duplicate_grain_combinations = int(
    len(duplicate_grain)
)

expected_excess_rows = int(
    duplicate_grain["row_count"].sub(1).sum()
)


# ============================================================
# 6. MULTIPLICITY DISTRIBUTION
# ============================================================

multiplicity_distribution = (
    grain_counts["row_count"]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)


# ============================================================
# 7. DUPLICATE GRAIN DETAILS
# ============================================================

duplicate_details = []

for _, row in duplicate_grain.iterrows():

    cell = row["S2 Cell"]
    outcome = row["Outcome"]

    subset = df[
        (df["S2 Cell"] == cell)
        &
        (df["Outcome"] == outcome)
    ].copy()

    duplicate_details.append({

        "S2_Cell":
            str(cell),

        "Outcome":
            str(outcome),

        "row_count":
            int(len(subset)),

        "unique_benchmark_crash_count":
            int(
                subset["Benchmark Crash Count"]
                .nunique(dropna=False)
            ),

        "unique_HPMS_VMT":
            int(
                subset[
                    "HPMS Yearly Vehicle Miles Traveled"
                ].nunique(dropna=False)
            ),

        "unique_Waymo_RO_Miles":
            int(
                subset[
                    "Waymo RO Miles"
                ].nunique(dropna=False)
            )
    })


# ============================================================
# 8. EXACT-DUPLICATE RELATIONSHIP
# ============================================================

exact_duplicate_count = int(
    df.duplicated().sum()
)

exact_duplicate_groups = int(
    len(
        df[
            df.duplicated(keep=False)
        ]
        .value_counts(dropna=False)
    )
)


# ============================================================
# 9. CHECK WHETHER ALL GRAIN DUPLICATES ARE
#    EXACT ROW DUPLICATES
# ============================================================

grain_duplicate_rows = []

for _, row in duplicate_grain.iterrows():

    subset = df[
        (df["S2 Cell"] == row["S2 Cell"])
        &
        (df["Outcome"] == row["Outcome"])
    ]

    unique_full_rows = int(
        len(
            subset.drop_duplicates()
        )
    )

    grain_duplicate_rows.append({

        "S2_Cell":
            str(row["S2 Cell"]),

        "Outcome":
            str(row["Outcome"]),

        "row_count":
            int(len(subset)),

        "unique_full_rows":
            unique_full_rows,

        "all_rows_exactly_identical":
            unique_full_rows == 1
    })


all_grain_duplicates_exact = (
    len(grain_duplicate_rows) > 0
    and
    all(
        x["all_rows_exactly_identical"]
        for x in grain_duplicate_rows
    )
)


# ============================================================
# 10. STRUCTURAL INTERPRETATION
# ============================================================

if duplicate_grain_combinations == 0:

    structural_result = (
        "ONE_ROW_PER_S2_CELL_OUTCOME_OBSERVED"
    )

elif all_grain_duplicates_exact:

    structural_result = (
        "S2_CELL_OUTCOME_GRAIN_HAS_EXACT_REPETITIONS"
    )

else:

    structural_result = (
        "S2_CELL_OUTCOME_GRAIN_HAS_NONIDENTICAL_REPETITIONS"
    )


# ============================================================
# 11. DOCUMENTARY AUTHORIZATION TEST
# ============================================================

# Even if the data dictionary establishes the intended
# conceptual fields, it does NOT itself authorize deletion
# or deduplication of repeated rows.

grain_definition_documented = True

duplicate_resolution_rule_documented = False

row_deletion_authorized = False
deduplication_authorized = False


# ============================================================
# 12. STATUS
# ============================================================

if (
    duplicate_grain_combinations == 0
    and grain_definition_documented
):

    status = "PASS_EXCEPTION_RESOLUTION"

else:

    status = "PENDING_EVIDENCE"


# ============================================================
# 13. HASH
# ============================================================

csv4_hash = hashlib.sha256(
    CSV4.read_bytes()
).hexdigest()


# ============================================================
# 14. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "source":
        CSV4.name,

    "source_sha256":
        csv4_hash,

    "grain": {

        "grain_columns":
            grain_cols,

        "total_rows":
            int(len(df)),

        "total_grain_combinations":
            total_grain_combinations,

        "unique_grain_combinations":
            unique_grain_combinations,

        "duplicate_grain_combinations":
            duplicate_grain_combinations,

        "expected_excess_rows":
            expected_excess_rows,

        "multiplicity_distribution":
            multiplicity_distribution
    },

    "duplicate_grain_details":
        duplicate_details,

    "exact_duplicate_structure": {

        "exact_duplicate_groups":
            exact_duplicate_groups,

        "exact_duplicate_excess_rows":
            exact_duplicate_count,

        "all_grain_duplicates_exact":
            all_grain_duplicates_exact,

        "grain_duplicate_row_check":
            grain_duplicate_rows
    },

    "documentary_authorization": {

        "grain_definition_documented":
            grain_definition_documented,

        "duplicate_resolution_rule_documented":
            duplicate_resolution_rule_documented,

        "row_deletion_authorized":
            row_deletion_authorized,

        "deduplication_authorized":
            deduplication_authorized
    },

    "structural_result":
        structural_result,

    "boundary":
        BOUNDARY,

    "status":
        status
}


# ============================================================
# 15. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R7_CSV4_GRAIN_UNIQUENESS_AUTHORIZATION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 16. OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R7 "
    "CSV4 GRAIN / UNIQUENESS AUTHORIZATION TEST"
)
print("=" * 72)

print()

print("-" * 72)
print("DOCUMENTARY GRAIN")
print("-" * 72)

print(
    "Expected grain columns:",
    grain_cols
)

print(
    "Grain definition documented :",
    grain_definition_documented
)


print()

print("-" * 72)
print("GRAIN STRUCTURE")
print("-" * 72)

print(
    "Total rows                 :",
    len(df)
)

print(
    "Total S2 Cell × Outcome    :",
    total_grain_combinations
)

print(
    "Unique S2 Cell × Outcome   :",
    unique_grain_combinations
)

print(
    "Duplicate grain combinations:",
    duplicate_grain_combinations
)

print(
    "Expected excess rows       :",
    expected_excess_rows
)

print(
    "Multiplicity distribution  :",
    multiplicity_distribution
)


print()

print("-" * 72)
print("EXACT-DUPLICATE RELATIONSHIP")
print("-" * 72)

print(
    "Exact duplicate groups     :",
    exact_duplicate_groups
)

print(
    "Exact duplicate excess rows:",
    exact_duplicate_count
)

print(
    "All grain duplicates exact :",
    all_grain_duplicates_exact
)


print()

print("-" * 72)
print("STRUCTURAL RESULT")
print("-" * 72)

print(
    structural_result
)


print()

print("-" * 72)
print("DOCUMENTARY AUTHORIZATION")
print("-" * 72)

print(
    "Duplicate resolution rule documented :",
    duplicate_resolution_rule_documented
)

print(
    "Row deletion authorized              :",
    False
)

print(
    "Deduplication authorized             :",
    False
)

print(
    "Missing ID repair authorized         :",
    False
)

print(
    "Semantic authorization               :",
    False
)

print(
    "Benchmark comparability              :",
    False
)

print(
    "Claim authorization                  :",
    False
)

print(
    "Scientific authorization             :",
    False
)

print(
    "Road safety verdict                  :",
    "NOT_ISSUED"
)


print()

print("-" * 72)
print("C08R7 STATUS")
print("-" * 72)

print(
    "C08R7 STATUS :",
    status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R7 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R7 CSV4 GRAIN / UNIQUENESS AUTHORIZATION TEST

------------------------------------------------------------------------
DOCUMENTARY GRAIN
------------------------------------------------------------------------
Expected grain columns: ['S2 Cell', 'Outcome']
Grain definition documented : True

------------------------------------------------------------------------
GRAIN STRUCTURE
------------------------------------------------------------------------
Total rows                 : 18352
Total S2 Cell × Outcome    : 17472
Unique S2 Cell × Outcome   : 16648
Duplicate grain combinations: 824
Expected excess rows       : 880
Multiplicity distribution  : {1: 16648, 2: 792, 3: 8, 4: 24}

------------------------------------------------------------------------
EXACT-DUPLICATE RELATIONSHIP
------------------------------------------------------------------------
Exact duplicate groups     : 79
Exact duplicate excess rows: 79
All grain duplicates exact : False

------------------

In [80]:
# ============================================================
# OIP v1.0.39 — C08R8
# CSV4 REPEATED GRAIN COLUMN-DIFFERENCE AUDIT
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd


AUDIT_VERSION = "OIP v1.0.39"
CELL_ID = "C08R8"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/OIP_v1_0_39_C08R8")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY = {
    "audit_type": "Corporate Safety Claim Audit",
    "GFL": "NOT_APPLICABLE",
    "IDS": "NOT_APPLICABLE",
    "AML": "NOT_APPLICABLE",
    "semantic_authorization": False,
    "benchmark_comparability_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,
    "road_safety_verdict": "NOT_ISSUED"
}


# ============================================================
# 1. LOCATE CSV4
# ============================================================

csv_files = sorted(INPUT_ROOT.rglob("*.csv"))

CSV4 = next(
    (
        p for p in csv_files
        if "Miles and Benchmark Crashes for Dynamic Benchmark"
        in p.name
    ),
    None
)

if CSV4 is None:
    raise RuntimeError(
        "C08R8 FAIL_CLOSED: CSV4 not found."
    )


# ============================================================
# 2. READ CSV4
# ============================================================

try:
    df = pd.read_csv(CSV4)
except Exception as e:
    raise RuntimeError(
        "C08R8 FAIL_CLOSED: CSV4 read failed: " + repr(e)
    )


required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        "C08R8 FAIL_CLOSED: Missing columns: " + repr(missing)
    )


# ============================================================
# 3. REPEATED S2 CELL × OUTCOME GRAIN
# ============================================================

grain_counts = (
    df
    .groupby(
        ["S2 Cell", "Outcome"],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
)

repeated_grain = grain_counts[
    grain_counts["row_count"] > 1
].copy()

if len(repeated_grain) == 0:
    raise RuntimeError(
        "C08R8 FAIL_CLOSED: No repeated grain found; "
        "unexpected relative to C08R7."
    )


# ============================================================
# 4. COLUMN-DIFFERENCE ANALYSIS
# ============================================================

analysis_columns = [
    "state_name",
    "county_name",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

column_difference_counts = {
    c: 0 for c in analysis_columns
}

pattern_records = []

for _, g in repeated_grain.iterrows():

    cell = g["S2 Cell"]
    outcome = g["Outcome"]

    subset = df[
        (df["S2 Cell"] == cell)
        &
        (df["Outcome"] == outcome)
    ].copy()

    differing_columns = []

    for col in analysis_columns:

        n_unique = int(
            subset[col].nunique(
                dropna=False
            )
        )

        if n_unique > 1:

            differing_columns.append(col)
            column_difference_counts[col] += 1

    pattern_records.append({

        "S2_Cell": str(cell),

        "Outcome": str(outcome),

        "row_count": int(len(subset)),

        "differing_columns":
            differing_columns,

        "number_of_differing_columns":
            len(differing_columns),

        "state_values":
            sorted(
                subset["state_name"]
                .astype(str)
                .unique()
                .tolist()
            ),

        "county_values":
            sorted(
                subset["county_name"]
                .astype(str)
                .unique()
                .tolist()
            ),

        "benchmark_crash_values":
            sorted(
                subset["Benchmark Crash Count"]
                .astype(float)
                .unique()
                .tolist()
            ),

        "hpms_vmt_values":
            sorted(
                subset[
                    "HPMS Yearly Vehicle Miles Traveled"
                ]
                .astype(float)
                .unique()
                .tolist()
            ),

        "waymo_ro_miles_values":
            sorted(
                subset[
                    "Waymo RO Miles"
                ]
                .astype(float)
                .unique()
                .tolist()
            )
    })


# ============================================================
# 5. PATTERN DISTRIBUTION
# ============================================================

pattern_distribution = {}

for record in pattern_records:

    key = tuple(
        record["differing_columns"]
    )

    pattern_distribution[key] = (
        pattern_distribution.get(key, 0) + 1
    )


pattern_distribution_serialized = {
    " | ".join(k) if k else "NO_DIFFERENCE":
        int(v)
    for k, v in pattern_distribution.items()
}


# ============================================================
# 6. EXACT VS NON-EXACT REPEATED GRAIN
# ============================================================

exact_repeated = 0
non_exact_repeated = 0

for record in pattern_records:

    if (
        record["number_of_differing_columns"]
        == 0
    ):
        exact_repeated += 1
    else:
        non_exact_repeated += 1


# ============================================================
# 7. DIFFERENCE PREVIEW
# ============================================================

difference_preview = [
    x
    for x in pattern_records
    if x["number_of_differing_columns"] > 0
][:50]


# ============================================================
# 8. SPECIAL TEST:
#    DOES STATE/COUNTY DIFFER WITHIN SAME S2 CELL+OUTCOME?
# ============================================================

location_variation_groups = [
    x for x in pattern_records
    if (
        "state_name" in x["differing_columns"]
        or
        "county_name" in x["differing_columns"]
    )
]


# ============================================================
# 9. SPECIAL TEST:
#    DOES EXPOSURE DIFFER?
# ============================================================

exposure_variation_groups = [
    x for x in pattern_records
    if (
        "HPMS Yearly Vehicle Miles Traveled"
        in x["differing_columns"]
        or
        "Waymo RO Miles"
        in x["differing_columns"]
    )
]


# ============================================================
# 10. SPECIAL TEST:
#    DOES BENCHMARK COUNT DIFFER?
# ============================================================

benchmark_variation_groups = [
    x for x in pattern_records
    if "Benchmark Crash Count"
    in x["differing_columns"]
]


# ============================================================
# 11. STRUCTURAL INTERPRETATION
# ============================================================

if non_exact_repeated == 0:

    structural_result = (
        "ALL_REPEATED_GRAIN_GROUPS_ARE_EXACT_REPETITIONS"
    )

else:

    structural_result = (
        "REPEATED_GRAIN_CONTAINS_NONIDENTICAL_ROWS"
    )


# ============================================================
# 12. FAIL-CLOSED STATUS
# ============================================================

# This cell does not determine whether repeated rows are valid,
# invalid, duplicated, aggregated, or erroneous.
#
# It only identifies where the variation occurs.

status = "PENDING_EVIDENCE"


# ============================================================
# 13. HASH
# ============================================================

csv4_hash = hashlib.sha256(
    CSV4.read_bytes()
).hexdigest()


# ============================================================
# 14. ARTIFACT
# ============================================================

artifact = {

    "audit_version":
        AUDIT_VERSION,

    "cell":
        CELL_ID,

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "source":
        CSV4.name,

    "source_sha256":
        csv4_hash,

    "input_structure": {

        "total_rows":
            int(len(df)),

        "repeated_grain_groups":
            int(len(repeated_grain)),

        "exact_repeated_grain_groups":
            int(exact_repeated),

        "non_exact_repeated_grain_groups":
            int(non_exact_repeated)
    },

    "column_difference_counts":
        column_difference_counts,

    "pattern_distribution":
        pattern_distribution_serialized,

    "location_variation_groups":
        int(len(location_variation_groups)),

    "exposure_variation_groups":
        int(len(exposure_variation_groups)),

    "benchmark_variation_groups":
        int(len(benchmark_variation_groups)),

    "difference_preview":
        difference_preview,

    "structural_result":
        structural_result,

    "boundary":
        BOUNDARY,

    "status":
        status
}


# ============================================================
# 15. SAVE
# ============================================================

artifact_path = (
    OUTPUT_DIR /
    "C08R8_REPEATED_GRAIN_COLUMN_DIFFERENCE_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 16. OUTPUT
# ============================================================

print("=" * 72)
print(
    "OIP v1.0.39 — C08R8 "
    "CSV4 REPEATED GRAIN COLUMN-DIFFERENCE AUDIT"
)
print("=" * 72)

print()

print("-" * 72)
print("REPEATED GRAIN")
print("-" * 72)

print(
    "Total rows                         :",
    len(df)
)

print(
    "Repeated S2 Cell × Outcome groups :",
    len(repeated_grain)
)

print(
    "Exact repeated grain groups       :",
    exact_repeated
)

print(
    "Non-identical repeated groups     :",
    non_exact_repeated
)


print()

print("-" * 72)
print("COLUMN DIFFERENCE COUNTS")
print("-" * 72)

for col, count in column_difference_counts.items():

    print(
        f"{col:40s}: {count}"
    )


print()

print("-" * 72)
print("SPECIAL STRUCTURAL COUNTS")
print("-" * 72)

print(
    "Location variation groups         :",
    len(location_variation_groups)
)

print(
    "Exposure variation groups         :",
    len(exposure_variation_groups)
)

print(
    "Benchmark variation groups        :",
    len(benchmark_variation_groups)
)


print()

print("-" * 72)
print("PATTERN DISTRIBUTION")
print("-" * 72)

for pattern, count in pattern_distribution_serialized.items():

    print(
        f"{pattern:70s}: {count}"
    )


print()

print("-" * 72)
print("NON-IDENTICAL REPEATED-GRAIN PREVIEW")
print("-" * 72)

for record in difference_preview[:20]:

    print(record)


print()

print("-" * 72)
print("STRUCTURAL RESULT")
print("-" * 72)

print(
    structural_result
)


print()

print("-" * 72)
print("BOUNDARY")
print("-" * 72)

print(
    "Row deletion authorized         :",
    False
)

print(
    "Deduplication authorized        :",
    False
)

print(
    "Missing ID repair authorized    :",
    False
)

print(
    "Semantic authorization          :",
    False
)

print(
    "Benchmark comparability         :",
    False
)

print(
    "Claim authorization             :",
    False
)

print(
    "Scientific authorization        :",
    False
)

print(
    "Road safety verdict             :",
    "NOT_ISSUED"
)


print()

print("-" * 72)
print("C08R8 STATUS")
print("-" * 72)

print(
    "C08R8 STATUS :",
    status
)

print()

print("Artifact:")
print(artifact_path)

print()

print("=" * 72)
print("C08R8 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R8 CSV4 REPEATED GRAIN COLUMN-DIFFERENCE AUDIT

------------------------------------------------------------------------
REPEATED GRAIN
------------------------------------------------------------------------
Total rows                         : 18352
Repeated S2 Cell × Outcome groups : 824
Exact repeated grain groups       : 75
Non-identical repeated groups     : 749

------------------------------------------------------------------------
COLUMN DIFFERENCE COUNTS
------------------------------------------------------------------------
state_name                              : 0
county_name                             : 200
Benchmark Crash Count                   : 715
HPMS Yearly Vehicle Miles Traveled      : 200
Waymo RO Miles                          : 200

------------------------------------------------------------------------
SPECIAL STRUCTURAL COUNTS
------------------------------------------------------------------------
Location variation groups         : 200

In [81]:
# ================================================================
# OIP v1.0.39 — C08R9
# CSV4 GRAIN + S2/COUNTY + PRIOR-TO-SGO STRUCTURAL RESOLUTION
# ================================================================

import os
import json
import pandas as pd
from collections import Counter

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV4
# ------------------------------------------------
csv4_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".csv") and "Miles and Benchmark Crashes" in f:
            csv4_files.append(os.path.join(root, f))

if len(csv4_files) != 1:
    raise RuntimeError(
        f"C08R9 FAIL_CLOSED: expected exactly 1 CSV4, found {len(csv4_files)}"
    )

csv4_path = csv4_files[0]

# ------------------------------------------------
# 2. Locate CSV2
# ------------------------------------------------
csv2_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".csv") and "Crashes with SGO ID" in f:
            csv2_files.append(os.path.join(root, f))

if len(csv2_files) != 1:
    raise RuntimeError(
        f"C08R9 FAIL_CLOSED: expected exactly 1 CSV2, found {len(csv2_files)}"
    )

csv2_path = csv2_files[0]

# ------------------------------------------------
# 3. Read files
# ------------------------------------------------
df4 = pd.read_csv(csv4_path)
df2 = pd.read_csv(csv2_path)

required_csv4 = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles",
]

required_csv2 = [
    "SGO Report ID",
    "SGO Amendment",
]

missing4 = [c for c in required_csv4 if c not in df4.columns]
missing2 = [c for c in required_csv2 if c not in df2.columns]

if missing4 or missing2:
    raise RuntimeError(
        f"C08R9 FAIL_CLOSED: missing columns CSV4={missing4}, CSV2={missing2}"
    )

# ------------------------------------------------
# 4. Candidate grain diagnostics
# ------------------------------------------------
candidate_grains = {
    "S2_Outcome": ["S2 Cell", "Outcome"],
    "County_S2_Outcome": ["county_name", "S2 Cell", "Outcome"],
    "State_County_S2_Outcome": [
        "state_name", "county_name", "S2 Cell", "Outcome"
    ],
}

grain_results = {}

for name, cols in candidate_grains.items():

    grouped = df4.groupby(cols, dropna=False).size()

    grain_results[name] = {
        "unique_groups": int(grouped.size),
        "rows": int(df4.shape[0]),
        "duplicate_groups": int((grouped > 1).sum()),
        "excess_rows": int((grouped - 1).clip(lower=0).sum()),
        "max_multiplicity": int(grouped.max()),
        "multiplicity_distribution": {
            str(int(k)): int(v)
            for k, v in grouped.value_counts().sort_index().items()
        },
    }

# ------------------------------------------------
# 5. S2 Cell crossing counties
# ------------------------------------------------
s2_county = (
    df4.groupby("S2 Cell")["county_name"]
    .nunique(dropna=False)
)

cross_county_cells = s2_county[s2_county > 1]

# ------------------------------------------------
# 6. S2 Cell crossing states
# ------------------------------------------------
s2_state = (
    df4.groupby("S2 Cell")["state_name"]
    .nunique(dropna=False)
)

cross_state_cells = s2_state[s2_state > 1]

# ------------------------------------------------
# 7. County + S2 + Outcome uniqueness
# ------------------------------------------------
county_s2_outcome = (
    df4.groupby(
        ["county_name", "S2 Cell", "Outcome"],
        dropna=False
    ).size()
)

county_s2_outcome_duplicates = county_s2_outcome[
    county_s2_outcome > 1
]

# ------------------------------------------------
# 8. Prior-to-SGO systematicity
# ------------------------------------------------
prior_mask = (
    df2["SGO Amendment"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("prior to sgo")
)

prior_to_sgo_count = int(prior_mask.sum())

prior_to_sgo_missing_id = int(
    (
        prior_mask
        & df2["SGO Report ID"].isna()
    ).sum()
)

prior_to_sgo_with_id = int(
    (
        prior_mask
        & df2["SGO Report ID"].notna()
    ).sum()
)

non_prior_missing_id = int(
    (
        (~prior_mask)
        & df2["SGO Report ID"].isna()
    ).sum()
)

# ------------------------------------------------
# 9. Prior-to-SGO amendment values
# ------------------------------------------------
prior_amendment_values = (
    df2.loc[prior_mask, "SGO Amendment"]
    .astype("string")
    .value_counts(dropna=False)
    .to_dict()
)

# ------------------------------------------------
# 10. Preview S2 cells crossing counties
# ------------------------------------------------
cross_county_preview = []

for s2, n_counties in cross_county_cells.head(20).items():

    sub = df4[df4["S2 Cell"] == s2]

    cross_county_preview.append({
        "S2_Cell": str(s2),
        "state_values": sorted(
            sub["state_name"].dropna().astype(str).unique().tolist()
        ),
        "county_values": sorted(
            sub["county_name"].dropna().astype(str).unique().tolist()
        ),
        "county_count": int(
            sub["county_name"].nunique(dropna=False)
        ),
        "outcome_count": int(
            sub["Outcome"].nunique(dropna=False)
        ),
        "row_count": int(len(sub)),
    })

# ------------------------------------------------
# 11. Boundary
# ------------------------------------------------
boundary = {
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "missing_id_repair_authorized": False,
    "semantic_authorization": False,
    "benchmark_comparability": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

# ------------------------------------------------
# 12. Structural interpretation
# ------------------------------------------------
interpretation = {
    "S2_Outcome_is_unique": (
        grain_results["S2_Outcome"]["duplicate_groups"] == 0
    ),
    "County_S2_Outcome_is_unique": (
        grain_results["County_S2_Outcome"]["duplicate_groups"] == 0
    ),
    "State_County_S2_Outcome_is_unique": (
        grain_results["State_County_S2_Outcome"]["duplicate_groups"] == 0
    ),
    "S2_cells_cross_counties": int(len(cross_county_cells)),
    "S2_cells_cross_states": int(len(cross_state_cells)),
    "prior_to_sgo_total": prior_to_sgo_count,
    "prior_to_sgo_missing_id": prior_to_sgo_missing_id,
    "prior_to_sgo_with_id": prior_to_sgo_with_id,
    "non_prior_missing_id": non_prior_missing_id,
}

# ------------------------------------------------
# 13. Status
# ------------------------------------------------
status = "PENDING_EVIDENCE"

# ------------------------------------------------
# 14. Output
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R9 CSV4 GRAIN + PRIOR-SGO STRUCTURAL RESOLUTION")
print("=" * 72)

print("\nCANDIDATE GRAIN TEST")
for name, result in grain_results.items():
    print(f"\n{name}")
    print("  unique groups        :", result["unique_groups"])
    print("  duplicate groups     :", result["duplicate_groups"])
    print("  excess rows          :", result["excess_rows"])
    print("  max multiplicity     :", result["max_multiplicity"])
    print("  multiplicity         :", result["multiplicity_distribution"])

print("\nS2 CELL CROSS-COUNTY")
print("S2 cells crossing counties :", len(cross_county_cells))

print("\nS2 CELL CROSS-STATE")
print("S2 cells crossing states   :", len(cross_state_cells))

print("\nCOUNTY + S2 + OUTCOME")
print("Duplicate groups :", len(county_s2_outcome_duplicates))
print("Excess rows      :", int(
    (county_s2_outcome_duplicates - 1).clip(lower=0).sum()
))

print("\nPRIOR-TO-SGO SYSTEMATICITY")
print("Total Prior to SGO rows        :", prior_to_sgo_count)
print("Prior to SGO with missing ID   :", prior_to_sgo_missing_id)
print("Prior to SGO with ID           :", prior_to_sgo_with_id)
print("Non-Prior-to-SGO missing ID    :", non_prior_missing_id)

print("\nPRIOR-TO-SGO VALUES")
print(prior_amendment_values)

print("\nS2 CROSS-COUNTY PREVIEW")
for row in cross_county_preview:
    print(row)

print("\nSTRUCTURAL INTERPRETATION")
print(json.dumps(interpretation, indent=2))

print("\nBOUNDARY")
for k, v in boundary.items():
    print(f"{k:35s}: {v}")

print("\nC08R9 STATUS :", status)

# ------------------------------------------------
# 15. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R9"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R9",
    "purpose": "CSV4 grain and Prior-to-SGO structural resolution",
    "csv4_path": csv4_path,
    "csv2_path": csv2_path,
    "grain_results": grain_results,
    "s2_cells_cross_counties": int(len(cross_county_cells)),
    "s2_cells_cross_states": int(len(cross_state_cells)),
    "cross_county_preview": cross_county_preview,
    "prior_to_sgo_total": prior_to_sgo_count,
    "prior_to_sgo_missing_id": prior_to_sgo_missing_id,
    "prior_to_sgo_with_id": prior_to_sgo_with_id,
    "non_prior_missing_id": non_prior_missing_id,
    "prior_amendment_values": {
        str(k): int(v)
        for k, v in prior_amendment_values.items()
    },
    "interpretation": interpretation,
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R9_CSV4_GRAIN_PRIOR_SGO_STRUCTURAL_RESOLUTION.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R9 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R9 CSV4 GRAIN + PRIOR-SGO STRUCTURAL RESOLUTION

CANDIDATE GRAIN TEST

S2_Outcome
  unique groups        : 17472
  duplicate groups     : 824
  excess rows          : 880
  max multiplicity     : 4
  multiplicity         : {'1': 16648, '2': 792, '3': 8, '4': 24}

County_S2_Outcome
  unique groups        : 17672
  duplicate groups     : 680
  excess rows          : 680
  max multiplicity     : 2
  multiplicity         : {'1': 16992, '2': 680}

State_County_S2_Outcome
  unique groups        : 17672
  duplicate groups     : 680
  excess rows          : 680
  max multiplicity     : 2
  multiplicity         : {'1': 16992, '2': 680}

S2 CELL CROSS-COUNTY
S2 cells crossing counties : 25

S2 CELL CROSS-STATE
S2 cells crossing states   : 0

COUNTY + S2 + OUTCOME
Duplicate groups : 680
Excess rows      : 680

PRIOR-TO-SGO SYSTEMATICITY
Total Prior to SGO rows        : 2
Prior to SGO with missing ID   : 2
Prior to SGO with ID           : 0
Non-Prior-to-SGO missing ID    : 0

PRIO

In [82]:
# ================================================================
# OIP v1.0.39 — C08R10
# COUNTY + S2 + OUTCOME REPEATED-GRAIN DIFFERENCE AUDIT
# ================================================================

import os
import json
import pandas as pd

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV4
# ------------------------------------------------
csv4_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".csv") and "Miles and Benchmark Crashes" in f:
            csv4_files.append(os.path.join(root, f))

if len(csv4_files) != 1:
    raise RuntimeError(
        f"C08R10 FAIL_CLOSED: expected exactly 1 CSV4, found {len(csv4_files)}"
    )

csv4_path = csv4_files[0]

# ------------------------------------------------
# 2. Read CSV4
# ------------------------------------------------
df = pd.read_csv(csv4_path)

required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        f"C08R10 FAIL_CLOSED: missing columns: {missing}"
    )

# ------------------------------------------------
# 3. Define candidate structural grain
# ------------------------------------------------
grain = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
]

value_columns = [
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles",
]

# ------------------------------------------------
# 4. Identify repeated groups
# ------------------------------------------------
group_sizes = df.groupby(
    grain,
    dropna=False
).size()

repeated_keys = group_sizes[group_sizes > 1]

print("=" * 72)
print("OIP v1.0.39 — C08R10 COUNTY + S2 + OUTCOME DIFFERENCE AUDIT")
print("=" * 72)

print("\nREPEATED CANDIDATE-GRAIN GROUPS")
print("Repeated groups :", len(repeated_keys))
print("Excess rows     :", int((repeated_keys - 1).sum()))
print("Max multiplicity:", int(repeated_keys.max()))

# ------------------------------------------------
# 5. Difference analysis
# ------------------------------------------------
difference_counts = {
    col: 0
    for col in value_columns
}

pattern_counts = {}

preview = []

for key, group in df.groupby(grain, dropna=False):

    if len(group) <= 1:
        continue

    differing = []

    for col in value_columns:

        n_unique = group[col].nunique(dropna=False)

        if n_unique > 1:
            differing.append(col)
            difference_counts[col] += 1

    pattern = " | ".join(differing) if differing else "NO_DIFFERENCE"

    pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1

    if differing and len(preview) < 25:

        preview.append({
            "state_name": str(key[0]),
            "county_name": str(key[1]),
            "S2_Cell": str(key[2]),
            "Outcome": str(key[3]),
            "row_count": int(len(group)),
            "differing_columns": differing,
            "benchmark_values": sorted(
                group["Benchmark Crash Count"]
                .drop_duplicates()
                .tolist()
            ),
            "hpms_vmt_values": sorted(
                group["HPMS Yearly Vehicle Miles Traveled"]
                .drop_duplicates()
                .tolist()
            ),
            "waymo_ro_miles_values": sorted(
                group["Waymo RO Miles"]
                .drop_duplicates()
                .tolist()
            ),
        })

# ------------------------------------------------
# 6. Exact repeated groups at this grain
# ------------------------------------------------
exact_repeated_groups = 0

for key, group in df.groupby(grain, dropna=False):

    if len(group) <= 1:
        continue

    if group.drop_duplicates().shape[0] == 1:
        exact_repeated_groups += 1

nonidentical_groups = int(len(repeated_keys) - exact_repeated_groups)

# ------------------------------------------------
# 7. Check whether the repeated rows differ only
#    in benchmark/exposure fields
# ------------------------------------------------
only_benchmark_difference = 0
only_exposure_difference = 0
benchmark_and_exposure_difference = 0
no_difference = 0

for pattern, count in pattern_counts.items():

    if pattern == "NO_DIFFERENCE":
        no_difference += count

    elif pattern == "Benchmark Crash Count":
        only_benchmark_difference += count

    elif (
        pattern
        == "HPMS Yearly Vehicle Miles Traveled | Waymo RO Miles"
    ):
        only_exposure_difference += count

    elif (
        "Benchmark Crash Count" in pattern
        and "HPMS Yearly Vehicle Miles Traveled" in pattern
        and "Waymo RO Miles" in pattern
    ):
        benchmark_and_exposure_difference += count

# ------------------------------------------------
# 8. Structural interpretation
# ------------------------------------------------
if len(repeated_keys) == 0:
    structural_result = "CANDIDATE_GRAIN_UNIQUE"
elif nonidentical_groups > 0:
    structural_result = "CANDIDATE_GRAIN_HAS_NONIDENTICAL_REPETITIONS"
else:
    structural_result = "CANDIDATE_GRAIN_HAS_EXACT_REPETITIONS_ONLY"

# ------------------------------------------------
# 9. Boundary
# ------------------------------------------------
boundary = {
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "aggregation_authorized": False,
    "missing_id_repair_authorized": False,
    "grain_authorization": False,
    "semantic_authorization": False,
    "benchmark_comparability": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

# ------------------------------------------------
# 10. Status
# ------------------------------------------------
status = "PENDING_EVIDENCE"

# ------------------------------------------------
# 11. Print
# ------------------------------------------------
print("\nCOLUMN DIFFERENCE COUNTS")

for col, count in difference_counts.items():
    print(f"{col:45s}: {count}")

print("\nPATTERN DISTRIBUTION")

for pattern, count in sorted(
    pattern_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(f"{pattern:75s}: {count}")

print("\nREPEATED-GRAIN CLASSIFICATION")
print("Exact repeated groups      :", exact_repeated_groups)
print("Non-identical groups       :", nonidentical_groups)
print("Only benchmark difference  :", only_benchmark_difference)
print("Only exposure difference   :", only_exposure_difference)
print("Benchmark + exposure       :", benchmark_and_exposure_difference)
print("No difference              :", no_difference)

print("\nSTRUCTURAL RESULT")
print(structural_result)

print("\nNON-IDENTICAL PREVIEW")

for row in preview:
    print(row)

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:35s}: {v}")

print("\nC08R10 STATUS :", status)

# ------------------------------------------------
# 12. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R10"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R10",
    "purpose": "County + S2 + Outcome repeated-grain difference audit",
    "csv4_path": csv4_path,
    "candidate_grain": grain,
    "repeated_groups": int(len(repeated_keys)),
    "excess_rows": int((repeated_keys - 1).sum()),
    "max_multiplicity": int(repeated_keys.max()),
    "difference_counts": difference_counts,
    "pattern_distribution": pattern_counts,
    "exact_repeated_groups": exact_repeated_groups,
    "nonidentical_groups": nonidentical_groups,
    "only_benchmark_difference": only_benchmark_difference,
    "only_exposure_difference": only_exposure_difference,
    "benchmark_and_exposure_difference": benchmark_and_exposure_difference,
    "no_difference": no_difference,
    "structural_result": structural_result,
    "preview": preview,
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R10_COUNTY_S2_OUTCOME_DIFFERENCE_AUDIT.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R10 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R10 COUNTY + S2 + OUTCOME DIFFERENCE AUDIT

REPEATED CANDIDATE-GRAIN GROUPS
Repeated groups : 680
Excess rows     : 680
Max multiplicity: 2

COLUMN DIFFERENCE COUNTS
Benchmark Crash Count                        : 601
HPMS Yearly Vehicle Miles Traveled           : 0
Waymo RO Miles                               : 0

PATTERN DISTRIBUTION
Benchmark Crash Count                                                      : 601
NO_DIFFERENCE                                                              : 79

REPEATED-GRAIN CLASSIFICATION
Exact repeated groups      : 79
Non-identical groups       : 601
Only benchmark difference  : 601
Only exposure difference   : 0
Benchmark + exposure       : 0
No difference              : 79

STRUCTURAL RESULT
CANDIDATE_GRAIN_HAS_NONIDENTICAL_REPETITIONS

NON-IDENTICAL PREVIEW
{'state_name': 'Georgia', 'county_name': 'DeKalb', 'S2_Cell': '9868795481302237184', 'Outcome': 'airbag', 'row_count': 2, 'differing_columns': ['Benchmark Crash Count'], 'benc

In [83]:
# ================================================================
# OIP v1.0.39 — C08R11
# CSV4 BENCHMARK-VALUE DUALITY + DOCUMENTARY RESOLUTION
# ================================================================

import os
import json
import pandas as pd
from pypdf import PdfReader

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV4
# ------------------------------------------------
csv4_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".csv") and "Miles and Benchmark Crashes" in f:
            csv4_files.append(os.path.join(root, f))

if len(csv4_files) != 1:
    raise RuntimeError(
        f"C08R11 FAIL_CLOSED: expected exactly 1 CSV4, found {len(csv4_files)}"
    )

csv4_path = csv4_files[0]

# ------------------------------------------------
# 2. Locate Release Notes PDF
# ------------------------------------------------
pdf_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".pdf") and "Release_Notes" in f:
            pdf_files.append(os.path.join(root, f))

if len(pdf_files) != 1:
    raise RuntimeError(
        f"C08R11 FAIL_CLOSED: expected exactly 1 Release Notes PDF, found {len(pdf_files)}"
    )

release_path = pdf_files[0]

# ------------------------------------------------
# 3. Read CSV4
# ------------------------------------------------
df = pd.read_csv(csv4_path)

required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        f"C08R11 FAIL_CLOSED: missing CSV4 columns: {missing}"
    )

# ------------------------------------------------
# 4. Read Release Notes
# ------------------------------------------------
reader = PdfReader(release_path)

pages = []

for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({
        "page": i,
        "text": text
    })

full_text = "\n".join(
    f"[PAGE {p['page']}]\n{p['text']}"
    for p in pages
)

# ------------------------------------------------
# 5. Documentary search terms
# ------------------------------------------------
terms = [
    "dynamic benchmark",
    "dynamic benchmarks",
    "benchmark crash count",
    "Benchmark Crash Count",
    "predicted benchmark",
    "benchmark",
    "S2 Cell",
    "geographic distribution",
    "HPMS",
    "Vehicle Miles Traveled",
    "Waymo RO Miles",
    "observed",
    "blincoe",
    "Blincoe",
    "calculated",
    "derived",
    "aggregation",
    "aggregate",
    "Georgia",
    "Atlanta",
    "Fulton",
    "DeKalb",
]

term_hits = {}

for term in terms:
    term_hits[term] = {
        "count": int(full_text.lower().count(term.lower())),
        "pages": [
            p["page"]
            for p in pages
            if term.lower() in p["text"].lower()
        ]
    }

# ------------------------------------------------
# 6. Repeated County + S2 + Outcome groups
# ------------------------------------------------
grain = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
]

grouped = df.groupby(grain, dropna=False)

dual_benchmark_groups = []
exact_groups = 0

for key, group in grouped:

    if len(group) <= 1:
        continue

    benchmark_values = (
        group["Benchmark Crash Count"]
        .drop_duplicates()
        .tolist()
    )

    if len(benchmark_values) == 1:
        exact_groups += 1
    else:
        dual_benchmark_groups.append({
            "state_name": str(key[0]),
            "county_name": str(key[1]),
            "S2_Cell": str(key[2]),
            "Outcome": str(key[3]),
            "row_count": int(len(group)),
            "benchmark_values": sorted(
                [float(x) for x in benchmark_values]
            ),
            "hpms_vmt_values": sorted(
                group["HPMS Yearly Vehicle Miles Traveled"]
                .drop_duplicates()
                .astype(float)
                .tolist()
            ),
            "waymo_ro_miles_values": sorted(
                group["Waymo RO Miles"]
                .drop_duplicates()
                .astype(float)
                .tolist()
            ),
        })

# ------------------------------------------------
# 7. Benchmark-value relationship diagnostics
# ------------------------------------------------
dual_df = pd.DataFrame(dual_benchmark_groups)

relationship_summary = {}

if len(dual_df) > 0:

    relationship_summary = {
        "dual_groups": int(len(dual_df)),
        "counties": sorted(
            dual_df["county_name"].unique().tolist()
        ),
        "states": sorted(
            dual_df["state_name"].unique().tolist()
        ),
        "outcomes": sorted(
            dual_df["Outcome"].unique().tolist()
        ),
    }

# ------------------------------------------------
# 8. Look for common mathematical relationship
#    WITHOUT authorizing interpretation
# ------------------------------------------------
ratio_records = []

for row in dual_benchmark_groups:

    vals = row["benchmark_values"]

    if len(vals) == 2 and vals[0] != 0:
        ratio_records.append({
            "county_name": row["county_name"],
            "S2_Cell": row["S2_Cell"],
            "Outcome": row["Outcome"],
            "low": vals[0],
            "high": vals[1],
            "high_low_ratio": vals[1] / vals[0],
        })

ratio_summary = {}

if ratio_records:

    ratios = pd.Series(
        [r["high_low_ratio"] for r in ratio_records]
    )

    ratio_summary = {
        "n": int(len(ratios)),
        "min": float(ratios.min()),
        "median": float(ratios.median()),
        "max": float(ratios.max()),
        "mean": float(ratios.mean()),
    }

# ------------------------------------------------
# 9. Documentary excerpts around key terms
# ------------------------------------------------
excerpt_terms = [
    "dynamic benchmark",
    "benchmark crash count",
    "S2 Cell",
    "geographic distribution",
    "calculated",
    "derived",
]

excerpts = []

for term in excerpt_terms:

    term_lower = term.lower()

    for p in pages:

        text = p["text"]
        lower = text.lower()

        start = lower.find(term_lower)

        if start == -1:
            continue

        excerpt = text[
            max(0, start - 500):
            min(len(text), start + 1200)
        ]

        excerpts.append({
            "term": term,
            "page": p["page"],
            "excerpt": excerpt
        })

        if len(excerpts) >= 20:
            break

    if len(excerpts) >= 20:
        break

# ------------------------------------------------
# 10. Boundary
# ------------------------------------------------
boundary = {
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_authorization": False,
    "benchmark_comparability": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

# ------------------------------------------------
# 11. Status
# ------------------------------------------------
status = "PENDING_EVIDENCE"

# ------------------------------------------------
# 12. Print
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R11 BENCHMARK-VALUE DUALITY AUDIT")
print("=" * 72)

print("\nREPEATED-GRAIN BENCHMARK STRUCTURE")
print("Dual benchmark groups :", len(dual_benchmark_groups))
print("Exact benchmark groups:", exact_groups)

print("\nDOCUMENTARY TERM HITS")

for term, info in term_hits.items():
    print(
        f"{term:30s} : count={info['count']} "
        f"pages={info['pages']}"
    )

print("\nDUAL-BENCHMARK SUMMARY")
print(json.dumps(
    relationship_summary,
    indent=2
))

print("\nRATIO DIAGNOSTIC")
print(json.dumps(
    ratio_summary,
    indent=2
))

print("\nDUAL-BENCHMARK PREVIEW")

for row in dual_benchmark_groups[:25]:
    print(row)

print("\nDOCUMENTARY EXCERPTS")

for e in excerpts:
    print(
        f"\n--- {e['term']} | PAGE {e['page']} ---\n"
        f"{e['excerpt']}"
    )

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:40s}: {v}")

print("\nC08R11 STATUS :", status)

# ------------------------------------------------
# 13. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R11"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R11",
    "purpose": "Benchmark-value duality and documentary resolution",
    "csv4_path": csv4_path,
    "release_notes_path": release_path,
    "dual_benchmark_groups": len(dual_benchmark_groups),
    "exact_benchmark_groups": exact_groups,
    "relationship_summary": relationship_summary,
    "ratio_summary": ratio_summary,
    "dual_benchmark_preview": dual_benchmark_groups[:50],
    "term_hits": term_hits,
    "documentary_excerpts": excerpts,
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R11_BENCHMARK_VALUE_DUALITY_DOCUMENTARY_RESOLUTION.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R11 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R11 BENCHMARK-VALUE DUALITY AUDIT

REPEATED-GRAIN BENCHMARK STRUCTURE
Dual benchmark groups : 601
Exact benchmark groups: 79

DOCUMENTARY TERM HITS
dynamic benchmark              : count=12 pages=[2, 3, 4, 5, 6, 11, 17]
dynamic benchmarks             : count=1 pages=[11]
benchmark crash count          : count=0 pages=[]
Benchmark Crash Count          : count=0 pages=[]
predicted benchmark            : count=1 pages=[16]
benchmark                      : count=106 pages=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17, 18]
S2 Cell                        : count=6 pages=[10, 17]
geographic distribution        : count=2 pages=[6, 10]
HPMS                           : count=2 pages=[17]
Vehicle Miles Traveled         : count=2 pages=[5, 7]
Waymo RO Miles                 : count=2 pages=[12, 17]
observed                       : count=3 pages=[14, 17]
blincoe                        : count=5 pages=[11, 17]
Blincoe                        : count=5 pages=[11, 17]
calculated   

In [84]:
# ================================================================
# OIP v1.0.39 — C08R12R1
# ROBUST CSV4 ↔ CSV3 BENCHMARK RECONCILIATION
# ================================================================

import os
import json
import pandas as pd

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV3 / CSV4
# ------------------------------------------------
csv3_files = []
csv4_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".csv"):
            if "Collision Counts and Comparison" in f:
                csv3_files.append(os.path.join(root, f))
            if "Miles and Benchmark Crashes" in f:
                csv4_files.append(os.path.join(root, f))

if len(csv3_files) != 1:
    raise RuntimeError(
        f"C08R12R1 FAIL_CLOSED: expected 1 CSV3, found {len(csv3_files)}"
    )

if len(csv4_files) != 1:
    raise RuntimeError(
        f"C08R12R1 FAIL_CLOSED: expected 1 CSV4, found {len(csv4_files)}"
    )

csv3_path = csv3_files[0]
csv4_path = csv4_files[0]

# ------------------------------------------------
# 2. Read
# ------------------------------------------------
df3 = pd.read_csv(csv3_path)
df4 = pd.read_csv(csv4_path)

required3 = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Predicted Benchmark Count",
]

required4 = [
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
]

missing3 = [c for c in required3 if c not in df3.columns]
missing4 = [c for c in required4 if c not in df4.columns]

if missing3 or missing4:
    raise RuntimeError(
        f"C08R12R1 FAIL_CLOSED: missing CSV3={missing3}, CSV4={missing4}"
    )

# ------------------------------------------------
# 3. Normalize text
# ------------------------------------------------
df3["_county_norm"] = (
    df3["County Name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df4["_county_norm"] = (
    df4["county_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df3["_outcome_norm"] = (
    df3["Outcome"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df4["_outcome_norm"] = (
    df4["Outcome"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# ------------------------------------------------
# 4. Print unique labels for diagnostic purposes
# ------------------------------------------------
csv3_counties = sorted(
    df3["_county_norm"].dropna().unique().tolist()
)

csv4_counties = sorted(
    df4["_county_norm"].dropna().unique().tolist()
)

csv3_outcomes = sorted(
    df3["_outcome_norm"].dropna().unique().tolist()
)

csv4_outcomes = sorted(
    df4["_outcome_norm"].dropna().unique().tolist()
)

common_counties = sorted(
    set(csv3_counties) & set(csv4_counties)
)

common_outcomes = sorted(
    set(csv3_outcomes) & set(csv4_outcomes)
)

# ------------------------------------------------
# 5. Build county/outcome aggregate tables
# ------------------------------------------------
csv4_agg = (
    df4.groupby(
        ["_county_norm", "_outcome_norm"],
        dropna=False
    )["Benchmark Crash Count"]
    .sum(min_count=1)
    .reset_index(name="CSV4_Benchmark_Sum")
)

csv3_agg = (
    df3.groupby(
        ["_county_norm", "_outcome_norm"],
        dropna=False
    )["Predicted Benchmark Count"]
    .sum(min_count=1)
    .reset_index(name="CSV3_Predicted_Benchmark_Sum")
)

# ------------------------------------------------
# 6. Outer merge
# ------------------------------------------------
recon = pd.merge(
    csv3_agg,
    csv4_agg,
    on=["_county_norm", "_outcome_norm"],
    how="outer"
)

# ------------------------------------------------
# 7. Create abs_diff safely
# ------------------------------------------------
recon["absolute_difference"] = (
    recon["CSV4_Benchmark_Sum"]
    - recon["CSV3_Predicted_Benchmark_Sum"]
)

valid_compare = recon[
    recon["CSV3_Predicted_Benchmark_Sum"].notna()
    & recon["CSV4_Benchmark_Sum"].notna()
].copy()

valid_compare["abs_diff"] = (
    valid_compare["absolute_difference"].abs()
)

# ------------------------------------------------
# 8. Safe reconciliation statistics
# ------------------------------------------------
if len(valid_compare) > 0:

    exact_matches = int(
        (valid_compare["abs_diff"] <= 1e-9).sum()
    )

    near_matches = int(
        (valid_compare["abs_diff"] <= 1e-6).sum()
    )

    outcome_recon = (
        valid_compare
        .groupby("_outcome_norm")
        .agg(
            groups=("abs_diff", "size"),
            exact_matches=(
                "abs_diff",
                lambda x: int((x <= 1e-9).sum())
            ),
            mean_abs_difference=("abs_diff", "mean"),
            max_abs_difference=("abs_diff", "max"),
        )
        .reset_index()
    )

else:

    exact_matches = 0
    near_matches = 0

    outcome_recon = pd.DataFrame(
        columns=[
            "_outcome_norm",
            "groups",
            "exact_matches",
            "mean_abs_difference",
            "max_abs_difference",
        ]
    )

# ------------------------------------------------
# 9. Missing-side diagnostics
# ------------------------------------------------
csv3_only = recon[
    recon["CSV3_Predicted_Benchmark_Sum"].notna()
    & recon["CSV4_Benchmark_Sum"].isna()
].copy()

csv4_only = recon[
    recon["CSV3_Predicted_Benchmark_Sum"].isna()
    & recon["CSV4_Benchmark_Sum"].notna()
].copy()

# ------------------------------------------------
# 10. Boundary
# ------------------------------------------------
boundary = {
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "benchmark_reconciliation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_authorization": False,
    "benchmark_comparability": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

status = "PENDING_EVIDENCE"

# ------------------------------------------------
# 11. Print
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R12R1 ROBUST BENCHMARK RECONCILIATION")
print("=" * 72)

print("\nCSV3 COUNTIES")
print(csv3_counties)

print("\nCSV4 COUNTIES")
print(csv4_counties)

print("\nCOMMON COUNTIES")
print(common_counties)

print("\nCSV3 OUTCOMES")
print(csv3_outcomes)

print("\nCSV4 OUTCOMES")
print(csv4_outcomes)

print("\nCOMMON OUTCOMES")
print(common_outcomes)

print("\nRECONCILIATION")

print("CSV3 county/outcome groups :", len(csv3_agg))
print("CSV4 county/outcome groups :", len(csv4_agg))
print("Comparable groups           :", len(valid_compare))
print("CSV3-only groups            :", len(csv3_only))
print("CSV4-only groups            :", len(csv4_only))
print("Exact matches               :", exact_matches)
print("Near matches                :", near_matches)

print("\nOUTCOME RECONCILIATION")

if len(outcome_recon) > 0:
    print(outcome_recon.to_string(index=False))
else:
    print("NO_COMPARABLE_GROUPS")

print("\nCSV3-ONLY PREVIEW")

for _, row in csv3_only.head(20).iterrows():
    print({
        "county": row["_county_norm"],
        "outcome": row["_outcome_norm"],
        "csv3_predicted_benchmark":
            row["CSV3_Predicted_Benchmark_Sum"]
    })

print("\nCSV4-ONLY PREVIEW")

for _, row in csv4_only.head(20).iterrows():
    print({
        "county": row["_county_norm"],
        "outcome": row["_outcome_norm"],
        "csv4_benchmark":
            row["CSV4_Benchmark_Sum"]
    })

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:42s}: {v}")

print("\nC08R12R1 STATUS :", status)

# ------------------------------------------------
# 12. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R12R1"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R12R1",
    "purpose": "Robust CSV4 to CSV3 benchmark reconciliation",
    "csv3_path": csv3_path,
    "csv4_path": csv4_path,
    "csv3_counties": csv3_counties,
    "csv4_counties": csv4_counties,
    "common_counties": common_counties,
    "csv3_outcomes": csv3_outcomes,
    "csv4_outcomes": csv4_outcomes,
    "common_outcomes": common_outcomes,
    "csv3_groups": int(len(csv3_agg)),
    "csv4_groups": int(len(csv4_agg)),
    "comparable_groups": int(len(valid_compare)),
    "csv3_only_groups": int(len(csv3_only)),
    "csv4_only_groups": int(len(csv4_only)),
    "exact_matches": exact_matches,
    "near_matches": near_matches,
    "outcome_reconciliation":
        outcome_recon.to_dict(orient="records"),
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R12R1_ROBUST_BENCHMARK_RECONCILIATION.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R12R1 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R12R1 ROBUST BENCHMARK RECONCILIATION

CSV3 COUNTIES
['all locations (mileage blended)', 'atlanta area', 'dekalb', 'fulton', 'los angeles', 'maricopa', 'san francisco', 'san francisco bay area', 'san mateo', 'santa clara', 'travis']

CSV4 COUNTIES
['dekalb', 'fulton', 'los angeles', 'maricopa', 'san francisco', 'san mateo', 'santa clara', 'travis']

COMMON COUNTIES
['dekalb', 'fulton', 'los angeles', 'maricopa', 'san francisco', 'san mateo', 'santa clara', 'travis']

CSV3 OUTCOMES
['any airbag deployment', 'any fatal', 'any injury', 'ego vehicle airbag deployment', 'police report', 'suspected serious injury+']

CSV4 OUTCOMES
['airbag', 'blincoe', 'blincoe_any_injury', 'ego_airbag', 'fatal', 'ka', 'observed_any_injury', 'police_reported']

COMMON OUTCOMES
[]

RECONCILIATION
CSV3 county/outcome groups : 66
CSV4 county/outcome groups : 64
Comparable groups           : 0
CSV3-only groups            : 66
CSV4-only groups            : 64
Exact matches               : 0
Near 

In [85]:
# ================================================================
# OIP v1.0.39 — C08R13
# CSV3 ↔ CSV4 OUTCOME SEMANTIC MAPPING EVIDENCE GATE
# ================================================================

import os
import json
import pandas as pd
from pypdf import PdfReader

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV3, CSV4 and Release Notes
# ------------------------------------------------
csv3_files = []
csv4_files = []
pdf_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()

        if fl.endswith(".csv"):
            if "Collision Counts and Comparison" in f:
                csv3_files.append(os.path.join(root, f))
            if "Miles and Benchmark Crashes" in f:
                csv4_files.append(os.path.join(root, f))

        if fl.endswith(".pdf") and "Release_Notes" in f:
            pdf_files.append(os.path.join(root, f))

if len(csv3_files) != 1:
    raise RuntimeError(
        f"C08R13 FAIL_CLOSED: expected 1 CSV3, found {len(csv3_files)}"
    )

if len(csv4_files) != 1:
    raise RuntimeError(
        f"C08R13 FAIL_CLOSED: expected 1 CSV4, found {len(csv4_files)}"
    )

if len(pdf_files) != 1:
    raise RuntimeError(
        f"C08R13 FAIL_CLOSED: expected 1 Release Notes PDF, found {len(pdf_files)}"
    )

csv3_path = csv3_files[0]
csv4_path = csv4_files[0]
release_path = pdf_files[0]

# ------------------------------------------------
# 2. Read CSVs
# ------------------------------------------------
df3 = pd.read_csv(csv3_path)
df4 = pd.read_csv(csv4_path)

if "Outcome" not in df3.columns:
    raise RuntimeError(
        "C08R13 FAIL_CLOSED: CSV3 Outcome column missing"
    )

if "Outcome" not in df4.columns:
    raise RuntimeError(
        "C08R13 FAIL_CLOSED: CSV4 Outcome column missing"
    )

csv3_outcomes = sorted(
    df3["Outcome"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

csv4_outcomes = sorted(
    df4["Outcome"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

# ------------------------------------------------
# 3. Read Release Notes
# ------------------------------------------------
reader = PdfReader(release_path)

pages = []

for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({
        "page": i,
        "text": text
    })

# ------------------------------------------------
# 4. Search every CSV4 outcome term
# ------------------------------------------------
outcome_search = {}

for outcome in csv4_outcomes:

    term = outcome.lower()

    hits = []

    for p in pages:

        txt = p["text"]
        lower = txt.lower()

        if term in lower:
            hits.append(p["page"])

    outcome_search[outcome] = {
        "exact_term": outcome,
        "documented": len(hits) > 0,
        "pages": hits
    }

# ------------------------------------------------
# 5. Search CSV3 outcome terms
# ------------------------------------------------
csv3_search = {}

for outcome in csv3_outcomes:

    term = outcome.lower()

    hits = []

    for p in pages:

        txt = p["text"]
        lower = txt.lower()

        if term in lower:
            hits.append(p["page"])

    csv3_search[outcome] = {
        "exact_term": outcome,
        "documented": len(hits) > 0,
        "pages": hits
    }

# ------------------------------------------------
# 6. Search likely semantic relationship phrases
#    WITHOUT authorizing a mapping
# ------------------------------------------------
relationship_terms = [
    "Any Airbag Deployment",
    "Any Vehicle Airbag Deployment",
    "Ego Vehicle Airbag Deployment",
    "Any Injury",
    "Any Injury Reported",
    "Observed_any_injury",
    "Blincoe_any_injury",
    "Blincoe",
    "Highest Injury Severity Alleged",
    "Serious Injury",
    "Suspected Serious Injury+",
    "Police Reported",
    "Police_reported",
    "Fatal",
    "KA",
    "airbag",
    "ego_airbag",
    "outcome group",
    "outcome groups",
    "outcome crash group",
]

relationship_hits = {}

for term in relationship_terms:

    hits = []

    for p in pages:

        txt = p["text"]
        lower = txt.lower()

        if term.lower() in lower:
            hits.append(p["page"])

    relationship_hits[term] = {
        "count": len(hits),
        "pages": hits
    }

# ------------------------------------------------
# 7. Extract documentary excerpts
# ------------------------------------------------
target_terms = [
    "outcome group",
    "Blincoe",
    "observed_any_injury",
    "Any Injury",
    "Any Airbag Deployment",
    "Ego Vehicle Airbag Deployment",
    "Police Reported",
    "Highest Injury Severity Alleged",
    "KA",
]

excerpts = []

for term in target_terms:

    found = False

    for p in pages:

        text = p["text"]
        lower = text.lower()

        pos = lower.find(term.lower())

        if pos == -1:
            continue

        excerpt = text[
            max(0, pos - 700):
            min(len(text), pos + 1800)
        ]

        excerpts.append({
            "term": term,
            "page": p["page"],
            "excerpt": excerpt
        })

        found = True
        break

# ------------------------------------------------
# 8. Candidate mapping discovery
#    ONLY lexical/documentary overlap.
#    No semantic authorization.
# ------------------------------------------------
candidate_mapping = []

for csv4_outcome in csv4_outcomes:

    c4 = csv4_outcome.lower()

    for csv3_outcome in csv3_outcomes:

        c3 = csv3_outcome.lower()

        score = 0

        # conservative lexical checks
        if "airbag" in c4 and "airbag" in c3:
            score += 1

        if "fatal" in c4 and "fatal" in c3:
            score += 1

        if "police" in c4 and "police" in c3:
            score += 1

        if "injury" in c4 and "injury" in c3:
            score += 1

        if "ego" in c4 and "ego" in c3:
            score += 1

        if "serious" in c3 and (
            "blincoe" in c4
            or "ka" in c4
        ):
            score += 0

        if score > 0:

            candidate_mapping.append({
                "csv4_outcome": csv4_outcome,
                "csv3_outcome": csv3_outcome,
                "lexical_overlap_score": score,
                "semantic_authorized": False
            })

# ------------------------------------------------
# 9. Explicit boundary
# ------------------------------------------------
boundary = {
    "outcome_mapping_authorized": False,
    "semantic_equivalence_authorized": False,
    "benchmark_reconciliation_authorized": False,
    "benchmark_value_selection_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

# ------------------------------------------------
# 10. Status
# ------------------------------------------------
status = "PENDING_EVIDENCE"

# ------------------------------------------------
# 11. Print
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R13 OUTCOME SEMANTIC MAPPING EVIDENCE GATE")
print("=" * 72)

print("\nCSV3 OUTCOMES")
for x in csv3_outcomes:
    print("-", x)

print("\nCSV4 OUTCOMES")
for x in csv4_outcomes:
    print("-", x)

print("\nCSV4 OUTCOME DOCUMENTATION")

for outcome, info in outcome_search.items():
    print(
        f"{outcome:30s} : "
        f"documented={info['documented']} "
        f"pages={info['pages']}"
    )

print("\nCSV3 OUTCOME DOCUMENTATION")

for outcome, info in csv3_search.items():
    print(
        f"{outcome:35s} : "
        f"documented={info['documented']} "
        f"pages={info['pages']}"
    )

print("\nRELATIONSHIP TERM SEARCH")

for term, info in relationship_hits.items():
    if info["count"] > 0:
        print(
            f"{term:40s} : "
            f"count={info['count']} "
            f"pages={info['pages']}"
        )

print("\nCANDIDATE MAPPINGS")
print("NOTE: lexical overlap is NOT semantic authorization.")

for row in candidate_mapping:
    print(row)

print("\nDOCUMENTARY EXCERPTS")

for item in excerpts:
    print(
        f"\n--- {item['term']} | PAGE {item['page']} ---\n"
        f"{item['excerpt']}"
    )

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:40s}: {v}")

print("\nC08R13 STATUS :", status)

# ------------------------------------------------
# 12. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R13"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R13",
    "purpose": "CSV3 to CSV4 outcome semantic mapping evidence gate",
    "csv3_path": csv3_path,
    "csv4_path": csv4_path,
    "release_notes_path": release_path,
    "csv3_outcomes": csv3_outcomes,
    "csv4_outcomes": csv4_outcomes,
    "csv4_outcome_documentation": outcome_search,
    "csv3_outcome_documentation": csv3_search,
    "relationship_hits": relationship_hits,
    "candidate_mapping": candidate_mapping,
    "documentary_excerpts": excerpts,
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R13_OUTCOME_SEMANTIC_MAPPING_EVIDENCE_GATE.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R13 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R13 OUTCOME SEMANTIC MAPPING EVIDENCE GATE

CSV3 OUTCOMES
- Any Airbag Deployment
- Any Fatal
- Any Injury
- Ego Vehicle Airbag Deployment
- Police Report
- Suspected Serious Injury+

CSV4 OUTCOMES
- airbag
- blincoe
- blincoe_any_injury
- ego_airbag
- fatal
- ka
- observed_any_injury
- police_reported

CSV4 OUTCOME DOCUMENTATION
airbag                         : documented=True pages=[2, 3, 4, 7, 8, 9, 10, 11, 13, 14, 17]
blincoe                        : documented=True pages=[11, 17]
blincoe_any_injury             : documented=True pages=[17]
ego_airbag                     : documented=False pages=[]
fatal                          : documented=True pages=[3, 8, 9, 14]
ka                             : documented=True pages=[2, 17]
observed_any_injury            : documented=True pages=[17]
police_reported                : documented=True pages=[17]

CSV3 OUTCOME DOCUMENTATION
Any Airbag Deployment               : documented=True pages=[8, 14]
Any Fatal                 

In [86]:
# ================================================================
# OIP v1.0.39 — C08R14
# DOCUMENTARY OUTCOME MAPPING AUTHORIZATION GATE
# ================================================================

import os
import json
import pandas as pd
from pypdf import PdfReader

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate files
# ------------------------------------------------
csv3_files = []
csv4_files = []
pdf_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()

        if fl.endswith(".csv"):
            if "Collision Counts and Comparison" in f:
                csv3_files.append(os.path.join(root, f))
            if "Miles and Benchmark Crashes" in f:
                csv4_files.append(os.path.join(root, f))

        if fl.endswith(".pdf") and "Release_Notes" in f:
            pdf_files.append(os.path.join(root, f))

if len(csv3_files) != 1:
    raise RuntimeError(
        f"C08R14 FAIL_CLOSED: expected 1 CSV3, found {len(csv3_files)}"
    )

if len(csv4_files) != 1:
    raise RuntimeError(
        f"C08R14 FAIL_CLOSED: expected 1 CSV4, found {len(csv4_files)}"
    )

if len(pdf_files) != 1:
    raise RuntimeError(
        f"C08R14 FAIL_CLOSED: expected 1 Release Notes PDF, found {len(pdf_files)}"
    )

csv3_path = csv3_files[0]
csv4_path = csv4_files[0]
release_path = pdf_files[0]

# ------------------------------------------------
# 2. Read datasets
# ------------------------------------------------
df3 = pd.read_csv(csv3_path)
df4 = pd.read_csv(csv4_path)

if "Outcome" not in df3.columns:
    raise RuntimeError(
        "C08R14 FAIL_CLOSED: CSV3 Outcome column missing"
    )

if "Outcome" not in df4.columns:
    raise RuntimeError(
        "C08R14 FAIL_CLOSED: CSV4 Outcome column missing"
    )

csv3_outcomes = sorted(
    df3["Outcome"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

csv4_outcomes = sorted(
    df4["Outcome"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

# ------------------------------------------------
# 3. Read Release Notes
# ------------------------------------------------
reader = PdfReader(release_path)

pages = []

for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({
        "page": i,
        "text": text
    })

full_text = "\n".join(
    f"[PAGE {p['page']}]\n{p['text']}"
    for p in pages
)

# ------------------------------------------------
# 4. Documentary mapping rules
#
# IMPORTANT:
# These are NOT model-inferred mappings.
# Each mapping requires direct documentary wording.
# ------------------------------------------------
mapping_rules = [

    {
        "csv4": "airbag",
        "csv3": "Any Airbag Deployment",
        "required_terms": [
            "Any Vehicle Airbag Deployment",
            "Airbag Deployment"
        ],
        "reason":
            "Release Notes explicitly states that the published "
            "Airbag Deployment outcome is the more complete "
            "Any Vehicle Airbag Deployment outcome."
    },

    {
        "csv4": "ego_airbag",
        "csv3": "Ego Vehicle Airbag Deployment",
        "required_terms": [
            "Ego Vehicle Airbag Deployment"
        ],
        "reason":
            "Release Notes explicitly defines Ego Vehicle Airbag Deployment."
    },

    {
        "csv4": "observed_any_injury",
        "csv3": "Any Injury",
        "required_terms": [
            "Observed_any_injury",
            "Any Injury Reported"
        ],
        "reason":
            "CSV4 dictionary defines Observed_any_injury as "
            "police-reported injury of any severity, while Release "
            "Notes defines Any Injury Reported."
    },

    {
        "csv4": "blincoe_any_injury",
        "csv3": "Any Injury",
        "required_terms": [
            "Blincoe_any_injury",
            "Any Injury Reported"
        ],
        "reason":
            "CSV4 dictionary defines Blincoe_any_injury as an estimate "
            "of crashes with any injury reported using an underreporting "
            "correction."
    },

    {
        "csv4": "police_reported",
        "csv3": "Police Report",
        "required_terms": [
            "Police_reported",
            "Police-Reported"
        ],
        "reason":
            "CSV4 dictionary defines Police_reported as police-reported crashes."
    },

    {
        "csv4": "fatal",
        "csv3": "Any Fatal",
        "required_terms": [
            "Fatal"
        ],
        "reason":
            "Both datasets/documentation explicitly contain fatal outcome terminology."
    },

    {
        "csv4": "ka",
        "csv3": "Suspected Serious Injury+",
        "required_terms": [
            "Killed",
            "Incapacitation",
            "KABCO"
        ],
        "reason":
            "CSV4 defines KA as Killed or Incapacitation (A). "
            "This mapping is intentionally NOT authorized merely "
            "because both relate to serious outcomes."
    },

    {
        "csv4": "blincoe",
        "csv3": "Any Injury",
        "required_terms": [
            "Blincoe",
            "any property damage or injury"
        ],
        "reason":
            "CSV4 defines Blincoe as an estimate of crashes with "
            "any property damage or injury."
    },
]

# ------------------------------------------------
# 5. Evaluate documentary support
# ------------------------------------------------
mapping_results = []

for rule in mapping_rules:

    found_terms = []
    pages_found = set()

    for term in rule["required_terms"]:

        term_lower = term.lower()

        for p in pages:

            if term_lower in p["text"].lower():
                found_terms.append(term)
                pages_found.add(p["page"])

    found_terms = sorted(set(found_terms))

    # Conservative authorization:
    # Explicitly authorize only mappings whose documentation
    # contains the relevant CSV4 and CSV3 concepts.
    #
    # KA and Blincoe are intentionally blocked from direct
    # equivalence to the CSV3 target because their definitions
    # are not equivalent to those CSV3 labels.
    #
    if rule["csv4"] == "ka":
        state = "PENDING_MAPPING"

    elif rule["csv4"] == "blincoe":
        state = "PENDING_MAPPING"

    elif len(found_terms) >= 1:
        state = "DOCUMENTED_MAPPING"

    else:
        state = "PENDING_MAPPING"

    mapping_results.append({
        "csv4_outcome": rule["csv4"],
        "csv3_target": rule["csv3"],
        "state": state,
        "evidence_terms_found": found_terms,
        "evidence_pages": sorted(pages_found),
        "reason": rule["reason"],
    })

# ------------------------------------------------
# 6. Explicit unresolved mappings
# ------------------------------------------------
unresolved_csv4 = [
    x for x in csv4_outcomes
    if x not in {
        "airbag",
        "blincoe",
        "blincoe_any_injury",
        "ego_airbag",
        "fatal",
        "ka",
        "observed_any_injury",
        "police_reported"
    }
]

# ------------------------------------------------
# 7. Summary
# ------------------------------------------------
documented_count = sum(
    1
    for x in mapping_results
    if x["state"] == "DOCUMENTED_MAPPING"
)

pending_count = sum(
    1
    for x in mapping_results
    if x["state"] == "PENDING_MAPPING"
)

# ------------------------------------------------
# 8. Boundary
# ------------------------------------------------
boundary = {
    "outcome_mapping_authorized": False,
    "documented_mapping_discovery_only": True,
    "semantic_equivalence_authorized": False,
    "benchmark_reconciliation_authorized": False,
    "benchmark_value_selection_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

status = "PENDING_EVIDENCE"

# ------------------------------------------------
# 9. Print
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R14 DOCUMENTARY OUTCOME MAPPING GATE")
print("=" * 72)

print("\nCSV3 OUTCOMES")
for x in csv3_outcomes:
    print("-", x)

print("\nCSV4 OUTCOMES")
for x in csv4_outcomes:
    print("-", x)

print("\nDOCUMENTARY MAPPING RESULTS")

for row in mapping_results:
    print("\n", row)

print("\nSUMMARY")
print("Documented mapping candidates :", documented_count)
print("Pending mappings              :", pending_count)
print("Unresolved CSV4 outcomes      :", unresolved_csv4)

print("\nIMPORTANT")
print("DOCUMENTED_MAPPING means documentary evidence was found.")
print("It does NOT authorize benchmark comparability or scientific validity.")

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:42s}: {v}")

print("\nC08R14 STATUS :", status)

# ------------------------------------------------
# 10. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R14"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R14",
    "purpose": "Documentary outcome mapping gate",
    "csv3_path": csv3_path,
    "csv4_path": csv4_path,
    "release_notes_path": release_path,
    "csv3_outcomes": csv3_outcomes,
    "csv4_outcomes": csv4_outcomes,
    "mapping_results": mapping_results,
    "documented_count": documented_count,
    "pending_count": pending_count,
    "unresolved_csv4_outcomes": unresolved_csv4,
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R14_DOCUMENTARY_OUTCOME_MAPPING_GATE.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R14 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R14 DOCUMENTARY OUTCOME MAPPING GATE

CSV3 OUTCOMES
- Any Airbag Deployment
- Any Fatal
- Any Injury
- Ego Vehicle Airbag Deployment
- Police Report
- Suspected Serious Injury+

CSV4 OUTCOMES
- airbag
- blincoe
- blincoe_any_injury
- ego_airbag
- fatal
- ka
- observed_any_injury
- police_reported

DOCUMENTARY MAPPING RESULTS

 {'csv4_outcome': 'airbag', 'csv3_target': 'Any Airbag Deployment', 'state': 'DOCUMENTED_MAPPING', 'evidence_terms_found': ['Airbag Deployment', 'Any Vehicle Airbag Deployment'], 'evidence_pages': [2, 3, 4, 7, 8, 9, 10, 11, 13, 14], 'reason': 'Release Notes explicitly states that the published Airbag Deployment outcome is the more complete Any Vehicle Airbag Deployment outcome.'}

 {'csv4_outcome': 'ego_airbag', 'csv3_target': 'Ego Vehicle Airbag Deployment', 'state': 'DOCUMENTED_MAPPING', 'evidence_terms_found': ['Ego Vehicle Airbag Deployment'], 'evidence_pages': [7], 'reason': 'Release Notes explicitly defines Ego Vehicle Airbag Deployment.'}



In [87]:
# ================================================================
# OIP v1.0.39 — C08R15R1
# ROBUST DOCUMENTED-MAPPING BENCHMARK RECONCILIATION
# ================================================================

import os
import json
import pandas as pd

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV3 / CSV4
# ------------------------------------------------
csv3_files = []
csv4_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".csv"):
            if "Collision Counts and Comparison" in f:
                csv3_files.append(os.path.join(root, f))
            if "Miles and Benchmark Crashes" in f:
                csv4_files.append(os.path.join(root, f))

if len(csv3_files) != 1:
    raise RuntimeError(
        f"C08R15R1 FAIL_CLOSED: expected 1 CSV3, found {len(csv3_files)}"
    )

if len(csv4_files) != 1:
    raise RuntimeError(
        f"C08R15R1 FAIL_CLOSED: expected 1 CSV4, found {len(csv4_files)}"
    )

csv3_path = csv3_files[0]
csv4_path = csv4_files[0]

# ------------------------------------------------
# 2. Read
# ------------------------------------------------
df3 = pd.read_csv(csv3_path)
df4 = pd.read_csv(csv4_path)

required3 = [
    "Outcome",
    "County Name",
    "Predicted Benchmark Count",
    "Benchmark IPMM",
]

required4 = [
    "county_name",
    "Outcome",
    "Benchmark Crash Count",
]

missing3 = [c for c in required3 if c not in df3.columns]
missing4 = [c for c in required4 if c not in df4.columns]

if missing3 or missing4:
    raise RuntimeError(
        f"C08R15R1 FAIL_CLOSED: missing CSV3={missing3}, CSV4={missing4}"
    )

# ------------------------------------------------
# 3. Documentary mappings from C08R14
# ------------------------------------------------
mapping = {
    "airbag": "Any Airbag Deployment",
    "ego_airbag": "Ego Vehicle Airbag Deployment",
    "observed_any_injury": "Any Injury",
    "blincoe_any_injury": "Any Injury",
    "police_reported": "Police Report",
    "fatal": "Any Fatal",
}

# ------------------------------------------------
# 4. Normalize
# ------------------------------------------------
df3["county_key"] = (
    df3["County Name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df3["csv3_outcome"] = (
    df3["Outcome"]
    .astype("string")
    .str.strip()
)

df4["county_key"] = (
    df4["county_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df4["csv4_outcome"] = (
    df4["Outcome"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# ------------------------------------------------
# 5. Apply only documented mappings
# ------------------------------------------------
df4_mapped = df4[
    df4["csv4_outcome"].isin(mapping.keys())
].copy()

df4_mapped["csv3_target"] = (
    df4_mapped["csv4_outcome"].map(mapping)
)

df3_mapped = df3[
    df3["csv3_outcome"].isin(mapping.values())
].copy()

# ------------------------------------------------
# 6. CSV4 benchmark aggregation
# ------------------------------------------------
csv4_benchmark = (
    df4_mapped
    .groupby(
        ["county_key", "csv4_outcome", "csv3_target"],
        dropna=False
    )
    .agg(
        csv4_benchmark_sum=(
            "Benchmark Crash Count",
            "sum"
        ),
        csv4_rows=(
            "Benchmark Crash Count",
            "size"
        ),
        csv4_unique_benchmark_values=(
            "Benchmark Crash Count",
            lambda x: int(x.dropna().nunique())
        )
    )
    .reset_index()
)

# ------------------------------------------------
# 7. CSV3 benchmark aggregation
# ------------------------------------------------
csv3_benchmark = (
    df3_mapped
    .groupby(
        ["county_key", "csv3_outcome"],
        dropna=False
    )
    .agg(
        csv3_predicted_benchmark_sum=(
            "Predicted Benchmark Count",
            "sum"
        ),
        csv3_rows=(
            "Predicted Benchmark Count",
            "size"
        ),
        csv3_unique_benchmark_ipmm=(
            "Benchmark IPMM",
            lambda x: int(x.dropna().nunique())
        )
    )
    .reset_index()
)

# ------------------------------------------------
# 8. Merge using explicit target columns
# ------------------------------------------------
recon = pd.merge(
    csv4_benchmark,
    csv3_benchmark,
    left_on=["county_key", "csv3_target"],
    right_on=["county_key", "csv3_outcome"],
    how="outer"
)

# ------------------------------------------------
# 9. Safe numeric reconciliation
# ------------------------------------------------
recon["benchmark_difference"] = (
    recon["csv4_benchmark_sum"]
    - recon["csv3_predicted_benchmark_sum"]
)

recon["absolute_difference"] = (
    recon["benchmark_difference"].abs()
)

comparable = recon[
    recon["csv4_benchmark_sum"].notna()
    & recon["csv3_predicted_benchmark_sum"].notna()
].copy()

# ------------------------------------------------
# 10. Summary
# ------------------------------------------------
if len(comparable) > 0:

    comparable_summary = (
        comparable
        .groupby("csv3_target")
        .agg(
            groups=("absolute_difference", "size"),
            exact_matches=(
                "absolute_difference",
                lambda x: int((x <= 1e-9).sum())
            ),
            mean_abs_difference=(
                "absolute_difference",
                "mean"
            ),
            max_abs_difference=(
                "absolute_difference",
                "max"
            )
        )
        .reset_index()
    )

else:

    comparable_summary = pd.DataFrame(
        columns=[
            "csv3_target",
            "groups",
            "exact_matches",
            "mean_abs_difference",
            "max_abs_difference"
        ]
    )

# ------------------------------------------------
# 11. Mapping summary
# ------------------------------------------------
mapping_summary = []

for csv4_outcome, csv3_target in mapping.items():

    c4 = df4_mapped[
        df4_mapped["csv4_outcome"] == csv4_outcome
    ]

    c3 = df3_mapped[
        df3_mapped["csv3_outcome"] == csv3_target
    ]

    mapping_summary.append({
        "csv4_outcome": csv4_outcome,
        "csv3_target": csv3_target,
        "csv4_rows": int(len(c4)),
        "csv3_rows": int(len(c3)),
        "csv4_counties": int(
            c4["county_key"].nunique()
        ),
        "csv3_counties": int(
            c3["county_key"].nunique()
        ),
    })

# ------------------------------------------------
# 12. Preview
# ------------------------------------------------
preview = []

for _, row in comparable.head(40).iterrows():

    preview.append({
        "county": row["county_key"],
        "csv4_outcome": row["csv4_outcome"],
        "csv3_target": row["csv3_target"],
        "csv4_benchmark_sum": float(
            row["csv4_benchmark_sum"]
        ),
        "csv3_predicted_benchmark_sum": float(
            row["csv3_predicted_benchmark_sum"]
        ),
        "absolute_difference": float(
            row["absolute_difference"]
        ),
        "csv4_rows": int(row["csv4_rows"]),
        "csv3_rows": int(row["csv3_rows"])
    })

# ------------------------------------------------
# 13. Boundary
# ------------------------------------------------
boundary = {
    "documented_mapping_discovery_only": True,
    "semantic_equivalence_authorized": False,
    "benchmark_reconciliation_authorized": False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED"
}

status = "PENDING_EVIDENCE"

# ------------------------------------------------
# 14. Print
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R15R1 ROBUST BENCHMARK RECONCILIATION")
print("=" * 72)

print("\nDOCUMENTED MAPPING MATRIX")

for k, v in mapping.items():
    print(f"{k:25s} -> {v}")

print("\nMAPPING SUMMARY")

for row in mapping_summary:
    print(row)

print("\nCOMPARABLE COUNTY/TARGET GROUPS")
print(len(comparable))

print("\nOUTCOME-LEVEL RECONCILIATION")

if len(comparable_summary) > 0:
    print(
        comparable_summary.to_string(
            index=False
        )
    )
else:
    print("NO_COMPARABLE_GROUPS")

print("\nRECONCILIATION PREVIEW")

for row in preview:
    print(row)

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:42s}: {v}")

print("\nC08R15R1 STATUS :", status)

# ------------------------------------------------
# 15. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R15R1"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R15R1",
    "purpose":
        "Robust reconciliation using documentary outcome mappings",
    "csv3_path": csv3_path,
    "csv4_path": csv4_path,
    "mapping": mapping,
    "mapping_summary": mapping_summary,
    "comparable_groups": int(len(comparable)),
    "comparable_summary":
        comparable_summary.to_dict(
            orient="records"
        ),
    "preview": preview,
    "boundary": boundary,
    "status": status
}

artifact_path = os.path.join(
    out_dir,
    "C08R15R1_ROBUST_DOCUMENTED_MAPPING_RECONCILIATION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R15R1 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R15R1 ROBUST BENCHMARK RECONCILIATION

DOCUMENTED MAPPING MATRIX
airbag                    -> Any Airbag Deployment
ego_airbag                -> Ego Vehicle Airbag Deployment
observed_any_injury       -> Any Injury
blincoe_any_injury        -> Any Injury
police_reported           -> Police Report
fatal                     -> Any Fatal

MAPPING SUMMARY
{'csv4_outcome': 'airbag', 'csv3_target': 'Any Airbag Deployment', 'csv4_rows': 2294, 'csv3_rows': 260, 'csv4_counties': 8, 'csv3_counties': 11}
{'csv4_outcome': 'ego_airbag', 'csv3_target': 'Ego Vehicle Airbag Deployment', 'csv4_rows': 2294, 'csv3_rows': 260, 'csv4_counties': 8, 'csv3_counties': 11}
{'csv4_outcome': 'observed_any_injury', 'csv3_target': 'Any Injury', 'csv4_rows': 2294, 'csv3_rows': 520, 'csv4_counties': 8, 'csv3_counties': 11}
{'csv4_outcome': 'blincoe_any_injury', 'csv3_target': 'Any Injury', 'csv4_rows': 2294, 'csv3_rows': 520, 'csv4_counties': 8, 'csv3_counties': 11}
{'csv4_outcome': 'police_reported'

In [88]:
# ================================================================
# OIP v1.0.39 — C08R16
# CSV3 BENCHMARK STRUCTURE / GROUPING AUDIT
# ================================================================

import os
import json
import pandas as pd

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV3
# ------------------------------------------------
csv3_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if (
            f.lower().endswith(".csv")
            and "Collision Counts and Comparison" in f
        ):
            csv3_files.append(os.path.join(root, f))

if len(csv3_files) != 1:
    raise RuntimeError(
        f"C08R16 FAIL_CLOSED: expected 1 CSV3, found {len(csv3_files)}"
    )

csv3_path = csv3_files[0]

# ------------------------------------------------
# 2. Read
# ------------------------------------------------
df = pd.read_csv(csv3_path)

required = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
    "Waymo Count",
    "Predicted Benchmark Count",
    "Benchmark IPMM",
    "Waymo IPMM",
    "Waymo Percent Reduction",
    "Waymo Rate Ratio",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        f"C08R16 FAIL_CLOSED: missing columns: {missing}"
    )

# ------------------------------------------------
# 3. Normalize text
# ------------------------------------------------
for col in [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
]:
    df[f"_{col}_norm"] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# ------------------------------------------------
# 4. Overall structural inventory
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R16 CSV3 BENCHMARK STRUCTURE AUDIT")
print("=" * 72)

print("\nTOTAL ROWS :", len(df))

print("\nOUTCOME × BENCHMARK COMPARISON")

outcome_comparison = (
    df.groupby(
        ["_Outcome_norm", "_Benchmark Comparison_norm"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
)

print(
    outcome_comparison.to_string(index=False)
)

print("\nOUTCOME × GROUPING TYPE")

outcome_grouping = (
    df.groupby(
        ["_Outcome_norm", "_Grouping Type_norm"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
)

print(
    outcome_grouping.to_string(index=False)
)

print("\nCOUNTY × BENCHMARK COMPARISON")

county_comparison = (
    df.groupby(
        ["_County Name_norm", "_Benchmark Comparison_norm"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
)

print(
    county_comparison.to_string(index=False)
)

print("\nCOUNTY × GROUPING TYPE")

county_grouping = (
    df.groupby(
        ["_County Name_norm", "_Grouping Type_norm"],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
)

print(
    county_grouping.to_string(index=False)
)

# ------------------------------------------------
# 5. Metric availability by comparison type
# ------------------------------------------------
print("\nMETRIC COVERAGE BY BENCHMARK COMPARISON")

metric_rows = []

for comparison, g in df.groupby(
    "_Benchmark Comparison_norm",
    dropna=False
):

    metric_rows.append({
        "benchmark_comparison": comparison,
        "rows": len(g),
        "predicted_benchmark_nonnull": int(
            g["Predicted Benchmark Count"].notna().sum()
        ),
        "benchmark_ipmm_nonnull": int(
            g["Benchmark IPMM"].notna().sum()
        ),
        "waymo_ipmm_nonnull": int(
            g["Waymo IPMM"].notna().sum()
        ),
        "percent_reduction_nonnull": int(
            g["Waymo Percent Reduction"].notna().sum()
        ),
        "rate_ratio_nonnull": int(
            g["Waymo Rate Ratio"].notna().sum()
        ),
    })

metric_df = pd.DataFrame(metric_rows)

print(
    metric_df.to_string(index=False)
)

# ------------------------------------------------
# 6. Exact comparison labels
# ------------------------------------------------
print("\nEXACT BENCHMARK COMPARISON LABELS")

for x in sorted(
    df["_Benchmark Comparison_norm"]
    .dropna()
    .unique()
):
    print("-", x)

# ------------------------------------------------
# 7. Exact grouping labels
# ------------------------------------------------
print("\nEXACT GROUPING TYPE LABELS")

for x in sorted(
    df["_Grouping Type_norm"]
    .dropna()
    .unique()
):
    print("-", x)

# ------------------------------------------------
# 8. County labels
# ------------------------------------------------
print("\nEXACT COUNTY LABELS")

for x in sorted(
    df["_County Name_norm"]
    .dropna()
    .unique()
):
    print("-", x)

# ------------------------------------------------
# 9. Dynamic vs non-dynamic structure
# ------------------------------------------------
df["_is_dynamic"] = (
    df["_Benchmark Comparison_norm"]
    .fillna("")
    .str.contains("dynamic", regex=False)
)

dynamic_summary = (
    df.groupby(
        [
            "_is_dynamic",
            "_Outcome_norm",
            "_County Name_norm",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
)

print("\nDYNAMIC / NON-DYNAMIC STRUCTURE")

print(
    dynamic_summary.to_string(index=False)
)

# ------------------------------------------------
# 10. Boundary
# ------------------------------------------------
boundary = {
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "benchmark_reconciliation_authorized": False,
    "dynamic_static_equivalence_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_authorization": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

status = "PENDING_EVIDENCE"

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:45s}: {v}")

print("\nC08R16 STATUS :", status)

# ------------------------------------------------
# 11. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R16"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R16",
    "purpose":
        "CSV3 benchmark comparison and grouping structure audit",
    "csv3_path": csv3_path,
    "total_rows": int(len(df)),
    "outcome_comparison":
        outcome_comparison.to_dict(orient="records"),
    "outcome_grouping":
        outcome_grouping.to_dict(orient="records"),
    "county_comparison":
        county_comparison.to_dict(orient="records"),
    "county_grouping":
        county_grouping.to_dict(orient="records"),
    "metric_coverage":
        metric_df.to_dict(orient="records"),
    "benchmark_comparison_labels":
        sorted(
            df["_Benchmark Comparison_norm"]
            .dropna()
            .unique()
            .tolist()
        ),
    "grouping_type_labels":
        sorted(
            df["_Grouping Type_norm"]
            .dropna()
            .unique()
            .tolist()
        ),
    "county_labels":
        sorted(
            df["_County Name_norm"]
            .dropna()
            .unique()
            .tolist()
        ),
    "dynamic_summary":
        dynamic_summary.to_dict(orient="records"),
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R16_CSV3_BENCHMARK_STRUCTURE_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R16 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R16 CSV3 BENCHMARK STRUCTURE AUDIT

TOTAL ROWS : 1804

OUTCOME × BENCHMARK COMPARISON
                _Outcome_norm                  _Benchmark Comparison_norm  rows
        any airbag deployment                       any airbag deployment   130
        any airbag deployment         any airbag deployment (non-dynamic)   130
                    any fatal                                   any fatal   123
                    any fatal                         any fatal (dynamic)   123
                   any injury                         any-injury-reported   130
                   any injury           any-injury-reported (non-dynamic)   130
                   any injury     any-injury-reported (observed, dynamic)   130
                   any injury any-injury-reported (observed, non-dynamic)   130
ego vehicle airbag deployment               ego vehicle airbag deployment   130
ego vehicle airbag deployment ego vehicle airbag deployment (non-dynamic)   130
                p

In [89]:
# ================================================================
# OIP v1.0.39 — C08R17
# CSV3 BENCHMARK VARIANT VALUE-DIFFERENCE AUDIT
# ================================================================

import os
import json
import pandas as pd

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV3
# ------------------------------------------------
csv3_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if (
            f.lower().endswith(".csv")
            and "Collision Counts and Comparison" in f
        ):
            csv3_files.append(os.path.join(root, f))

if len(csv3_files) != 1:
    raise RuntimeError(
        f"C08R17 FAIL_CLOSED: expected 1 CSV3, found {len(csv3_files)}"
    )

csv3_path = csv3_files[0]

# ------------------------------------------------
# 2. Read CSV3
# ------------------------------------------------
df = pd.read_csv(csv3_path)

required = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
    "Waymo Count",
    "Predicted Benchmark Count",
    "Benchmark IPMM",
    "Benchmark IPMM CI Lower",
    "Benchmark IPMM CI Upper",
    "Waymo IPMM",
    "Waymo Percent Reduction",
    "Waymo Rate Ratio",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        f"C08R17 FAIL_CLOSED: missing columns: {missing}"
    )

# ------------------------------------------------
# 3. Normalize structural fields
# ------------------------------------------------
for col in [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
]:
    df[f"_{col}_norm"] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# ------------------------------------------------
# 4. Basic benchmark variant inventory
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R17 CSV3 BENCHMARK VARIANT VALUE AUDIT")
print("=" * 72)

print("\nBENCHMARK COMPARISON LABELS")

labels = sorted(
    df["_Benchmark Comparison_norm"]
    .dropna()
    .unique()
    .tolist()
)

for x in labels:
    print("-", x)

# ------------------------------------------------
# 5. Create structural unit
#
# IMPORTANT:
# We deliberately exclude Benchmark Comparison.
# This lets us inspect whether different benchmark
# variants occupy the same structural unit.
# ------------------------------------------------
unit_cols = [
    "_Outcome_norm",
    "_County Name_norm",
    "_Grouping Type_norm",
    "_Conflict Partner Type Group_norm",
]

unit_counts = (
    df.groupby(unit_cols, dropna=False)
    .size()
    .reset_index(name="rows")
)

print("\nSTRUCTURAL UNIT SUMMARY")

print(
    "Unique structural units :",
    len(unit_counts)
)

print(
    "Units containing >1 benchmark comparison :",
    int((unit_counts["rows"] > 1).sum())
)

print(
    "Units containing exactly 1 benchmark comparison :",
    int((unit_counts["rows"] == 1).sum())
)

# ------------------------------------------------
# 6. Variant count per structural unit
# ------------------------------------------------
variant_counts = (
    df.groupby(unit_cols, dropna=False)
    ["_Benchmark Comparison_norm"]
    .nunique(dropna=False)
    .reset_index(name="benchmark_variant_count")
)

print("\nBENCHMARK VARIANT COUNT DISTRIBUTION")

print(
    variant_counts["benchmark_variant_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------
# 7. Pivot benchmark predicted counts
# ------------------------------------------------
pivot = df.pivot_table(
    index=unit_cols,
    columns="_Benchmark Comparison_norm",
    values="Predicted Benchmark Count",
    aggfunc="first"
).reset_index()

print("\nPREDICTED BENCHMARK VARIANT PIVOT")

comparison_columns = [
    c for c in pivot.columns
    if c not in unit_cols
]

print(
    "Variant columns found :",
    len(comparison_columns)
)

# ------------------------------------------------
# 8. Within-unit benchmark value ranges
# ------------------------------------------------
value_summary = (
    df.groupby(unit_cols, dropna=False)
    ["Predicted Benchmark Count"]
    .agg(
        benchmark_value_count="count",
        benchmark_value_min="min",
        benchmark_value_max="max",
    )
    .reset_index()
)

value_summary["absolute_range"] = (
    value_summary["benchmark_value_max"]
    - value_summary["benchmark_value_min"]
)

print("\nWITHIN-UNIT BENCHMARK VALUE RANGE")

print(
    "Units with multiple benchmark values :",
    int(
        (
            value_summary["absolute_range"]
            .fillna(0)
            > 0
        ).sum()
    )
)

print(
    "Units with identical benchmark values :",
    int(
        (
            value_summary["absolute_range"]
            .fillna(0)
            == 0
        ).sum()
    )
)

print(
    "Maximum within-unit benchmark range :",
    float(
        value_summary["absolute_range"]
        .fillna(0)
        .max()
    )
)

# ------------------------------------------------
# 9. Show representative variant rows
# ------------------------------------------------
print("\nREPRESENTATIVE BENCHMARK VARIANT ROWS")

preview_cols = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
    "Predicted Benchmark Count",
    "Benchmark IPMM",
    "Waymo IPMM",
    "Waymo Percent Reduction",
    "Waymo Rate Ratio",
]

preview = (
    df[preview_cols]
    .sort_values(
        [
            "County Name",
            "Outcome",
            "Grouping Type",
            "Conflict Partner Type Group",
            "Benchmark Comparison",
        ]
    )
    .head(80)
)

print(
    preview.to_string(index=False)
)

# ------------------------------------------------
# 10. Explicit Any Injury variant audit
# ------------------------------------------------
injury = df[
    df["_Outcome_norm"].eq("any injury")
].copy()

print("\nANY INJURY BENCHMARK VARIANTS")

injury_summary = (
    injury.groupby(
        [
            "_County Name_norm",
            "_Grouping Type_norm",
            "_Conflict Partner Type Group_norm",
            "_Benchmark Comparison_norm",
        ],
        dropna=False
    )
    ["Predicted Benchmark Count"]
    .agg(
        rows="count",
        min_value="min",
        max_value="max",
    )
    .reset_index()
)

print(
    injury_summary.to_string(index=False)
)

# ------------------------------------------------
# 11. Exact same structural unit + different values
# ------------------------------------------------
different_value_units = value_summary[
    value_summary["absolute_range"].fillna(0) > 0
].copy()

print("\nDIFFERENT-VALUE STRUCTURAL UNITS")

print(
    "Count :",
    len(different_value_units)
)

if len(different_value_units) > 0:
    print(
        different_value_units
        .sort_values(
            "absolute_range",
            ascending=False
        )
        .head(30)
        .to_string(index=False)
    )

# ------------------------------------------------
# 12. Boundary
# ------------------------------------------------
boundary = {
    "benchmark_variant_semantic_equivalence_authorized": False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "benchmark_reconciliation_authorized": False,
    "dynamic_static_equivalence_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
}

status = "PENDING_EVIDENCE"

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:55s}: {v}")

print("\nC08R17 STATUS :", status)

# ------------------------------------------------
# 13. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R17"
os.makedirs(out_dir, exist_ok=True)

artifact = {
    "cell": "C08R17",
    "purpose":
        "CSV3 benchmark variant value-difference audit",
    "csv3_path": csv3_path,
    "total_rows": int(len(df)),
    "benchmark_labels": labels,
    "unique_structural_units": int(len(unit_counts)),
    "multi_variant_units": int(
        (unit_counts["rows"] > 1).sum()
    ),
    "single_variant_units": int(
        (unit_counts["rows"] == 1).sum()
    ),
    "variant_count_distribution":
        variant_counts[
            "benchmark_variant_count"
        ]
        .value_counts()
        .sort_index()
        .to_dict(),
    "units_with_different_benchmark_values": int(
        len(different_value_units)
    ),
    "maximum_within_unit_benchmark_range": float(
        value_summary["absolute_range"]
        .fillna(0)
        .max()
    ),
    "boundary": boundary,
    "status": status,
}

artifact_path = os.path.join(
    out_dir,
    "C08R17_CSV3_BENCHMARK_VARIANT_VALUE_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R17 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R17 CSV3 BENCHMARK VARIANT VALUE AUDIT

BENCHMARK COMPARISON LABELS
- any airbag deployment
- any airbag deployment (non-dynamic)
- any fatal
- any fatal (dynamic)
- any-injury-reported
- any-injury-reported (non-dynamic)
- any-injury-reported (observed, dynamic)
- any-injury-reported (observed, non-dynamic)
- ego vehicle airbag deployment
- ego vehicle airbag deployment (non-dynamic)
- police-reported
- police-reported (non-dynamic)
- suspected serious injury+
- suspected serious injury+ (non-dynamic)

STRUCTURAL UNIT SUMMARY
Unique structural units : 772
Units containing >1 benchmark comparison : 772
Units containing exactly 1 benchmark comparison : 0

BENCHMARK VARIANT COUNT DISTRIBUTION
benchmark_variant_count
2    642
4    130

PREDICTED BENCHMARK VARIANT PIVOT
Variant columns found : 14

WITHIN-UNIT BENCHMARK VALUE RANGE
Units with multiple benchmark values : 717
Units with identical benchmark values : 55
Maximum within-unit benchmark range : 442.063419367164

RE

In [90]:
# ================================================================
# OIP v1.0.39 — C08R18
# RELEASE NOTES BENCHMARK VARIANT DOCUMENTATION GATE
# ================================================================

import os
import json
import re
from pypdf import PdfReader

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate Release Notes PDF
# ------------------------------------------------
pdf_files = []

for root, dirs, files in os.walk(BASE):
    for f in files:
        if (
            f.lower().endswith(".pdf")
            and "Release_Notes" in f
        ):
            pdf_files.append(os.path.join(root, f))

if len(pdf_files) != 1:
    raise RuntimeError(
        f"C08R18 FAIL_CLOSED: expected 1 Release Notes PDF, found {len(pdf_files)}"
    )

pdf_path = pdf_files[0]

# ------------------------------------------------
# 2. Extract page-level text
# ------------------------------------------------
reader = PdfReader(pdf_path)

pages = []

for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({
        "page": i,
        "text": text
    })

# ------------------------------------------------
# 3. Target documentary concepts
# ------------------------------------------------
targets = {
    "dynamic benchmark": [
        "dynamic benchmark",
        "dynamic benchmarks",
        "dynamic"
    ],

    "overall human benchmark": [
        "overall human benchmark",
        "overall population human benchmark",
        "overall population human benchmarks"
    ],

    "non-dynamic": [
        "non-dynamic",
        "non dynamic"
    ],

    "observed any injury": [
        "observed_any_injury",
        "observed any injury",
        "observed"
    ],

    "any-injury-reported": [
        "any-injury-reported",
        "any injury reported",
        "any injury"
    ],

    "benchmark comparison": [
        "benchmark comparison",
        "comparison using",
        "comparisons using"
    ],

    "predicted benchmark": [
        "predicted benchmark",
        "predicted benchmark count"
    ],

    "CSV3": [
        "CSV3",
        "csv3"
    ],

    "CSV4": [
        "CSV4",
        "csv4"
    ],

    "geographic distribution": [
        "geographic distribution",
        "S2 cell",
        "spatial distribution"
    ],

    "human benchmark": [
        "human benchmark",
        "human benchmarks"
    ]
}

# ------------------------------------------------
# 4. Page-level target search
# ------------------------------------------------
evidence = []

for label, variants in targets.items():

    for item in pages:

        text_lower = item["text"].lower()

        matched = []

        for term in variants:
            if term.lower() in text_lower:
                matched.append(term)

        if matched:
            evidence.append({
                "target": label,
                "page": item["page"],
                "matched_terms": matched
            })

# ------------------------------------------------
# 5. Print documentary index
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R18 BENCHMARK VARIANT DOCUMENTATION GATE")
print("=" * 72)

print("\nPDF :", os.path.basename(pdf_path))
print("PAGES :", len(pages))

print("\nDOCUMENTARY TARGET INDEX")

for label in targets:

    rows = [
        x for x in evidence
        if x["target"] == label
    ]

    page_list = sorted(
        set(x["page"] for x in rows)
    )

    print(
        f"{label:30s} | "
        f"found={len(rows) > 0} | "
        f"pages={page_list}"
    )

# ------------------------------------------------
# 6. Extract high-value contextual snippets
# ------------------------------------------------
context_terms = [
    "dynamic benchmark",
    "overall population human benchmark",
    "Benchmark Comparison",
    "non-dynamic",
    "observed",
    "CSV3",
    "CSV4",
    "Predicted Benchmark",
    "S2 Cell"
]

snippets = []

for item in pages:

    text = item["text"]
    lower = text.lower()

    for term in context_terms:

        pos = lower.find(term.lower())

        if pos == -1:
            continue

        start = max(0, pos - 500)
        end = min(len(text), pos + 1200)

        snippet = text[start:end].replace(
            "\n", " "
        )

        snippets.append({
            "page": item["page"],
            "term": term,
            "snippet": snippet
        })

# ------------------------------------------------
# 7. Deduplicate snippets
# ------------------------------------------------
unique_snippets = []
seen = set()

for s in snippets:

    key = (
        s["page"],
        s["term"],
        s["snippet"]
    )

    if key not in seen:
        seen.add(key)
        unique_snippets.append(s)

print("\n" + "=" * 72)
print("DOCUMENTARY CONTEXT SNIPPETS")
print("=" * 72)

for s in unique_snippets:

    print(
        f"\n--- PAGE {s['page']} | TERM: {s['term']} ---"
    )

    print(s["snippet"])

# ------------------------------------------------
# 8. Explicit documentation checks
# ------------------------------------------------
page_text = "\n".join(
    x["text"].lower()
    for x in pages
)

checks = {

    "dynamic_benchmark_documented":
        "dynamic benchmark" in page_text,

    "overall_human_benchmark_documented":
        "overall population human benchmark" in page_text
        or "overall human benchmark" in page_text,

    "non_dynamic_documented":
        "non-dynamic" in page_text
        or "non dynamic" in page_text,

    "benchmark_comparison_documented":
        "benchmark comparison" in page_text,

    "predicted_benchmark_documented":
        "predicted benchmark" in page_text,

    "csv3_documented":
        "csv3" in page_text,

    "csv4_documented":
        "csv4" in page_text,

    "s2_cell_documented":
        "s2 cell" in page_text,

    "observed_injury_documented":
        "observed_any_injury" in page_text
        or "observed" in page_text,

    "any_injury_reported_documented":
        "any-injury-reported" in page_text
        or "any injury reported" in page_text
}

print("\n" + "=" * 72)
print("DOCUMENTATION CHECKS")
print("=" * 72)

for k, v in checks.items():
    print(f"{k:45s}: {v}")

# ------------------------------------------------
# 9. Critical boundary
# ------------------------------------------------
boundary = {

    "documentation_discovery_only": True,

    "dynamic_variant_authorized": False,

    "non_dynamic_variant_authorized": False,

    "observed_variant_authorized": False,

    "overall_benchmark_variant_authorized": False,

    "benchmark_value_selection_authorized": False,

    "benchmark_aggregation_authorized": False,

    "benchmark_reconciliation_authorized": False,

    "semantic_equivalence_authorized": False,

    "claim_authorization": False,

    "scientific_authorization": False,

    "road_safety_verdict": "NOT_ISSUED"
}

status = "PENDING_EVIDENCE"

print("\nBOUNDARY")

for k, v in boundary.items():
    print(f"{k:50s}: {v}")

print("\nC08R18 STATUS :", status)

# ------------------------------------------------
# 10. Artifact
# ------------------------------------------------
out_dir = "/kaggle/working/OIP_v1_0_39_C08R18"
os.makedirs(out_dir, exist_ok=True)

artifact = {

    "cell": "C08R18",

    "purpose":
        "Release Notes documentary evidence gate for CSV3 benchmark variants",

    "pdf_path": pdf_path,

    "page_count": len(pages),

    "documentation_checks": checks,

    "evidence_index": evidence,

    "context_snippets": unique_snippets,

    "boundary": boundary,

    "status": status
}

artifact_path = os.path.join(
    out_dir,
    "C08R18_RELEASE_NOTES_BENCHMARK_VARIANT_GATE.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R18 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R18 BENCHMARK VARIANT DOCUMENTATION GATE

PDF : Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf
PAGES : 18

DOCUMENTARY TARGET INDEX
dynamic benchmark              | found=True | pages=[2, 3, 4, 5, 6, 10, 11, 14, 17]
overall human benchmark        | found=True | pages=[10, 11]
non-dynamic                    | found=True | pages=[14]
observed any injury            | found=True | pages=[14, 17]
any-injury-reported            | found=True | pages=[4, 5, 6, 8, 9, 10, 11, 13, 14, 17]
benchmark comparison           | found=True | pages=[11]
predicted benchmark            | found=True | pages=[16]
CSV3                           | found=True | pages=[2, 3, 4, 5, 6, 7, 8, 9, 11, 13, 14]
CSV4                           | found=True | pages=[2, 3, 4, 5, 6, 10, 17]
geographic distribution        | found=True | pages=[6, 10, 17]
human benchmark                | found=True | pages=[1, 2, 5, 6, 8, 9, 10, 11, 15, 16, 18]

DOCUMENTARY CONTEXT SNIPPETS

--- PAGE 2 | TERM: dynamic

In [91]:
# ================================================================
# OIP v1.0.39 — C08R19
# CSV3 BENCHMARK VARIANT ACTUAL-DATA RESOLUTION
# ================================================================

import os
import json
import pandas as pd

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV3
# ------------------------------------------------
csv3_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.lower().endswith(".csv") and "collision" in f.lower() and "benchmark" in f.lower():
            csv3_path = os.path.join(root, f)
            break
    if csv3_path:
        break

if csv3_path is None:
    raise RuntimeError("C08R19 FAIL_CLOSED: CSV3 not found.")

df = pd.read_csv(csv3_path)

required = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Benchmark IPMM",
    "Predicted Benchmark Count",
    "Waymo Count",
    "Waymo Percent Reduction",
    "Waymo Rate Ratio"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        f"C08R19 FAIL_CLOSED: required CSV3 columns missing: {missing}"
    )

# ------------------------------------------------
# 2. Normalize text
# ------------------------------------------------
df["_outcome"] = (
    df["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_benchmark_comparison"] = (
    df["Benchmark Comparison"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_county"] = (
    df["County Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ------------------------------------------------
# 3. Exact Benchmark Comparison inventory
# ------------------------------------------------
comparison_counts = (
    df["Benchmark Comparison"]
    .value_counts(dropna=False)
    .sort_index()
)

print("=" * 72)
print("OIP v1.0.39 — C08R19")
print("CSV3 BENCHMARK VARIANT ACTUAL-DATA RESOLUTION")
print("=" * 72)

print("\nCSV3 FILE:")
print(csv3_path)

print("\nTOTAL ROWS:", len(df))

print("\n" + "-" * 72)
print("BENCHMARK COMPARISON VALUES")
print("-" * 72)

for value, count in comparison_counts.items():
    print(f"{value} | rows={count}")

# ------------------------------------------------
# 4. Classify documented benchmark variants
# ------------------------------------------------
def classify_variant(x):
    s = str(x).strip().lower()

    if "dynamic" in s and "non-dynamic" not in s:
        return "DYNAMIC"

    if "non-dynamic" in s:
        return "NON_DYNAMIC"

    # According to Page 11, overall human benchmark has
    # no additional label.
    return "OVERALL_OR_UNLABELED"

df["_variant"] = df["_benchmark_comparison"].apply(classify_variant)

print("\n" + "-" * 72)
print("VARIANT CLASSIFICATION")
print("-" * 72)

print(
    df["_variant"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------
# 5. Variant × Outcome
# ------------------------------------------------
print("\n" + "-" * 72)
print("VARIANT × OUTCOME")
print("-" * 72)

variant_outcome = pd.crosstab(
    df["_variant"],
    df["_outcome"]
)

print(variant_outcome.to_string())

# ------------------------------------------------
# 6. Variant × County
# ------------------------------------------------
print("\n" + "-" * 72)
print("VARIANT × COUNTY")
print("-" * 72)

variant_county = pd.crosstab(
    df["_variant"],
    df["_county"]
)

print(variant_county.to_string())

# ------------------------------------------------
# 7. Outcome-level variant coverage
# ------------------------------------------------
print("\n" + "-" * 72)
print("OUTCOME-LEVEL VARIANT COVERAGE")
print("-" * 72)

coverage_rows = []

for outcome in sorted(df["_outcome"].unique()):

    sub = df[df["_outcome"] == outcome]

    variants = sorted(sub["_variant"].unique().tolist())

    coverage_rows.append({
        "Outcome": outcome,
        "Rows": len(sub),
        "Variants": "|".join(variants),
        "Dynamic_Rows": int((sub["_variant"] == "DYNAMIC").sum()),
        "NonDynamic_Rows": int((sub["_variant"] == "NON_DYNAMIC").sum()),
        "Overall_Unlabeled_Rows": int(
            (sub["_variant"] == "OVERALL_OR_UNLABELED").sum()
        )
    })

coverage_df = pd.DataFrame(coverage_rows)

print(coverage_df.to_string(index=False))

# ------------------------------------------------
# 8. Same county + outcome appearing in multiple variants
# ------------------------------------------------
variant_overlap = (
    df.groupby(["_county", "_outcome"])["_variant"]
      .nunique()
      .reset_index(name="variant_count")
)

multi_variant_groups = variant_overlap[
    variant_overlap["variant_count"] > 1
]

print("\n" + "-" * 72)
print("COUNTY × OUTCOME MULTI-VARIANT GROUPS")
print("-" * 72)

print(
    "Total county-outcome groups:",
    len(variant_overlap)
)

print(
    "Groups containing >1 benchmark variant:",
    len(multi_variant_groups)
)

if len(multi_variant_groups) > 0:
    print("\nPreview:")
    print(
        multi_variant_groups
        .head(25)
        .to_string(index=False)
    )

# ------------------------------------------------
# 9. Check whether multiple benchmark values occur
#    inside the same county × outcome
# ------------------------------------------------
value_check = (
    df.groupby(["_county", "_outcome"])
      .agg(
          rows=("Benchmark IPMM", "size"),
          unique_benchmark_ipmm=("Benchmark IPMM", "nunique"),
          unique_predicted_count=("Predicted Benchmark Count", "nunique")
      )
      .reset_index()
)

multi_value_groups = value_check[
    (value_check["unique_benchmark_ipmm"] > 1) |
    (value_check["unique_predicted_count"] > 1)
]

print("\n" + "-" * 72)
print("COUNTY × OUTCOME BENCHMARK-VALUE VARIATION")
print("-" * 72)

print(
    "Groups with >1 Benchmark IPMM:",
    int((value_check["unique_benchmark_ipmm"] > 1).sum())
)

print(
    "Groups with >1 Predicted Benchmark Count:",
    int((value_check["unique_predicted_count"] > 1).sum())
)

if len(multi_value_groups) > 0:
    print("\nPreview:")
    print(
        multi_value_groups
        .head(25)
        .to_string(index=False)
    )

# ------------------------------------------------
# 10. Boundary
# ------------------------------------------------
boundary = {
    "actual_variant_inventory_only": True,
    "dynamic_variant_authorized": False,
    "non_dynamic_variant_authorized": False,
    "overall_benchmark_variant_authorized": False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "benchmark_reconciliation_authorized": False,
    "semantic_equivalence_authorized": False,
    "deduplication_authorized": False,
    "row_deletion_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED"
}

# ------------------------------------------------
# 11. Save artifact
# ------------------------------------------------
outdir = "/kaggle/working/OIP_v1_0_39_C08R19"
os.makedirs(outdir, exist_ok=True)

artifact = {
    "cell": "C08R19",
    "purpose": "CSV3 benchmark variant actual-data resolution",
    "csv3_file": csv3_path,
    "total_rows": int(len(df)),
    "benchmark_comparison_values": {
        str(k): int(v)
        for k, v in comparison_counts.items()
    },
    "variant_counts": {
        str(k): int(v)
        for k, v in df["_variant"].value_counts().items()
    },
    "multi_variant_county_outcome_groups": int(
        len(multi_variant_groups)
    ),
    "benchmark_value_variation_groups": int(
        len(multi_value_groups)
    ),
    "boundary": boundary,
    "status": "PENDING_EVIDENCE"
}

artifact_path = os.path.join(
    outdir,
    "C08R19_CSV3_BENCHMARK_VARIANT_ACTUAL_DATA_RESOLUTION.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 72)
print("BOUNDARY")
print("=" * 72)

for k, v in boundary.items():
    print(f"{k:45} : {v}")

print("\nC08R19 STATUS : PENDING_EVIDENCE")

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R19 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R19
CSV3 BENCHMARK VARIANT ACTUAL-DATA RESOLUTION

CSV3 FILE:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv

TOTAL ROWS: 1804

------------------------------------------------------------------------
BENCHMARK COMPARISON VALUES
------------------------------------------------------------------------
Any Airbag Deployment | rows=130
Any Airbag Deployment (non-Dynamic) | rows=130
Any Fatal | rows=123
Any Fatal (Dynamic) | rows=123
Any-injury-reported | rows=130
Any-injury-reported (Observed, Dynamic) | rows=130
Any-injury-reported (non-Dynamic) | rows=130
Any-injury-reported (observed, non-Dynamic) | rows=130
Ego Vehicle Airbag Deployment | rows=130
Ego Vehicle Airbag Deployment (non-Dynamic) | rows=130
Police-reported | rows=130
Police-reported (non-Dynamic) | rows=130
Suspected Serious Injury+ | rows=129
Suspected Serious Injury+ (non-Dynamic) | rows=129

-------

In [92]:
# ================================================================
# OIP v1.0.39 — C08R20
# CLAIM × BENCHMARK VARIANT DOCUMENTARY ALIGNMENT AUDIT
# ================================================================

import os
import json
import pandas as pd
from pypdf import PdfReader

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV3 and Release Notes
# ------------------------------------------------
csv3_path = None
release_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()

        if fl.endswith(".csv") and "collision" in fl and "benchmark" in fl:
            csv3_path = os.path.join(root, f)

        if fl.endswith(".pdf") and "release" in fl and "note" in fl:
            release_path = os.path.join(root, f)

if csv3_path is None:
    raise RuntimeError("C08R20 FAIL_CLOSED: CSV3 not found.")

if release_path is None:
    raise RuntimeError("C08R20 FAIL_CLOSED: Release Notes PDF not found.")

# ------------------------------------------------
# 2. Load CSV3
# ------------------------------------------------
df = pd.read_csv(csv3_path)

required = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Predicted Benchmark Count",
    "Benchmark IPMM",
    "Waymo IPMM",
    "Waymo Percent Reduction",
    "Waymo Rate Ratio"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        f"C08R20 FAIL_CLOSED: missing required columns: {missing}"
    )

# ------------------------------------------------
# 3. Normalize
# ------------------------------------------------
df["_outcome"] = (
    df["Outcome"].astype(str).str.strip().str.lower()
)

df["_benchmark"] = (
    df["Benchmark Comparison"].astype(str).str.strip()
)

df["_benchmark_norm"] = (
    df["_benchmark"].str.lower()
)

# ------------------------------------------------
# 4. Documentary text
# ------------------------------------------------
reader = PdfReader(release_path)

pages = []

for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append(text)

full_text = "\n".join(pages)

# ------------------------------------------------
# 5. Published claim families
# ------------------------------------------------
claims = {
    "serious_injury_or_worse": [
        "95% fewer",
        "serious injury",
        "Suspected Serious Injury+"
    ],
    "airbag": [
        "82% fewer",
        "airbag deployment",
        "Any Airbag Deployment"
    ],
    "any_injury": [
        "82% fewer",
        "injury-causing",
        "Any Injury",
        "Any-injury-reported"
    ],
    "pedestrian": [
        "93% fewer",
        "pedestrian"
    ],
    "cyclist": [
        "86% fewer",
        "cyclist"
    ],
    "motorcycle": [
        "82% fewer",
        "motorcycle"
    ]
}

# ------------------------------------------------
# 6. Documentary variant references
# ------------------------------------------------
variant_terms = {
    "dynamic": [
        "dynamic benchmark",
        "dynamic human benchmark",
        "dynamic"
    ],
    "overall": [
        "overall population human benchmark",
        "overall human benchmark",
        "overall population"
    ],
    "non_dynamic": [
        "non-Dynamic",
        "non-dynamic"
    ],
    "observed": [
        "observed",
        "observed, dynamic",
        "observed, non-Dynamic"
    ]
}

doc_findings = {}

for variant, terms in variant_terms.items():

    hits = []

    for term in terms:
        term_low = term.lower()

        for page_no, text in enumerate(pages, start=1):
            if term_low in text.lower():
                hits.append({
                    "term": term,
                    "page": page_no
                })

    doc_findings[variant] = hits

# ------------------------------------------------
# 7. Actual CSV3 benchmark variants
# ------------------------------------------------
actual_variants = sorted(
    df["Benchmark Comparison"].dropna().unique().tolist()
)

# ------------------------------------------------
# 8. Claim-related outcome inventory
# ------------------------------------------------
claim_outcome_map = {
    "serious_injury_or_worse": [
        "suspected serious injury+"
    ],
    "airbag": [
        "any airbag deployment"
    ],
    "any_injury": [
        "any injury"
    ]
}

print("=" * 72)
print("OIP v1.0.39 — C08R20")
print("CLAIM × BENCHMARK VARIANT DOCUMENTARY ALIGNMENT AUDIT")
print("=" * 72)

print("\nCSV3:", csv3_path)
print("RELEASE NOTES:", release_path)

print("\n" + "-" * 72)
print("ACTUAL CSV3 BENCHMARK VARIANTS")
print("-" * 72)

for v in actual_variants:
    print(v)

# ------------------------------------------------
# 9. Documentary evidence
# ------------------------------------------------
print("\n" + "-" * 72)
print("DOCUMENTARY VARIANT EVIDENCE")
print("-" * 72)

for variant, hits in doc_findings.items():

    pages_found = sorted(
        set(h["page"] for h in hits)
    )

    print(
        f"{variant:15} | found={len(hits) > 0} "
        f"| pages={pages_found}"
    )

# ------------------------------------------------
# 10. Claim × actual benchmark variants
# ------------------------------------------------
print("\n" + "-" * 72)
print("CLAIM × ACTUAL CSV3 BENCHMARK VARIANTS")
print("-" * 72)

claim_variant_rows = []

for claim, outcomes in claim_outcome_map.items():

    sub = df[df["_outcome"].isin(outcomes)]

    variants = sorted(
        sub["Benchmark Comparison"]
        .dropna()
        .unique()
        .tolist()
    )

    claim_variant_rows.append({
        "Claim": claim,
        "CSV3_Rows": len(sub),
        "Actual_Variants": " | ".join(variants)
    })

claim_variant_df = pd.DataFrame(claim_variant_rows)

print(
    claim_variant_df.to_string(index=False)
)

# ------------------------------------------------
# 11. Detect whether documentary evidence explicitly
#     selects a final variant for each claim.
# ------------------------------------------------
print("\n" + "-" * 72)
print("EXPLICIT VARIANT SELECTION TEST")
print("-" * 72)

selection_results = []

for claim in claim_outcome_map:

    # We deliberately do NOT infer selection from
    # numerical values or website wording.

    selection_results.append({
        "Claim": claim,
        "Explicit_Final_Variant_Selected_By_Documentation": False,
        "Reason": (
            "Documentation identifies benchmark variants, "
            "but this audit does not authorize selecting "
            "one variant for scientific claim evaluation."
        )
    })

selection_df = pd.DataFrame(selection_results)

print(
    selection_df.to_string(index=False)
)

# ------------------------------------------------
# 12. Important documentary statement check
# ------------------------------------------------
page11 = pages[10] if len(pages) >= 11 else ""

page11_terms = [
    "dynamic human benchmarks",
    "overall population human benchmarks",
    "Benchmark Comparison",
    "Airbag Deployment (Dynamic)",
    "Airbag Deployment"
]

page11_checks = {
    term: (term.lower() in page11.lower())
    for term in page11_terms
}

print("\n" + "-" * 72)
print("PAGE 11 BENCHMARK-VARIANT STATEMENT CHECK")
print("-" * 72)

for term, found in page11_checks.items():
    print(f"{term:45} : {found}")

# ------------------------------------------------
# 13. Boundary
# ------------------------------------------------
boundary = {
    "documentary_variant_alignment_completed": True,
    "claim_to_variant_final_selection_authorized": False,
    "dynamic_variant_authorized": False,
    "overall_variant_authorized": False,
    "non_dynamic_variant_authorized": False,
    "observed_variant_authorized": False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "semantic_equivalence_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED"
}

# ------------------------------------------------
# 14. Save artifact
# ------------------------------------------------
outdir = "/kaggle/working/OIP_v1_0_39_C08R20"
os.makedirs(outdir, exist_ok=True)

artifact = {
    "cell": "C08R20",
    "purpose": "Claim to benchmark variant documentary alignment audit",
    "csv3_file": csv3_path,
    "release_notes_file": release_path,
    "actual_csv3_variants": actual_variants,
    "documentary_variant_findings": doc_findings,
    "claim_variant_inventory":
        claim_variant_df.to_dict(orient="records"),
    "explicit_variant_selection":
        selection_df.to_dict(orient="records"),
    "page11_checks": page11_checks,
    "boundary": boundary,
    "status": "PENDING_EVIDENCE"
}

artifact_path = os.path.join(
    outdir,
    "C08R20_CLAIM_BENCHMARK_VARIANT_ALIGNMENT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 72)
print("BOUNDARY")
print("=" * 72)

for k, v in boundary.items():
    print(f"{k:50} : {v}")

print("\nC08R20 STATUS : PENDING_EVIDENCE")

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R20 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R20
CLAIM × BENCHMARK VARIANT DOCUMENTARY ALIGNMENT AUDIT

CSV3: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv
RELEASE NOTES: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf

------------------------------------------------------------------------
ACTUAL CSV3 BENCHMARK VARIANTS
------------------------------------------------------------------------
Any Airbag Deployment
Any Airbag Deployment (non-Dynamic)
Any Fatal
Any Fatal (Dynamic)
Any-injury-reported
Any-injury-reported (Observed, Dynamic)
Any-injury-reported (non-Dynamic)
Any-injury-reported (observed, non-Dynamic)
Ego Vehicle Airbag Deployment
Ego Vehicle Airbag Deployment (non-Dynamic)
Police-reported
Police-reported (non-Dynamic)
Suspected Serious Injury+
Suspected Serious Injury+ (non-Dynamic)

---------------------------------

In [93]:
# ================================================================
# OIP v1.0.39 — C08R21
# CSV4 BENCHMARK DUPLICATE / REPETITION PROVENANCE RESOLUTION
# ================================================================

import os
import json
import pandas as pd
from pypdf import PdfReader

BASE = "/kaggle/input"

# ------------------------------------------------
# 1. Locate CSV4 + Release Notes
# ------------------------------------------------
csv4_path = None
release_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()

        if fl.endswith(".csv") and "miles" in fl and "benchmark" in fl:
            if "dynamic" in fl:
                csv4_path = os.path.join(root, f)

        if fl.endswith(".pdf") and "release" in fl and "note" in fl:
            release_path = os.path.join(root, f)

if csv4_path is None:
    raise RuntimeError("C08R21 FAIL_CLOSED: CSV4 not found.")

if release_path is None:
    raise RuntimeError("C08R21 FAIL_CLOSED: Release Notes not found.")

# ------------------------------------------------
# 2. Load CSV4
# ------------------------------------------------
df = pd.read_csv(csv4_path)

required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise RuntimeError(
        f"C08R21 FAIL_CLOSED: missing CSV4 columns: {missing}"
    )

# ------------------------------------------------
# 3. Normalize structural fields
# ------------------------------------------------
for c in ["state_name", "county_name", "S2 Cell", "Outcome"]:
    df[f"_{c}"] = (
        df[c]
        .astype(str)
        .str.strip()
        .str.lower()
    )

# ------------------------------------------------
# 4. Grain diagnostics
# ------------------------------------------------
grain_cols = [
    "_state_name",
    "_county_name",
    "_S2 Cell",
    "_Outcome"
]

county_grain = [
    "_county_name",
    "_S2 Cell",
    "_Outcome"
]

s2_grain = [
    "_S2 Cell",
    "_Outcome"
]

county_groups = (
    df.groupby(county_grain, dropna=False)
      .size()
      .reset_index(name="rows")
)

county_dup = county_groups[
    county_groups["rows"] > 1
].copy()

s2_groups = (
    df.groupby(s2_grain, dropna=False)
      .size()
      .reset_index(name="rows")
)

s2_dup = s2_groups[
    s2_groups["rows"] > 1
].copy()

# ------------------------------------------------
# 5. Exact duplicate analysis
# ------------------------------------------------
exact_cols = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

exact_groups = (
    df.groupby(exact_cols, dropna=False)
      .size()
      .reset_index(name="multiplicity")
)

exact_dup = exact_groups[
    exact_groups["multiplicity"] > 1
].copy()

exact_excess = int(
    (exact_dup["multiplicity"] - 1).sum()
)

# ------------------------------------------------
# 6. Benchmark-only differences within
#    County × S2 Cell × Outcome
# ------------------------------------------------
benchmark_variation = (
    df.groupby(county_grain, dropna=False)
      .agg(
          rows=("Benchmark Crash Count", "size"),
          unique_benchmark_counts=(
              "Benchmark Crash Count",
              "nunique"
          ),
          unique_vmt=(
              "HPMS Yearly Vehicle Miles Traveled",
              "nunique"
          ),
          unique_waymo_miles=(
              "Waymo RO Miles",
              "nunique"
          )
      )
      .reset_index()
)

benchmark_only_groups = benchmark_variation[
    (benchmark_variation["rows"] > 1) &
    (benchmark_variation["unique_benchmark_counts"] > 1) &
    (benchmark_variation["unique_vmt"] == 1) &
    (benchmark_variation["unique_waymo_miles"] == 1)
].copy()

# ------------------------------------------------
# 7. Documentary evidence
# ------------------------------------------------
reader = PdfReader(release_path)

pages = [
    page.extract_text() or ""
    for page in reader.pages
]

full_text = "\n".join(pages).lower()

doc_terms = {
    "dynamic_benchmark": "dynamic benchmark",
    "spatial_distribution": "spatial distribution",
    "s2_cell": "s2 cell",
    "benchmark_crash_count": "benchmark crash count",
    "replicated_across_outcome": "replicated across outcome",
    "geographic_distribution": "geographic distribution",
    "geographic_adjustment": "geographic adjustment",
    "county": "county",
    "state": "state",
    "duplicate": "duplicate",
    "duplicates": "duplicates",
    "repeated": "repeated",
    "replicated": "replicated"
}

doc_hits = {}

for key, term in doc_terms.items():
    doc_hits[key] = {
        "found": term in full_text,
        "pages": [
            i + 1
            for i, page_text in enumerate(pages)
            if term in page_text.lower()
        ]
    }

# ------------------------------------------------
# 8. Print diagnostics
# ------------------------------------------------
print("=" * 72)
print("OIP v1.0.39 — C08R21")
print("CSV4 BENCHMARK DUPLICATE / REPETITION PROVENANCE RESOLUTION")
print("=" * 72)

print("\nCSV4:", csv4_path)
print("RELEASE NOTES:", release_path)

print("\n" + "-" * 72)
print("STRUCTURAL COUNTS")
print("-" * 72)

print("Total CSV4 rows:", len(df))
print("County × S2 Cell × Outcome groups:", len(county_groups))
print("Repeated County × S2 Cell × Outcome groups:", len(county_dup))
print("Repeated S2 Cell × Outcome groups:", len(s2_dup))
print("Exact duplicate groups:", len(exact_dup))
print("Exact duplicate excess rows:", exact_excess)

print("\n" + "-" * 72)
print("BENCHMARK-ONLY REPETITION")
print("-" * 72)

print(
    "Groups with benchmark variation only:",
    len(benchmark_only_groups)
)

print(
    "All repeated groups with VMT unchanged:",
    int(
        (
            (benchmark_variation["rows"] > 1) &
            (benchmark_variation["unique_vmt"] == 1)
        ).sum()
    )
)

print(
    "All repeated groups with Waymo miles unchanged:",
    int(
        (
            (benchmark_variation["rows"] > 1) &
            (benchmark_variation["unique_waymo_miles"] == 1)
        ).sum()
    )
)

# ------------------------------------------------
# 9. Benchmark variation preview
# ------------------------------------------------
if len(benchmark_only_groups) > 0:

    print("\nPreview of benchmark-only repetitions:")

    preview_keys = county_grain

    preview = (
        df.merge(
            benchmark_only_groups[
                preview_keys
            ],
            on=preview_keys,
            how="inner"
        )
        [
            preview_keys +
            [
                "Benchmark Crash Count",
                "HPMS Yearly Vehicle Miles Traveled",
                "Waymo RO Miles"
            ]
        ]
        .drop_duplicates()
        .head(30)
    )

    print(
        preview.to_string(index=False)
    )

# ------------------------------------------------
# 10. Documentary evidence
# ------------------------------------------------
print("\n" + "-" * 72)
print("DOCUMENTARY PROVENANCE TERMS")
print("-" * 72)

for key, info in doc_hits.items():
    print(
        f"{key:30} | "
        f"found={info['found']} | "
        f"pages={info['pages']}"
    )

# ------------------------------------------------
# 11. Targeted documentary contexts
# ------------------------------------------------
targets = [
    "replicated across Outcome levels",
    "dynamic adjustment",
    "spatial distribution",
    "Benchmark Crash Count",
    "S2 Cell"
]

print("\n" + "-" * 72)
print("TARGETED DOCUMENTARY CONTEXT")
print("-" * 72)

for target in targets:

    found = False

    for page_no, text in enumerate(pages, start=1):

        low = text.lower()
        pos = low.find(target.lower())

        if pos >= 0:

            found = True

            start = max(0, pos - 350)
            end = min(len(text), pos + 700)

            print(
                f"\n--- PAGE {page_no} | TERM: {target} ---"
            )
            print(
                text[start:end]
                .replace("\n", " ")
            )

            break

    if not found:
        print(
            f"\n--- TERM NOT FOUND: {target} ---"
        )

# ------------------------------------------------
# 12. Boundary
# ------------------------------------------------
boundary = {
    "duplicate_provenance_discovery_only": True,
    "benchmark_repetition_explained": (
        len(benchmark_only_groups) > 0
    ),
    "exact_duplicate_resolution_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "semantic_equivalence_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED"
}

# ------------------------------------------------
# 13. Save artifact
# ------------------------------------------------
outdir = "/kaggle/working/OIP_v1_0_39_C08R21"
os.makedirs(outdir, exist_ok=True)

artifact = {
    "cell": "C08R21",
    "purpose": "CSV4 benchmark duplicate/repetition provenance resolution",
    "csv4_file": csv4_path,
    "total_rows": int(len(df)),
    "county_s2_outcome_groups": int(len(county_groups)),
    "repeated_county_s2_outcome_groups": int(len(county_dup)),
    "repeated_s2_outcome_groups": int(len(s2_dup)),
    "exact_duplicate_groups": int(len(exact_dup)),
    "exact_duplicate_excess_rows": exact_excess,
    "benchmark_only_repetition_groups": int(
        len(benchmark_only_groups)
    ),
    "documentary_hits": doc_hits,
    "boundary": boundary,
    "status": "PENDING_EVIDENCE"
}

artifact_path = os.path.join(
    outdir,
    "C08R21_CSV4_BENCHMARK_REPETITION_PROVENANCE.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 72)
print("BOUNDARY")
print("=" * 72)

for k, v in boundary.items():
    print(f"{k:50} : {v}")

print("\nC08R21 STATUS : PENDING_EVIDENCE")

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 72)
print("C08R21 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R21
CSV4 BENCHMARK DUPLICATE / REPETITION PROVENANCE RESOLUTION

CSV4: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv
RELEASE NOTES: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf

------------------------------------------------------------------------
STRUCTURAL COUNTS
------------------------------------------------------------------------
Total CSV4 rows: 18352
County × S2 Cell × Outcome groups: 17672
Repeated County × S2 Cell × Outcome groups: 680
Repeated S2 Cell × Outcome groups: 824
Exact duplicate groups: 79
Exact duplicate excess rows: 79

------------------------------------------------------------------------
BENCHMARK-ONLY REPETITION
------------------------------------------------------------------------
Groups with benchmark variation only: 601
All repeated groups with

In [94]:
# ================================================================
# OIP v1.0.39 — C08R22R2
# CORRECTED DATASET-SEPARATED REPETITION + ID AUDIT
# ================================================================

import os
import json
import pandas as pd

BASE = "/kaggle/input"

# ================================================================
# 1. Locate CSV2 and CSV4 separately
# ================================================================

csv2_path = None
csv4_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()

        if (
            fl.endswith(".csv")
            and "crashes" in fl
            and "sgo" in fl
        ):
            csv2_path = os.path.join(root, f)

        if (
            fl.endswith(".csv")
            and "miles" in fl
            and "benchmark" in fl
            and "dynamic" in fl
        ):
            csv4_path = os.path.join(root, f)

if csv2_path is None:
    raise RuntimeError(
        "C08R22R2 FAIL_CLOSED: CSV2 not found."
    )

if csv4_path is None:
    raise RuntimeError(
        "C08R22R2 FAIL_CLOSED: CSV4 not found."
    )

# ================================================================
# 2. Load CSV4
# ================================================================

csv4 = pd.read_csv(csv4_path)

csv4_required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing_csv4 = [
    c for c in csv4_required
    if c not in csv4.columns
]

if missing_csv4:
    raise RuntimeError(
        f"C08R22R2 FAIL_CLOSED: CSV4 missing columns: "
        f"{missing_csv4}"
    )

# ================================================================
# 3. Load CSV2
# ================================================================

csv2 = pd.read_csv(csv2_path)

csv2_required = [
    "SGO Report ID",
    "SGO Amendment"
]

missing_csv2 = [
    c for c in csv2_required
    if c not in csv2.columns
]

if missing_csv2:
    raise RuntimeError(
        f"C08R22R2 FAIL_CLOSED: CSV2 missing columns: "
        f"{missing_csv2}"
    )

# ================================================================
# 4. CSV4 normalization for structural audit
# ================================================================

csv4["_state"] = (
    csv4["state_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv4["_county"] = (
    csv4["county_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv4["_s2"] = (
    csv4["S2 Cell"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv4["_outcome"] = (
    csv4["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ================================================================
# 5. CSV4 structural grain
# ================================================================

grain = [
    "_state",
    "_county",
    "_s2",
    "_outcome"
]

grouped = (
    csv4.groupby(
        grain,
        dropna=False
    )
    .agg(
        rows=("Benchmark Crash Count", "size"),
        unique_benchmark=(
            "Benchmark Crash Count",
            "nunique"
        ),
        unique_vmt=(
            "HPMS Yearly Vehicle Miles Traveled",
            "nunique"
        ),
        unique_waymo=(
            "Waymo RO Miles",
            "nunique"
        )
    )
    .reset_index()
)

repeated = grouped[
    grouped["rows"] > 1
].copy()

# ================================================================
# 6. Repetition classification
# ================================================================

def classify_pattern(row):

    if (
        row["unique_benchmark"] == 1
        and row["unique_vmt"] == 1
        and row["unique_waymo"] == 1
    ):
        return "EXACT_VALUE_REPETITION"

    if (
        row["unique_benchmark"] > 1
        and row["unique_vmt"] == 1
        and row["unique_waymo"] == 1
    ):
        return "BENCHMARK_ONLY_VARIATION"

    if (
        row["unique_benchmark"] > 1
        and (
            row["unique_vmt"] > 1
            or row["unique_waymo"] > 1
        )
    ):
        return "MULTI_FIELD_VARIATION"

    return "OTHER_REPETITION"

repeated["_pattern"] = repeated.apply(
    classify_pattern,
    axis=1
)

# ================================================================
# 7. Exact duplicate audit
# ================================================================

exact_cols = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

exact_groups = (
    csv4.groupby(
        exact_cols,
        dropna=False
    )
    .size()
    .reset_index(
        name="multiplicity"
    )
)

exact_duplicate_groups = exact_groups[
    exact_groups["multiplicity"] > 1
].copy()

exact_duplicate_excess_rows = int(
    (
        exact_duplicate_groups["multiplicity"] - 1
    ).sum()
)

# ================================================================
# 8. Non-identical benchmark repetitions
# ================================================================

nonidentical_groups = repeated[
    repeated["unique_benchmark"] > 1
].copy()

nonidentical_group_count = len(
    nonidentical_groups
)

nonidentical_rows_total = int(
    nonidentical_groups["rows"].sum()
)

nonidentical_excess_rows = int(
    (
        nonidentical_groups["rows"] - 1
    ).sum()
)

# ================================================================
# 9. Exact-value repetitions
# ================================================================

exact_value_repetition = repeated[
    repeated["_pattern"] ==
    "EXACT_VALUE_REPETITION"
].copy()

exact_value_group_count = len(
    exact_value_repetition
)

exact_value_rows_total = int(
    exact_value_repetition["rows"].sum()
)

# ================================================================
# 10. Row reconciliation
# ================================================================

single_groups = grouped[
    grouped["rows"] == 1
]

single_row_count = int(
    single_groups["rows"].sum()
)

repeated_row_count = int(
    repeated["rows"].sum()
)

reconciled_total = (
    single_row_count
    + repeated_row_count
)

row_reconciliation_pass = (
    reconciled_total ==
    len(csv4)
)

# ================================================================
# 11. Multiplicity distribution
# ================================================================

multiplicity_distribution = (
    repeated["rows"]
    .value_counts()
    .sort_index()
    .to_dict()
)

# ================================================================
# 12. S2 × Outcome structure
# ================================================================

s2_outcome_counts = (
    csv4.groupby("_s2")["_outcome"]
    .nunique()
)

outcome_distribution = (
    s2_outcome_counts
    .value_counts()
    .sort_index()
    .to_dict()
)

non_8_s2_cells = int(
    (
        s2_outcome_counts != 8
    ).sum()
)

# ================================================================
# 13. CSV2 Prior-to-SGO audit
# ================================================================

prior_to_sgo_mask = (
    csv2["SGO Amendment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("prior to sgo")
)

missing_sgo_id = (
    csv2["SGO Report ID"].isna()
)

prior_to_sgo_rows = int(
    prior_to_sgo_mask.sum()
)

prior_to_sgo_missing_id = int(
    (
        prior_to_sgo_mask
        & missing_sgo_id
    ).sum()
)

non_prior_missing_id = int(
    (
        (~prior_to_sgo_mask)
        & missing_sgo_id
    ).sum()
)

total_csv2_missing_id = int(
    missing_sgo_id.sum()
)

# ================================================================
# 14. CSV4 hidden provenance fields
# ================================================================

csv4_source_columns = [
    c for c in csv4.columns
    if not c.startswith("_")
]

hidden_provenance_columns = [
    c for c in csv4_source_columns
    if any(
        term in c.lower()
        for term in [
            "version",
            "release",
            "date",
            "source",
            "method"
        ]
    )
]

# ================================================================
# 15. Print results
# ================================================================

print("=" * 72)
print("OIP v1.0.39 — C08R22R2")
print("DATASET-SEPARATED REPETITION + ID AUDIT")
print("=" * 72)

print("\nCSV2:")
print(csv2_path)

print("CSV4:")
print(csv4_path)

# ------------------------------------------------
print("\n" + "-" * 72)
print("CSV4 ROW RECONCILIATION")
print("-" * 72)

print(
    "Original CSV4 rows:",
    len(csv4)
)

print(
    "Single-row groups:",
    len(single_groups)
)

print(
    "Rows inside repeated groups:",
    repeated_row_count
)

print(
    "Reconciled total:",
    reconciled_total
)

print(
    "ROW RECONCILIATION PASS:",
    row_reconciliation_pass
)

# ------------------------------------------------
print("\n" + "-" * 72)
print("CSV4 EXACT DUPLICATES")
print("-" * 72)

print(
    "Exact duplicate groups:",
    len(exact_duplicate_groups)
)

print(
    "Exact duplicate excess rows:",
    exact_duplicate_excess_rows
)

# ------------------------------------------------
print("\n" + "-" * 72)
print("CSV4 NON-IDENTICAL REPETITIONS")
print("-" * 72)

print(
    "Non-identical repetition groups:",
    nonidentical_group_count
)

print(
    "Rows contained in those groups:",
    nonidentical_rows_total
)

print(
    "Excess rows above one-per-group:",
    nonidentical_excess_rows
)

# ------------------------------------------------
print("\n" + "-" * 72)
print("CSV4 REPETITION PATTERN")
print("-" * 72)

for pattern, count in (
    repeated["_pattern"]
    .value_counts()
    .items()
):

    rows_for_pattern = int(
        repeated.loc[
            repeated["_pattern"] == pattern,
            "rows"
        ].sum()
    )

    print(
        f"{pattern:35} "
        f"groups={count} "
        f"rows={rows_for_pattern}"
    )

# ------------------------------------------------
print("\n" + "-" * 72)
print("REPETITION MULTIPLICITY")
print("-" * 72)

print(
    multiplicity_distribution
)

# ------------------------------------------------
print("\n" + "-" * 72)
print("S2 × OUTCOME STRUCTURE")
print("-" * 72)

print(
    "S2 cells with non-8 outcome count:",
    non_8_s2_cells
)

print(
    "Outcome-count distribution:",
    outcome_distribution
)

# ------------------------------------------------
print("\n" + "-" * 72)
print("CSV2 PRIOR-TO-SGO / MISSING ID")
print("-" * 72)

print(
    "Total CSV2 rows:",
    len(csv2)
)

print(
    "Prior-to-SGO rows:",
    prior_to_sgo_rows
)

print(
    "Prior-to-SGO rows with missing SGO ID:",
    prior_to_sgo_missing_id
)

print(
    "Non-Prior-to-SGO rows with missing SGO ID:",
    non_prior_missing_id
)

print(
    "Total missing SGO IDs:",
    total_csv2_missing_id
)

# ------------------------------------------------
print("\n" + "-" * 72)
print("CSV4 HIDDEN PROVENANCE FIELD TEST")
print("-" * 72)

print(
    "CSV4 source columns:",
    csv4_source_columns
)

print(
    "Version/source/date/method-like columns:",
    hidden_provenance_columns
)

# ================================================================
# 16. Formal closure determination
# ================================================================

# Structural checks
structure_verified = (
    row_reconciliation_pass
    and len(exact_duplicate_groups) == 79
    and exact_duplicate_excess_rows == 79
    and nonidentical_group_count == 601
    and prior_to_sgo_rows == 2
    and prior_to_sgo_missing_id == 2
    and non_prior_missing_id == 0
)

# Documentary row-level provenance remains unresolved.
repetition_provenance_resolved = False

if not row_reconciliation_pass:
    status = "FAIL_CLOSED"

elif not structure_verified:
    status = "PENDING_EVIDENCE"

else:
    status = "PENDING_EVIDENCE"

# ================================================================
# 17. Boundary
# ================================================================

boundary = {

    "row_reconciliation_verified":
        bool(row_reconciliation_pass),

    "exact_duplicate_structure_verified":
        bool(
            len(exact_duplicate_groups) == 79
            and exact_duplicate_excess_rows == 79
        ),

    "nonidentical_repetition_structure_verified":
        bool(
            nonidentical_group_count == 601
        ),

    "benchmark_repetition_structurally_observed":
        True,

    "benchmark_repetition_provenance_resolved":
        repetition_provenance_resolved,

    "benchmark_repetition_authorization":
        "NOT_AUTHORIZED",

    "exact_duplicate_resolution_authorized":
        False,

    "nonidentical_repetition_resolution_authorized":
        False,

    "row_deletion_authorized":
        False,

    "deduplication_authorized":
        False,

    "benchmark_value_selection_authorized":
        False,

    "benchmark_aggregation_authorized":
        False,

    "semantic_equivalence_authorized":
        False,

    "claim_authorization":
        False,

    "scientific_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED"
}

# ================================================================
# 18. Artifact
# ================================================================

outdir = (
    "/kaggle/working/"
    "OIP_v1_0_39_C08R22R2"
)

os.makedirs(
    outdir,
    exist_ok=True
)

artifact = {

    "cell":
        "C08R22R2",

    "purpose":
        "Corrected dataset-separated repetition and ID audit",

    "csv2_file":
        csv2_path,

    "csv4_file":
        csv4_path,

    "csv4_row_reconciliation": {

        "original_rows":
            int(len(csv4)),

        "single_row_groups":
            int(len(single_groups)),

        "repeated_rows":
            int(repeated_row_count),

        "reconciled_total":
            int(reconciled_total),

        "pass":
            bool(row_reconciliation_pass)
    },

    "exact_duplicates": {

        "groups":
            int(len(exact_duplicate_groups)),

        "excess_rows":
            int(exact_duplicate_excess_rows)
    },

    "nonidentical_repetitions": {

        "groups":
            int(nonidentical_group_count),

        "rows":
            int(nonidentical_rows_total),

        "excess_rows":
            int(nonidentical_excess_rows)
    },

    "repetition_multiplicity":
        {
            str(k): int(v)
            for k, v in
            multiplicity_distribution.items()
        },

    "s2_outcome_structure": {

        "non_8_s2_cells":
            int(non_8_s2_cells),

        "distribution":
            {
                str(k): int(v)
                for k, v in
                outcome_distribution.items()
            }
    },

    "csv2_prior_sgo": {

        "rows":
            int(prior_to_sgo_rows),

        "missing_sgo_id":
            int(prior_to_sgo_missing_id),

        "non_prior_missing_sgo_id":
            int(non_prior_missing_id),

        "total_missing_sgo_id":
            int(total_csv2_missing_id)
    },

    "csv4_hidden_provenance_columns":
        hidden_provenance_columns,

    "benchmark_repetition_structurally_observed":
        True,

    "benchmark_repetition_provenance_resolved":
        repetition_provenance_resolved,

    "benchmark_repetition_authorization":
        "NOT_AUTHORIZED",

    "boundary":
        boundary,

    "status":
        status
}

artifact_path = os.path.join(
    outdir,
    "C08R22R2_CORRECTED_DATASET_SEPARATED_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================
# 19. Final print
# ================================================================

print("\n" + "=" * 72)
print("FINAL BOUNDARY")
print("=" * 72)

for k, v in boundary.items():

    print(
        f"{k:55} : {v}"
    )

print(
    "\nC08R22R2 STATUS :",
    status
)

print(
    "\nArtifact:"
)

print(
    artifact_path
)

print("\n" + "=" * 72)
print("C08R22R2 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R22R2
DATASET-SEPARATED REPETITION + ID AUDIT

CSV2:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV2 - Crashes with SGO ID and Group Membership 202009-202606-2022benchmark.csv
CSV4:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

------------------------------------------------------------------------
CSV4 ROW RECONCILIATION
------------------------------------------------------------------------
Original CSV4 rows: 18352
Single-row groups: 16992
Rows inside repeated groups: 1360
Reconciled total: 18352
ROW RECONCILIATION PASS: True

------------------------------------------------------------------------
CSV4 EXACT DUPLICATES
------------------------------------------------------------------------
Exact duplicate groups: 79
Exact duplicate excess rows: 79

------------------------------------------------------------------------
CSV4 

In [95]:
# ================================================================
# OIP v1.0.39 — C08R23
# RELEASE-NOTES PROVENANCE RESOLUTION GATE
# ================================================================

import os
import re
import json
import pandas as pd

BASE = "/kaggle/input"

# ================================================================
# 1. Locate Release Notes PDF
# ================================================================

release_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()

        if (
            fl.endswith(".pdf")
            and "release" in fl
            and "notes" in fl
        ):
            release_path = os.path.join(root, f)

if release_path is None:
    raise RuntimeError(
        "C08R23 FAIL_CLOSED: Waymo Release Notes PDF not found."
    )

print("=" * 72)
print("OIP v1.0.39 — C08R23")
print("RELEASE-NOTES PROVENANCE RESOLUTION GATE")
print("=" * 72)

print("\nRelease Notes:")
print(release_path)

# ================================================================
# 2. Extract PDF text
# ================================================================

try:
    from pypdf import PdfReader
except Exception as e:
    raise RuntimeError(
        f"C08R23 FAIL_CLOSED: pypdf unavailable: {e}"
    )

reader = PdfReader(release_path)

pages = []

for i, page in enumerate(reader.pages, start=1):

    text = page.extract_text() or ""

    pages.append({
        "page": i,
        "text": text
    })

full_text = "\n".join(
    p["text"] for p in pages
)

print(
    "\nPDF pages:",
    len(pages)
)

print(
    "Extracted characters:",
    len(full_text)
)

# ================================================================
# 3. Documentary search helper
# ================================================================

def search_terms(term_list):

    results = []

    lower_full = full_text.lower()

    for term in term_list:

        term_lower = term.lower()

        occurrences = []

        for page in pages:

            txt = page["text"]
            lower_txt = txt.lower()

            if term_lower in lower_txt:

                occurrences.append(
                    page["page"]
                )

        results.append({
            "term": term,
            "found": len(occurrences) > 0,
            "pages": occurrences
        })

    return results


# ================================================================
# 4. Targeted provenance terms
# ================================================================

terms = [

    # Repetition / duplicate handling
    "duplicate",
    "duplicates",
    "duplicate rows",
    "duplicate records",
    "repeated",
    "replicated",
    "replication",

    # Benchmark derivation
    "benchmark crash count",
    "benchmark counts",
    "benchmark calculation",
    "benchmark calculations",
    "benchmark derived",
    "derived benchmark",
    "calculated benchmark",
    "benchmark methodology",

    # Spatial benchmark
    "dynamic benchmark",
    "dynamic adjustment",
    "geographic adjustment",
    "geographic distribution",
    "spatial distribution",
    "S2 Cell",

    # CSV4
    "CSV4",
    "Miles and Benchmark Crashes",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles",

    # Outcome replication
    "replicated across Outcome levels",
    "replicated across outcomes",
    "Outcome levels",

    # Source / methodology
    "data source",
    "data sources",
    "methodology",
    "calculated",
    "derived",
    "aggregation",
    "aggregate",

    # Georgia-specific context
    "Georgia",
    "DeKalb",
    "Fulton",
    "Atlanta"
]

term_results = search_terms(terms)

print("\n" + "-" * 72)
print("TARGETED DOCUMENTARY SEARCH")
print("-" * 72)

for r in term_results:

    print(
        f"{r['term']:45} "
        f"found={r['found']} "
        f"pages={r['pages']}"
    )

# ================================================================
# 5. Context extraction helper
# ================================================================

def contexts_for_term(term, window=450):

    contexts = []

    term_lower = term.lower()

    for p in pages:

        txt = p["text"]
        lower_txt = txt.lower()

        start = 0

        while True:

            pos = lower_txt.find(
                term_lower,
                start
            )

            if pos == -1:
                break

            left = max(
                0,
                pos - window
            )

            right = min(
                len(txt),
                pos + len(term) + window
            )

            snippet = txt[left:right]

            contexts.append({
                "page": p["page"],
                "term": term,
                "snippet": snippet
            })

            start = (
                pos
                + len(term_lower)
            )

    return contexts


# ================================================================
# 6. High-value documentary contexts
# ================================================================

high_value_terms = [

    "duplicate",
    "replicated",
    "dynamic benchmark",
    "benchmark crash count",
    "CSV4",
    "derived",
    "calculated",
    "aggregation",
    "S2 Cell",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

high_value_contexts = []

for term in high_value_terms:

    high_value_contexts.extend(
        contexts_for_term(
            term,
            window=500
        )
    )

# Limit displayed contexts while retaining full evidence artifact.
display_contexts = (
    high_value_contexts[:30]
)

print("\n" + "-" * 72)
print("HIGH-VALUE DOCUMENTARY CONTEXTS")
print("-" * 72)

for c in display_contexts:

    print(
        f"\n[Page {c['page']}] "
        f"TERM: {c['term']}"
    )

    cleaned = re.sub(
        r"\s+",
        " ",
        c["snippet"]
    ).strip()

    print(
        cleaned[:1000]
    )

# ================================================================
# 7. Specific resolution tests
# ================================================================

def pages_containing_all(terms_required):

    matching_pages = []

    for p in pages:

        txt = p["text"].lower()

        if all(
            t.lower() in txt
            for t in terms_required
        ):
            matching_pages.append(
                p["page"]
            )

    return matching_pages


# Exact duplicate handling:
exact_duplicate_rule_pages = (
    pages_containing_all([
        "duplicate",
        "CSV4"
    ])
)

# Benchmark repetition / replication:
benchmark_replication_pages = (
    pages_containing_all([
        "benchmark",
        "replicated"
    ])
)

# Dynamic benchmark:
dynamic_benchmark_pages = (
    pages_containing_all([
        "dynamic benchmark"
    ])
)

# Spatial S2 benchmark:
spatial_benchmark_pages = (
    pages_containing_all([
        "S2 Cell",
        "benchmark"
    ])
)

# CSV4 benchmark source:
csv4_benchmark_pages = (
    pages_containing_all([
        "CSV4",
        "benchmark"
    ])
)

# ================================================================
# 8. Important documentary distinctions
# ================================================================

# We explicitly distinguish:
#
# A) Documentation that CSV4 contains replicated fields
# B) Documentation that benchmark values can vary by spatial unit
# C) Documentation that exact duplicate rows are intentional
#
# A/B do NOT automatically establish C.

replication_documented = (
    len(benchmark_replication_pages) > 0
    or any(
        r["found"]
        for r in term_results
        if r["term"] == "replicated"
    )
)

exact_duplicate_handling_documented = (
    len(exact_duplicate_rule_pages) > 0
)

dynamic_method_documented = (
    len(dynamic_benchmark_pages) > 0
)

spatial_method_documented = (
    len(spatial_benchmark_pages) > 0
)

csv4_benchmark_documented = (
    len(csv4_benchmark_pages) > 0
)

# ================================================================
# 9. Formal provenance status
# ================================================================

# IMPORTANT:
# Documentary presence of dynamic/spatial benchmark methodology
# does NOT authorize treating 601 repeated benchmark values as
# resolved provenance.
#
# Exact duplicate handling is separately tested.

benchmark_repetition_provenance_resolved = False

if exact_duplicate_handling_documented:

    exact_duplicate_resolution_state = (
        "DOCUMENTED_RULE_FOUND"
    )

else:

    exact_duplicate_resolution_state = (
        "NO_EXACT_DUPLICATE_RULE_FOUND"
    )

# Unless documentary evidence explicitly explains the observed
# same-grain benchmark repetitions, remain unresolved.

if (
    replication_documented
    and spatial_method_documented
    and csv4_benchmark_documented
):

    benchmark_methodology_context = (
        "DOCUMENTED_CONTEXT_FOUND"
    )

else:

    benchmark_methodology_context = (
        "DOCUMENTARY_CONTEXT_INCOMPLETE"
    )

# ================================================================
# 10. Boundary
# ================================================================

boundary = {

    "benchmark_repetition_structurally_observed":
        True,

    "benchmark_repetition_documentary_context_found":
        bool(
            benchmark_methodology_context ==
            "DOCUMENTED_CONTEXT_FOUND"
        ),

    "benchmark_repetition_provenance_resolved":
        benchmark_repetition_provenance_resolved,

    "benchmark_repetition_authorization":
        "NOT_AUTHORIZED",

    "exact_duplicate_handling_documented":
        exact_duplicate_handling_documented,

    "exact_duplicate_resolution_authorized":
        False,

    "row_deletion_authorized":
        False,

    "deduplication_authorized":
        False,

    "benchmark_value_selection_authorized":
        False,

    "benchmark_aggregation_authorized":
        False,

    "semantic_equivalence_authorized":
        False,

    "claim_authorization":
        False,

    "scientific_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED"
}

# ================================================================
# 11. Final status
# ================================================================

# This cell is a documentary resolution gate.
#
# Even if methodology is documented, C08 remains PENDING unless
# the actual observed repetition provenance is explicitly resolved.

if benchmark_repetition_provenance_resolved:

    status = "PASS_C08_PROVENANCE_RESOLVED"

else:

    status = "PENDING_EVIDENCE"

# ================================================================
# 12. Artifact
# ================================================================

outdir = (
    "/kaggle/working/"
    "OIP_v1_0_39_C08R23"
)

os.makedirs(
    outdir,
    exist_ok=True
)

artifact = {

    "cell":
        "C08R23",

    "purpose":
        "Release Notes documentary provenance resolution gate",

    "release_notes":
        release_path,

    "pages":
        len(pages),

    "extracted_characters":
        len(full_text),

    "targeted_term_results":
        term_results,

    "high_value_contexts":
        high_value_contexts,

    "documentary_resolution": {

        "benchmark_replication_pages":
            benchmark_replication_pages,

        "dynamic_benchmark_pages":
            dynamic_benchmark_pages,

        "spatial_benchmark_pages":
            spatial_benchmark_pages,

        "csv4_benchmark_pages":
            csv4_benchmark_pages,

        "exact_duplicate_rule_pages":
            exact_duplicate_rule_pages,

        "benchmark_methodology_context":
            benchmark_methodology_context,

        "exact_duplicate_handling_state":
            exact_duplicate_resolution_state
    },

    "boundary":
        boundary,

    "benchmark_repetition_structurally_observed":
        True,

    "benchmark_repetition_provenance_resolved":
        benchmark_repetition_provenance_resolved,

    "status":
        status
}

artifact_path = os.path.join(
    outdir,
    "C08R23_RELEASE_NOTES_PROVENANCE_GATE.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================
# 13. Final output
# ================================================================

print("\n" + "=" * 72)
print("DOCUMENTARY RESOLUTION SUMMARY")
print("=" * 72)

print(
    "Benchmark replication context:",
    benchmark_methodology_context
)

print(
    "Dynamic benchmark documented:",
    dynamic_method_documented
)

print(
    "Spatial/S2 benchmark context documented:",
    spatial_method_documented
)

print(
    "CSV4 benchmark context documented:",
    csv4_benchmark_documented
)

print(
    "Exact duplicate handling rule documented:",
    exact_duplicate_handling_documented
)

print(
    "Benchmark repetition provenance resolved:",
    benchmark_repetition_provenance_resolved
)

print("\n" + "-" * 72)
print("FINAL BOUNDARY")
print("-" * 72)

for k, v in boundary.items():

    print(
        f"{k:55} : {v}"
    )

print(
    "\nC08R23 STATUS :",
    status
)

print(
    "\nArtifact:"
)

print(
    artifact_path
)

print("\n" + "=" * 72)
print("C08R23 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R23
RELEASE-NOTES PROVENANCE RESOLUTION GATE

Release Notes:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf

PDF pages: 18
Extracted characters: 46984

------------------------------------------------------------------------
TARGETED DOCUMENTARY SEARCH
------------------------------------------------------------------------
duplicate                                     found=False pages=[]
duplicates                                    found=False pages=[]
duplicate rows                                found=False pages=[]
duplicate records                             found=False pages=[]
repeated                                      found=False pages=[]
replicated                                    found=True pages=[17]
replication                                   found=True pages=[6]
benchmark crash count                         found=False pages=[]
benchmark counts                          

In [96]:
# ================================================================
# OIP v1.0.39 — C08R24
# BENCHMARK REPETITION NUMERICAL STRUCTURE DIAGNOSTIC
# ================================================================

import os
import json
import numpy as np
import pandas as pd

BASE = "/kaggle/input"

# ================================================================
# 1. Locate CSV4
# ================================================================

csv4_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:

        fl = f.lower()

        if (
            fl.endswith(".csv")
            and "miles" in fl
            and "benchmark" in fl
            and "dynamic" in fl
        ):
            csv4_path = os.path.join(root, f)

if csv4_path is None:
    raise RuntimeError(
        "C08R24 FAIL_CLOSED: CSV4 not found."
    )

df = pd.read_csv(csv4_path)

required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        f"C08R24 FAIL_CLOSED: missing columns: {missing}"
    )

# ================================================================
# 2. Normalize structural identifiers
# ================================================================

df["_state"] = (
    df["state_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_county"] = (
    df["county_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_s2"] = (
    df["S2 Cell"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_outcome"] = (
    df["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ================================================================
# 3. Identify repeated structural groups
# ================================================================

grain = [
    "_state",
    "_county",
    "_s2",
    "_outcome"
]

grouped = (
    df.groupby(
        grain,
        dropna=False
    )
    .agg(
        row_count=(
            "Benchmark Crash Count",
            "size"
        ),
        benchmark_nunique=(
            "Benchmark Crash Count",
            "nunique"
        ),
        vmt_nunique=(
            "HPMS Yearly Vehicle Miles Traveled",
            "nunique"
        ),
        waymo_nunique=(
            "Waymo RO Miles",
            "nunique"
        )
    )
    .reset_index()
)

benchmark_repeated = grouped[
    (grouped["row_count"] > 1)
    &
    (grouped["benchmark_nunique"] > 1)
].copy()

print("=" * 72)
print("OIP v1.0.39 — C08R24")
print("BENCHMARK REPETITION NUMERICAL STRUCTURE DIAGNOSTIC")
print("=" * 72)

print("\nCSV4:")
print(csv4_path)

print("\nTotal rows:", len(df))

print(
    "Benchmark-only repetition groups:",
    len(benchmark_repeated)
)

# ================================================================
# 4. Extract benchmark values per repeated group
# ================================================================

rows = []

for _, g in benchmark_repeated.iterrows():

    mask = (
        (df["_state"] == g["_state"])
        &
        (df["_county"] == g["_county"])
        &
        (df["_s2"] == g["_s2"])
        &
        (df["_outcome"] == g["_outcome"])
    )

    sub = df.loc[
        mask,
        [
            "_state",
            "_county",
            "_s2",
            "_outcome",
            "Benchmark Crash Count",
            "HPMS Yearly Vehicle Miles Traveled",
            "Waymo RO Miles"
        ]
    ].copy()

    values = sorted(
        pd.to_numeric(
            sub["Benchmark Crash Count"],
            errors="coerce"
        )
        .dropna()
        .unique()
        .tolist()
    )

    if len(values) < 2:
        continue

    low = float(values[0])
    high = float(values[-1])

    ratio = (
        high / low
        if low != 0
        else np.nan
    )

    difference = high - low

    rows.append({

        "state": g["_state"],
        "county": g["_county"],
        "s2": g["_s2"],
        "outcome": g["_outcome"],

        "benchmark_values":
            values,

        "min_benchmark":
            low,

        "max_benchmark":
            high,

        "difference":
            difference,

        "ratio_high_to_low":
            ratio,

        "has_zero":
            any(
                abs(v) < 1e-12
                for v in values
            ),

        "value_count":
            len(values),

        "vmt_unique":
            int(g["vmt_nunique"]),

        "waymo_unique":
            int(g["waymo_nunique"])
    })

rep = pd.DataFrame(rows)

# ================================================================
# 5. Numerical classification
# ================================================================

rep["pattern"] = np.select(

    [
        rep["has_zero"],

        rep["ratio_high_to_low"].notna()
        &
        np.isclose(
            rep["ratio_high_to_low"],
            2.0,
            rtol=1e-6,
            atol=1e-9
        ),

        rep["ratio_high_to_low"].notna()
        &
        np.isclose(
            rep["ratio_high_to_low"],
            1.5,
            rtol=1e-6,
            atol=1e-9
        ),

        rep["ratio_high_to_low"].notna()
        &
        np.isclose(
            rep["ratio_high_to_low"],
            1.0,
            rtol=1e-6,
            atol=1e-9
        )
    ],

    [
        "ZERO_VS_NONZERO",
        "APPROX_2X",
        "APPROX_1_5X",
        "EQUAL_UNEXPECTED"
    ],

    default="OTHER_NUMERICAL_RELATION"
)

# ================================================================
# 6. Summary statistics
# ================================================================

zero_vs_nonzero = int(
    (rep["pattern"] == "ZERO_VS_NONZERO").sum()
)

approx_2x = int(
    (rep["pattern"] == "APPROX_2X").sum()
)

approx_1_5x = int(
    (rep["pattern"] == "APPROX_1_5X").sum()
)

equal_unexpected = int(
    (rep["pattern"] == "EQUAL_UNEXPECTED").sum()
)

other_pattern = int(
    (rep["pattern"] == "OTHER_NUMERICAL_RELATION").sum()
)

# ================================================================
# 7. VMT / Waymo consistency
# ================================================================

vmt_changed = int(
    (
        rep["vmt_unique"] > 1
    ).sum()
)

waymo_changed = int(
    (
        rep["waymo_unique"] > 1
    ).sum()
)

# ================================================================
# 8. Print pattern summary
# ================================================================

print("\n" + "-" * 72)
print("NUMERICAL PATTERN SUMMARY")
print("-" * 72)

print(
    "ZERO_VS_NONZERO:",
    zero_vs_nonzero
)

print(
    "APPROX_2X:",
    approx_2x
)

print(
    "APPROX_1_5X:",
    approx_1_5x
)

print(
    "EQUAL_UNEXPECTED:",
    equal_unexpected
)

print(
    "OTHER_NUMERICAL_RELATION:",
    other_pattern
)

print("\n" + "-" * 72)
print("EXPOSURE CONSISTENCY")
print("-" * 72)

print(
    "Groups with VMT variation:",
    vmt_changed
)

print(
    "Groups with Waymo RO Miles variation:",
    waymo_changed
)

# ================================================================
# 9. Benchmark difference distribution
# ================================================================

print("\n" + "-" * 72)
print("BENCHMARK DIFFERENCE DISTRIBUTION")
print("-" * 72)

if len(rep) > 0:

    print(
        rep["difference"]
        .describe()
        .to_string()
    )

else:

    print(
        "No benchmark repetition groups available."
    )

# ================================================================
# 10. Representative examples
# ================================================================

print("\n" + "-" * 72)
print("REPRESENTATIVE REPETITION EXAMPLES")
print("-" * 72)

if len(rep) > 0:

    preview = (
        rep.sort_values(
            "difference",
            ascending=False
        )
        .head(15)
    )

    for _, r in preview.iterrows():

        print(
            f"\nState={r['state']}"
            f" | County={r['county']}"
            f" | S2={r['s2']}"
            f" | Outcome={r['outcome']}"
        )

        print(
            "Benchmark values:",
            r["benchmark_values"]
        )

        print(
            "Difference:",
            r["difference"]
        )

        print(
            "High/Low ratio:",
            r["ratio_high_to_low"]
        )

        print(
            "Pattern:",
            r["pattern"]
        )

# ================================================================
# 11. Check whether benchmark differences are accompanied by
#     exposure differences
# ================================================================

exposure_consistent = (
    vmt_changed == 0
    and waymo_changed == 0
)

# ================================================================
# 12. IMPORTANT: no provenance authorization
# ================================================================

# Numerical structure alone cannot establish why the benchmark
# differs. Therefore it cannot authorize selection, deletion,
# deduplication, aggregation, or semantic equivalence.

provenance_resolved = False

boundary = {

    "benchmark_repetition_structurally_observed":
        True,

    "benchmark_repetition_numerically_characterized":
        True,

    "exposure_fields_consistent_across_repeated_groups":
        bool(exposure_consistent),

    "benchmark_repetition_provenance_resolved":
        False,

    "benchmark_value_selection_authorized":
        False,

    "benchmark_aggregation_authorized":
        False,

    "row_deletion_authorized":
        False,

    "deduplication_authorized":
        False,

    "semantic_equivalence_authorized":
        False,

    "claim_authorization":
        False,

    "scientific_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED"
}

# ================================================================
# 13. Status
# ================================================================

status = "PENDING_EVIDENCE"

# ================================================================
# 14. Artifact
# ================================================================

outdir = (
    "/kaggle/working/"
    "OIP_v1_0_39_C08R24"
)

os.makedirs(
    outdir,
    exist_ok=True
)

artifact = {

    "cell":
        "C08R24",

    "purpose":
        "Numerical characterization of CSV4 benchmark repetition",

    "total_csv4_rows":
        int(len(df)),

    "benchmark_repetition_groups":
        int(len(rep)),

    "pattern_counts": {

        "ZERO_VS_NONZERO":
            zero_vs_nonzero,

        "APPROX_2X":
            approx_2x,

        "APPROX_1_5X":
            approx_1_5x,

        "EQUAL_UNEXPECTED":
            equal_unexpected,

        "OTHER_NUMERICAL_RELATION":
            other_pattern
    },

    "exposure_consistency": {

        "groups_with_vmt_variation":
            vmt_changed,

        "groups_with_waymo_variation":
            waymo_changed,

        "exposure_consistent":
            exposure_consistent
    },

    "representations":
        rep.to_dict(
            orient="records"
        ),

    "benchmark_repetition_provenance_resolved":
        provenance_resolved,

    "boundary":
        boundary,

    "status":
        status
}

artifact_path = os.path.join(
    outdir,
    "C08R24_BENCHMARK_REPETITION_NUMERICAL_DIAGNOSTIC.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================
# 15. Final output
# ================================================================

print("\n" + "=" * 72)
print("FINAL BOUNDARY")
print("=" * 72)

for k, v in boundary.items():

    print(
        f"{k:55} : {v}"
    )

print(
    "\nC08R24 STATUS :",
    status
)

print(
    "\nArtifact:"
)

print(
    artifact_path
)

print("\n" + "=" * 72)
print("C08R24 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R24
BENCHMARK REPETITION NUMERICAL STRUCTURE DIAGNOSTIC

CSV4:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

Total rows: 18352
Benchmark-only repetition groups: 601

------------------------------------------------------------------------
NUMERICAL PATTERN SUMMARY
------------------------------------------------------------------------
ZERO_VS_NONZERO: 141
APPROX_2X: 6
APPROX_1_5X: 0
EQUAL_UNEXPECTED: 0
OTHER_NUMERICAL_RELATION: 454

------------------------------------------------------------------------
EXPOSURE CONSISTENCY
------------------------------------------------------------------------
Groups with VMT variation: 0
Groups with Waymo RO Miles variation: 0

------------------------------------------------------------------------
BENCHMARK DIFFERENCE DISTRIBUTION
------------------------------------------------------------------------
count     601.0

In [97]:
# ================================================================
# OIP v1.0.39 — C08R25
# BENCHMARK REPETITION GEOGRAPHIC / OUTCOME / SOURCE-PATTERN AUDIT
# ================================================================

import os
import json
import numpy as np
import pandas as pd

BASE = "/kaggle/input"

# ================================================================
# 1. Locate CSV4
# ================================================================

csv4_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()

        if (
            fl.endswith(".csv")
            and "miles" in fl
            and "benchmark" in fl
            and "dynamic" in fl
        ):
            csv4_path = os.path.join(root, f)

if csv4_path is None:
    raise RuntimeError(
        "C08R25 FAIL_CLOSED: CSV4 not found."
    )

df = pd.read_csv(csv4_path)

required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        f"C08R25 FAIL_CLOSED: missing columns: {missing}"
    )

# ================================================================
# 2. Normalize
# ================================================================

df["_state"] = (
    df["state_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_county"] = (
    df["county_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_outcome"] = (
    df["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["_benchmark"] = pd.to_numeric(
    df["Benchmark Crash Count"],
    errors="coerce"
)

# ================================================================
# 3. Repeated structural groups
# ================================================================

grain = [
    "_state",
    "_county",
    "S2 Cell",
    "_outcome"
]

grouped = (
    df.groupby(
        grain,
        dropna=False
    )
    .agg(
        rows=(
            "Benchmark Crash Count",
            "size"
        ),
        benchmark_nunique=(
            "Benchmark Crash Count",
            "nunique"
        ),
        vmt_nunique=(
            "HPMS Yearly Vehicle Miles Traveled",
            "nunique"
        ),
        waymo_nunique=(
            "Waymo RO Miles",
            "nunique"
        )
    )
    .reset_index()
)

rep_groups = grouped[
    (grouped["rows"] > 1)
    &
    (grouped["benchmark_nunique"] > 1)
].copy()

# ================================================================
# 4. Attach benchmark values
# ================================================================

records = []

for _, g in rep_groups.iterrows():

    mask = (
        (df["_state"] == g["_state"])
        &
        (df["_county"] == g["_county"])
        &
        (df["S2 Cell"] == g["S2 Cell"])
        &
        (df["_outcome"] == g["_outcome"])
    )

    sub = df.loc[
        mask,
        [
            "_state",
            "_county",
            "S2 Cell",
            "_outcome",
            "_benchmark",
            "HPMS Yearly Vehicle Miles Traveled",
            "Waymo RO Miles"
        ]
    ].copy()

    vals = sorted(
        sub["_benchmark"]
        .dropna()
        .unique()
        .tolist()
    )

    if len(vals) < 2:
        continue

    records.append({

        "state":
            g["_state"],

        "county":
            g["_county"],

        "s2":
            g["S2 Cell"],

        "outcome":
            g["_outcome"],

        "benchmark_values":
            vals,

        "low":
            float(vals[0]),

        "high":
            float(vals[-1]),

        "difference":
            float(vals[-1] - vals[0]),

        "ratio":
            (
                float(vals[-1] / vals[0])
                if vals[0] != 0
                else np.nan
            ),

        "vmt_nunique":
            int(g["vmt_nunique"]),

        "waymo_nunique":
            int(g["waymo_nunique"])
    })

rep = pd.DataFrame(records)

# ================================================================
# 5. Geographic distribution
# ================================================================

print("=" * 72)
print("OIP v1.0.39 — C08R25")
print("BENCHMARK REPETITION GEOGRAPHIC / OUTCOME AUDIT")
print("=" * 72)

print("\nTotal CSV4 rows:", len(df))
print(
    "Benchmark repetition groups:",
    len(rep)
)

print("\n" + "-" * 72)
print("REPETITION GROUPS BY STATE")
print("-" * 72)

print(
    rep["state"]
    .value_counts()
    .to_string()
)

print("\n" + "-" * 72)
print("REPETITION GROUPS BY COUNTY")
print("-" * 72)

print(
    rep["county"]
    .value_counts()
    .to_string()
)

print("\n" + "-" * 72)
print("REPETITION GROUPS BY OUTCOME")
print("-" * 72)

print(
    rep["outcome"]
    .value_counts()
    .to_string()
)

# ================================================================
# 6. State × outcome
# ================================================================

print("\n" + "-" * 72)
print("STATE × OUTCOME")
print("-" * 72)

state_outcome = pd.crosstab(
    rep["state"],
    rep["outcome"]
)

print(
    state_outcome.to_string()
)

# ================================================================
# 7. County × outcome
# ================================================================

print("\n" + "-" * 72)
print("COUNTY × OUTCOME")
print("-" * 72)

county_outcome = pd.crosstab(
    rep["county"],
    rep["outcome"]
)

print(
    county_outcome.to_string()
)

# ================================================================
# 8. Compare Georgia vs non-Georgia
# ================================================================

rep["is_georgia"] = (
    rep["state"] == "georgia"
)

print("\n" + "-" * 72)
print("GEORGIA VS NON-GEORGIA")
print("-" * 72)

print(
    rep["is_georgia"]
    .value_counts()
    .rename(
        index={
            True: "Georgia",
            False: "Non-Georgia"
        }
    )
    .to_string()
)

# ================================================================
# 9. Zero/non-zero distribution by geography
# ================================================================

rep["zero_vs_nonzero"] = np.where(
    rep["low"] == 0,
    "ZERO_TO_NONZERO",
    "NONZERO_TO_NONZERO"
)

print("\n" + "-" * 72)
print("ZERO/NONZERO BY STATE")
print("-" * 72)

zero_state = pd.crosstab(
    rep["state"],
    rep["zero_vs_nonzero"]
)

print(
    zero_state.to_string()
)

# ================================================================
# 10. Ratio distribution
# ================================================================

print("\n" + "-" * 72)
print("RATIO DISTRIBUTION")
print("-" * 72)

valid_ratio = rep[
    rep["ratio"].notna()
].copy()

print(
    valid_ratio["ratio"]
    .describe()
    .to_string()
)

# ================================================================
# 11. Identify whether every repeated group is Georgia
# ================================================================

all_georgia = (
    len(rep) > 0
    and rep["is_georgia"].all()
)

print(
    "\nAll benchmark repetition groups Georgia:",
    all_georgia
)

# ================================================================
# 12. Compare benchmark repetition with county
# ================================================================

county_summary = (
    rep.groupby("county")
    .agg(
        repetition_groups=("difference", "size"),
        mean_difference=("difference", "mean"),
        max_difference=("difference", "max"),
        median_difference=("difference", "median")
    )
    .sort_values(
        "repetition_groups",
        ascending=False
    )
)

print("\n" + "-" * 72)
print("COUNTY BENCHMARK DIFFERENCE SUMMARY")
print("-" * 72)

print(
    county_summary.to_string()
)

# ================================================================
# 13. Outcome summary
# ================================================================

outcome_summary = (
    rep.groupby("outcome")
    .agg(
        repetition_groups=("difference", "size"),
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        max_difference=("difference", "max")
    )
    .sort_values(
        "repetition_groups",
        ascending=False
    )
)

print("\n" + "-" * 72)
print("OUTCOME BENCHMARK DIFFERENCE SUMMARY")
print("-" * 72)

print(
    outcome_summary.to_string()
)

# ================================================================
# 14. Look for a single dominant benchmark value pair
# ================================================================

rep["value_pair"] = rep.apply(
    lambda r:
        tuple(
            round(
                float(x),
                8
            )
            for x in r["benchmark_values"]
        ),
    axis=1
)

pair_counts = (
    rep["value_pair"]
    .value_counts()
)

print("\n" + "-" * 72)
print("MOST COMMON BENCHMARK VALUE PAIRS")
print("-" * 72)

print(
    pair_counts.head(20).to_string()
)

# ================================================================
# 15. Check whether repetition occurs across multiple outcomes
#     within the same S2 cell
# ================================================================

cell_repetition_outcomes = (
    rep.groupby(
        ["state", "county", "s2"]
    )["outcome"]
    .nunique()
)

multi_outcome_cells = int(
    (
        cell_repetition_outcomes > 1
    ).sum()
)

print("\n" + "-" * 72)
print("MULTI-OUTCOME REPETITION CELLS")
print("-" * 72)

print(
    "S2 cells with repetition in >1 outcome:",
    multi_outcome_cells
)

print(
    "Maximum repeated outcomes in one S2 cell:",
    int(
        cell_repetition_outcomes.max()
    )
    if len(cell_repetition_outcomes)
    else 0
)

# ================================================================
# 16. Exposure consistency
# ================================================================

exposure_consistent = (
    rep["vmt_nunique"].eq(1).all()
    and rep["waymo_nunique"].eq(1).all()
)

print(
    "\nExposure fields consistent:",
    exposure_consistent
)

# ================================================================
# 17. Provenance decision
# ================================================================

# This cell can establish a geographic/outcome pattern.
# It cannot establish the causal/documentary origin of the
# benchmark differences.

pattern_identified = (
    len(rep) == 601
)

provenance_resolved = False

# ================================================================
# 18. Boundary
# ================================================================

boundary = {

    "benchmark_repetition_structurally_observed":
        True,

    "benchmark_repetition_geographically_characterized":
        bool(pattern_identified),

    "benchmark_repetition_outcome_characterized":
        bool(pattern_identified),

    "exposure_fields_consistent":
        bool(exposure_consistent),

    "benchmark_repetition_provenance_resolved":
        False,

    "benchmark_value_selection_authorized":
        False,

    "benchmark_aggregation_authorized":
        False,

    "row_deletion_authorized":
        False,

    "deduplication_authorized":
        False,

    "semantic_equivalence_authorized":
        False,

    "claim_authorization":
        False,

    "scientific_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED"
}

status = "PENDING_EVIDENCE"

# ================================================================
# 19. Artifact
# ================================================================

outdir = (
    "/kaggle/working/"
    "OIP_v1_0_39_C08R25"
)

os.makedirs(
    outdir,
    exist_ok=True
)

artifact = {

    "cell":
        "C08R25",

    "purpose":
        "Geographic/outcome characterization of benchmark repetition",

    "total_rows":
        int(len(df)),

    "benchmark_repetition_groups":
        int(len(rep)),

    "all_georgia":
        bool(all_georgia),

    "multi_outcome_repetition_cells":
        int(multi_outcome_cells),

    "max_repeated_outcomes_per_s2":
        (
            int(cell_repetition_outcomes.max())
            if len(cell_repetition_outcomes)
            else 0
        ),

    "exposure_consistent":
        bool(exposure_consistent),

    "state_distribution":
        rep["state"].value_counts().to_dict(),

    "county_distribution":
        rep["county"].value_counts().to_dict(),

    "outcome_distribution":
        rep["outcome"].value_counts().to_dict(),

    "county_summary":
        county_summary.reset_index().to_dict(
            orient="records"
        ),

    "outcome_summary":
        outcome_summary.reset_index().to_dict(
            orient="records"
        ),

    "most_common_value_pairs":
        {
            str(k): int(v)
            for k, v in
            pair_counts.head(20).items()
        },

    "boundary":
        boundary,

    "benchmark_repetition_provenance_resolved":
        provenance_resolved,

    "status":
        status
}

artifact_path = os.path.join(
    outdir,
    "C08R25_GEOGRAPHIC_OUTCOME_PATTERN_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================
# 20. Final
# ================================================================

print("\n" + "=" * 72)
print("FINAL BOUNDARY")
print("=" * 72)

for k, v in boundary.items():

    print(
        f"{k:55} : {v}"
    )

print(
    "\nC08R25 STATUS :",
    status
)

print(
    "\nArtifact:"
)

print(
    artifact_path
)

print("\n" + "=" * 72)
print("C08R25 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R25
BENCHMARK REPETITION GEOGRAPHIC / OUTCOME AUDIT

Total CSV4 rows: 18352
Benchmark repetition groups: 601

------------------------------------------------------------------------
REPETITION GROUPS BY STATE
------------------------------------------------------------------------
state
georgia    601

------------------------------------------------------------------------
REPETITION GROUPS BY COUNTY
------------------------------------------------------------------------
county
fulton    478
dekalb    123

------------------------------------------------------------------------
REPETITION GROUPS BY OUTCOME
------------------------------------------------------------------------
outcome
blincoe                85
police_reported        85
observed_any_injury    84
blincoe_any_injury     84
ego_airbag             82
airbag                 82
ka                     72
fatal                  27

------------------------------------------------------------------------
STA

In [98]:
# ================================================================
# OIP v1.0.39 — C08R26
# GEORGIA / ATLANTA BENCHMARK CROSS-FILE PROVENANCE AUDIT
# ================================================================

import os
import json
import numpy as np
import pandas as pd

BASE = "/kaggle/input"

# ================================================================
# 1. Locate CSV3 and CSV4
# ================================================================

csv3_path = None
csv4_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:

        fl = f.lower()

        if (
            fl.endswith(".csv")
            and "collision" in fl
            and "benchmark" in fl
        ):
            csv3_path = os.path.join(root, f)

        if (
            fl.endswith(".csv")
            and "miles" in fl
            and "benchmark" in fl
            and "dynamic" in fl
        ):
            csv4_path = os.path.join(root, f)

if csv3_path is None:
    raise RuntimeError(
        "C08R26 FAIL_CLOSED: CSV3 not found."
    )

if csv4_path is None:
    raise RuntimeError(
        "C08R26 FAIL_CLOSED: CSV4 not found."
    )

csv3 = pd.read_csv(csv3_path)
csv4 = pd.read_csv(csv4_path)

# ================================================================
# 2. Required columns
# ================================================================

csv3_required = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
    "Benchmark IPMM",
    "Predicted Benchmark Count"
]

csv4_required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing3 = [
    c for c in csv3_required
    if c not in csv3.columns
]

missing4 = [
    c for c in csv4_required
    if c not in csv4.columns
]

if missing3:
    raise RuntimeError(
        f"C08R26 FAIL_CLOSED: CSV3 missing {missing3}"
    )

if missing4:
    raise RuntimeError(
        f"C08R26 FAIL_CLOSED: CSV4 missing {missing4}"
    )

# ================================================================
# 3. Normalize CSV3
# ================================================================

for c in [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group"
]:

    csv3[f"_{c.lower().replace(' ', '_')}"] = (
        csv3[c]
        .astype(str)
        .str.strip()
        .str.lower()
    )

csv3["_benchmark_ipmm"] = pd.to_numeric(
    csv3["Benchmark IPMM"],
    errors="coerce"
)

csv3["_predicted_count"] = pd.to_numeric(
    csv3["Predicted Benchmark Count"],
    errors="coerce"
)

# ================================================================
# 4. Normalize CSV4
# ================================================================

csv4["_state"] = (
    csv4["state_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv4["_county"] = (
    csv4["county_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv4["_outcome"] = (
    csv4["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv4["_benchmark"] = pd.to_numeric(
    csv4["Benchmark Crash Count"],
    errors="coerce"
)

# ================================================================
# 5. Restrict CSV4 to Georgia repetition groups
# ================================================================

ga4 = csv4[
    csv4["_state"] == "georgia"
].copy()

grain = [
    "_state",
    "_county",
    "S2 Cell",
    "_outcome"
]

g4 = (
    ga4.groupby(
        grain,
        dropna=False
    )
    .agg(
        rows=(
            "Benchmark Crash Count",
            "size"
        ),
        benchmark_nunique=(
            "Benchmark Crash Count",
            "nunique"
        ),
        benchmark_min=(
            "Benchmark Crash Count",
            "min"
        ),
        benchmark_max=(
            "Benchmark Crash Count",
            "max"
        ),
        vmt_nunique=(
            "HPMS Yearly Vehicle Miles Traveled",
            "nunique"
        ),
        waymo_nunique=(
            "Waymo RO Miles",
            "nunique"
        )
    )
    .reset_index()
)

repeated_ga = g4[
    (g4["rows"] > 1)
    &
    (g4["benchmark_nunique"] > 1)
].copy()

print("=" * 72)
print("OIP v1.0.39 — C08R26")
print("GEORGIA / ATLANTA BENCHMARK CROSS-FILE PROVENANCE AUDIT")
print("=" * 72)

print("\nCSV3:")
print(csv3_path)

print("\nCSV4:")
print(csv4_path)

print("\nCSV4 Georgia rows:", len(ga4))

print(
    "Georgia benchmark repetition groups:",
    len(repeated_ga)
)

# ================================================================
# 6. CSV3 Georgia / Atlanta benchmark inventory
# ================================================================

county3 = (
    csv3["_county_name"]
    .value_counts()
)

print("\n" + "-" * 72)
print("CSV3 COUNTY LABELS")
print("-" * 72)

print(
    county3.to_string()
)

# ================================================================
# 7. Atlanta-related CSV3 rows
# ================================================================

atlanta_mask = (
    csv3["_county_name"].isin([
        "atlanta area",
        "fulton",
        "dekalb"
    ])
)

atlanta3 = csv3[
    atlanta_mask
].copy()

print("\n" + "-" * 72)
print("CSV3 ATLANTA / FULTON / DEKALB STRUCTURE")
print("-" * 72)

print(
    "Rows:",
    len(atlanta3)
)

if len(atlanta3):

    summary_cols = [
        "Outcome",
        "Benchmark Comparison",
        "County Name",
        "Grouping Type",
        "Conflict Partner Type Group"
    ]

    print(
        atlanta3[
            summary_cols
        ]
        .drop_duplicates()
        .sort_values(
            [
                "County Name",
                "Outcome",
                "Benchmark Comparison"
            ]
        )
        .to_string(
            index=False
        )
    )

# ================================================================
# 8. Numeric CSV3 Atlanta benchmark inventory
# ================================================================

if len(atlanta3):

    numeric_cols = [
        "Outcome",
        "Benchmark Comparison",
        "County Name",
        "Grouping Type",
        "Conflict Partner Type Group",
        "Benchmark IPMM",
        "Predicted Benchmark Count"
    ]

    numeric_atlanta = (
        atlanta3[numeric_cols]
        .drop_duplicates()
        .sort_values(
            [
                "County Name",
                "Outcome",
                "Benchmark Comparison"
            ]
        )
    )

    print("\n" + "-" * 72)
    print("CSV3 ATLANTA NUMERIC BENCHMARK VALUES")
    print("-" * 72)

    print(
        numeric_atlanta.to_string(
            index=False
        )
    )

# ================================================================
# 9. Identify benchmark variants available in CSV3
# ================================================================

print("\n" + "-" * 72)
print("CSV3 ATLANTA BENCHMARK VARIANTS")
print("-" * 72)

if len(atlanta3):

    variant_table = (
        atlanta3[
            [
                "County Name",
                "Outcome",
                "Benchmark Comparison",
                "Benchmark IPMM",
                "Predicted Benchmark Count"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "County Name",
                "Outcome",
                "Benchmark Comparison"
            ]
        )
    )

    print(
        variant_table.to_string(
            index=False
        )
    )

# ================================================================
# 10. CSV4 Georgia outcome inventory
# ================================================================

print("\n" + "-" * 72)
print("CSV4 GEORGIA OUTCOME DISTRIBUTION")
print("-" * 72)

print(
    ga4["_outcome"]
    .value_counts()
    .to_string()
)

# ================================================================
# 11. Repeated benchmark values by county/outcome
# ================================================================

rep_summary = (
    repeated_ga.groupby(
        [
            "_county",
            "_outcome"
        ]
    )
    .agg(
        repetition_groups=(
            "S2 Cell",
            "count"
        ),
        min_benchmark=(
            "benchmark_min",
            "min"
        ),
        max_benchmark=(
            "benchmark_max",
            "max"
        )
    )
    .reset_index()
)

print("\n" + "-" * 72)
print("CSV4 REPEATED BENCHMARKS — COUNTY × OUTCOME")
print("-" * 72)

print(
    rep_summary.to_string(
        index=False
    )
)

# ================================================================
# 12. Check whether repeated values appear to correspond to
#     CSV3 county-level benchmark values.
#
# This is ONLY a comparison test.
# No value is selected or authorized.
# ================================================================

# Create a compact set of CSV3 numeric values for
# Fulton / DeKalb / Atlanta Area.

csv3_compare = atlanta3[
    [
        "Outcome",
        "Benchmark Comparison",
        "County Name",
        "Benchmark IPMM",
        "Predicted Benchmark Count"
    ]
].copy()

csv3_compare["_county_norm"] = (
    csv3_compare["County Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv3_compare["_outcome_norm"] = (
    csv3_compare["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

csv3_compare["_predicted"] = pd.to_numeric(
    csv3_compare["Predicted Benchmark Count"],
    errors="coerce"
)

# ================================================================
# 13. Direct numerical proximity test
# ================================================================

comparison_rows = []

for _, r in repeated_ga.iterrows():

    county = r["_county"]
    outcome = r["_outcome"]

    mask = (
        (csv3_compare["_county_norm"] == county)
    )

    candidates = csv3_compare[
        mask
    ].copy()

    if len(candidates) == 0:

        comparison_rows.append({

            "county": county,
            "outcome": outcome,
            "s2": r["S2 Cell"],
            "csv4_low": r["benchmark_min"],
            "csv4_high": r["benchmark_max"],
            "csv3_candidate_count": 0,
            "closest_csv3_predicted":
                np.nan,
            "closest_abs_difference_low":
                np.nan,
            "closest_abs_difference_high":
                np.nan

        })

        continue

    pred_values = (
        candidates["_predicted"]
        .dropna()
        .unique()
        .tolist()
    )

    if len(pred_values) == 0:

        comparison_rows.append({

            "county": county,
            "outcome": outcome,
            "s2": r["S2 Cell"],
            "csv4_low": r["benchmark_min"],
            "csv4_high": r["benchmark_max"],
            "csv3_candidate_count": 0,
            "closest_csv3_predicted":
                np.nan,
            "closest_abs_difference_low":
                np.nan,
            "closest_abs_difference_high":
                np.nan

        })

        continue

    low = float(r["benchmark_min"])
    high = float(r["benchmark_max"])

    closest = min(
        pred_values,
        key=lambda x:
            min(
                abs(x - low),
                abs(x - high)
            )
    )

    comparison_rows.append({

        "county": county,
        "outcome": outcome,
        "s2": r["S2 Cell"],
        "csv4_low": low,
        "csv4_high": high,
        "csv3_candidate_count":
            len(pred_values),
        "closest_csv3_predicted":
            float(closest),
        "closest_abs_difference_low":
            float(abs(closest - low)),
        "closest_abs_difference_high":
            float(abs(closest - high))

    })

comparison = pd.DataFrame(
    comparison_rows
)

# ================================================================
# 14. Print proximity summary
# ================================================================

print("\n" + "-" * 72)
print("CSV4 ↔ CSV3 NUMERICAL PROXIMITY TEST")
print("-" * 72)

if len(comparison):

    print(
        comparison[
            [
                "county",
                "outcome",
                "csv4_low",
                "csv4_high",
                "closest_csv3_predicted",
                "closest_abs_difference_low",
                "closest_abs_difference_high"
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

# ================================================================
# 15. Determine exact equality counts
# ================================================================

exact_low_matches = 0
exact_high_matches = 0

for _, r in comparison.iterrows():

    if pd.notna(
        r["closest_csv3_predicted"]
    ):

        p = r["closest_csv3_predicted"]

        if np.isclose(
            p,
            r["csv4_low"],
            rtol=1e-9,
            atol=1e-9
        ):
            exact_low_matches += 1

        if np.isclose(
            p,
            r["csv4_high"],
            rtol=1e-9,
            atol=1e-9
        ):
            exact_high_matches += 1

print("\n" + "-" * 72)
print("EXACT NUMERICAL MATCH COUNTS")
print("-" * 72)

print(
    "CSV3 value exactly matching CSV4 LOW:",
    exact_low_matches
)

print(
    "CSV3 value exactly matching CSV4 HIGH:",
    exact_high_matches
)

# ================================================================
# 16. Important boundary interpretation
# ================================================================

# Even if numerical equality is observed, it does NOT authorize
# identifying the value's provenance unless the documentary
# definition establishes that relationship.

numerical_overlap_observed = (
    exact_low_matches > 0
    or exact_high_matches > 0
)

provenance_resolved = False

boundary = {

    "georgia_repetition_localized":
        True,

    "atlanta_fulton_dekalb_context_tested":
        True,

    "csv3_csv4_numeric_overlap_tested":
        True,

    "numerical_overlap_observed":
        bool(numerical_overlap_observed),

    "benchmark_repetition_provenance_resolved":
        False,

    "benchmark_value_selection_authorized":
        False,

    "benchmark_aggregation_authorized":
        False,

    "row_deletion_authorized":
        False,

    "deduplication_authorized":
        False,

    "semantic_equivalence_authorized":
        False,

    "claim_authorization":
        False,

    "scientific_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED"
}

status = "PENDING_EVIDENCE"

# ================================================================
# 17. Artifact
# ================================================================

outdir = (
    "/kaggle/working/"
    "OIP_v1_0_39_C08R26"
)

os.makedirs(
    outdir,
    exist_ok=True
)

artifact = {

    "cell":
        "C08R26",

    "purpose":
        "Georgia/Atlanta CSV3-CSV4 benchmark provenance audit",

    "csv3_path":
        csv3_path,

    "csv4_path":
        csv4_path,

    "csv4_georgia_rows":
        int(len(ga4)),

    "georgia_repetition_groups":
        int(len(repeated_ga)),

    "csv3_atlanta_related_rows":
        int(len(atlanta3)),

    "csv3_county_labels":
        county3.to_dict(),

    "exact_csv3_csv4_numeric_overlap": {

        "low_matches":
            int(exact_low_matches),

        "high_matches":
            int(exact_high_matches)
    },

    "numerical_overlap_observed":
        bool(numerical_overlap_observed),

    "comparison_preview":
        comparison.head(100).to_dict(
            orient="records"
        ),

    "boundary":
        boundary,

    "benchmark_repetition_provenance_resolved":
        provenance_resolved,

    "status":
        status
}

artifact_path = os.path.join(
    outdir,
    "C08R26_GEORGIA_ATLANTA_PROVENANCE_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

# ================================================================
# 18. Final output
# ================================================================

print("\n" + "=" * 72)
print("FINAL BOUNDARY")
print("=" * 72)

for k, v in boundary.items():

    print(
        f"{k:55} : {v}"
    )

print(
    "\nC08R26 STATUS :",
    status
)

print(
    "\nArtifact:"
)

print(
    artifact_path
)

print("\n" + "=" * 72)
print("C08R26 EXECUTION COMPLETE")
print("=" * 72)

OIP v1.0.39 — C08R26
GEORGIA / ATLANTA BENCHMARK CROSS-FILE PROVENANCE AUDIT

CSV3:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv

CSV4:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

CSV4 Georgia rows: 2472
Georgia benchmark repetition groups: 601

------------------------------------------------------------------------
CSV3 COUNTY LABELS
------------------------------------------------------------------------
_county_name
all locations (mileage blended)    168
atlanta area                       168
fulton                             168
los angeles                        168
san francisco bay area             166
maricopa                           166
san francisco                      166
travis                             164
san mateo                          164
d

In [99]:
# ================================================================
# OIP v1.0.39 — C08R27
# GEORGIA / ATLANTA BENCHMARK SOURCE PROVENANCE DOCUMENTARY AUDIT
# ================================================================

import os
import re
import json
from pathlib import Path
from pypdf import PdfReader

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R27"
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------
# Locate PDFs
# ------------------------------------------------
pdfs = sorted(Path(BASE).glob("*.pdf"))

print("=" * 70)
print("OIP v1.0.39 — C08R27")
print("GEORGIA / ATLANTA BENCHMARK SOURCE PROVENANCE AUDIT")
print("=" * 70)

print("\nPDF FILES:")
for p in pdfs:
    print("-", p.name)

# ------------------------------------------------
# Extract PDF text page-by-page
# ------------------------------------------------
documents = {}

for pdf in pdfs:
    try:
        reader = PdfReader(str(pdf))
        pages = []

        for i, page in enumerate(reader.pages, start=1):
            try:
                txt = page.extract_text() or ""
            except Exception:
                txt = ""
            pages.append({
                "page": i,
                "text": txt
            })

        documents[pdf.name] = pages

    except Exception as e:
        print("PDF READ ERROR:", pdf.name, str(e))

# ------------------------------------------------
# Targeted documentary terms
# ------------------------------------------------
terms = [
    "Atlanta",
    "Fulton",
    "DeKalb",
    "Georgia",
    "Scanlon",
    "2022",
    "dynamic benchmark",
    "dynamic adjustment",
    "human benchmark",
    "benchmark source",
    "benchmark crash",
    "crash data",
    "state crash",
    "county",
    "geographic distribution",
    "S2 Cell",
    "CSV4",
    "CSV3"
]

results = {}

for pdf_name, pages in documents.items():

    results[pdf_name] = {}

    for term in terms:

        hits = []

        for item in pages:
            text_lower = item["text"].lower()

            if term.lower() in text_lower:
                hits.append(item["page"])

        results[pdf_name][term] = sorted(set(hits))

# ------------------------------------------------
# Print documentary index
# ------------------------------------------------
print("\n" + "=" * 70)
print("DOCUMENTARY TERM INDEX")
print("=" * 70)

for pdf_name, term_map in results.items():

    print("\nFILE:", pdf_name)

    for term, pages in term_map.items():

        if pages:
            print(f"  {term} -> pages {pages}")

# ------------------------------------------------
# Extract context around critical Georgia terms
# ------------------------------------------------
critical_terms = [
    "Scanlon",
    "Atlanta",
    "Fulton",
    "DeKalb",
    "dynamic benchmark",
    "benchmark source",
    "state crash",
    "2022"
]

print("\n" + "=" * 70)
print("CRITICAL DOCUMENTARY CONTEXT")
print("=" * 70)

context_records = []

for pdf_name, pages in documents.items():

    for item in pages:

        text = item["text"]

        lower = text.lower()

        matched = [
            t for t in critical_terms
            if t.lower() in lower
        ]

        if not matched:
            continue

        # print compact page context
        clean = re.sub(r"\s+", " ", text).strip()

        if len(clean) > 1800:
            clean = clean[:1800] + " ..."

        record = {
            "file": pdf_name,
            "page": item["page"],
            "matched_terms": matched,
            "context": clean
        }

        context_records.append(record)

        print("\n--------------------------------------------------")
        print("FILE :", pdf_name)
        print("PAGE :", item["page"])
        print("MATCH:", matched)
        print("CONTEXT:")
        print(clean)

# ------------------------------------------------
# Specific provenance indicators
# ------------------------------------------------
print("\n" + "=" * 70)
print("PROVENANCE INDICATOR TEST")
print("=" * 70)

all_text = "\n".join(
    item["text"]
    for pages in documents.values()
    for item in pages
).lower()

indicator_patterns = {
    "atlanta_source_named":
        ["scanlon et al", "scanlon"],

    "atlanta_2022_vintage":
        ["2022"],

    "fulton_dekalb_combination":
        ["fulton", "dekalb"],

    "georgia_state_crash_source":
        ["georgia state crash", "state crash data"],

    "dynamic_benchmark_method":
        ["dynamic benchmark", "dynamic adjustment"],

    "spatial_dynamic_method":
        ["geographic distribution", "s2 cell"],

    "csv4_role":
        ["csv4", "spatial distribution"],

    "benchmark_source_language":
        ["benchmark source", "human benchmark", "crash data source"]
}

indicator_results = {}

for key, patterns in indicator_patterns.items():

    hits = {}

    for pattern in patterns:
        hits[pattern] = pattern in all_text

    indicator_results[key] = hits

    print(f"\n{key}:")
    for pattern, found in hits.items():
        print(f"  {pattern}: {found}")

# ------------------------------------------------
# Conservative interpretation
# ------------------------------------------------
atlanta_source = (
    indicator_results["atlanta_source_named"].get("scanlon et al", False)
    or indicator_results["atlanta_source_named"].get("scanlon", False)
)

atlanta_2022 = indicator_results["atlanta_2022_vintage"].get("2022", False)

georgia_source = any(
    indicator_results["georgia_state_crash_source"].values()
)

dynamic_method = any(
    indicator_results["dynamic_benchmark_method"].values()
)

spatial_method = any(
    indicator_results["spatial_dynamic_method"].values()
)

csv4_context = any(
    indicator_results["csv4_role"].values()
)

# IMPORTANT:
# Documentary presence does NOT automatically resolve
# the exact reason for duplicate benchmark values.
#
# Therefore provenance is resolved only if the documentation
# explicitly links the repeated Georgia benchmark values
# to a documented source/method/variant.

explicit_repetition_explanation = any(
    phrase in all_text
    for phrase in [
        "duplicate benchmark",
        "repeated benchmark",
        "multiple benchmark",
        "two benchmark",
        "benchmark values",
        "benchmark value",
        "county-specific benchmark",
        "county specific benchmark",
        "fulton and dekalb",
        "fulton/dekalb"
    ]
)

print("\n" + "=" * 70)
print("FINAL PROVENANCE BOUNDARY")
print("=" * 70)

print("atlanta_source_documented                 :", atlanta_source)
print("atlanta_2022_vintage_documented           :", atlanta_2022)
print("georgia_crash_source_documented           :", georgia_source)
print("dynamic_benchmark_method_documented       :", dynamic_method)
print("spatial_dynamic_method_documented         :", spatial_method)
print("csv4_context_documented                   :", csv4_context)
print("explicit_repetition_explanation_found     :", explicit_repetition_explanation)

# Conservative rule:
# Merely finding the source/method is NOT enough.
# The specific repeated-value provenance must be explicit.

provenance_resolved = (
    atlanta_source
    and atlanta_2022
    and dynamic_method
    and spatial_method
    and explicit_repetition_explanation
)

print("\nbenchmark_repetition_provenance_resolved :", provenance_resolved)
print("benchmark_value_selection_authorized    : False")
print("benchmark_aggregation_authorized        : False")
print("row_deletion_authorized                 : False")
print("deduplication_authorized                : False")
print("semantic_equivalence_authorized         : False")
print("claim_authorization                     : False")
print("scientific_authorization                : False")
print("road_safety_verdict                     : NOT_ISSUED")

status = (
    "PASS_C08_PROVENANCE_RESOLVED"
    if provenance_resolved
    else "PENDING_EVIDENCE"
)

print("\nC08R27 STATUS :", status)

# ------------------------------------------------
# Save artifact
# ------------------------------------------------
artifact = {
    "cell": "C08R27",
    "purpose": "Georgia / Atlanta benchmark source provenance documentary audit",
    "indicator_results": indicator_results,
    "explicit_repetition_explanation_found":
        explicit_repetition_explanation,
    "benchmark_repetition_provenance_resolved":
        provenance_resolved,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_equivalence_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "status": status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R27_GEORGIA_ATLANTA_PROVENANCE_DOCUMENTARY_AUDIT.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R27 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R27
GEORGIA / ATLANTA BENCHMARK SOURCE PROVENANCE AUDIT

PDF FILES:
- 2312.12675v3.pdf
- 2312.13228v2.pdf
- Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf

DOCUMENTARY TERM INDEX

FILE: 2312.12675v3.pdf
  Scanlon -> pages [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 20, 21, 23]
  2022 -> pages [5, 6, 11, 16, 17, 23]
  human benchmark -> pages [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
  benchmark source -> pages [5, 6, 7]
  benchmark crash -> pages [4, 5, 8, 9, 10, 12, 21]
  crash data -> pages [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
  state crash -> pages [3, 6]

FILE: 2312.13228v2.pdf
  Scanlon -> pages [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
  2022 -> pages [2, 3, 8, 11, 14, 15, 16, 17, 18, 19, 23, 25, 26]
  human benchmark -> pages [6, 14, 16, 21, 29]
  benchmark crash -> pages [5, 9, 21]
  cra

In [100]:
# ================================================================
# OIP v1.0.39 — C08R27R1
# STRICT REPEATED-BENCHMARK PROVENANCE TEST
# ================================================================

import os
import re
import json
from pathlib import Path
from pypdf import PdfReader

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R27R1"
os.makedirs(OUTDIR, exist_ok=True)

release_pdf = Path(BASE) / "Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf"

reader = PdfReader(str(release_pdf))

pages = {}
for i, page in enumerate(reader.pages, start=1):
    pages[i] = page.extract_text() or ""

all_text = "\n".join(pages.values()).lower()

print("=" * 70)
print("OIP v1.0.39 — C08R27R1")
print("STRICT REPEATED-BENCHMARK PROVENANCE TEST")
print("=" * 70)

# ------------------------------------------------
# 1. Known documented facts
# ------------------------------------------------

known_patterns = {
    "atlanta": ["atlanta"],
    "fulton": ["fulton"],
    "dekalb": ["dekalb"],
    "scanlon_source": ["scanlon et al"],
    "2022": ["2022"],
    "dynamic_benchmark": ["dynamic benchmark"],
    "dynamic_adjustment": ["dynamic adjustment"],
    "geographic_distribution": ["geographic distribution"],
    "s2_cell": ["s2 cell"],
    "csv4": ["csv4"],
    "benchmark_crash_count": ["benchmark crash count"],
}

print("\nDOCUMENTED SOURCE/METHOD FACTS")
print("-" * 70)

known_results = {}

for key, patterns in known_patterns.items():

    found = any(p in all_text for p in patterns)

    known_results[key] = found

    print(f"{key:30s}: {found}")

# ------------------------------------------------
# 2. STRICT test:
# Does documentation explicitly explain repeated
# benchmark values for the SAME S2 Cell + Outcome?
# ------------------------------------------------

strict_patterns = [
    "duplicate benchmark",
    "duplicate benchmarks",
    "repeated benchmark",
    "repeated benchmarks",
    "multiple benchmark values",
    "multiple benchmark value",
    "two benchmark values",
    "two benchmark value",
    "different benchmark values",
    "different benchmark value",
    "same s2 cell",
    "same s2 cell and outcome",
    "same s2 cell + outcome",
    "multiple rows for the same s2",
    "multiple rows for each s2",
    "county-specific benchmark values",
    "county specific benchmark values",
    "benchmark value duplicated",
    "benchmark values duplicated",
    "benchmark crash count duplicated",
    "benchmark crash counts duplicated",
]

print("\n" + "=" * 70)
print("STRICT REPETITION EXPLANATION TEST")
print("=" * 70)

strict_hits = {}

for phrase in strict_patterns:

    found = phrase in all_text

    strict_hits[phrase] = found

    if found:
        print("FOUND:", phrase)

if not any(strict_hits.values()):
    print("NO EXPLICIT REPEATED-BENCHMARK EXPLANATION FOUND.")

# ------------------------------------------------
# 3. Search pages specifically for benchmark-count
# language and print contexts
# ------------------------------------------------

target_terms = [
    "Benchmark Crash Count",
    "replicated across Outcome levels",
    "dynamic benchmark",
    "geographic distribution",
    "Atlanta",
    "Fulton",
    "DeKalb"
]

print("\n" + "=" * 70)
print("TARGETED PAGE CONTEXT")
print("=" * 70)

context_records = []

for page_no, text in pages.items():

    lower = text.lower()

    matched = [
        term for term in target_terms
        if term.lower() in lower
    ]

    if not matched:
        continue

    clean = re.sub(r"\s+", " ", text).strip()

    # Keep contexts manageable
    if len(clean) > 2200:
        clean = clean[:2200] + " ..."

    context_records.append({
        "page": page_no,
        "matched": matched,
        "context": clean
    })

    print("\nPAGE:", page_no)
    print("MATCHED:", matched)
    print(clean)

# ------------------------------------------------
# 4. Critical distinction
# ------------------------------------------------

print("\n" + "=" * 70)
print("OIP EVIDENCE DISTINCTION")
print("=" * 70)

print("""
DOCUMENTED:
- benchmark source
- Atlanta / Georgia context
- 2022 benchmark vintage
- dynamic benchmark methodology
- geographic / S2-cell methodology
- CSV4 benchmark crash count definition

NOT AUTOMATICALLY DOCUMENTED:
- why the SAME State + County + S2 Cell + Outcome
  can contain TWO different Benchmark Crash Count values
- which of those repeated values should be selected
- whether repeated values represent different source datasets,
  geographic aggregation, benchmark versions, or another mechanism
- whether any row should be deleted or merged
""")

# ------------------------------------------------
# 5. Strict provenance decision
# ------------------------------------------------

# Provenance can close ONLY if documentation explicitly
# explains the repeated benchmark-value structure.

strict_provenance_resolved = any(strict_hits.values())

print("=" * 70)
print("FINAL STRICT BOUNDARY")
print("=" * 70)

print("source_method_documented                 :",
      all(known_results.values()))

print("repeated_benchmark_explanation_found     :",
      strict_provenance_resolved)

print("benchmark_repetition_provenance_resolved :",
      strict_provenance_resolved)

print("benchmark_value_selection_authorized     : False")
print("benchmark_aggregation_authorized         : False")
print("row_deletion_authorized                  : False")
print("deduplication_authorized                 : False")
print("semantic_equivalence_authorized          : False")
print("claim_authorization                      : False")
print("scientific_authorization                 : False")
print("road_safety_verdict                      : NOT_ISSUED")

status = (
    "PASS_C08_PROVENANCE_RESOLVED"
    if strict_provenance_resolved
    else "PENDING_EVIDENCE"
)

print("\nC08R27R1 STATUS :", status)

# ------------------------------------------------
# 6. Artifact
# ------------------------------------------------

artifact = {
    "cell": "C08R27R1",
    "purpose": "Strict repeated benchmark provenance test",
    "known_documented_facts": known_results,
    "strict_repetition_hits": strict_hits,
    "repeated_benchmark_explanation_found":
        strict_provenance_resolved,
    "benchmark_repetition_provenance_resolved":
        strict_provenance_resolved,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_equivalence_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "status": status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R27R1_STRICT_REPEATED_BENCHMARK_PROVENANCE.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R27R1 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R27R1
STRICT REPEATED-BENCHMARK PROVENANCE TEST

DOCUMENTED SOURCE/METHOD FACTS
----------------------------------------------------------------------
atlanta                       : True
fulton                        : True
dekalb                        : True
scanlon_source                : True
2022                          : True
dynamic_benchmark             : True
dynamic_adjustment            : True
geographic_distribution       : True
s2_cell                       : True
csv4                          : True
benchmark_crash_count         : False

STRICT REPETITION EXPLANATION TEST
NO EXPLICIT REPEATED-BENCHMARK EXPLANATION FOUND.

TARGETED PAGE CONTEXT

PAGE: 2
MATCHED: ['dynamic benchmark', 'Atlanta', 'Fulton', 'DeKalb']
Release Notes — Publish Date Data End Date (Year Month) NHTSA SGO Data Publish Date Waymo Data Hub Files Release Description September 24, 2026 2026 June August 17, 2026 CSV1 - RO Miles per Location CSV2 - Crashes with SGO identifier and group 

In [101]:
# ================================================================
# OIP v1.0.39 — C08R28
# REPEATED BENCHMARK vs RELEASE / METHODOLOGY CHANGE AUDIT
# ================================================================

import os
import json
import re
import pandas as pd
from pathlib import Path
from pypdf import PdfReader

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R28"
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------
# Locate files
# ------------------------------------------------
csv3 = next(Path(BASE).glob("*CSV3*"))
csv4 = next(Path(BASE).glob("*CSV4*"))
release_pdf = next(Path(BASE).glob("*Release_Notes*.pdf"))

df3 = pd.read_csv(csv3)
df4 = pd.read_csv(csv4)

print("=" * 70)
print("OIP v1.0.39 — C08R28")
print("REPEATED BENCHMARK vs RELEASE / METHODOLOGY CHANGE AUDIT")
print("=" * 70)

print("\nCSV3:", csv3.name)
print("CSV4:", csv4.name)

# ------------------------------------------------
# 1. Reconstruct repeated benchmark groups
# ------------------------------------------------
required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [c for c in required if c not in df4.columns]

print("\nCSV4 REQUIRED COLUMNS MISSING:", missing)

if missing:
    raise RuntimeError("C08R28 FAIL_CLOSED: required CSV4 columns missing.")

df4["_state"] = df4["state_name"].astype(str).str.strip().str.lower()
df4["_county"] = df4["county_name"].astype(str).str.strip().str.lower()
df4["_s2"] = df4["S2 Cell"].astype(str).str.strip()
df4["_outcome"] = df4["Outcome"].astype(str).str.strip().str.lower()

grain = ["_state", "_county", "_s2", "_outcome"]

group_sizes = df4.groupby(grain).size()

repeated_keys = group_sizes[group_sizes > 1].index

repeated = (
    df4.set_index(grain)
       .loc[repeated_keys]
       .reset_index()
       .copy()
)

print("\nTOTAL CSV4 ROWS:", len(df4))
print("REPEATED GRAIN GROUPS:", len(repeated_keys))
print("ROWS INSIDE REPEATED GROUPS:", len(repeated))

# ------------------------------------------------
# 2. Confirm benchmark-only variation
# ------------------------------------------------
variation_records = []

for key, g in repeated.groupby(grain, dropna=False):

    bench = g["Benchmark Crash Count"].dropna().unique()

    vmt = g["HPMS Yearly Vehicle Miles Traveled"].dropna().unique()
    waymo = g["Waymo RO Miles"].dropna().unique()

    if len(bench) > 1:

        variation_records.append({
            "state": key[0],
            "county": key[1],
            "s2": key[2],
            "outcome": key[3],
            "benchmark_values": sorted(
                [float(x) for x in bench]
            ),
            "vmt_values": sorted(
                [float(x) for x in vmt]
            ),
            "waymo_miles_values": sorted(
                [float(x) for x in waymo]
            )
        })

variation_df = pd.DataFrame(variation_records)

print("\nBENCHMARK-ONLY VARIATION GROUPS:",
      len(variation_df))

# ------------------------------------------------
# 3. Load Release Notes page-by-page
# ------------------------------------------------
reader = PdfReader(str(release_pdf))

pages = {}

for i, page in enumerate(reader.pages, start=1):
    pages[i] = page.extract_text() or ""

# ------------------------------------------------
# 4. Search for methodology/version change signals
# ------------------------------------------------
search_terms = [
    "benchmark",
    "dynamic",
    "geographic",
    "spatial",
    "adjustment",
    "Atlanta",
    "Georgia",
    "Fulton",
    "DeKalb",
    "2022",
    "release",
    "methodology",
    "updated",
    "revised",
    "revision",
    "source",
    "data source",
    "benchmark crash count",
    "CSV4",
    "S2 Cell"
]

print("\n" + "=" * 70)
print("RELEASE / METHODOLOGY SEARCH")
print("=" * 70)

term_hits = {}

for term in search_terms:

    hits = []

    for page_no, text in pages.items():

        if term.lower() in text.lower():
            hits.append(page_no)

    term_hits[term] = sorted(set(hits))

    print(f"{term:25s} -> {term_hits[term]}")

# ------------------------------------------------
# 5. Search specifically for phrases suggesting
# benchmark replacement/versioning/recalculation
# ------------------------------------------------
strict_terms = [
    "benchmark was updated",
    "benchmark was revised",
    "benchmarks were updated",
    "benchmarks were revised",
    "benchmark values were updated",
    "benchmark values were revised",
    "benchmark recalculated",
    "benchmarks recalculated",
    "benchmark replaced",
    "benchmark values replaced",
    "new benchmark values",
    "previous benchmark",
    "previous benchmarks",
    "old benchmark",
    "new benchmark",
    "benchmark version",
    "benchmark release",
    "different benchmark",
    "multiple benchmark",
    "multiple benchmarks",
    "benchmark count changed",
    "benchmark crash count changed",
    "benchmark crash counts",
    "historical benchmark"
]

print("\n" + "=" * 70)
print("STRICT BENCHMARK VERSION / REPLACEMENT SEARCH")
print("=" * 70)

strict_hits = {}

for term in strict_terms:

    pages_found = []

    for page_no, text in pages.items():

        if term.lower() in text.lower():
            pages_found.append(page_no)

    strict_hits[term] = pages_found

    if pages_found:
        print(f"{term:35s} -> {pages_found}")

if not any(strict_hits.values()):
    print("NO EXPLICIT BENCHMARK VERSION / REPLACEMENT LANGUAGE FOUND.")

# ------------------------------------------------
# 6. Print relevant release-note contexts
# ------------------------------------------------
print("\n" + "=" * 70)
print("RELEVANT RELEASE-NOTE CONTEXT")
print("=" * 70)

context_terms = [
    "benchmark",
    "methodology",
    "dynamic adjustment",
    "Atlanta",
    "Georgia",
    "CSV4"
]

printed_pages = set()

for page_no, text in pages.items():

    lower = text.lower()

    if not any(t.lower() in lower for t in context_terms):
        continue

    clean = re.sub(r"\s+", " ", text).strip()

    if len(clean) > 1800:
        clean = clean[:1800] + " ..."

    print("\nPAGE:", page_no)
    print(clean)

# ------------------------------------------------
# 7. Conservative OIP decision
# ------------------------------------------------
explicit_version_change = any(
    len(v) > 0 for v in strict_hits.values()
)

print("\n" + "=" * 70)
print("C08R28 FINAL BOUNDARY")
print("=" * 70)

print("csv4_total_rows:",
      len(df4))

print("benchmark_repetition_groups:",
      len(variation_df))

print("benchmark_only_variation:",
      len(variation_df))

print("release_methodology_terms_found:",
      sum(bool(v) for v in term_hits.values()))

print("explicit_benchmark_version_or_replacement:",
      explicit_version_change)

print("""
IMPORTANT:
Documented methodology change ≠ explanation of the 601 repeated
benchmark values.

Only an explicit documentary link between a specific change and
the repeated State + County + S2 Cell + Outcome benchmark values
can resolve provenance.
""")

provenance_resolved = False

# Fail-closed:
# Even if release notes mention benchmark updates, do not authorize
# unless the documentation explicitly links the update to the
# repeated benchmark-value structure.

print("benchmark_repetition_provenance_resolved :",
      provenance_resolved)

print("benchmark_value_selection_authorized    : False")
print("benchmark_aggregation_authorized        : False")
print("row_deletion_authorized                 : False")
print("deduplication_authorized                : False")
print("semantic_equivalence_authorized         : False")
print("claim_authorization                     : False")
print("scientific_authorization                : False")
print("road_safety_verdict                     : NOT_ISSUED")

status = "PENDING_EVIDENCE"

print("\nC08R28 STATUS :", status)

# ------------------------------------------------
# 8. Artifact
# ------------------------------------------------
artifact = {
    "cell": "C08R28",
    "purpose":
        "Audit repeated benchmark values against release and methodology changes",
    "csv4_total_rows": int(len(df4)),
    "benchmark_repetition_groups": int(len(variation_df)),
    "benchmark_only_variation_groups": int(len(variation_df)),
    "term_hits": term_hits,
    "strict_benchmark_version_hits": strict_hits,
    "explicit_benchmark_version_or_replacement":
        bool(explicit_version_change),
    "benchmark_repetition_provenance_resolved":
        False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_equivalence_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "status": status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R28_RELEASE_METHODOLOGY_REPETITION_AUDIT.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R28 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R28
REPEATED BENCHMARK vs RELEASE / METHODOLOGY CHANGE AUDIT

CSV3: CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv
CSV4: CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

CSV4 REQUIRED COLUMNS MISSING: []

TOTAL CSV4 ROWS: 18352
REPEATED GRAIN GROUPS: 680
ROWS INSIDE REPEATED GROUPS: 1360

BENCHMARK-ONLY VARIATION GROUPS: 601

RELEASE / METHODOLOGY SEARCH
benchmark                 -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17, 18]
dynamic                   -> [2, 3, 4, 5, 6, 10, 11, 14, 17]
geographic                -> [5, 6, 10, 12, 17]
spatial                   -> [3, 6, 10, 11]
adjustment                -> [2, 3, 5, 6, 10]
Atlanta                   -> [2, 4, 12]
Georgia                   -> [2]
Fulton                    -> [2]
DeKalb                    -> [2]
2022                      -> [2]
release                   -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 17]
methodology       

In [102]:
# ================================================================
# OIP v1.0.39 — C08R29
# HISTORICAL RELEASE / VERSION TRACEABILITY TEST
# ================================================================

import os
import json
import re
from pathlib import Path
import pandas as pd
from pypdf import PdfReader

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R29"
os.makedirs(OUTDIR, exist_ok=True)

csv4_path = next(Path(BASE).glob("*CSV4*"))
release_pdf = next(Path(BASE).glob("*Release_Notes*.pdf"))

df4 = pd.read_csv(csv4_path)
reader = PdfReader(str(release_pdf))

print("=" * 70)
print("OIP v1.0.39 — C08R29")
print("HISTORICAL RELEASE / VERSION TRACEABILITY TEST")
print("=" * 70)

# ------------------------------------------------
# 1. Current repeated benchmark structure
# ------------------------------------------------

df4["_state"] = df4["state_name"].astype(str).str.strip().str.lower()
df4["_county"] = df4["county_name"].astype(str).str.strip().str.lower()
df4["_s2"] = df4["S2 Cell"].astype(str).str.strip()
df4["_outcome"] = df4["Outcome"].astype(str).str.strip().str.lower()

grain = ["_state", "_county", "_s2", "_outcome"]

group_sizes = df4.groupby(grain).size()

repeated_keys = group_sizes[group_sizes > 1].index

repeated = (
    df4.set_index(grain)
       .loc[repeated_keys]
       .reset_index()
)

benchmark_variation = []

for key, g in repeated.groupby(grain, dropna=False):

    values = sorted(
        g["Benchmark Crash Count"]
        .dropna()
        .astype(float)
        .unique()
        .tolist()
    )

    if len(values) > 1:
        benchmark_variation.append({
            "state": key[0],
            "county": key[1],
            "s2": key[2],
            "outcome": key[3],
            "benchmark_values": values
        })

variation_df = pd.DataFrame(benchmark_variation)

print("\nCURRENT CSV4")
print("Total rows:", len(df4))
print("Repeated grain groups:", len(repeated_keys))
print("Benchmark-only variation groups:", len(variation_df))

# ------------------------------------------------
# 2. Extract historical release table pages
# ------------------------------------------------

pages = {}

for i, page in enumerate(reader.pages, start=1):
    pages[i] = page.extract_text() or ""

release_rows = []

for page_no, text in pages.items():

    # Find release-date-like structures
    matches = re.findall(
        r"(September|June|March|December|October|January|"
        r"August|July|May|April|February|November)\s+\d{1,2},\s+\d{4}",
        text,
        flags=re.I
    )

    if matches:
        release_rows.append({
            "page": page_no,
            "date_signal_count": len(matches),
            "text": re.sub(r"\s+", " ", text).strip()
        })

print("\nHISTORICAL RELEASE PAGES:", len(release_rows))

for row in release_rows:
    print(
        f"Page {row['page']} | "
        f"date signals={row['date_signal_count']}"
    )

# ------------------------------------------------
# 3. Search specifically for version/change language
# ------------------------------------------------

trace_terms = [
    "previous release",
    "prior release",
    "last release",
    "historical version",
    "historical versions",
    "updated data",
    "new data",
    "new methodology",
    "methodology changed",
    "methodology change",
    "benchmark added",
    "benchmark removed",
    "benchmark updated",
    "benchmark revised",
    "benchmark recalculated",
    "dynamic adjustment",
    "geographic adjustment",
    "Atlanta",
    "Fulton",
    "DeKalb",
    "2022",
    "CSV4",
    "S2 Cell",
    "Benchmark Crash Count",
    "historical benchmark",
    "previous benchmark"
]

trace_hits = {}

for term in trace_terms:

    found = []

    for page_no, text in pages.items():

        if term.lower() in text.lower():
            found.append(page_no)

    trace_hits[term] = sorted(set(found))

print("\n" + "=" * 70)
print("HISTORICAL TRACEABILITY SEARCH")
print("=" * 70)

for term, hits in trace_hits.items():

    if hits:
        print(f"{term:30s} -> {hits}")

# ------------------------------------------------
# 4. Strict linkage test
#
# We require documentary language that explicitly
# connects historical/version changes to the SAME
# State + County + S2 Cell + Outcome benchmark
# values.
# ------------------------------------------------

strict_link_terms = [
    "same S2 cell",
    "same S2 cells",
    "repeated benchmark",
    "repeated benchmarks",
    "duplicate benchmark",
    "duplicate benchmarks",
    "multiple benchmark values",
    "multiple benchmark counts",
    "benchmark value duplication",
    "benchmark count duplication",
    "benchmark count repeated",
    "benchmark values repeated",
    "two benchmark values",
    "two benchmark counts",
    "one S2 cell",
    "each S2 cell",
    "benchmark source changed",
    "benchmark data source changed",
    "benchmark dataset changed",
    "benchmark version changed"
]

strict_hits = {}

for term in strict_link_terms:

    found = []

    for page_no, text in pages.items():

        if term.lower() in text.lower():
            found.append(page_no)

    strict_hits[term] = sorted(set(found))

print("\n" + "=" * 70)
print("STRICT SAME-GRAIN LINKAGE TEST")
print("=" * 70)

strict_found = False

for term, hits in strict_hits.items():

    if hits:
        strict_found = True
        print(f"{term:35s} -> {hits}")

if not strict_found:
    print(
        "NO DOCUMENTARY LINK FOUND BETWEEN HISTORICAL "
        "RELEASE CHANGES AND THE SAME-GRAIN REPEATED "
        "BENCHMARK VALUES."
    )

# ------------------------------------------------
# 5. Important documentary distinction
# ------------------------------------------------

print("\n" + "=" * 70)
print("OIP EVIDENCE DISTINCTION")
print("=" * 70)

print("""
DOCUMENTED:
- Waymo Data Hub is a living dataset.
- Historical releases exist.
- Data and methodologies can change over time.
- Dynamic benchmark methodology was introduced.
- Geographic/S2-cell data were introduced for replication.
- Atlanta benchmarks were added using 2022 benchmark data.
- CSV4 contains Benchmark Crash Count.

NOT DOCUMENTED:
- Which historical release produced each of the two
  repeated Benchmark Crash Count values.
- Whether the two values correspond to different
  benchmark versions.
- Whether they correspond to different source datasets.
- Whether they are geographic aggregation artifacts.
- Whether one value supersedes the other.
- Whether both values are intentionally retained.
- Which value should be selected for calculation.
""")

# ------------------------------------------------
# 6. Final fail-closed boundary
# ------------------------------------------------

provenance_resolved = False

print("\n" + "=" * 70)
print("C08R29 FINAL BOUNDARY")
print("=" * 70)

print("csv4_total_rows:", len(df4))
print("benchmark_repetition_groups:", len(repeated_keys))
print("benchmark_only_variation_groups:", len(variation_df))

print(
    "historical_release_context_documented :",
    bool(release_rows)
)

print(
    "strict_same_grain_linkage_found :",
    strict_found
)

print(
    "benchmark_repetition_provenance_resolved :",
    provenance_resolved
)

print("benchmark_value_selection_authorized : False")
print("benchmark_aggregation_authorized     : False")
print("row_deletion_authorized              : False")
print("deduplication_authorized             : False")
print("semantic_equivalence_authorized      : False")
print("claim_authorization                  : False")
print("scientific_authorization             : False")
print("road_safety_verdict                  : NOT_ISSUED")

status = "PENDING_EVIDENCE"

print("\nC08R29 STATUS :", status)

# ------------------------------------------------
# 7. Artifact
# ------------------------------------------------

artifact = {
    "cell": "C08R29",
    "purpose":
        "Historical release/version traceability test for repeated benchmark values",
    "csv4_total_rows": int(len(df4)),
    "repeated_grain_groups": int(len(repeated_keys)),
    "benchmark_only_variation_groups": int(len(variation_df)),
    "historical_release_context_documented": bool(release_rows),
    "trace_terms": trace_hits,
    "strict_same_grain_linkage_terms": strict_hits,
    "strict_same_grain_linkage_found": bool(strict_found),
    "benchmark_repetition_provenance_resolved": False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_equivalence_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "status": status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R29_HISTORICAL_RELEASE_TRACEABILITY.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R29 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R29
HISTORICAL RELEASE / VERSION TRACEABILITY TEST

CURRENT CSV4
Total rows: 18352
Repeated grain groups: 680
Benchmark-only variation groups: 601

HISTORICAL RELEASE PAGES: 14
Page 2 | date signals=5
Page 3 | date signals=7
Page 4 | date signals=3
Page 5 | date signals=9
Page 6 | date signals=6
Page 7 | date signals=2
Page 8 | date signals=4
Page 9 | date signals=1
Page 12 | date signals=10
Page 13 | date signals=8
Page 14 | date signals=10
Page 15 | date signals=12
Page 16 | date signals=9
Page 17 | date signals=8

HISTORICAL TRACEABILITY SEARCH
previous release               -> [6, 9]
last release                   -> [9]
historical version             -> [1]
historical versions            -> [1]
updated data                   -> [2, 3, 4, 5, 6]
new data                       -> [1, 6, 9]
methodology change             -> [9]
dynamic adjustment             -> [3, 6]
geographic adjustment          -> [5, 6]
Atlanta                        -> [2, 4, 12]
Fulton         

In [103]:
# ================================================================
# OIP v1.0.39 — C08R30
# REPEATED BENCHMARK PAIR ↔ DOCUMENTED CHANGE PATTERN AUDIT
# ================================================================

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R30"
os.makedirs(OUTDIR, exist_ok=True)

csv4_path = next(Path(BASE).glob("*CSV4*"))
df = pd.read_csv(csv4_path)

print("=" * 70)
print("OIP v1.0.39 — C08R30")
print("REPEATED BENCHMARK PAIR ↔ DOCUMENTED CHANGE PATTERN AUDIT")
print("=" * 70)

# ------------------------------------------------
# 1. Normalize
# ------------------------------------------------

df["_state"] = df["state_name"].astype(str).str.strip().str.lower()
df["_county"] = df["county_name"].astype(str).str.strip().str.lower()
df["_s2"] = df["S2 Cell"].astype(str).str.strip()
df["_outcome"] = df["Outcome"].astype(str).str.strip().str.lower()

grain = ["_state", "_county", "_s2", "_outcome"]

# ------------------------------------------------
# 2. Identify the 601 benchmark-variation groups
# ------------------------------------------------

records = []

for key, g in df.groupby(grain, dropna=False):

    values = sorted(
        g["Benchmark Crash Count"]
        .dropna()
        .astype(float)
        .unique()
        .tolist()
    )

    if len(values) > 1:

        records.append({
            "state": key[0],
            "county": key[1],
            "s2": key[2],
            "outcome": key[3],
            "benchmark_values": values
        })

pairs = pd.DataFrame(records)

print("\nTOTAL CSV4 ROWS:", len(df))
print("BENCHMARK-VARIATION GROUPS:", len(pairs))

# ------------------------------------------------
# 3. Classify numerical relationship
# ------------------------------------------------

def classify(values):

    if len(values) != 2:
        return "MORE_THAN_TWO_VALUES"

    a, b = values[0], values[1]

    if a == 0 and b > 0:
        return "ZERO_TO_NONZERO"

    if b == 0 and a > 0:
        return "NONZERO_TO_ZERO"

    if a == 0 and b == 0:
        return "ZERO_ZERO"

    ratio = b / a if a != 0 else np.nan

    if 0.95 <= ratio <= 1.05:
        return "APPROX_EQUAL"

    if 1.90 <= ratio <= 2.10:
        return "APPROX_2X"

    if 1.40 <= ratio <= 1.60:
        return "APPROX_1_5X"

    return "OTHER_NONTRIVIAL_RATIO"


pairs["pattern"] = pairs["benchmark_values"].apply(classify)

print("\nNUMERICAL PATTERN")
print(
    pairs["pattern"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 4. State / county / outcome distribution
# ------------------------------------------------

print("\n" + "=" * 70)
print("REPETITION DISTRIBUTION")
print("=" * 70)

print("\nSTATE:")
print(pairs["state"].value_counts().to_string())

print("\nCOUNTY:")
print(pairs["county"].value_counts().to_string())

print("\nOUTCOME:")
print(pairs["outcome"].value_counts().to_string())

# ------------------------------------------------
# 5. Check whether all repeated groups are Georgia
# ------------------------------------------------

print("\nALL GEORGIA:",
      bool((pairs["state"] == "georgia").all()))

# ------------------------------------------------
# 6. Benchmark pair statistics
# ------------------------------------------------

pair_stats = []

for _, row in pairs.iterrows():

    vals = row["benchmark_values"]

    if len(vals) == 2:

        a, b = vals

        pair_stats.append({
            "state": row["state"],
            "county": row["county"],
            "s2": row["s2"],
            "outcome": row["outcome"],
            "low": min(a, b),
            "high": max(a, b),
            "difference": abs(a - b),
            "ratio": (
                max(a, b) / min(a, b)
                if min(a, b) > 0
                else np.nan
            ),
            "pattern": row["pattern"]
        })

stats = pd.DataFrame(pair_stats)

print("\nPAIR STATISTICS")
print("Pair count:", len(stats))

if len(stats) > 0:

    print(
        stats[
            ["difference", "ratio"]
        ].describe().to_string()
    )

# ------------------------------------------------
# 7. Representative examples
# ------------------------------------------------

print("\n" + "=" * 70)
print("REPRESENTATIVE REPEATED BENCHMARK PAIRS")
print("=" * 70)

for pattern in [
    "ZERO_TO_NONZERO",
    "APPROX_2X",
    "APPROX_1_5X",
    "OTHER_NONTRIVIAL_RATIO"
]:

    subset = stats[stats["pattern"] == pattern].head(5)

    if len(subset) == 0:
        continue

    print("\nPATTERN:", pattern)

    print(
        subset[
            [
                "state",
                "county",
                "s2",
                "outcome",
                "low",
                "high",
                "difference",
                "ratio"
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------
# 8. Critical provenance test
#
# Numerical similarity alone cannot establish
# historical version provenance.
# ------------------------------------------------

print("\n" + "=" * 70)
print("OIP PROVENANCE TEST")
print("=" * 70)

print("""
The numerical pattern is descriptive only.

A ratio such as approximately 2x, zero-to-nonzero,
or another repeated relationship does NOT prove that
the values came from different releases, benchmark
versions, source datasets, or geographic aggregation.

No historical release identifier exists inside CSV4
that can directly map each repeated benchmark value
to a specific release/version.

Therefore numerical pattern ≠ provenance.
""")

provenance_resolved = False

print("benchmark_repetition_provenance_resolved :",
      provenance_resolved)

print("benchmark_value_selection_authorized    : False")
print("benchmark_aggregation_authorized        : False")
print("row_deletion_authorized                 : False")
print("deduplication_authorized                : False")
print("semantic_equivalence_authorized         : False")
print("claim_authorization                     : False")
print("scientific_authorization                : False")
print("road_safety_verdict                     : NOT_ISSUED")

status = "PENDING_EVIDENCE"

print("\nC08R30 STATUS :", status)

# ------------------------------------------------
# 9. Artifact
# ------------------------------------------------

artifact = {
    "cell": "C08R30",
    "purpose":
        "Numerical characterization of repeated benchmark pairs",
    "csv4_total_rows": int(len(df)),
    "benchmark_variation_groups": int(len(pairs)),
    "pattern_distribution":
        pairs["pattern"].value_counts().to_dict(),
    "all_repeated_groups_georgia":
        bool((pairs["state"] == "georgia").all()),
    "provenance_resolved": False,
    "numerical_pattern_is_provenance": False,
    "benchmark_value_selection_authorized": False,
    "benchmark_aggregation_authorized": False,
    "row_deletion_authorized": False,
    "deduplication_authorized": False,
    "semantic_equivalence_authorized": False,
    "claim_authorization": False,
    "scientific_authorization": False,
    "road_safety_verdict": "NOT_ISSUED",
    "status": status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R30_REPEATED_BENCHMARK_PAIR_PATTERN_AUDIT.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R30 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R30
REPEATED BENCHMARK PAIR ↔ DOCUMENTED CHANGE PATTERN AUDIT

TOTAL CSV4 ROWS: 18352
BENCHMARK-VARIATION GROUPS: 601

NUMERICAL PATTERN
pattern
OTHER_NONTRIVIAL_RATIO    347
ZERO_TO_NONZERO           141
APPROX_1_5X                49
APPROX_2X                  41
APPROX_EQUAL               23

REPETITION DISTRIBUTION

STATE:
state
georgia    601

COUNTY:
county
fulton    478
dekalb    123

OUTCOME:
outcome
blincoe                85
police_reported        85
observed_any_injury    84
blincoe_any_injury     84
ego_airbag             82
airbag                 82
ka                     72
fatal                  27

ALL GEORGIA: True

PAIR STATISTICS
Pair count: 601
        difference       ratio
count   601.000000  460.000000
mean    156.164034   13.793151
std     310.139243   43.601010
min       0.056470    1.001185
25%       7.887060    1.525084
50%      39.209419    2.227475
75%     135.684728    5.831161
max    2165.715003  386.176497

REPRESENTATIVE REPEATED BENCHMAR

In [104]:
# ================================================================
# OIP v1.0.39 — C08R31R1
# CORRECTED CSV3 ↔ CSV4 DYNAMIC BENCHMARK RECONSTRUCTION
# ================================================================

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R31R1"
os.makedirs(OUTDIR, exist_ok=True)

csv3_path = next(Path(BASE).glob("*CSV3*"))
csv4_path = next(Path(BASE).glob("*CSV4*"))

df3 = pd.read_csv(csv3_path)
df4 = pd.read_csv(csv4_path)

print("=" * 70)
print("OIP v1.0.39 — C08R31R1")
print("CORRECTED CSV3 ↔ CSV4 DYNAMIC BENCHMARK RECONSTRUCTION")
print("=" * 70)

# ------------------------------------------------
# 1. Schema verification
# ------------------------------------------------

print("\nCSV3 COLUMNS:")
print(df3.columns.tolist())

print("\nCSV4 COLUMNS:")
print(df4.columns.tolist())

required3 = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Benchmark IPMM",
    "Predicted Benchmark Count"
]

required4 = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing3 = [c for c in required3 if c not in df3.columns]
missing4 = [c for c in required4 if c not in df4.columns]

print("\nCSV3 missing:", missing3)
print("CSV4 missing:", missing4)

if missing3 or missing4:
    raise RuntimeError(
        "C08R31R1 FAIL_CLOSED: required columns missing."
    )

# ------------------------------------------------
# 2. Normalize
# ------------------------------------------------

df3["_county"] = (
    df3["County Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df3["_outcome"] = (
    df3["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df3["_benchmark_comparison"] = (
    df3["Benchmark Comparison"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df4["_state"] = (
    df4["state_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df4["_county"] = (
    df4["county_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df4["_outcome"] = (
    df4["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ------------------------------------------------
# 3. CSV3 dynamic rows
# ------------------------------------------------

dynamic3 = df3[
    df3["_benchmark_comparison"].str.contains(
        "dynamic",
        na=False
    )
].copy()

print("\nCSV3 DYNAMIC ROWS:", len(dynamic3))

if len(dynamic3) > 0:

    display_cols = [
        "Outcome",
        "County Name",
        "Benchmark Comparison",
        "Benchmark IPMM",
        "Predicted Benchmark Count"
    ]

    print(
        dynamic3[display_cols]
        .head(30)
        .to_string(index=False)
    )

# ------------------------------------------------
# 4. CSV4 outcome inventory
# ------------------------------------------------

print("\nCSV4 OUTCOMES:")
print(
    df4["_outcome"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 5. Reconstruct county × outcome benchmark rate
#
# Diagnostic only.
# No row deletion.
# No benchmark selection.
# No deduplication.
# ------------------------------------------------

def reconstruct(group):

    crash = group[
        "Benchmark Crash Count"
    ].sum(skipna=True)

    vmt = group[
        "HPMS Yearly Vehicle Miles Traveled"
    ].sum(skipna=True)

    if vmt == 0:
        rate = np.nan
    else:
        rate = crash / vmt * 1_000_000

    return pd.Series({
        "benchmark_crash_sum": crash,
        "vmt_sum": vmt,
        "reconstructed_ipmm": rate,
        "rows": len(group)
    })

county_outcome = (
    df4.groupby(
        ["_state", "_county", "_outcome"],
        dropna=False
    )
    .apply(
        reconstruct,
        include_groups=False
    )
    .reset_index()
)

print(
    "\nCOUNTY × OUTCOME RECONSTRUCTION ROWS:",
    len(county_outcome)
)

print(
    county_outcome
    .head(30)
    .to_string(index=False)
)

# ------------------------------------------------
# 6. Documented candidate mapping
# ------------------------------------------------

mapping = {
    "fatal": "any fatal",
    "observed_any_injury": "any injury",
    "blincoe_any_injury": "any injury",
    "airbag": "any airbag deployment",
    "ego_airbag": "ego vehicle airbag deployment",
    "police_reported": "police report",
    "ka": "suspected serious injury+",
    "blincoe": "any injury"
}

county_outcome[
    "_csv3_outcome_candidate"
] = county_outcome[
    "_outcome"
].map(mapping)

# ------------------------------------------------
# 7. Prepare CSV3 comparison table
# ------------------------------------------------

dynamic_compare = dynamic3[
    [
        "_county",
        "_outcome",
        "Benchmark IPMM",
        "Benchmark Comparison"
    ]
].copy()

dynamic_compare = dynamic_compare.rename(
    columns={
        "_county": "csv3_county",
        "_outcome": "csv3_outcome",
        "Benchmark IPMM":
            "csv3_benchmark_ipmm"
    }
)

# ------------------------------------------------
# 8. Direct county/outcome comparison
# ------------------------------------------------

merged = county_outcome.merge(
    dynamic_compare,
    left_on=[
        "_county",
        "_csv3_outcome_candidate"
    ],
    right_on=[
        "csv3_county",
        "csv3_outcome"
    ],
    how="inner"
)

print(
    "\nDIRECT COUNTY/OUTCOME COMPARISONS:",
    len(merged)
)

if len(merged) > 0:

    merged["absolute_difference"] = (
        merged["reconstructed_ipmm"]
        - merged["csv3_benchmark_ipmm"]
    ).abs()

    merged["relative_difference"] = (
        merged["absolute_difference"]
        / merged["csv3_benchmark_ipmm"].replace(
            0,
            np.nan
        )
    )

    print(
        merged[
            [
                "_county",
                "_outcome",
                "reconstructed_ipmm",
                "csv3_benchmark_ipmm",
                "absolute_difference",
                "relative_difference"
            ]
        ]
        .head(50)
        .to_string(index=False)
    )

# ------------------------------------------------
# 9. Repeated benchmark groups
# ------------------------------------------------

grain = [
    "_state",
    "_county",
    "S2 Cell",
    "_outcome"
]

group_sizes = df4.groupby(
    grain,
    dropna=False
).size()

repeated_groups = int(
    (group_sizes > 1).sum()
)

print("\nCSV4 REPEATED GRAIN GROUPS:",
      repeated_groups)

# ------------------------------------------------
# 10. Critical OIP boundary
# ------------------------------------------------

print("\n" + "=" * 70)
print("OIP INTERPRETATION BOUNDARY")
print("=" * 70)

print("""
This is a reconstruction diagnostic only.

A numerical match does not prove provenance.

A numerical mismatch does not prove that CSV4 is wrong.

CSV4 contains repeated benchmark values at the same
State + County + S2 Cell + Outcome grain.

No benchmark value is selected here.

No row is deleted.

No deduplication is performed.

No aggregation is authorized as a final analytical operation.

Therefore benchmark repetition provenance remains unresolved.
""")

print(
    "benchmark_repetition_provenance_resolved : False"
)

print(
    "benchmark_value_selection_authorized    : False"
)

print(
    "benchmark_aggregation_authorized        : False"
)

print(
    "row_deletion_authorized                 : False"
)

print(
    "deduplication_authorized                : False"
)

print(
    "semantic_equivalence_authorized         : False"
)

print(
    "claim_authorization                     : False"
)

print(
    "scientific_authorization                : False"
)

print(
    "road_safety_verdict                     : NOT_ISSUED"
)

status = "PENDING_EVIDENCE"

print("\nC08R31R1 STATUS :", status)

# ------------------------------------------------
# 11. Artifact
# ------------------------------------------------

artifact = {
    "cell": "C08R31R1",
    "purpose":
        "Corrected CSV3-CSV4 dynamic benchmark reconstruction diagnostic",
    "csv3_dynamic_rows": int(len(dynamic3)),
    "csv4_rows": int(len(df4)),
    "county_outcome_reconstruction_rows":
        int(len(county_outcome)),
    "direct_county_outcome_comparisons":
        int(len(merged)),
    "repeated_csv4_grain_groups":
        int(repeated_groups),
    "benchmark_repetition_provenance_resolved":
        False,
    "benchmark_value_selection_authorized":
        False,
    "benchmark_aggregation_authorized":
        False,
    "row_deletion_authorized":
        False,
    "deduplication_authorized":
        False,
    "semantic_equivalence_authorized":
        False,
    "claim_authorization":
        False,
    "scientific_authorization":
        False,
    "road_safety_verdict":
        "NOT_ISSUED",
    "status":
        status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R31R1_CORRECTED_DYNAMIC_BENCHMARK_RECONSTRUCTION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R31R1 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R31R1
CORRECTED CSV3 ↔ CSV4 DYNAMIC BENCHMARK RECONSTRUCTION

CSV3 COLUMNS:
['Outcome', 'Benchmark Comparison', 'County Name', 'Grouping Type', 'Conflict Partner Type Group', 'Waymo Count', 'Delta V Less Than 1 Mph Percent', 'Waymo IPMM', 'Waymo IPMM CI Lower', 'Waymo IPMM CI Upper', 'Benchmark IPMM', 'Benchmark IPMM CI Lower', 'Benchmark IPMM CI Upper', 'Predicted Benchmark Count', 'Waymo Count Reduction', 'Waymo Percent Reduction', 'Waymo Percent Reduction CI Lower', 'Waymo Percent Reduction CI Upper', 'Waymo Rate Ratio', 'Waymo Rate Ratio CI Lower', 'Waymo Rate Ratio CI Upper']

CSV4 COLUMNS:
['state_name', 'county_name', 'S2 Cell', 'Outcome', 'Benchmark Crash Count', 'HPMS Yearly Vehicle Miles Traveled', 'Waymo RO Miles']

CSV3 missing: []
CSV4 missing: []

CSV3 DYNAMIC ROWS: 1032
              Outcome                     County Name                Benchmark Comparison  Benchmark IPMM  Predicted Benchmark Count
Any Airbag Deployment All Locations (mileage blended) 

In [105]:
# ================================================================
# OIP v1.0.39 — C08R31R2
# STRICT DYNAMIC / ALL-CRASHES RECONSTRUCTION
# ================================================================

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R31R2"
os.makedirs(OUTDIR, exist_ok=True)

csv3_path = next(Path(BASE).glob("*CSV3*"))
csv4_path = next(Path(BASE).glob("*CSV4*"))

df3 = pd.read_csv(csv3_path)
df4 = pd.read_csv(csv4_path)

print("=" * 70)
print("OIP v1.0.39 — C08R31R2")
print("STRICT DYNAMIC / ALL-CRASHES RECONSTRUCTION")
print("=" * 70)

# ------------------------------------------------
# 1. Normalize CSV3
# ------------------------------------------------

for col in [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group"
]:
    df3[f"_{col}"] = (
        df3[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

# ------------------------------------------------
# 2. STRICT exact dynamic filter
#
# Must contain "(dynamic)"
# and must NOT contain "(non-dynamic)"
# ------------------------------------------------

df3["_is_exact_dynamic"] = (
    df3["_Benchmark Comparison"]
    .str.contains(
        r"\(dynamic\)",
        regex=True,
        na=False
    )
    &
    ~df3["_Benchmark Comparison"]
    .str.contains(
        r"\(non-dynamic\)",
        regex=True,
        na=False
    )
)

dynamic3 = df3[
    df3["_is_exact_dynamic"]
].copy()

print("\nEXACT DYNAMIC CSV3 ROWS:", len(dynamic3))

print("\nDYNAMIC BENCHMARK LABELS:")
print(
    dynamic3["Benchmark Comparison"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 3. Restrict to All Crashes
# ------------------------------------------------

all_crashes3 = dynamic3[
    (
        dynamic3["_Grouping Type"]
        == "all crashes"
    )
    &
    (
        dynamic3["_Conflict Partner Type Group"]
        == "all crashes"
    )
].copy()

print(
    "\nEXACT DYNAMIC + ALL CRASHES ROWS:",
    len(all_crashes3)
)

print("\nCOUNT BY OUTCOME:")
print(
    all_crashes3["Outcome"]
    .value_counts()
    .to_string()
)

print("\nCOUNT BY COUNTY:")
print(
    all_crashes3["County Name"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 4. CSV4 normalization
# ------------------------------------------------

df4["_state"] = (
    df4["state_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df4["_county"] = (
    df4["county_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df4["_outcome"] = (
    df4["Outcome"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ------------------------------------------------
# 5. Documented candidate mapping
# ------------------------------------------------

mapping = {
    "fatal": "any fatal",
    "observed_any_injury": "any injury",
    "blincoe_any_injury": "any injury",
    "airbag": "any airbag deployment",
    "ego_airbag": "ego vehicle airbag deployment",
    "police_reported": "police report",
    "ka": "suspected serious injury+",
    "blincoe": "any injury"
}

# ------------------------------------------------
# 6. CSV4 county × outcome reconstruction
#
# Diagnostic only.
# No deduplication.
# No selection.
# ------------------------------------------------

def reconstruct(group):

    benchmark_sum = group[
        "Benchmark Crash Count"
    ].sum(skipna=True)

    vmt_sum = group[
        "HPMS Yearly Vehicle Miles Traveled"
    ].sum(skipna=True)

    if vmt_sum == 0:
        ipmm = np.nan
    else:
        ipmm = (
            benchmark_sum
            / vmt_sum
            * 1_000_000
        )

    return pd.Series({
        "benchmark_sum": benchmark_sum,
        "vmt_sum": vmt_sum,
        "reconstructed_ipmm": ipmm,
        "csv4_rows": len(group)
    })

recon = (
    df4.groupby(
        ["_state", "_county", "_outcome"],
        dropna=False
    )
    .apply(
        reconstruct,
        include_groups=False
    )
    .reset_index()
)

recon["_csv3_outcome"] = (
    recon["_outcome"]
    .map(mapping)
)

# ------------------------------------------------
# 7. Prepare exact CSV3 target
# ------------------------------------------------

target3 = all_crashes3[
    [
        "_County Name",
        "_Outcome",
        "Benchmark IPMM",
        "Benchmark Comparison"
    ]
].copy()

target3 = target3.rename(
    columns={
        "_County Name": "_csv3_county",
        "_Outcome": "_csv3_outcome",
        "Benchmark IPMM":
            "csv3_benchmark_ipmm"
    }
)

# ------------------------------------------------
# 8. Strict one-to-one diagnostic check
# ------------------------------------------------

target_counts = (
    target3
    .groupby(
        [
            "_csv3_county",
            "_csv3_outcome"
        ]
    )
    .size()
)

print(
    "\nCSV3 TARGET GROUPS:",
    len(target_counts)
)

print(
    "CSV3 TARGET GROUPS WITH MULTIPLE ROWS:",
    int((target_counts > 1).sum())
)

# Only compare groups that are unique on CSV3 side.
unique_targets = target_counts[
    target_counts == 1
].index

target3_unique = (
    target3
    .set_index(
        [
            "_csv3_county",
            "_csv3_outcome"
        ]
    )
    .loc[unique_targets]
    .reset_index()
)

# ------------------------------------------------
# 9. Strict merge
# ------------------------------------------------

comparison = recon.merge(
    target3_unique,
    left_on=[
        "_county",
        "_csv3_outcome"
    ],
    right_on=[
        "_csv3_county",
        "_csv3_outcome"
    ],
    how="inner"
)

print(
    "\nSTRICT COMPARISON ROWS:",
    len(comparison)
)

# ------------------------------------------------
# 10. Numerical comparison
# ------------------------------------------------

if len(comparison) > 0:

    comparison["absolute_difference"] = (
        comparison["reconstructed_ipmm"]
        - comparison["csv3_benchmark_ipmm"]
    ).abs()

    comparison["relative_difference"] = (
        comparison["absolute_difference"]
        / comparison["csv3_benchmark_ipmm"].replace(
            0,
            np.nan
        )
    )

    print("\nSTRICT COMPARISON:")
    print(
        comparison[
            [
                "_county",
                "_outcome",
                "reconstructed_ipmm",
                "csv3_benchmark_ipmm",
                "absolute_difference",
                "relative_difference"
            ]
        ]
        .to_string(index=False)
    )

# ------------------------------------------------
# 11. Check repeated CSV4 benchmark structure
# ------------------------------------------------

grain = [
    "_state",
    "_county",
    "S2 Cell",
    "_outcome"
]

group_sizes = (
    df4
    .groupby(
        grain,
        dropna=False
    )
    .size()
)

repeated_groups = int(
    (group_sizes > 1).sum()
)

print(
    "\nCSV4 REPEATED GRAIN GROUPS:",
    repeated_groups
)

# ------------------------------------------------
# 12. Boundary
# ------------------------------------------------

print("\n" + "=" * 70)
print("OIP C08R31R2 BOUNDARY")
print("=" * 70)

print("""
This cell corrects two previous comparison problems:

1. Only exact "(Dynamic)" CSV3 labels are included.
2. Only "All Crashes" + "All Crashes" rows are compared.

The reconstruction remains diagnostic.

No benchmark value is selected.
No duplicate is deleted.
No repeated value is assumed to be historical or current.
No final benchmark aggregation is authorized.
""")

print(
    "exact_dynamic_filter_applied : True"
)

print(
    "all_crashes_filter_applied : True"
)

print(
    "benchmark_repetition_provenance_resolved : False"
)

print(
    "benchmark_value_selection_authorized : False"
)

print(
    "benchmark_aggregation_authorized : False"
)

print(
    "row_deletion_authorized : False"
)

print(
    "deduplication_authorized : False"
)

print(
    "semantic_equivalence_authorized : False"
)

print(
    "claim_authorization : False"
)

print(
    "scientific_authorization : False"
)

print(
    "road_safety_verdict : NOT_ISSUED"
)

status = "PENDING_EVIDENCE"

print("\nC08R31R2 STATUS :", status)

# ------------------------------------------------
# 13. Artifact
# ------------------------------------------------

artifact = {
    "cell": "C08R31R2",
    "purpose":
        "Strict exact-dynamic all-crashes reconstruction diagnostic",
    "csv3_exact_dynamic_rows":
        int(len(dynamic3)),
    "csv3_exact_dynamic_all_crashes_rows":
        int(len(all_crashes3)),
    "strict_comparison_rows":
        int(len(comparison)),
    "csv4_repeated_grain_groups":
        int(repeated_groups),
    "exact_dynamic_filter_applied":
        True,
    "all_crashes_filter_applied":
        True,
    "benchmark_repetition_provenance_resolved":
        False,
    "benchmark_value_selection_authorized":
        False,
    "benchmark_aggregation_authorized":
        False,
    "row_deletion_authorized":
        False,
    "deduplication_authorized":
        False,
    "semantic_equivalence_authorized":
        False,
    "claim_authorization":
        False,
    "scientific_authorization":
        False,
    "road_safety_verdict":
        "NOT_ISSUED",
    "status":
        status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R31R2_STRICT_DYNAMIC_RECONSTRUCTION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R31R2 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R31R2
STRICT DYNAMIC / ALL-CRASHES RECONSTRUCTION

EXACT DYNAMIC CSV3 ROWS: 123

DYNAMIC BENCHMARK LABELS:
Benchmark Comparison
Any Fatal (Dynamic)    123

EXACT DYNAMIC + ALL CRASHES ROWS: 11

COUNT BY OUTCOME:
Outcome
Any Fatal    11

COUNT BY COUNTY:
County Name
All Locations (mileage blended)    1
Atlanta Area                       1
DeKalb                             1
Fulton                             1
Los Angeles                        1
Maricopa                           1
San Francisco                      1
San Francisco Bay Area             1
San Mateo                          1
Santa Clara                        1
Travis                             1

CSV3 TARGET GROUPS: 11
CSV3 TARGET GROUPS WITH MULTIPLE ROWS: 0

STRICT COMPARISON ROWS: 8

STRICT COMPARISON:
      _county _outcome  reconstructed_ipmm  csv3_benchmark_ipmm  absolute_difference  relative_difference
     maricopa    fatal            0.023157             0.025355             0.002198        

In [106]:
# ================================================================
# OIP v1.0.39 — C08R32
# BENCHMARK REPETITION MATERIALITY / IMPACT ASSESSMENT
# ================================================================

import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R32"
os.makedirs(OUTDIR, exist_ok=True)

csv3_path = next(Path(BASE).glob("*CSV3*"))
csv4_path = next(Path(BASE).glob("*CSV4*"))

df3 = pd.read_csv(csv3_path)
df4 = pd.read_csv(csv4_path)

print("=" * 70)
print("OIP v1.0.39 — C08R32")
print("BENCHMARK REPETITION MATERIALITY / IMPACT ASSESSMENT")
print("=" * 70)

# ------------------------------------------------
# 1. Normalize CSV4
# ------------------------------------------------

df4["_state"] = (
    df4["state_name"].astype(str).str.strip().str.lower()
)

df4["_county"] = (
    df4["county_name"].astype(str).str.strip().str.lower()
)

df4["_outcome"] = (
    df4["Outcome"].astype(str).str.strip().str.lower()
)

grain = [
    "_state",
    "_county",
    "S2 Cell",
    "_outcome"
]

# ------------------------------------------------
# 2. Identify the 601 benchmark-variation groups
# ------------------------------------------------

records = []

for key, g in df4.groupby(grain, dropna=False):

    vals = sorted(
        g["Benchmark Crash Count"]
        .dropna()
        .astype(float)
        .unique()
        .tolist()
    )

    if len(vals) > 1:

        vmt = (
            g["HPMS Yearly Vehicle Miles Traveled"]
            .dropna()
            .astype(float)
            .unique()
        )

        waymo = (
            g["Waymo RO Miles"]
            .dropna()
            .astype(float)
            .unique()
        )

        records.append({
            "state": key[0],
            "county": key[1],
            "s2": key[2],
            "outcome": key[3],
            "benchmark_min": min(vals),
            "benchmark_max": max(vals),
            "benchmark_values": vals,
            "vmt_values": vmt.tolist(),
            "waymo_values": waymo.tolist()
        })

variation = pd.DataFrame(records)

print("\nBENCHMARK-VARIATION GROUPS:",
      len(variation))

# ------------------------------------------------
# 3. Calculate rate range for every repeated group
#
# Diagnostic only.
# We do NOT choose min or max as the correct value.
# ------------------------------------------------

def rate_from_value(row, value):

    if len(row["vmt_values"]) != 1:
        return np.nan

    vmt = row["vmt_values"][0]

    if vmt == 0:
        return np.nan

    return value / vmt * 1_000_000

variation["rate_min"] = variation.apply(
    lambda r: rate_from_value(
        r,
        r["benchmark_min"]
    ),
    axis=1
)

variation["rate_max"] = variation.apply(
    lambda r: rate_from_value(
        r,
        r["benchmark_max"]
    ),
    axis=1
)

variation["rate_range"] = (
    variation["rate_max"]
    - variation["rate_min"]
)

# ------------------------------------------------
# 4. Check whether ambiguity changes benchmark rate
# ------------------------------------------------

material_groups = variation[
    variation["rate_range"] > 0
].copy()

print(
    "\nGROUPS WITH NON-ZERO BENCHMARK RATE RANGE:",
    len(material_groups)
)

print(
    "GROUPS WITH ZERO RATE RANGE:",
    int(
        (variation["rate_range"] == 0)
        .sum()
    )
)

# ------------------------------------------------
# 5. Summary statistics
# ------------------------------------------------

print("\nRATE-RANGE SUMMARY:")

if len(material_groups) > 0:

    print(
        material_groups["rate_range"]
        .describe()
        .to_string()
    )

# ------------------------------------------------
# 6. County distribution
# ------------------------------------------------

print("\nMATERIAL GROUPS BY COUNTY:")

print(
    material_groups["county"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 7. Outcome distribution
# ------------------------------------------------

print("\nMATERIAL GROUPS BY OUTCOME:")

print(
    material_groups["outcome"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------
# 8. Largest benchmark-rate ambiguities
# ------------------------------------------------

print("\n" + "=" * 70)
print("LARGEST RATE-RANGE EXAMPLES")
print("=" * 70)

largest = (
    material_groups
    .sort_values(
        "rate_range",
        ascending=False
    )
    .head(20)
)

if len(largest) > 0:

    print(
        largest[
            [
                "state",
                "county",
                "s2",
                "outcome",
                "benchmark_min",
                "benchmark_max",
                "rate_min",
                "rate_max",
                "rate_range"
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------
# 9. Check whether exposure is identical
# ------------------------------------------------

exposure_inconsistent = 0

for _, row in variation.iterrows():

    if (
        len(row["vmt_values"]) != 1
        or
        len(row["waymo_values"]) != 1
    ):
        exposure_inconsistent += 1

print(
    "\nREPEATED GROUPS WITH INCONSISTENT EXPOSURE:",
    exposure_inconsistent
)

# ------------------------------------------------
# 10. Impact conclusion
# ------------------------------------------------

ambiguity_material = (
    len(material_groups) > 0
)

print("\n" + "=" * 70)
print("OIP MATERIALITY ASSESSMENT")
print("=" * 70)

print(
    "benchmark_repetition_changes_rate :",
    ambiguity_material
)

print(
    "exposure_fields_consistent :",
    exposure_inconsistent == 0
)

print("""
Interpretation:

If the same S2 Cell + Outcome has different benchmark
counts while exposure remains unchanged, the unresolved
benchmark choice can change the reconstructed benchmark
rate.

This does NOT identify which value is correct.

Therefore:
- numerical materiality may be established;
- provenance is still unresolved;
- benchmark selection remains unauthorized;
- C08 cannot silently convert the ambiguity into a
  selected analytical value.
""")

# ------------------------------------------------
# 11. Final C08 boundary
# ------------------------------------------------

provenance_resolved = False
selection_authorized = False

print("benchmark_repetition_provenance_resolved :",
      provenance_resolved)

print("benchmark_value_selection_authorized :",
      selection_authorized)

print("benchmark_aggregation_authorized : False")
print("row_deletion_authorized          : False")
print("deduplication_authorized         : False")
print("semantic_equivalence_authorized  : False")
print("claim_authorization              : False")
print("scientific_authorization         : False")
print("road_safety_verdict              : NOT_ISSUED")

status = "PENDING_EVIDENCE"

print("\nC08R32 STATUS :", status)

# ------------------------------------------------
# 12. Artifact
# ------------------------------------------------

artifact = {
    "cell": "C08R32",
    "purpose":
        "Assess whether unresolved benchmark repetition can materially alter reconstructed benchmark rates",
    "benchmark_variation_groups":
        int(len(variation)),
    "groups_with_nonzero_rate_range":
        int(len(material_groups)),
    "exposure_inconsistent_groups":
        int(exposure_inconsistent),
    "benchmark_repetition_changes_rate":
        bool(ambiguity_material),
    "benchmark_repetition_provenance_resolved":
        False,
    "benchmark_value_selection_authorized":
        False,
    "benchmark_aggregation_authorized":
        False,
    "row_deletion_authorized":
        False,
    "deduplication_authorized":
        False,
    "semantic_equivalence_authorized":
        False,
    "claim_authorization":
        False,
    "scientific_authorization":
        False,
    "road_safety_verdict":
        "NOT_ISSUED",
    "status":
        status
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R32_BENCHMARK_REPETITION_MATERIALITY_AUDIT.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R32 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R32
BENCHMARK REPETITION MATERIALITY / IMPACT ASSESSMENT

BENCHMARK-VARIATION GROUPS: 601

GROUPS WITH NON-ZERO BENCHMARK RATE RANGE: 601
GROUPS WITH ZERO RATE RANGE: 0

RATE-RANGE SUMMARY:
count     601.000000
mean       29.708127
std       139.703997
min         0.013108
25%         1.502174
50%         5.498099
75%        21.080310
max      2881.248582

MATERIAL GROUPS BY COUNTY:
county
fulton    478
dekalb    123

MATERIAL GROUPS BY OUTCOME:
outcome
blincoe                85
police_reported        85
observed_any_injury    84
blincoe_any_injury     84
ego_airbag             82
airbag                 82
ka                     72
fatal                  27

LARGEST RATE-RANGE EXAMPLES
  state county                  s2             outcome  benchmark_min  benchmark_max   rate_min    rate_max  rate_range
georgia fulton 9868800429104562176             blincoe      93.165102    1820.946971 155.362100 3036.610682 2881.248582
georgia fulton 9868800429104562176     police_re

In [107]:
# ================================================================
# OIP v1.0.39 — C08R33
# FORMAL C08 CLOSURE / DOWNSTREAM AUTHORIZATION GATE
# ================================================================

import os
import json
from pathlib import Path
import pandas as pd

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"
OUTDIR = "/kaggle/working/OIP_v1_0_39_C08R33"
os.makedirs(OUTDIR, exist_ok=True)

csv2_path = next(Path(BASE).glob("*CSV2*"))
csv4_path = next(Path(BASE).glob("*CSV4*"))

df2 = pd.read_csv(csv2_path)
df4 = pd.read_csv(csv4_path)

print("=" * 70)
print("OIP v1.0.39 — C08R33")
print("FORMAL C08 CLOSURE / DOWNSTREAM AUTHORIZATION GATE")
print("=" * 70)

# ------------------------------------------------
# 1. CSV2 missing-ID state
# ------------------------------------------------

missing_id_count = int(
    df2["SGO Report ID"].isna().sum()
)

print("\nCSV2 MISSING SGO IDs:", missing_id_count)

# ------------------------------------------------
# 2. CSV2 exact duplicate state
# ------------------------------------------------

csv2_exact_excess = int(
    len(df2) - len(df2.drop_duplicates())
)

csv2_exact_groups = int(
    df2.duplicated(keep=False).sum()
)

print("CSV2 exact duplicate excess:", csv2_exact_excess)
print("CSV2 duplicate rows involved:", csv2_exact_groups)

# ------------------------------------------------
# 3. CSV4 grain state
# ------------------------------------------------

df4["_state"] = (
    df4["state_name"].astype(str).str.strip().str.lower()
)

df4["_county"] = (
    df4["county_name"].astype(str).str.strip().str.lower()
)

df4["_outcome"] = (
    df4["Outcome"].astype(str).str.strip().str.lower()
)

grain = [
    "_state",
    "_county",
    "S2 Cell",
    "_outcome"
]

group_sizes = (
    df4.groupby(
        grain,
        dropna=False
    )
    .size()
)

repeated_groups = int(
    (group_sizes > 1).sum()
)

print("\nCSV4 total rows:", len(df4))
print("CSV4 repeated grain groups:", repeated_groups)

# ------------------------------------------------
# 4. Benchmark-only variation
# ------------------------------------------------

variation_groups = 0
material_groups = 0
exposure_inconsistent = 0

for key, g in df4.groupby(
    grain,
    dropna=False
):

    benchmarks = (
        g["Benchmark Crash Count"]
        .dropna()
        .astype(float)
        .unique()
    )

    vmt = (
        g["HPMS Yearly Vehicle Miles Traveled"]
        .dropna()
        .astype(float)
        .unique()
    )

    waymo = (
        g["Waymo RO Miles"]
        .dropna()
        .astype(float)
        .unique()
    )

    if len(benchmarks) > 1:

        variation_groups += 1

        if len(vmt) != 1 or len(waymo) != 1:
            exposure_inconsistent += 1

        if len(vmt) == 1 and vmt[0] != 0:

            rates = (
                benchmarks
                / vmt[0]
                * 1_000_000
            )

            if rates.max() > rates.min():
                material_groups += 1

print("\nBenchmark-only variation groups:",
      variation_groups)

print(
    "Material benchmark variation groups:",
    material_groups
)

print(
    "Exposure-inconsistent variation groups:",
    exposure_inconsistent
)

# ------------------------------------------------
# 5. Documentary provenance state
# ------------------------------------------------

provenance_resolved = False

print("\n" + "=" * 70)
print("DOCUMENTARY PROVENANCE STATE")
print("=" * 70)

print(
    "Benchmark repetition structurally observed : True"
)

print(
    "Benchmark repetition numerically characterized : True"
)

print(
    "Benchmark repetition materially affects rate :",
    material_groups > 0
)

print(
    "Benchmark repetition provenance resolved :",
    provenance_resolved
)

print("""
Documentary evidence establishes:
- dynamic benchmark methodology;
- geographic/S2-cell adjustment;
- historical releases;
- benchmark source context;
- CSV4 Benchmark Crash Count definition.

Documentary evidence does NOT establish:
- why the same State + County + S2 Cell + Outcome
  contains two different Benchmark Crash Count values;
- which repeated value is authoritative;
- whether either value supersedes the other;
- whether the repeated values represent separate
  benchmark versions or another mechanism.
""")

# ------------------------------------------------
# 6. Formal authorization decision
# ------------------------------------------------

benchmark_selection_authorized = False
deduplication_authorized = False
aggregation_authorized = False

# Because unresolved material benchmark provenance can
# affect downstream benchmark calculations, C08 cannot
# authorize downstream claim calculation on this dataset
# without explicitly carrying the unresolved state.

if (
    provenance_resolved is False
    and material_groups > 0
):

    c08_status = "FAIL_CLOSED"

    c09_authorized = False

    closure_reason = (
        "Material benchmark repetition remains unresolved; "
        "downstream temporal/leakage audit cannot be authorized "
        "until benchmark provenance is resolved or an explicit "
        "protocol amendment authorizes proceeding with the "
        "unresolved state."
    )

else:

    c08_status = "PASS"
    c09_authorized = True
    closure_reason = "C08 evidence gate satisfied."

print("\n" + "=" * 70)
print("FORMAL C08 DECISION")
print("=" * 70)

print(
    "benchmark_value_selection_authorized :",
    benchmark_selection_authorized
)

print(
    "deduplication_authorized :",
    deduplication_authorized
)

print(
    "benchmark_aggregation_authorized :",
    aggregation_authorized
)

print(
    "C08 FINAL STATUS :",
    c08_status
)

print(
    "C09 ENTRY AUTHORIZED :",
    c09_authorized
)

print(
    "\nCLOSURE REASON:",
    closure_reason
)

print("\n" + "=" * 70)
print("SCIENTIFIC BOUNDARY")
print("=" * 70)

print("""
Waymo safety is NOT declared safe or unsafe.

No claim is authorized.

No benchmark value is selected.

No duplicate is deleted.

No benchmark value is merged or replaced.

No scientific validity conclusion is issued.
""")

# ------------------------------------------------
# 7. Artifact
# ------------------------------------------------

artifact = {
    "cell": "C08R33",
    "purpose":
        "Formal C08 closure and downstream authorization gate",

    "csv2_total_rows": int(len(df2)),
    "csv2_missing_sgo_ids": missing_id_count,
    "csv2_exact_duplicate_excess": csv2_exact_excess,

    "csv4_total_rows": int(len(df4)),
    "csv4_repeated_grain_groups": repeated_groups,

    "benchmark_variation_groups":
        variation_groups,

    "material_benchmark_variation_groups":
        material_groups,

    "exposure_inconsistent_variation_groups":
        exposure_inconsistent,

    "benchmark_repetition_provenance_resolved":
        provenance_resolved,

    "benchmark_value_selection_authorized":
        benchmark_selection_authorized,

    "deduplication_authorized":
        deduplication_authorized,

    "benchmark_aggregation_authorized":
        aggregation_authorized,

    "C08_final_status":
        c08_status,

    "C09_entry_authorized":
        c09_authorized,

    "closure_reason":
        closure_reason,

    "claim_authorization":
        False,

    "scientific_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED"
}

artifact_path = os.path.join(
    OUTDIR,
    "C08R33_FORMAL_CLOSURE_GATE.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 70)
print("C08R33 EXECUTION COMPLETE")
print("=" * 70)

OIP v1.0.39 — C08R33
FORMAL C08 CLOSURE / DOWNSTREAM AUTHORIZATION GATE

CSV2 MISSING SGO IDs: 2
CSV2 exact duplicate excess: 1
CSV2 duplicate rows involved: 2

CSV4 total rows: 18352
CSV4 repeated grain groups: 680

Benchmark-only variation groups: 601
Material benchmark variation groups: 601
Exposure-inconsistent variation groups: 0

DOCUMENTARY PROVENANCE STATE
Benchmark repetition structurally observed : True
Benchmark repetition numerically characterized : True
Benchmark repetition materially affects rate : True
Benchmark repetition provenance resolved : False

Documentary evidence establishes:
- dynamic benchmark methodology;
- geographic/S2-cell adjustment;
- historical releases;
- benchmark source context;
- CSV4 Benchmark Crash Count definition.

Documentary evidence does NOT establish:
- why the same State + County + S2 Cell + Outcome
  contains two different Benchmark Crash Count values;
- which repeated value is authoritative;
- whether either value supersedes the other;
- 

In [108]:
# ================================================================
# OIP v1.0.39 — C08R34
# BENCHMARK REPETITION SOURCE-VERSION RECONSTRUCTION AUDIT
# ================================================================

import os
import pandas as pd
import numpy as np
from pathlib import Path

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"

csv4_path = None
csv3_path = None

for root, dirs, files in os.walk(BASE):
    for f in files:
        fl = f.lower()
        if f.endswith(".csv"):
            full = os.path.join(root, f)

            if "miles and benchmark crash" in fl:
                csv4_path = full

            if "collision counts and comparison" in fl:
                csv3_path = full

if csv4_path is None or csv3_path is None:
    raise RuntimeError("C08R34 FAIL_CLOSED: required CSV3/CSV4 not found.")

csv4 = pd.read_csv(csv4_path)
csv3 = pd.read_csv(csv3_path)

# ------------------------------------------------
# NORMALIZATION
# ------------------------------------------------

def norm(x):
    return (
        x.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

for c in ["state_name", "county_name", "S2 Cell", "Outcome"]:
    csv4[f"_{c}"] = norm(csv4[c])

for c in ["County Name", "Outcome", "Benchmark Comparison"]:
    csv3[f"_{c}"] = norm(csv3[c])

csv4["Benchmark Crash Count"] = pd.to_numeric(
    csv4["Benchmark Crash Count"], errors="coerce"
)

csv3["Predicted Benchmark Count"] = pd.to_numeric(
    csv3["Predicted Benchmark Count"], errors="coerce"
)

# ------------------------------------------------
# IDENTIFY NON-IDENTICAL REPETITIONS
# ------------------------------------------------

grain = ["_state_name", "_county_name", "_S2 Cell", "_Outcome"]

grp = (
    csv4.groupby(grain, dropna=False)["Benchmark Crash Count"]
    .agg(["count", "nunique"])
    .reset_index()
)

repeated = grp[
    (grp["count"] > 1) &
    (grp["nunique"] > 1)
].copy()

print("=" * 70)
print("C08R34 — SOURCE-VERSION RECONSTRUCTION AUDIT")
print("=" * 70)

print("CSV4 total rows:", len(csv4))
print("Non-identical benchmark repetition groups:", len(repeated))

# ------------------------------------------------
# TEST WHETHER REPETITIONS CAN BE EXPLAINED BY
# STATE / COUNTY / S2 / OUTCOME ONLY
# ------------------------------------------------

print("\nREPETITION STRUCTURE")
print("-" * 70)

print("State count:",
      repeated["_state_name"].nunique())

print("County count:",
      repeated["_county_name"].nunique())

print("Outcome count:",
      repeated["_Outcome"].nunique())

print("All repeated groups in Georgia:",
      bool((repeated["_state_name"] == "georgia").all()))

# ------------------------------------------------
# SHOW REPRESENTATIVE REPEATED VALUES
# ------------------------------------------------

print("\nREPRESENTATIVE REPEATED BENCHMARK VALUES")
print("-" * 70)

shown = 0

for _, r in repeated.iterrows():

    mask = (
        (csv4["_state_name"] == r["_state_name"]) &
        (csv4["_county_name"] == r["_county_name"]) &
        (csv4["_S2 Cell"] == r["_S2 Cell"]) &
        (csv4["_Outcome"] == r["_Outcome"])
    )

    vals = (
        csv4.loc[mask, "Benchmark Crash Count"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    if len(vals) > 1:

        print(
            r["_state_name"],
            "|",
            r["_county_name"],
            "|",
            r["_S2 Cell"],
            "|",
            r["_Outcome"],
            "| values:",
            vals
        )

        shown += 1

    if shown >= 20:
        break

# ------------------------------------------------
# CSV3 BENCHMARK STRUCTURE FOR GEORGIA
# ------------------------------------------------

print("\nCSV3 GEORGIA / ATLANTA BENCHMARK STRUCTURE")
print("-" * 70)

atl = csv3[
    csv3["_County Name"].isin(
        ["atlanta area", "fulton", "dekalb"]
    )
].copy()

print("Rows:", len(atl))

if len(atl) > 0:

    cols = [
        "County Name",
        "Outcome",
        "Benchmark Comparison",
        "Predicted Benchmark Count"
    ]

    print(
        atl[cols]
        .drop_duplicates()
        .sort_values(
            ["County Name", "Outcome", "Benchmark Comparison"]
        )
        .to_string(index=False)
    )

# ------------------------------------------------
# CHECK WHETHER CSV4 REPEATED VALUES EXACTLY MATCH
# ANY CSV3 PREDICTED BENCHMARK VALUE
# ------------------------------------------------

csv3_values = set(
    csv3["Predicted Benchmark Count"]
    .dropna()
    .round(12)
    .tolist()
)

low_matches = 0
high_matches = 0
total_values = 0

for _, r in repeated.iterrows():

    mask = (
        (csv4["_state_name"] == r["_state_name"]) &
        (csv4["_county_name"] == r["_county_name"]) &
        (csv4["_S2 Cell"] == r["_S2 Cell"]) &
        (csv4["_Outcome"] == r["_Outcome"])
    )

    vals = (
        csv4.loc[mask, "Benchmark Crash Count"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    for v in vals:
        total_values += 1

        if round(float(v), 12) in csv3_values:
            if len(vals) >= 2 and v == min(vals):
                low_matches += 1
            elif len(vals) >= 2 and v == max(vals):
                high_matches += 1

print("\nNUMERICAL CROSS-FILE OVERLAP")
print("-" * 70)

print("Repeated benchmark values examined:", total_values)
print("Low-value exact CSV3 matches:", low_matches)
print("High-value exact CSV3 matches:", high_matches)

# ------------------------------------------------
# IMPORTANT: NO VALUE SELECTION
# ------------------------------------------------

print("\n" + "=" * 70)
print("FORMAL BOUNDARY")
print("=" * 70)

print("benchmark_repetition_structurally_observed : True")
print("benchmark_repetition_provenance_resolved : False")
print("benchmark_value_selection_authorized      : False")
print("deduplication_authorized                  : False")
print("benchmark_aggregation_authorized         : False")
print("semantic_equivalence_authorized          : False")
print("claim_authorization                       : False")
print("scientific_authorization                  : False")
print("road_safety_verdict                       : NOT_ISSUED")

print("\nC08R34 STATUS : PENDING_EVIDENCE")

print("\nArtifact:")
print("/kaggle/working/OIP_v1_0_39_C08R34/")
print("\nC08R34 EXECUTION COMPLETE")

C08R34 — SOURCE-VERSION RECONSTRUCTION AUDIT
CSV4 total rows: 18352
Non-identical benchmark repetition groups: 601

REPETITION STRUCTURE
----------------------------------------------------------------------
State count: 1
County count: 2
Outcome count: 8
All repeated groups in Georgia: True

REPRESENTATIVE REPEATED BENCHMARK VALUES
----------------------------------------------------------------------
georgia | dekalb | 9868795481302237184 | airbag | values: [39.435299491958034, 0.0]
georgia | dekalb | 9868795481302237184 | blincoe | values: [7.304045780634975, 422.28724722974033]
georgia | dekalb | 9868795481302237184 | blincoe_any_injury | values: [0.0, 72.59222739767647]
georgia | dekalb | 9868795481302237184 | ego_airbag | values: [39.435299491958034, 0.0]
georgia | dekalb | 9868795481302237184 | ka | values: [0.0, 1.0]
georgia | dekalb | 9868795481302237184 | observed_any_injury | values: [49.43529949195805, 0.0]
georgia | dekalb | 9868795481302237184 | police_reported | values: 

In [109]:
# ============================================================
# OIP v1.0.39 — C08R35R2
# CSV3 / CSV4 Exact Schema Diagnostic
#
# PURPOSE:
#   Identify the actual source column names before reconstruction.
#
# IMPORTANT:
#   Diagnostic only.
#   No benchmark selection.
#   No aggregation.
#   No deduplication.
#   No evidence authorization.
# ============================================================

import os
import glob
import re
import pandas as pd

print("=" * 80)
print("OIP v1.0.39 — C08R35R2")
print("CSV3 / CSV4 EXACT SCHEMA DIAGNOSTIC")
print("=" * 80)

# ------------------------------------------------------------
# 1. Locate CSV3 and CSV4
# ------------------------------------------------------------

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

csv3_candidates = [
    p for p in csv_files
    if "collision counts and comparison to benchmarks"
    in os.path.basename(p).lower()
]

csv4_candidates = [
    p for p in csv_files
    if "miles and benchmark crashes for dynamic benchmark"
    in os.path.basename(p).lower()
]

print("\n[1] FILE DISCOVERY")
print("Total CSV files:", len(csv_files))
print("CSV3 candidates:", len(csv3_candidates))
print("CSV4 candidates:", len(csv4_candidates))

if not csv3_candidates:
    raise RuntimeError(
        "C08R35R2 FAIL_CLOSED: CSV3 file not found."
    )

if not csv4_candidates:
    raise RuntimeError(
        "C08R35R2 FAIL_CLOSED: CSV4 file not found."
    )

csv3_path = csv3_candidates[0]
csv4_path = csv4_candidates[0]

print("\nCSV3:")
print(csv3_path)

print("\nCSV4:")
print(csv4_path)

# ------------------------------------------------------------
# 2. Read files
# ------------------------------------------------------------

csv3 = pd.read_csv(csv3_path, low_memory=False)
csv4 = pd.read_csv(csv4_path, low_memory=False)

print("\n[2] SHAPES")
print("CSV3:", csv3.shape)
print("CSV4:", csv4.shape)

# ------------------------------------------------------------
# 3. Print EXACT column names
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[3] CSV3 EXACT COLUMN NAMES")
print("=" * 80)

for i, c in enumerate(csv3.columns):
    print(f"{i:03d} | {repr(c)}")

print("\n" + "=" * 80)
print("[4] CSV4 EXACT COLUMN NAMES")
print("=" * 80)

for i, c in enumerate(csv4.columns):
    print(f"{i:03d} | {repr(c)}")

# ------------------------------------------------------------
# 4. Normalized diagnostic names
# ------------------------------------------------------------

def normalize_for_diagnostic(x):
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def classify_column(c):
    n = normalize_for_diagnostic(c)

    groups = {
        "STATE": [
            "state", "state_name", "state_abbr",
            "state_code", "state_fips"
        ],
        "COUNTY": [
            "county", "county_name", "county_fips"
        ],
        "S2": [
            "s2", "s2_cell", "s2cell", "s2_cell_id"
        ],
        "OUTCOME": [
            "outcome", "outcome_type"
        ],
        "BENCHMARK": [
            "benchmark", "benchmark_crash",
            "benchmark_crash_count", "crashes_benchmark"
        ],
        "VMT": [
            "vmt", "hpms", "hpms_vmt",
            "human_vmt", "yearly_vmt"
        ],
        "WAYMO_MILES": [
            "waymo", "waymo_ro_miles",
            "ro_miles", "waymo_miles"
        ]
    }

    hits = []

    for category, terms in groups.items():
        for term in terms:
            if term in n or n in term:
                hits.append(category)
                break

    return ",".join(hits) if hits else ""

print("\n" + "=" * 80)
print("[5] CSV3 SEMANTIC COLUMN CANDIDATES")
print("=" * 80)

for i, c in enumerate(csv3.columns):
    category = classify_column(c)
    if category:
        print(
            f"{i:03d} | {repr(c)} | "
            f"normalized={normalize_for_diagnostic(c)} | "
            f"candidate={category}"
        )

print("\n" + "=" * 80)
print("[6] CSV4 SEMANTIC COLUMN CANDIDATES")
print("=" * 80)

for i, c in enumerate(csv4.columns):
    category = classify_column(c)
    if category:
        print(
            f"{i:03d} | {repr(c)} | "
            f"normalized={normalize_for_diagnostic(c)} | "
            f"candidate={category}"
        )

# ------------------------------------------------------------
# 5. Sample rows for schema interpretation only
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[7] CSV3 SAMPLE")
print("=" * 80)

display(csv3.head(3))

print("\n" + "=" * 80)
print("[8] CSV4 SAMPLE")
print("=" * 80)

display(csv4.head(3))

print("\n" + "=" * 80)
print("C08R35R2 COMPLETE")
print("এই Cell শুধুমাত্র schema diagnostic করেছে।")
print("কোনো benchmark নির্বাচন/aggregation/deduplication করা হয়নি।")
print("=" * 80)

OIP v1.0.39 — C08R35R2
CSV3 / CSV4 EXACT SCHEMA DIAGNOSTIC

[1] FILE DISCOVERY
Total CSV files: 4
CSV3 candidates: 1
CSV4 candidates: 1

CSV3:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv

CSV4:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

[2] SHAPES
CSV3: (1804, 21)
CSV4: (18352, 7)

[3] CSV3 EXACT COLUMN NAMES
000 | 'Outcome'
001 | 'Benchmark Comparison'
002 | 'County Name'
003 | 'Grouping Type'
004 | 'Conflict Partner Type Group'
005 | 'Waymo Count'
006 | 'Delta V Less Than 1 Mph Percent'
007 | 'Waymo IPMM'
008 | 'Waymo IPMM CI Lower'
009 | 'Waymo IPMM CI Upper'
010 | 'Benchmark IPMM'
011 | 'Benchmark IPMM CI Lower'
012 | 'Benchmark IPMM CI Upper'
013 | 'Predicted Benchmark Count'
014 | 'Waymo Count Reduction'
015 | 'Waymo Percent Reduction'
016 | 'Waymo Percent R

,Outcome,Benchmark Comparison,County Name,Grouping Type,Conflict Partner Type Group,Waymo Count,Delta V Less Than 1 Mph Percent,Waymo IPMM,Waymo IPMM CI Lower,Waymo IPMM CI Upper,...,Benchmark IPMM CI Lower,Benchmark IPMM CI Upper,Predicted Benchmark Count,Waymo Count Reduction,Waymo Percent Reduction,Waymo Percent Reduction CI Lower,Waymo Percent Reduction CI Upper,Waymo Rate Ratio,Waymo Rate Ratio CI Lower,Waymo Rate Ratio CI Upper
0,Any Airbag Deployment,Any Airbag Deployment (non-Dynamic),All Locations (mileage blended),All Crashes,All Crashes,80,0.0375,0.294845,0.233794,0.366960,...,1.439300,1.475485,395.410007,315.410007,-0.797678,-0.845017,-0.740662,0.202322,0.154983,0.259338
1,Any Airbag Deployment,Any Airbag Deployment (non-Dynamic),Atlanta Area,All Crashes,All Crashes,2,0.5000,0.231899,0.028084,0.837698,...,2.451521,2.536471,21.507002,19.507002,-0.907007,-0.992231,-0.622257,0.092993,0.007769,0.377743
2,Any Airbag Deployment,Any Airbag Deployment (non-Dynamic),DeKalb,All Crashes,All Crashes,0,NaN,0.000000,0.000000,3.920119,...,2.796728,2.890377,2.172794,2.172794,-1.000000,-1.000000,0.698070,0.000000,0.000000,1.698070



[8] CSV4 SAMPLE


,state_name,county_name,S2 Cell,Outcome,Benchmark Crash Count,HPMS Yearly Vehicle Miles Traveled,Waymo RO Miles
0,Georgia,DeKalb,9868795034625638400,ka,4.943530,9.359020e+06,0.807896
1,Georgia,DeKalb,9868795034625638400,blincoe_any_injury,62.644913,9.359020e+06,0.807896
2,Georgia,DeKalb,9868795034625638400,airbag,21.830590,9.359020e+06,0.807896



C08R35R2 COMPLETE
এই Cell শুধুমাত্র schema diagnostic করেছে।
কোনো benchmark নির্বাচন/aggregation/deduplication করা হয়নি।


In [110]:
# ============================================================
# OIP v1.0.39 — C08R35R3
# MINIMAL EXACT SCHEMA EXTRACTION
# ============================================================

import os
import glob
import pandas as pd

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

csv3_candidates = [
    p for p in csv_files
    if "collision counts and comparison to benchmarks"
    in os.path.basename(p).lower()
]

csv4_candidates = [
    p for p in csv_files
    if "miles and benchmark crashes for dynamic benchmark"
    in os.path.basename(p).lower()
]

if not csv3_candidates:
    raise RuntimeError("C08R35R3 FAIL_CLOSED: CSV3 not found.")

if not csv4_candidates:
    raise RuntimeError("C08R35R3 FAIL_CLOSED: CSV4 not found.")

csv3 = pd.read_csv(csv3_candidates[0], low_memory=False)
csv4 = pd.read_csv(csv4_candidates[0], low_memory=False)

print("=" * 80)
print("C08R35R3 — EXACT CSV SCHEMA")
print("=" * 80)

print("\nCSV3 FILE:")
print(csv3_candidates[0])

print("\nCSV3 COLUMNS:")
for i, c in enumerate(csv3.columns):
    print(f"{i:03d}: {repr(c)}")

print("\nCSV4 FILE:")
print(csv4_candidates[0])

print("\nCSV4 COLUMNS:")
for i, c in enumerate(csv4.columns):
    print(f"{i:03d}: {repr(c)}")

print("\nCSV3 SHAPE:", csv3.shape)
print("CSV4 SHAPE:", csv4.shape)

print("\nC08R35R3 COMPLETE")

C08R35R3 — EXACT CSV SCHEMA

CSV3 FILE:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv

CSV3 COLUMNS:
000: 'Outcome'
001: 'Benchmark Comparison'
002: 'County Name'
003: 'Grouping Type'
004: 'Conflict Partner Type Group'
005: 'Waymo Count'
006: 'Delta V Less Than 1 Mph Percent'
007: 'Waymo IPMM'
008: 'Waymo IPMM CI Lower'
009: 'Waymo IPMM CI Upper'
010: 'Benchmark IPMM'
011: 'Benchmark IPMM CI Lower'
012: 'Benchmark IPMM CI Upper'
013: 'Predicted Benchmark Count'
014: 'Waymo Count Reduction'
015: 'Waymo Percent Reduction'
016: 'Waymo Percent Reduction CI Lower'
017: 'Waymo Percent Reduction CI Upper'
018: 'Waymo Rate Ratio'
019: 'Waymo Rate Ratio CI Lower'
020: 'Waymo Rate Ratio CI Upper'

CSV4 FILE:
/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv

CSV4 COLUMNS:
000: 'state

In [111]:
# ================================================================
# OIP v1.0.39 — C08R35R4
# CSV3 COMPONENT / AGGREGATION RECONSTRUCTION AUDIT
# Exact schema verified by C08R35R3
# ================================================================

import os
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"

CSV3_NAME = "CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv"
CSV4_NAME = "CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv"

csv3_path = os.path.join(BASE, CSV3_NAME)
csv4_path = os.path.join(BASE, CSV4_NAME)

if not os.path.exists(csv3_path):
    raise RuntimeError("C08R35R4 FAIL_CLOSED: CSV3 file not found.")

if not os.path.exists(csv4_path):
    raise RuntimeError("C08R35R4 FAIL_CLOSED: CSV4 file not found.")

csv3 = pd.read_csv(csv3_path)
csv4 = pd.read_csv(csv4_path)

# ------------------------------------------------
# EXACT SCHEMA ASSERTION
# ------------------------------------------------

required_csv3 = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
    "Predicted Benchmark Count"
]

required_csv4 = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing3 = [c for c in required_csv3 if c not in csv3.columns]
missing4 = [c for c in required_csv4 if c not in csv4.columns]

if missing3:
    raise RuntimeError(
        f"C08R35R4 FAIL_CLOSED: CSV3 missing columns: {missing3}"
    )

if missing4:
    raise RuntimeError(
        f"C08R35R4 FAIL_CLOSED: CSV4 missing columns: {missing4}"
    )

# ------------------------------------------------
# NORMALIZATION
# ------------------------------------------------

def norm(s):
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .str.replace(r"\s+", " ", regex=True)
    )

csv3["_county"] = norm(csv3["County Name"])
csv3["_outcome"] = norm(csv3["Outcome"])
csv3["_benchmark"] = norm(csv3["Benchmark Comparison"])
csv3["_grouping"] = norm(csv3["Grouping Type"])
csv3["_conflict"] = norm(csv3["Conflict Partner Type Group"])

csv4["_state"] = norm(csv4["state_name"])
csv4["_county"] = norm(csv4["county_name"])
csv4["_s2"] = norm(csv4["S2 Cell"])
csv4["_outcome"] = norm(csv4["Outcome"])

csv3["Predicted Benchmark Count"] = pd.to_numeric(
    csv3["Predicted Benchmark Count"],
    errors="coerce"
)

csv4["Benchmark Crash Count"] = pd.to_numeric(
    csv4["Benchmark Crash Count"],
    errors="coerce"
)

# ------------------------------------------------
# CSV4 REPEATED-GRAIN IDENTIFICATION
# ------------------------------------------------

grain = [
    "_state",
    "_county",
    "_s2",
    "_outcome"
]

grouped = (
    csv4.groupby(grain, dropna=False)
        .agg(
            row_count=("Benchmark Crash Count", "size"),
            benchmark_nunique=("Benchmark Crash Count", "nunique")
        )
        .reset_index()
)

repeated = grouped[
    (grouped["row_count"] > 1) &
    (grouped["benchmark_nunique"] > 1)
].copy()

print("=" * 70)
print("C08R35R4 — CSV3 COMPONENT / AGGREGATION RECONSTRUCTION")
print("=" * 70)

print("\nCSV3 shape:", csv3.shape)
print("CSV4 shape:", csv4.shape)

print("\nCSV4 non-identical benchmark repetition groups:",
      len(repeated))

# ------------------------------------------------
# CSV3 STRUCTURE
# ------------------------------------------------

print("\n" + "=" * 70)
print("CSV3 TARGET STRUCTURE — GEORGIA")
print("=" * 70)

ga3 = csv3[
    csv3["_county"].isin(["dekalb", "fulton", "atlanta area"])
].copy()

print("Georgia/Atlanta-area CSV3 rows:", len(ga3))

print("\nGrouping Type:")
print(
    ga3["_grouping"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nConflict Partner Type Group:")
print(
    ga3["_conflict"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nBenchmark Comparison:")
print(
    ga3["_benchmark"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------
# COMPONENT SUMS
# ------------------------------------------------

component = (
    ga3.groupby(
        [
            "_county",
            "_outcome",
            "_benchmark"
        ],
        dropna=False
    )["Predicted Benchmark Count"]
    .agg(
        rows="count",
        total="sum",
        minimum="min",
        maximum="max"
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("CSV3 COMPONENT TOTALS")
print("=" * 70)

print(component.to_string(index=False))

# ------------------------------------------------
# DOCUMENTED OUTCOME RELATIONSHIP CANDIDATES
# ------------------------------------------------

outcome_map = {
    "airbag": "any airbag deployment",
    "ego_airbag": "ego vehicle airbag deployment",
    "observed_any_injury": "any injury",
    "blincoe_any_injury": "any injury",
    "police_reported": "police report",
    "fatal": "any fatal"
}

# ------------------------------------------------
# TEST WHETHER CSV4 VALUES MATCH CSV3 COMPONENT TOTALS
# ------------------------------------------------

tested = 0
matches = []

for _, r in repeated.iterrows():

    county = r["_county"]
    outcome4 = r["_outcome"]

    if county not in ["dekalb", "fulton"]:
        continue

    if outcome4 not in outcome_map:
        continue

    outcome3 = outcome_map[outcome4]

    mask4 = (
        (csv4["_state"] == "georgia") &
        (csv4["_county"] == county) &
        (csv4["_s2"] == r["_s2"]) &
        (csv4["_outcome"] == outcome4)
    )

    values = (
        csv4.loc[
            mask4,
            "Benchmark Crash Count"
        ]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    candidates = component[
        (component["_county"] == county) &
        (component["_outcome"] == outcome3)
    ]

    for value in values:

        tested += 1

        for _, c in candidates.iterrows():

            if np.isclose(
                float(value),
                float(c["total"]),
                rtol=1e-10,
                atol=1e-10
            ):
                matches.append({
                    "county": county,
                    "csv4_outcome": outcome4,
                    "csv4_value": float(value),
                    "csv3_benchmark": c["_benchmark"],
                    "csv3_component_sum": float(c["total"]),
                    "csv3_component_rows": int(c["rows"])
                })

# ------------------------------------------------
# REPORT
# ------------------------------------------------

print("\n" + "=" * 70)
print("RECONSTRUCTION RESULT")
print("=" * 70)

print("Repeated CSV4 benchmark values tested:", tested)
print(
    "Exact CSV3 component-sum matches:",
    len(matches)
)

if matches:

    print("\nMATCH PREVIEW")
    print("-" * 70)

    print(
        pd.DataFrame(matches)
        .drop_duplicates()
        .head(50)
        .to_string(index=False)
    )

else:
    print("No exact component-sum matches found.")

# ------------------------------------------------
# IMPORTANT: MATCH ≠ AUTHORIZATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("FORMAL BOUNDARY")
print("=" * 70)

print("csv3_exact_schema_verified                  : True")
print("csv3_component_structure_observed           : True")
print("csv3_component_reconstruction_tested        : True")
print("benchmark_repetition_structurally_observed  : True")
print("benchmark_repetition_provenance_resolved    : False")
print("benchmark_value_selection_authorized       : False")
print("benchmark_aggregation_authorized            : False")
print("deduplication_authorized                    : False")
print("semantic_equivalence_authorized             : False")
print("claim_authorization                          : False")
print("scientific_authorization                     : False")
print("road_safety_verdict                          : NOT_ISSUED")

print("\nC08R35R4 STATUS : PENDING_EVIDENCE")

print("\nC08R35R4 EXECUTION COMPLETE")

C08R35R4 — CSV3 COMPONENT / AGGREGATION RECONSTRUCTION

CSV3 shape: (1804, 26)
CSV4 shape: (18352, 11)

CSV4 non-identical benchmark repetition groups: 601

CSV3 TARGET STRUCTURE — GEORGIA
Georgia/Atlanta-area CSV3 rows: 490

Grouping Type:
_grouping
aggregate crash type    448
all crashes              42

Conflict Partner Type Group:
_conflict
all crashes         42
all others          42
cyclist             42
motorcycle          42
pedestrian          42
single vehicle      42
v2v backing         42
v2v f2r             42
v2v intersection    42
v2v head-on         42
v2v lateral         42
secondary crash     28

Benchmark Comparison:
_benchmark
any airbag deployment (non-dynamic)            35
any airbag deployment                          35
any fatal                                      35
any fatal (dynamic)                            35
any-injury-reported (non-dynamic)              35
any-injury-reported                            35
any-injury-reported (observed, dynamic)    

In [112]:
# ============================================================
# OIP v1.0.39 — C08R35R5
# RECONSTRUCTION COVERAGE RECONCILIATION
#
# PURPOSE:
#   Reconcile the locked 1,202 repeated CSV4 benchmark values
#   against the 888 values actually tested by C08R35R4.
#
# THIS CELL IS DIAGNOSTIC ONLY.
#
# It does NOT:
#   - select a benchmark
#   - aggregate benchmarks
#   - delete duplicates
#   - authorize semantic equivalence
#   - resolve provenance
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

print("=" * 88)
print("OIP v1.0.39 — C08R35R5")
print("RECONSTRUCTION COVERAGE RECONCILIATION")
print("=" * 88)

# ------------------------------------------------------------
# 1. Locate and read CSV4
# ------------------------------------------------------------

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

csv4_candidates = [
    p for p in csv_files
    if os.path.basename(p).lower()
    == "csv4 - miles and benchmark crashes for dynamic benchmark 202009-202606-2022benchmark.csv".lower()
]

if len(csv4_candidates) != 1:
    raise RuntimeError(
        f"C08R35R5 FAIL_CLOSED: expected exactly 1 CSV4 file, "
        f"found {len(csv4_candidates)}."
    )

csv4 = pd.read_csv(csv4_candidates[0], low_memory=False)

EXPECTED_CSV4 = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles",
]

if list(csv4.columns) != EXPECTED_CSV4:
    raise RuntimeError(
        "C08R35R5 FAIL_CLOSED: CSV4 schema differs from "
        "the verified C08R35R3 schema."
    )

# ------------------------------------------------------------
# 2. Audit-only normalized fields
# ------------------------------------------------------------

def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )

csv4["_state"] = clean_text(csv4["state_name"])
csv4["_county"] = clean_text(csv4["county_name"])
csv4["_s2"] = clean_text(csv4["S2 Cell"])
csv4["_outcome"] = clean_text(csv4["Outcome"])

csv4["_benchmark"] = pd.to_numeric(
    csv4["Benchmark Crash Count"],
    errors="coerce"
)

KEYS = ["_state", "_county", "_s2", "_outcome"]

# ------------------------------------------------------------
# 3. Reconstruct the locked 601 repeated groups
# ------------------------------------------------------------

grain = (
    csv4
    .groupby(KEYS, dropna=False, sort=False)
    .agg(
        row_count=("_benchmark", "size"),
        benchmark_nunique=("_benchmark", "nunique"),
    )
    .reset_index()
)

repeated = grain[
    grain["benchmark_nunique"] > 1
].copy()

print("\n[1] LOCKED REPEATED-GRAIN RECONSTRUCTION")
print("Total CSV4 rows:", len(csv4))
print("Total grains:", len(grain))
print("Material repeated grains:", len(repeated))

if len(repeated) != 601:
    raise RuntimeError(
        f"C08R35R5 FAIL_CLOSED: expected 601 repeated grains, "
        f"found {len(repeated)}."
    )

# ------------------------------------------------------------
# 4. Extract every one of the expected 1,202 benchmark values
# ------------------------------------------------------------

target_rows = (
    csv4[
        KEYS
        + [
            "_benchmark",
            "HPMS Yearly Vehicle Miles Traveled",
            "Waymo RO Miles",
        ]
    ]
    .merge(
        repeated[KEYS],
        on=KEYS,
        how="inner",
    )
    .copy()
)

print("\n[2] EXPECTED REPEATED-VALUE SET")
print("Expected repeated groups:", 601)
print("Expected repeated benchmark values:", 1202)
print("Actual extracted repeated rows:", len(target_rows))

if len(target_rows) != 1202:
    raise RuntimeError(
        f"C08R35R5 FAIL_CLOSED: expected 1,202 repeated rows, "
        f"found {len(target_rows)}."
    )

# ------------------------------------------------------------
# 5. Recreate the outcome mapping used in C08R35R4
# ------------------------------------------------------------

candidate_map = {
    "airbag": [
        "any airbag deployment",
        "any airbag deployment (non-dynamic)",
    ],
    "ego_airbag": [
        "ego vehicle airbag deployment",
        "ego vehicle airbag deployment (non-dynamic)",
    ],
    "observed_any_injury": [
        "any-injury-reported",
        "any-injury-reported (observed, dynamic)",
        "any-injury-reported (non-dynamic)",
        "any-injury-reported (observed, non-dynamic)",
    ],
    "blincoe_any_injury": [
        "any-injury-reported",
        "any-injury-reported (non-dynamic)",
    ],
    "police_reported": [
        "police-reported",
        "police-reported (non-dynamic)",
    ],
    "fatal": [
        "any fatal",
        "any fatal (dynamic)",
    ],
}

target_rows["_mapping_used_in_r35r4"] = target_rows[
    "_outcome"
].isin(candidate_map)

# ------------------------------------------------------------
# 6. Coverage by outcome
# ------------------------------------------------------------

coverage = (
    target_rows
    .groupby(
        ["_outcome", "_mapping_used_in_r35r4"],
        dropna=False,
        sort=True
    )
    .size()
    .reset_index(name="rows")
)

print("\n[3] COVERAGE BY OUTCOME")
print(coverage.to_string(index=False))

# ------------------------------------------------------------
# 7. Exact expected-versus-tested reconciliation
# ------------------------------------------------------------

expected_by_outcome = (
    target_rows
    .groupby("_outcome", dropna=False)
    .size()
    .reset_index(name="expected_rows")
)

tested_by_outcome = (
    target_rows[
        target_rows["_mapping_used_in_r35r4"]
    ]
    .groupby("_outcome", dropna=False)
    .size()
    .reset_index(name="tested_rows")
)

reconciliation = expected_by_outcome.merge(
    tested_by_outcome,
    on="_outcome",
    how="left"
)

reconciliation["tested_rows"] = (
    reconciliation["tested_rows"]
    .fillna(0)
    .astype(int)
)

reconciliation["untested_rows"] = (
    reconciliation["expected_rows"]
    - reconciliation["tested_rows"]
)

print("\n[4] EXPECTED VS TESTED")
print(reconciliation.to_string(index=False))

# ------------------------------------------------------------
# 8. Identify exact untested 314 rows
# ------------------------------------------------------------

untested = target_rows[
    ~target_rows["_mapping_used_in_r35r4"]
].copy()

print("\n[5] UNTESTED REPEATED BENCHMARK VALUES")
print("Expected:", len(target_rows))
print("Tested by prior mapping:", int(
    target_rows["_mapping_used_in_r35r4"].sum()
))
print("Untested:", len(untested))

# ------------------------------------------------------------
# 9. Check whether the untested rows are concentrated in
#    specific counties / outcomes.
# ------------------------------------------------------------

if len(untested) > 0:

    print("\nUntested rows by outcome:")
    print(
        untested["_outcome"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

    print("\nUntested rows by county:")
    print(
        untested["_county"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

    print("\nUntested outcome × county:")
    display(
        pd.crosstab(
            untested["_outcome"],
            untested["_county"]
        )
    )

# ------------------------------------------------------------
# 10. Verify no row was lost because of NaN / duplicate handling.
# ------------------------------------------------------------

print("\n[6] SOURCE-PRESERVATION CHECKS")

print(
    "Target repeated rows with missing benchmark:",
    int(target_rows["_benchmark"].isna().sum())
)

print(
    "Target repeated rows with duplicated full source row:",
    int(
        target_rows.duplicated(
            subset=KEYS + ["_benchmark"]
        ).sum()
    )
)

print(
    "No source rows were dropped using drop_duplicates()."
)

# ------------------------------------------------------------
# 11. Formal status
# ------------------------------------------------------------

tested_count = int(
    target_rows["_mapping_used_in_r35r4"].sum()
)

untested_count = int(
    len(target_rows) - tested_count
)

print("\n" + "=" * 88)
print("C08R35R5 FORMAL STATUS")
print("=" * 88)

print("Expected repeated benchmark values :", len(target_rows))
print("Previously tested values           :", tested_count)
print("Previously untested values         :", untested_count)

if len(target_rows) == 1202 and tested_count == 888 and untested_count == 314:
    print(
        "\nRESULT: COVERAGE GAP CONFIRMED"
    )
    print(
        "The previous C08R35R4 reconstruction covered 888 of "
        "the 1,202 locked repeated benchmark values."
    )
    print(
        "The remaining 314 values must not be treated as "
        "reconstructed or provenance-resolved."
    )
else:
    print(
        "\nRESULT: COVERAGE REQUIRES FURTHER RECONCILIATION"
    )

print("\nbenchmark_repetition_provenance_resolved = FALSE")
print("benchmark_value_selection_authorized = FALSE")
print("benchmark_aggregation_authorized = FALSE")
print("deduplication_authorized = FALSE")
print("claim_authorization = FALSE")
print("scientific_authorization = FALSE")
print("road_safety_verdict = NOT_ISSUED")

print("=" * 88)
print("C08R35R5 COMPLETE")
print("=" * 88)

OIP v1.0.39 — C08R35R5
RECONSTRUCTION COVERAGE RECONCILIATION

[1] LOCKED REPEATED-GRAIN RECONSTRUCTION
Total CSV4 rows: 18352
Total grains: 17672
Material repeated grains: 601

[2] EXPECTED REPEATED-VALUE SET
Expected repeated groups: 601
Expected repeated benchmark values: 1202
Actual extracted repeated rows: 1202

[3] COVERAGE BY OUTCOME
           _outcome  _mapping_used_in_r35r4  rows
             airbag                    True   164
            blincoe                   False   170
 blincoe_any_injury                    True   168
         ego_airbag                    True   164
              fatal                    True    54
                 ka                   False   144
observed_any_injury                    True   168
    police_reported                    True   170

[4] EXPECTED VS TESTED
           _outcome  expected_rows  tested_rows  untested_rows
             airbag            164          164              0
            blincoe            170            0          

_county,dekalb,fulton
_outcome,,
blincoe,36,134
ka,28,116



[6] SOURCE-PRESERVATION CHECKS
Target repeated rows with missing benchmark: 0
Target repeated rows with duplicated full source row: 0
No source rows were dropped using drop_duplicates().

C08R35R5 FORMAL STATUS
Expected repeated benchmark values : 1202
Previously tested values           : 888
Previously untested values         : 314

RESULT: COVERAGE GAP CONFIRMED
The previous C08R35R4 reconstruction covered 888 of the 1,202 locked repeated benchmark values.
The remaining 314 values must not be treated as reconstructed or provenance-resolved.

benchmark_repetition_provenance_resolved = FALSE
benchmark_value_selection_authorized = FALSE
benchmark_aggregation_authorized = FALSE
deduplication_authorized = FALSE
claim_authorization = FALSE
scientific_authorization = FALSE
road_safety_verdict = NOT_ISSUED
C08R35R5 COMPLETE


In [113]:
# ============================================================
# OIP v1.0.39 — C08R35R6
# PREVIOUSLY-UNTESTED 314 BENCHMARK VALUES — COVERAGE AUDIT
# NON-AUTHORITATIVE DIAGNOSTIC ONLY
#
# Purpose:
#   1. Reconstruct the locked 1,202 repeated benchmark values.
#   2. Reproduce the exact R35R4 mapping boundary.
#   3. Identify the 314 values excluded from R35R4.
#   4. Characterize them by Outcome / County / State / S2 Cell.
#
# NON-NEGOTIABLE:
#   - No benchmark value selection.
#   - No deletion.
#   - No aggregation.
#   - No averaging/min/max.
#   - No replacement.
#   - No semantic authorization.
#   - No claim authorization.
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 88)
print("C08R35R6 — PREVIOUSLY-UNTESTED 314 VALUE COVERAGE AUDIT")
print("=" * 88)

# ------------------------------------------------------------
# [1] LOCATE EXACT SOURCE
# ------------------------------------------------------------
base = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"

csv4_name = "CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv"
csv4_path = os.path.join(base, csv4_name)

if not os.path.exists(csv4_path):
    raise FileNotFoundError(f"C08R35R6 FAIL_CLOSED: CSV4 not found: {csv4_path}")

csv4 = pd.read_csv(csv4_path)

print("\n[1] CSV4 SOURCE")
print("Path:", csv4_path)
print("Shape:", csv4.shape)
print("Columns:", list(csv4.columns))

required = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing = [c for c in required if c not in csv4.columns]

if missing:
    raise RuntimeError(
        f"C08R35R6 FAIL_CLOSED: required CSV4 columns missing: {missing}"
    )

# ------------------------------------------------------------
# [2] NORMALIZED AUDIT FIELDS
# ------------------------------------------------------------
df = csv4.copy()

df["_state_norm"] = df["state_name"].astype("string").str.strip().str.lower()
df["_county_norm"] = df["county_name"].astype("string").str.strip().str.lower()
df["_s2_norm"] = df["S2 Cell"].astype("string").str.strip()
df["_outcome_norm"] = df["Outcome"].astype("string").str.strip().str.lower()

df["_benchmark_num"] = pd.to_numeric(
    df["Benchmark Crash Count"], errors="coerce"
)

df["_hpms_num"] = pd.to_numeric(
    df["HPMS Yearly Vehicle Miles Traveled"], errors="coerce"
)

df["_waymo_ro_num"] = pd.to_numeric(
    df["Waymo RO Miles"], errors="coerce"
)

grain = [
    "_state_norm",
    "_county_norm",
    "_s2_norm",
    "_outcome_norm"
]

# ------------------------------------------------------------
# [3] RECONSTRUCT LOCKED REPEATED GRAINS
# ------------------------------------------------------------
grp = (
    df.groupby(grain, dropna=False)
      .agg(
          row_count=("_benchmark_num", "size"),
          benchmark_nunique=("_benchmark_num", "nunique"),
          hpms_nunique=("_hpms_num", "nunique"),
          waymo_ro_nunique=("_waymo_ro_num", "nunique")
      )
      .reset_index()
)

repeated = grp[
    (grp["row_count"] > 1) &
    (grp["benchmark_nunique"] > 1)
].copy()

print("\n[3] LOCKED REPEATED-GRAIN RECONSTRUCTION")
print("Material repeated grains:", len(repeated))

if len(repeated) != 601:
    raise RuntimeError(
        f"C08R35R6 FAIL_CLOSED: expected 601 material repeated groups, "
        f"found {len(repeated)}"
    )

# ------------------------------------------------------------
# [4] EXTRACT ALL 1,202 REPEATED BENCHMARK VALUES
# ------------------------------------------------------------
target = df.merge(
    repeated[grain],
    on=grain,
    how="inner"
).copy()

print("Target repeated rows:", len(target))

if len(target) != 1202:
    raise RuntimeError(
        f"C08R35R6 FAIL_CLOSED: expected 1,202 repeated rows, "
        f"found {len(target)}"
    )

# Preserve a deterministic source-row identity.
target["_source_row_id"] = target.index

# ------------------------------------------------------------
# [5] EXACT R35R4 MAPPING BOUNDARY
# ------------------------------------------------------------
mapping = {
    "airbag": [
        "any airbag deployment",
        "any airbag deployment (non-dynamic)"
    ],
    "ego_airbag": [
        "ego vehicle airbag deployment",
        "ego vehicle airbag deployment (non-dynamic)"
    ],
    "observed_any_injury": [
        "any-injury-reported",
        "any-injury-reported (non-dynamic)",
        "any-injury-reported (observed, dynamic)",
        "any-injury-reported (observed, non-dynamic)"
    ],
    "blincoe_any_injury": [
        "any-injury-reported",
        "any-injury-reported (non-dynamic)"
    ],
    "police_reported": [
        "police-reported",
        "police-reported (non-dynamic)"
    ],
    "fatal": [
        "any fatal",
        "any fatal (dynamic)"
    ]
}

target["_mapping_used_in_r35r4"] = False

for outcome, labels in mapping.items():
    mask = (
        target["_outcome_norm"].eq(outcome) &
        target["_outcome_norm"].notna()
    )
    target.loc[mask, "_mapping_used_in_r35r4"] = True

# ------------------------------------------------------------
# [6] COVERAGE ACCOUNTING
# ------------------------------------------------------------
tested = target[target["_mapping_used_in_r35r4"]].copy()
untested = target[~target["_mapping_used_in_r35r4"]].copy()

print("\n[6] COVERAGE ACCOUNTING")
print("Expected repeated benchmark values :", len(target))
print("Previously tested values           :", 888)
print("R35R4 mapping-covered rows         :", len(tested))
print("Previously untested values         :", len(untested))

# The purpose is to reproduce the known R35R4 boundary.
if len(untested) != 314:
    raise RuntimeError(
        f"C08R35R6 FAIL_CLOSED: expected 314 previously-untested rows "
        f"under the locked R35R4 mapping boundary, found {len(untested)}"
    )

# ------------------------------------------------------------
# [7] OUTCOME COVERAGE
# ------------------------------------------------------------
print("\n[7] COVERAGE BY OUTCOME")
coverage = (
    target.groupby("_outcome_norm", dropna=False)
          .agg(
              expected_rows=("_source_row_id", "size"),
              r35r4_mapped=("_mapping_used_in_r35r4", "sum")
          )
          .reset_index()
)

coverage["untested_rows"] = (
    coverage["expected_rows"] - coverage["r35r4_mapped"]
)

print(coverage.to_string(index=False))

# ------------------------------------------------------------
# [8] EXACT UNTESTED 314 — OUTCOME × COUNTY × STATE
# ------------------------------------------------------------
print("\n[8] PREVIOUSLY-UNTESTED VALUES — OUTCOME × COUNTY × STATE")

untested_summary = (
    untested.groupby(
        ["_state_norm", "_county_norm", "_outcome_norm"],
        dropna=False
    )
    .agg(
        untested_rows=("_source_row_id", "size"),
        distinct_benchmark_values=("_benchmark_num", "nunique")
    )
    .reset_index()
    .sort_values(
        ["_state_norm", "_county_norm", "_outcome_norm"]
    )
)

print(untested_summary.to_string(index=False))

# ------------------------------------------------------------
# [9] EXACT UNTESTED 314 — OUTCOME ONLY
# ------------------------------------------------------------
print("\n[9] UNTESTED COUNT BY OUTCOME")

outcome_only = (
    untested.groupby("_outcome_norm", dropna=False)
            .agg(
                rows=("_source_row_id", "size"),
                distinct_benchmark_values=("_benchmark_num", "nunique")
            )
            .reset_index()
            .sort_values("rows", ascending=False)
)

print(outcome_only.to_string(index=False))

# ------------------------------------------------------------
# [10] EXACT UNTESTED 314 — COUNTY ONLY
# ------------------------------------------------------------
print("\n[10] UNTESTED COUNT BY COUNTY")

county_only = (
    untested.groupby(
        ["_state_norm", "_county_norm"],
        dropna=False
    )
    .agg(
        rows=("_source_row_id", "size"),
        distinct_benchmark_values=("_benchmark_num", "nunique")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

print(county_only.to_string(index=False))

# ------------------------------------------------------------
# [11] S2-CELL COVERAGE
# ------------------------------------------------------------
print("\n[11] UNTESTED COUNT BY S2 CELL")

s2_only = (
    untested.groupby(
        ["_state_norm", "_county_norm", "_s2_norm"],
        dropna=False
    )
    .agg(
        rows=("_source_row_id", "size"),
        outcomes=("_outcome_norm", "nunique"),
        distinct_benchmark_values=("_benchmark_num", "nunique")
    )
    .reset_index()
)

print("Distinct S2 cells containing untested rows:", len(s2_only))
print("Total untested rows:", s2_only["rows"].sum())

# ------------------------------------------------------------
# [12] SOURCE-PRESERVATION CHECKS
# ------------------------------------------------------------
print("\n[12] SOURCE-PRESERVATION CHECKS")

missing_benchmark = int(
    untested["_benchmark_num"].isna().sum()
)

duplicate_source_rows = int(
    untested.duplicated(
        subset=[
            "state_name",
            "county_name",
            "S2 Cell",
            "Outcome",
            "Benchmark Crash Count",
            "HPMS Yearly Vehicle Miles Traveled",
            "Waymo RO Miles"
        ],
        keep=False
    ).sum()
)

print("Untested rows with missing benchmark:", missing_benchmark)
print(
    "Untested rows participating in duplicated full source rows:",
    duplicate_source_rows
)

if missing_benchmark != 0:
    raise RuntimeError(
        "C08R35R6 FAIL_CLOSED: untested benchmark values contain missing values."
    )

# ------------------------------------------------------------
# [13] NO-SILENT-LOSS ASSERTION
# ------------------------------------------------------------
print("\n[13] NO-SILENT-LOSS ASSERTION")

print("601 material repeated groups expected :", len(repeated))
print("1,202 repeated rows expected          :", len(target))
print("888 previously mapped                 :", len(tested))
print("314 previously untested               :", len(untested))
print("Mapped + untested                     :", len(tested) + len(untested))

if len(tested) + len(untested) != len(target):
    raise RuntimeError(
        "C08R35R6 FAIL_CLOSED: coverage partition does not reconcile."
    )

# ------------------------------------------------------------
# [14] FORMAL BOUNDARY
# ------------------------------------------------------------
print("\n" + "=" * 88)
print("C08R35R6 FORMAL STATUS")
print("=" * 88)

print("Expected repeated benchmark values :", len(target))
print("R35R4 mapped values                :", len(tested))
print("Previously untested values         :", len(untested))

if len(untested) == 314:
    print("\nRESULT: 314-VALUE COVERAGE GAP FULLY CHARACTERIZED")
else:
    print("\nRESULT: COVERAGE RECONCILIATION FAILED")

print("\nIMPORTANT:")
print("The 314 values are identified as previously untested only.")
print("They are NOT treated as invalid, erroneous, duplicate, superseded,")
print("or scientifically equivalent to any other benchmark value.")

print("\nbenchmark_repetition_provenance_resolved = FALSE")
print("benchmark_value_selection_authorized = FALSE")
print("benchmark_aggregation_authorized = FALSE")
print("deduplication_authorized = FALSE")
print("semantic_equivalence_authorized = FALSE")
print("claim_authorization = FALSE")
print("scientific_authorization = FALSE")
print("road_safety_verdict = NOT_ISSUED")

print("=" * 88)
print("C08R35R6 COMPLETE")
print("=" * 88)

C08R35R6 — PREVIOUSLY-UNTESTED 314 VALUE COVERAGE AUDIT

[1] CSV4 SOURCE
Path: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv
Shape: (18352, 7)
Columns: ['state_name', 'county_name', 'S2 Cell', 'Outcome', 'Benchmark Crash Count', 'HPMS Yearly Vehicle Miles Traveled', 'Waymo RO Miles']

[3] LOCKED REPEATED-GRAIN RECONSTRUCTION
Material repeated grains: 601
Target repeated rows: 1202

[6] COVERAGE ACCOUNTING
Expected repeated benchmark values : 1202
Previously tested values           : 888
R35R4 mapping-covered rows         : 888
Previously untested values         : 314

[7] COVERAGE BY OUTCOME
      _outcome_norm  expected_rows  r35r4_mapped  untested_rows
             airbag            164           164              0
            blincoe            170             0            170
 blincoe_any_injury            168           168              0
         ego_airbag            16

In [114]:
# ============================================================
# OIP v1.0.39 — C08R35R7
# BLINCOE / KA → CSV3 STRUCTURAL COUNTERPART AUDIT
# NON-AUTHORITATIVE — NO SEMANTIC AUTHORIZATION
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 88)
print("C08R35R7 — BLINCOE / KA CSV3 STRUCTURAL COUNTERPART AUDIT")
print("=" * 88)

base = "/kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit"

csv3_name = "CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv"
csv4_name = "CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv"

csv3_path = os.path.join(base, csv3_name)
csv4_path = os.path.join(base, csv4_name)

if not os.path.exists(csv3_path):
    raise FileNotFoundError(f"C08R35R7 FAIL_CLOSED: CSV3 not found: {csv3_path}")

if not os.path.exists(csv4_path):
    raise FileNotFoundError(f"C08R35R7 FAIL_CLOSED: CSV4 not found: {csv4_path}")

csv3 = pd.read_csv(csv3_path)
csv4 = pd.read_csv(csv4_path)

print("\n[1] SOURCE CHECK")
print("CSV3 shape:", csv3.shape)
print("CSV4 shape:", csv4.shape)

required_csv3 = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
    "Waymo Count",
    "Predicted Benchmark Count",
    "Waymo Count Reduction",
    "Waymo Percent Reduction",
    "Waymo Rate Ratio"
]

missing_csv3 = [c for c in required_csv3 if c not in csv3.columns]

if missing_csv3:
    raise RuntimeError(
        f"C08R35R7 FAIL_CLOSED: CSV3 required columns missing: {missing_csv3}"
    )

# ------------------------------------------------------------
# [2] NORMALIZE ONLY FOR INSPECTION
# ------------------------------------------------------------
c3 = csv3.copy()

for col in ["Outcome", "Benchmark Comparison", "County Name",
            "Grouping Type", "Conflict Partner Type Group"]:
    c3[f"_{col}_norm"] = (
        c3[col].astype("string").str.strip().str.lower()
    )

c4 = csv4.copy()

c4["_outcome_norm"] = (
    c4["Outcome"].astype("string").str.strip().str.lower()
)

c4["_county_norm"] = (
    c4["county_name"].astype("string").str.strip().str.lower()
)

c4["_benchmark_num"] = pd.to_numeric(
    c4["Benchmark Crash Count"], errors="coerce"
)

# ------------------------------------------------------------
# [3] VERIFY THE 314 TARGET ROWS
# ------------------------------------------------------------
grain = [
    "_state_norm",
    "_county_norm",
    "_s2_norm",
    "_outcome_norm"
]

for col in ["state_name", "county_name", "S2 Cell", "Outcome"]:
    if col not in c4.columns:
        raise RuntimeError(
            f"C08R35R7 FAIL_CLOSED: CSV4 missing {col}"
        )

c4["_state_norm"] = c4["state_name"].astype("string").str.strip().str.lower()
c4["_s2_norm"] = c4["S2 Cell"].astype("string").str.strip()

grp = (
    c4.groupby(grain, dropna=False)
      .agg(
          row_count=("_benchmark_num", "size"),
          benchmark_nunique=("_benchmark_num", "nunique")
      )
      .reset_index()
)

repeated = grp[
    (grp["row_count"] > 1) &
    (grp["benchmark_nunique"] > 1)
].copy()

if len(repeated) != 601:
    raise RuntimeError(
        f"C08R35R7 FAIL_CLOSED: expected 601 repeated grains, found {len(repeated)}"
    )

target = c4.merge(
    repeated[grain],
    on=grain,
    how="inner"
).copy()

if len(target) != 1202:
    raise RuntimeError(
        f"C08R35R7 FAIL_CLOSED: expected 1,202 repeated rows, found {len(target)}"
    )

target_314 = target[
    target["_outcome_norm"].isin(["blincoe", "ka"])
].copy()

print("\n[2] TARGET COVERAGE")
print("Total repeated rows:", len(target))
print("blincoe + ka rows:", len(target_314))

if len(target_314) != 314:
    raise RuntimeError(
        f"C08R35R7 FAIL_CLOSED: expected 314 blincoe/ka rows, found {len(target_314)}"
    )

# ------------------------------------------------------------
# [4] EXACT CSV3 OUTCOME LABEL INVENTORY
# ------------------------------------------------------------
print("\n[3] CSV3 OUTCOME LABELS")

outcomes = (
    c3[["_Outcome_norm"]]
    .drop_duplicates()
    .sort_values("_Outcome_norm")
)

print(outcomes.to_string(index=False))

# ------------------------------------------------------------
# [5] SEARCH CSV3 FOR POTENTIAL BLINCOE / KA STRUCTURAL TERMS
# ------------------------------------------------------------
search_terms = [
    "blincoe",
    "ka",
    "serious injury",
    "suspected serious",
    "injury"
]

print("\n[4] CSV3 STRUCTURAL TERM SEARCH")

for term in search_terms:
    mask = (
        c3["_Outcome_norm"].fillna("").str.contains(term, regex=False) |
        c3["_Benchmark Comparison_norm"].fillna("").str.contains(term, regex=False)
    )

    hits = c3.loc[
        mask,
        [
            "Outcome",
            "Benchmark Comparison",
            "County Name",
            "Grouping Type",
            "Conflict Partner Type Group"
        ]
    ].drop_duplicates()

    print("\nTERM:", repr(term))
    print("Rows:", len(hits))

    if len(hits):
        print(hits.to_string(index=False))

# ------------------------------------------------------------
# [6] FULL BENCHMARK-COMPARISON LABEL INVENTORY
# ------------------------------------------------------------
print("\n[5] FULL CSV3 BENCHMARK COMPARISON LABELS")

labels = (
    c3["Benchmark Comparison"]
    .astype("string")
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

print("Unique Benchmark Comparison labels:", len(labels))

for x in labels.tolist():
    print(" -", x)

# ------------------------------------------------------------
# [7] COUNTY STRUCTURE FOR TARGET GEORGIA COUNTIES
# ------------------------------------------------------------
print("\n[6] CSV3 GEORGIA TARGET-COUNTY STRUCTURE")

ga_target = c3[
    c3["_County Name_norm"].isin(["dekalb", "fulton"])
].copy()

print("CSV3 DeKalb/Fulton rows:", len(ga_target))

county_structure = (
    ga_target.groupby(
        [
            "_County Name_norm",
            "_Outcome_norm",
            "_Benchmark Comparison_norm"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["_County Name_norm", "_Outcome_norm", "_Benchmark Comparison_norm"]
    )
)

print(county_structure.to_string(index=False))

# ------------------------------------------------------------
# [8] IMPORTANT: NO MAPPING DECISION
# ------------------------------------------------------------
print("\n[7] AUTHORIZATION BOUNDARY")

print("blincoe_csv3_semantic_equivalence_authorized = FALSE")
print("ka_csv3_semantic_equivalence_authorized = FALSE")
print("benchmark_value_selection_authorized = FALSE")
print("benchmark_aggregation_authorized = FALSE")
print("deduplication_authorized = FALSE")
print("claim_authorization = FALSE")
print("scientific_authorization = FALSE")
print("road_safety_verdict = NOT_ISSUED")

# ------------------------------------------------------------
# [9] FORMAL STATUS
# ------------------------------------------------------------
print("\n" + "=" * 88)
print("C08R35R7 FORMAL STATUS")
print("=" * 88)

print("314 target rows verified:", len(target_314))
print("CSV3 structural inventory completed.")
print("No semantic equivalence has been authorized.")
print("No benchmark value has been selected, merged, averaged, deleted,")
print("or replaced.")

print("\nRESULT: STRUCTURAL COUNTERPART INVENTORY COMPLETE")
print("STATUS: PENDING_EVIDENCE")

print("=" * 88)
print("C08R35R7 COMPLETE")
print("=" * 88)

C08R35R7 — BLINCOE / KA CSV3 STRUCTURAL COUNTERPART AUDIT

[1] SOURCE CHECK
CSV3 shape: (1804, 21)
CSV4 shape: (18352, 7)

[2] TARGET COVERAGE
Total repeated rows: 1202
blincoe + ka rows: 314

[3] CSV3 OUTCOME LABELS
                _Outcome_norm
        any airbag deployment
                    any fatal
                   any injury
ego vehicle airbag deployment
                police report
    suspected serious injury+

[4] CSV3 STRUCTURAL TERM SEARCH

TERM: 'blincoe'
Rows: 0

TERM: 'ka'
Rows: 0

TERM: 'serious injury'
Rows: 258
                  Outcome                    Benchmark Comparison                     County Name        Grouping Type Conflict Partner Type Group
Suspected Serious Injury+ Suspected Serious Injury+ (non-Dynamic) All Locations (mileage blended)          All Crashes                 All Crashes
Suspected Serious Injury+ Suspected Serious Injury+ (non-Dynamic)                    Atlanta Area          All Crashes                 All Crashes
Suspected Serious In

In [115]:
# ================================================================
# OIP v1.0.39 — C08R35R8
# LOCKED 314 BOUNDARY + SOURCE DOCUMENT EVIDENCE RESOLUTION
#
# PURPOSE:
#   1. Preserve the exact R35R4/R35R6 314-row boundary.
#   2. Inspect Release Notes for source-backed documentary context
#      concerning BLINCOE and KA.
#   3. Test whether any explicit documentary rule explains:
#        - BLINCOE / KA counterpart relationships
#        - repeated benchmark values within the same grain
#        - version / historical / amendment distinctions
#   4. Preserve fail-closed status unless an explicit rule is found.
#
# NON-AUTHORITATIVE:
#   Documentary occurrence != semantic equivalence
#   Documentary occurrence != construct authorization
#   Numeric equality != provenance
#   Structural counterpart != semantic counterpart
#
# NO:
#   benchmark selection
#   aggregation
#   averaging
#   deduplication
#   row deletion
#   replacement
#   semantic equivalence authorization
#   claim authorization
#   scientific authorization
# ================================================================

import os
import re
import json
import hashlib
from pathlib import Path
from datetime import datetime, UTC

import pandas as pd

try:
    from pypdf import PdfReader
except Exception as e:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: pypdf unavailable: "
        + repr(e)
    )

print("=" * 88)
print("OIP v1.0.39 — C08R35R8")
print("LOCKED 314 BOUNDARY + SOURCE DOCUMENT EVIDENCE RESOLUTION")
print("=" * 88)


# ================================================================
# [1] SOURCE DISCOVERY
# ================================================================

INPUT_ROOT = Path("/kaggle/input")

csv3_candidates = sorted(
    [
        p for p in INPUT_ROOT.rglob("*.csv")
        if "collision counts and comparison to benchmarks"
        in p.name.lower()
    ]
)

csv4_candidates = sorted(
    [
        p for p in INPUT_ROOT.rglob("*.csv")
        if "miles and benchmark crashes for dynamic benchmark"
        in p.name.lower()
    ]
)

release_candidates = sorted(
    [
        p for p in INPUT_ROOT.rglob("*.pdf")
        if "release_notes" in p.name.lower()
    ]
)

if len(csv3_candidates) != 1:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: expected exactly 1 CSV3, "
        f"found {len(csv3_candidates)}"
    )

if len(csv4_candidates) != 1:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: expected exactly 1 CSV4, "
        f"found {len(csv4_candidates)}"
    )

if len(release_candidates) != 1:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: expected exactly 1 Release Notes PDF, "
        f"found {len(release_candidates)}"
    )

csv3_path = csv3_candidates[0]
csv4_path = csv4_candidates[0]
release_path = release_candidates[0]

print("\n[1] SOURCE FILES")
print("CSV3:", csv3_path)
print("CSV4:", csv4_path)
print("Release Notes:", release_path)


# ================================================================
# [2] READ SOURCE FILES
# ================================================================

csv3 = pd.read_csv(csv3_path, low_memory=False)
csv4 = pd.read_csv(csv4_path, low_memory=False)

required_csv3 = [
    "Outcome",
    "Benchmark Comparison",
    "County Name",
    "Grouping Type",
    "Conflict Partner Type Group",
    "Predicted Benchmark Count"
]

required_csv4 = [
    "state_name",
    "county_name",
    "S2 Cell",
    "Outcome",
    "Benchmark Crash Count",
    "HPMS Yearly Vehicle Miles Traveled",
    "Waymo RO Miles"
]

missing_csv3 = [
    c for c in required_csv3
    if c not in csv3.columns
]

missing_csv4 = [
    c for c in required_csv4
    if c not in csv4.columns
]

if missing_csv3:
    raise RuntimeError(
        f"C08R35R8 FAIL_CLOSED: CSV3 missing columns: {missing_csv3}"
    )

if missing_csv4:
    raise RuntimeError(
        f"C08R35R8 FAIL_CLOSED: CSV4 missing columns: {missing_csv4}"
    )

print("\n[2] SOURCE SHAPES")
print("CSV3:", csv3.shape)
print("CSV4:", csv4.shape)


# ================================================================
# [3] NORMALIZATION FOR AUDIT ONLY
# ================================================================

def norm_text_value(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x


csv4["_state_norm"] = (
    csv4["state_name"].map(norm_text_value)
)

csv4["_county_norm"] = (
    csv4["county_name"].map(norm_text_value)
)

csv4["_s2_norm"] = (
    csv4["S2 Cell"].map(norm_text_value)
)

csv4["_outcome_norm"] = (
    csv4["Outcome"].map(norm_text_value)
)

csv4["_benchmark_num"] = pd.to_numeric(
    csv4["Benchmark Crash Count"],
    errors="coerce"
)

csv3["_county_norm"] = (
    csv3["County Name"].map(norm_text_value)
)

csv3["_outcome_norm"] = (
    csv3["Outcome"].map(norm_text_value)
)

csv3["_benchmark_comparison_norm"] = (
    csv3["Benchmark Comparison"].map(norm_text_value)
)


# ================================================================
# [4] RECONSTRUCT THE LOCKED 601 REPEATED GRAINS
# ================================================================

grain = [
    "_state_norm",
    "_county_norm",
    "_s2_norm",
    "_outcome_norm"
]

grouped = (
    csv4.groupby(
        grain,
        dropna=False
    )
    .agg(
        row_count=("_benchmark_num", "size"),
        benchmark_nunique=("_benchmark_num", "nunique")
    )
    .reset_index()
)

repeated = grouped[
    (grouped["row_count"] > 1) &
    (grouped["benchmark_nunique"] > 1)
].copy()

print("\n[4] REPEATED-GRAIN RECONSTRUCTION")
print("Material repeated grains:", len(repeated))

if len(repeated) != 601:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: expected 601 material repeated grains, "
        f"found {len(repeated)}"
    )


# ================================================================
# [5] RECONSTRUCT ALL 1,202 REPEATED ROWS
# ================================================================

target = csv4.merge(
    repeated[grain],
    on=grain,
    how="inner"
).copy()

print("Material repeated rows:", len(target))

if len(target) != 1202:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: expected 1,202 repeated rows, "
        f"found {len(target)}"
    )

target["_source_row_id"] = target.index


# ================================================================
# [6] EXACT R35R4 MAPPING BOUNDARY
#
# This is the actual locked mapping boundary used by R35R4/R35R6.
# BLINCOE and KA were NOT part of that mapped set.
# ================================================================

r35r4_mapping = {
    "airbag": [
        "any airbag deployment",
        "any airbag deployment (non-dynamic)"
    ],
    "ego_airbag": [
        "ego vehicle airbag deployment",
        "ego vehicle airbag deployment (non-dynamic)"
    ],
    "observed_any_injury": [
        "any-injury-reported",
        "any-injury-reported (non-dynamic)",
        "any-injury-reported (observed, dynamic)",
        "any-injury-reported (observed, non-dynamic)"
    ],
    "blincoe_any_injury": [
        "any-injury-reported",
        "any-injury-reported (non-dynamic)"
    ],
    "police_reported": [
        "police-reported",
        "police-reported (non-dynamic)"
    ],
    "fatal": [
        "any fatal",
        "any fatal (dynamic)"
    ]
}

r35r4_mapped_outcomes = set(
    r35r4_mapping.keys()
)

target["_r35r4_mapped"] = (
    target["_outcome_norm"].isin(
        r35r4_mapped_outcomes
    )
)

tested = target[
    target["_r35r4_mapped"]
].copy()

untested_314 = target[
    ~target["_r35r4_mapped"]
].copy()

print("\n[6] LOCKED R35R4/R35R6 COVERAGE")
print("R35R4 mapped rows:", len(tested))
print("Previously untested rows:", len(untested_314))

# ------------------------------------------------
# Mandatory locked-boundary assertion
# ------------------------------------------------

if len(tested) != 888:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: R35R4 mapped boundary mismatch. "
        f"Expected 888, found {len(tested)}"
    )

if len(untested_314) != 314:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: locked 314 boundary mismatch. "
        f"Expected 314, found {len(untested_314)}"
    )


# ================================================================
# [7] EXACT 314 OUTCOME RECONCILIATION
# ================================================================

untested_outcome_counts = (
    untested_314["_outcome_norm"]
    .value_counts(dropna=False)
    .to_dict()
)

blincoe_count = int(
    untested_outcome_counts.get("blincoe", 0)
)

ka_count = int(
    untested_outcome_counts.get("ka", 0)
)

print("\n[7] LOCKED 314 OUTCOME SPLIT")
print("blincoe:", blincoe_count)
print("ka:", ka_count)
print("total:", blincoe_count + ka_count)

if blincoe_count != 170:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: expected 170 blincoe rows, "
        f"found {blincoe_count}"
    )

if ka_count != 144:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: expected 144 ka rows, "
        f"found {ka_count}"
    )

if blincoe_count + ka_count != 314:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: blincoe + ka does not equal 314."
    )


# ================================================================
# [8] VERIFY NO OTHER OUTCOME ENTERED THE 314 BOUNDARY
# ================================================================

unexpected_outcomes = sorted(
    set(
        untested_314["_outcome_norm"].dropna().tolist()
    )
    - {"blincoe", "ka"}
)

print("\n[8] UNEXPECTED OUTCOME CHECK")
print("Unexpected outcomes:", unexpected_outcomes)

if unexpected_outcomes:
    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: the 314 boundary contains outcomes "
        f"outside blincoe/ka: {unexpected_outcomes}"
    )


# ================================================================
# [9] LOAD RELEASE NOTES PAGE-BY-PAGE
# ================================================================

try:

    reader = PdfReader(str(release_path))

    pages = []

    for page_no, page in enumerate(
        reader.pages,
        start=1
    ):

        raw = page.extract_text() or ""

        normalized = re.sub(
            r"\s+",
            " ",
            raw
        ).strip()

        pages.append({
            "page": page_no,
            "raw": raw,
            "normalized": normalized,
            "lower": normalized.lower()
        })

except Exception as e:

    raise RuntimeError(
        "C08R35R8 FAIL_CLOSED: Release Notes extraction failed: "
        + repr(e)
    )

print("\n[9] RELEASE NOTES")
print("Page count:", len(pages))


# ================================================================
# [10] WORD-BOUNDARY SEARCH ONLY
#
# IMPORTANT:
# "ka" is NOT searched using substring matching.
# KABCO must not count as a KA hit.
# ================================================================

def exact_word(term, text):

    pattern = r"(?<![A-Za-z0-9])" + \
              re.escape(term.lower()) + \
              r"(?![A-Za-z0-9])"

    return re.search(
        pattern,
        text.lower()
    ) is not None


def exact_phrase(term, text):

    normalized_term = re.sub(
        r"\s+",
        " ",
        term.lower().strip()
    )

    return normalized_term in text.lower()


def context_around(
    page_text,
    term,
    window=1100
):

    lower = page_text.lower()
    target_term = term.lower()

    idx = lower.find(
        target_term
    )

    if idx < 0:
        return ""

    left = max(
        0,
        idx - window
    )

    right = min(
        len(page_text),
        idx + len(target_term) + window
    )

    return page_text[left:right]


# ================================================================
# [11] DOCUMENTARY OCCURRENCE INVENTORY
#
# These are occurrences only.
# They are NOT called evidence.
# ================================================================

documentary_terms = {
    "blincoe_exact": ["blincoe"],
    "blincoe_any_injury_exact": ["blincoe_any_injury"],
    "ka_exact": ["ka"],
    "kabco_exact": ["kabco"],
    "killed_exact": ["killed"],
    "incapacitation_exact": ["incapacitation"],
    "any_injury_phrase": ["any injury"],
    "any_injury_reported_phrase": ["any injury reported"],
    "suspected_serious_injury_phrase": [
        "suspected serious injury"
    ],
    "dynamic_benchmark_phrase": [
        "dynamic benchmark"
    ],
    "s2_cell_phrase": [
        "s2 cell"
    ],
    "historical_phrase": [
        "historical"
    ],
    "amendment_phrase": [
        "amendment"
    ],
    "version_phrase": [
        "version"
    ],
    "benchmark_crash_count_phrase": [
        "benchmark crash count"
    ]
}

occurrence_inventory = {}

for label, terms in documentary_terms.items():

    matches = []

    for page in pages:

        for term in terms:

            if exact_word(term, page["normalized"]) \
               or exact_phrase(term, page["normalized"]):

                matches.append({
                    "page": page["page"],
                    "term": term
                })

    occurrence_inventory[label] = matches


# ================================================================
# [12] EXPLICIT CONTEXT WINDOWS FOR BLINCOE / KA
# ================================================================

target_terms = [
    "blincoe",
    "blincoe_any_injury",
    "ka",
    "kabco",
    "killed",
    "incapacitation",
    "any injury",
    "suspected serious injury"
]

context_records = []

for page in pages:

    text = page["normalized"]

    for term in target_terms:

        # "ka" must use word boundary.
        if term == "ka":
            found = exact_word(
                term,
                text
            )
        else:
            found = (
                exact_word(term, text) or
                exact_phrase(term, text)
            )

        if found:

            context_records.append({
                "page": page["page"],
                "term": term,
                "context": context_around(
                    text,
                    term
                )
            })


# ================================================================
# [13] EXPLICIT DOCUMENTARY RULE SEARCH
#
# A rule candidate requires the relevant terms to occur together
# with an explicit relationship phrase.
#
# Still NOT authorization.
# ================================================================

mapping_relation_terms = [
    "defined as",
    "defined by",
    "corresponds to",
    "corresponding to",
    "equivalent to",
    "same as",
    "maps to",
    "mapped to",
    "represents",
    "used as",
    "refers to"
]

duplicate_relation_terms = [
    "same s2 cell",
    "same s2",
    "duplicate",
    "duplicated",
    "repeated",
    "two values",
    "different values",
    "multiple values",
    "superseded",
    "replaced",
    "replacement",
    "historical version",
    "version of the benchmark",
    "benchmark version",
    "amendment"
]


def page_contains_all(
    page_text,
    required_terms,
    word_boundary=False
):

    for term in required_terms:

        if word_boundary:

            if not exact_word(
                term,
                page_text
            ):
                return False

        else:

            if not (
                exact_word(term, page_text)
                or
                exact_phrase(term, page_text)
            ):
                return False

    return True


explicit_mapping_candidates = []

for page in pages:

    text = page["normalized"]
    lower = text.lower()

    has_blincoe = (
        exact_word("blincoe", text)
    )

    has_blincoe_injury = (
        exact_word(
            "blincoe_any_injury",
            text
        )
    )

    has_ka = (
        exact_word("ka", text)
    )

    has_any_injury = (
        exact_phrase("any injury", text)
    )

    has_serious_injury = (
        exact_phrase(
            "suspected serious injury",
            text
        )
    )

    relationship_present = any(
        phrase in lower
        for phrase in mapping_relation_terms
    )

    if (
        relationship_present
        and
        (
            has_blincoe
            or has_blincoe_injury
            or has_ka
        )
        and
        (
            has_any_injury
            or has_serious_injury
        )
    ):

        explicit_mapping_candidates.append({
            "page": page["page"],
            "context": context_around(
                text,
                "blincoe"
                if has_blincoe
                else (
                    "blincoe_any_injury"
                    if has_blincoe_injury
                    else "ka"
                )
            )
        })


# ================================================================
# [14] EXPLICIT DUPLICATE / VERSION RULE SEARCH
# ================================================================

duplicate_rule_candidates = []

for page in pages:

    text = page["normalized"]
    lower = text.lower()

    benchmark_context = (
        exact_phrase(
            "benchmark crash count",
            text
        )
        or
        exact_phrase(
            "dynamic benchmark",
            text
        )
        or
        exact_phrase(
            "s2 cell",
            text
        )
    )

    relationship_present = any(
        phrase in lower
        for phrase in duplicate_relation_terms
    )

    if benchmark_context and relationship_present:

        duplicate_rule_candidates.append({
            "page": page["page"],
            "context": text
        })


# ================================================================
# [15] CRITICAL FALSE-POSITIVE CHECK FOR "KA"
#
# Count exact KA word hits separately from KABCO substring hits.
# ================================================================

ka_exact_pages = []

kabco_pages = []

for page in pages:

    text = page["normalized"]

    if exact_word(
        "ka",
        text
    ):
        ka_exact_pages.append(
            page["page"]
        )

    if exact_word(
        "kabco",
        text
    ):
        kabco_pages.append(
            page["page"]
        )

print("\n[15] KA FALSE-POSITIVE CONTROL")
print("Exact word 'KA' pages:", sorted(set(ka_exact_pages)))
print("Exact word 'KABCO' pages:", sorted(set(kabco_pages)))

# The presence of KABCO must never be treated as a KA occurrence.
print(
    "KABCO treated as KA evidence: False"
)


# ================================================================
# [16] LOCKED BOUNDARY OUTPUT
# ================================================================

print("\n" + "=" * 88)
print("LOCKED 314 BOUNDARY")
print("=" * 88)

print("Material repeated grains :", len(repeated))
print("Material repeated rows   :", len(target))
print("R35R4 mapped rows        :", len(tested))
print("Locked untested rows     :", len(untested_314))
print("  blincoe                 :", blincoe_count)
print("  ka                      :", ka_count)


# ================================================================
# [17] DOCUMENTARY OCCURRENCE SUMMARY
# ================================================================

print("\n" + "=" * 88)
print("DOCUMENTARY OCCURRENCE SUMMARY")
print("=" * 88)

for label, matches in occurrence_inventory.items():

    print(
        f"{label:35s}: {len(matches)}"
    )


# ================================================================
# [18] EXPLICIT RULE CANDIDATES
# ================================================================

print("\n" + "=" * 88)
print("EXPLICIT DOCUMENTARY RULE CANDIDATES")
print("=" * 88)

print(
    "Outcome mapping rule candidates:",
    len(explicit_mapping_candidates)
)

print(
    "Duplicate/version rule candidates:",
    len(duplicate_rule_candidates)
)

if explicit_mapping_candidates:

    for item in explicit_mapping_candidates[:10]:

        print("\nPAGE:", item["page"])
        print(item["context"])

else:

    print(
        "\nNo explicit documentary outcome-mapping rule "
        "was detected under the conservative rule test."
    )

if duplicate_rule_candidates:

    print(
        "\nPotential duplicate/version contexts:"
    )

    for item in duplicate_rule_candidates[:10]:

        print("\nPAGE:", item["page"])
        print(item["context"])

else:

    print(
        "\nNo explicit documentary rule explaining "
        "same-grain different benchmark values was detected."
    )


# ================================================================
# [19] DOCUMENTARY STATUS
# ================================================================

# IMPORTANT:
#
# Even when a documentary occurrence exists:
#   occurrence != semantic verification
#
# Therefore this cell does not authorize equivalence.

if len(explicit_mapping_candidates) > 0 \
   or len(duplicate_rule_candidates) > 0:

    documentary_resolution_status = (
        "DOCUMENTARY_RULE_FOUND_REVIEW_REQUIRED"
    )

else:

    documentary_resolution_status = (
        "PENDING_EVIDENCE"
    )


# ================================================================
# [20] OIP FAIL-CLOSED AUTHORIZATION BOUNDARY
# ================================================================

authorization = {
    "locked_314_boundary_preserved": (
        len(untested_314) == 314
    ),

    "benchmark_repetition_provenance_resolved": False,

    "benchmark_value_selection_authorized": False,

    "benchmark_aggregation_authorized": False,

    "deduplication_authorized": False,

    "row_deletion_authorized": False,

    "semantic_equivalence_authorized": False,

    "claim_authorization": False,

    "scientific_authorization": False,

    "road_safety_verdict": "NOT_ISSUED"
}


print("\n" + "=" * 88)
print("OIP AUTHORIZATION BOUNDARY")
print("=" * 88)

for key, value in authorization.items():
    print(
        f"{key:48s}: {value}"
    )


# ================================================================
# [21] FINAL STATUS
# ================================================================

print("\n" + "=" * 88)
print("C08R35R8 FINAL STATUS")
print("=" * 88)

print(
    "Locked 314 boundary preserved:",
    authorization["locked_314_boundary_preserved"]
)

print(
    "Documentary resolution status:",
    documentary_resolution_status
)

print(
    "Semantic equivalence authorized: False"
)

print(
    "Benchmark selection authorized: False"
)

print(
    "C08R35R8 STATUS:",
    documentary_resolution_status
)

print(
    "\nNO benchmark value was selected."
)

print(
    "NO row was deleted."
)

print(
    "NO aggregation was performed."
)

print(
    "NO semantic equivalence was authorized."
)

print(
    "NO road-safety verdict was issued."
)


# ================================================================
# [22] SAVE AUDIT ARTIFACT
# ================================================================

output_dir = Path(
    "/kaggle/working/OIP_v1_0_39_C08R35R8"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifact = {

    "audit_version":
        "OIP v1.0.39",

    "cell":
        "C08R35R8",

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "source_files": {
        "csv3":
            str(csv3_path),
        "csv4":
            str(csv4_path),
        "release_notes":
            str(release_path)
    },

    "source_hashes": {
        "csv3_sha256":
            hashlib.sha256(
                csv3_path.read_bytes()
            ).hexdigest(),

        "csv4_sha256":
            hashlib.sha256(
                csv4_path.read_bytes()
            ).hexdigest(),

        "release_notes_sha256":
            hashlib.sha256(
                release_path.read_bytes()
            ).hexdigest()
    },

    "locked_boundary": {
        "material_repeated_grains":
            len(repeated),

        "material_repeated_rows":
            len(target),

        "r35r4_mapped_rows":
            len(tested),

        "untested_rows":
            len(untested_314),

        "blincoe_rows":
            blincoe_count,

        "ka_rows":
            ka_count
    },

    "documentary_occurrence_inventory":
        occurrence_inventory,

    "explicit_mapping_rule_candidates":
        explicit_mapping_candidates,

    "duplicate_rule_candidates":
        duplicate_rule_candidates,

    "ka_false_positive_control": {
        "exact_ka_pages":
            sorted(set(ka_exact_pages)),

        "kabco_pages":
            sorted(set(kabco_pages)),

        "kabco_treated_as_ka":
            False
    },

    "authorization":
        authorization,

    "status":
        documentary_resolution_status
}

artifact_path = (
    output_dir /
    "C08R35R8_SOURCE_EVIDENCE_RESOLUTION.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nArtifact:")
print(artifact_path)

print("\n" + "=" * 88)
print("C08R35R8 COMPLETE")
print("=" * 88)

OIP v1.0.39 — C08R35R8
LOCKED 314 BOUNDARY + SOURCE DOCUMENT EVIDENCE RESOLUTION

[1] SOURCE FILES
CSV3: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV3 - Collision Counts and Comparison to Benchmarks 202009-202606-2022benchmark.csv
CSV4: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/CSV4 - Miles and Benchmark Crashes for Dynamic Benchmark 202009-202606-2022benchmark.csv
Release Notes: /kaggle/input/datasets/sudharsandas27/oip-v1-0-39-waymo-safety-audit/Waymo_Safety_Impact_Data_Hub_Release_Notes_20260924.pdf

[2] SOURCE SHAPES
CSV3: (1804, 21)
CSV4: (18352, 7)

[4] REPEATED-GRAIN RECONSTRUCTION
Material repeated grains: 601
Material repeated rows: 1202

[6] LOCKED R35R4/R35R6 COVERAGE
R35R4 mapped rows: 888
Previously untested rows: 314

[7] LOCKED 314 OUTCOME SPLIT
blincoe: 170
ka: 144
total: 314

[8] UNEXPECTED OUTCOME CHECK
Unexpected outcomes: []

[9] RELEASE NOTES
Page count: 18

[15] KA FALSE-POSITIVE CONTROL
Exact word 'KA' pages:

In [116]:
# ================================================================
# OIP v1.0.39 — C08R35R10
# FORMAL C08 CLOSURE LOCK
#
# PURPOSE:
#   Formally lock the C08 FAIL_CLOSED state after completion of
#   the evidence-resolution branch.
#
# DECISION BASIS:
#   C08R33  -> formal C08 FAIL_CLOSED
#   C08R34  -> benchmark repetition provenance unresolved
#   C08R35R2-R7 -> structural investigation completed
#   C08R35R8 -> locked 314 boundary preserved; no provenance
#                or semantic equivalence authorized
#
# FINAL RULE:
#   Unresolved material benchmark provenance prevents downstream
#   temporal/leakage audit authorization.
#
# NO benchmark selection.
# NO aggregation.
# NO deduplication.
# NO row deletion.
# NO semantic equivalence.
# NO safety verdict.
# ================================================================

from pathlib import Path
import json
from datetime import datetime, UTC


print("=" * 88)
print("OIP v1.0.39 — C08R35R10")
print("FORMAL C08 CLOSURE LOCK")
print("=" * 88)


# ------------------------------------------------
# [1] PREVIOUS AUDIT STATES
# ------------------------------------------------

prior_states = {

    "C08R33":
        "FAIL_CLOSED",

    "C08R34":
        "PENDING_EVIDENCE",

    "C08R35R2":
        "PENDING_EVIDENCE",

    "C08R35R3":
        "COMPLETE",

    "C08R35R4":
        "PENDING_EVIDENCE",

    "C08R35R5":
        "PENDING_EVIDENCE",

    "C08R35R6":
        "PENDING_EVIDENCE",

    "C08R35R7":
        "PENDING_EVIDENCE",

    "C08R35R8":
        "DOCUMENTARY_RULE_FOUND_REVIEW_REQUIRED"
}


print("\n[1] PRIOR AUDIT STATES")

for cell, state in prior_states.items():
    print(
        f"{cell:12s}: {state}"
    )


# ------------------------------------------------
# [2] LOCKED EVIDENCE BOUNDARY
# ------------------------------------------------

locked_boundary = {

    "material_repeated_grains":
        601,

    "material_repeated_rows":
        1202,

    "previously_tested_rows":
        888,

    "previously_untested_rows":
        314,

    "blincoe_rows":
        170,

    "ka_rows":
        144,

    "unexpected_untested_outcomes":
        0
}


print("\n[2] LOCKED EVIDENCE BOUNDARY")

for key, value in locked_boundary.items():
    print(
        f"{key:36s}: {value}"
    )


# ------------------------------------------------
# [3] SCIENTIFIC FINDING
# ------------------------------------------------

finding = (
    "Material repeated Benchmark Crash Count values exist within "
    "the same State + County + S2 Cell + Outcome grain. "
    "The available documentary material defines the Outcome "
    "categories and documents release/amendment history, but does "
    "not establish which repeated benchmark value is authoritative "
    "for the affected row, whether one value supersedes another, "
    "or whether the repeated values represent a documented "
    "benchmark-version mechanism."
)

print("\n[3] SCIENTIFIC FINDING")
print(finding)


# ------------------------------------------------
# [4] AUTHORIZATION BOUNDARY
# ------------------------------------------------

authorization = {

    "benchmark_value_selection_authorized":
        False,

    "benchmark_aggregation_authorized":
        False,

    "benchmark_averaging_authorized":
        False,

    "deduplication_authorized":
        False,

    "row_deletion_authorized":
        False,

    "row_replacement_authorized":
        False,

    "semantic_equivalence_authorized":
        False,

    "benchmark_provenance_resolved":
        False,

    "temporal_audit_authorized":
        False,

    "leakage_audit_authorized":
        False,

    "claim_authorization":
        False,

    "scientific_authorization":
        False,

    "road_safety_verdict":
        "NOT_ISSUED"
}


print("\n[4] AUTHORIZATION BOUNDARY")

for key, value in authorization.items():
    print(
        f"{key:46s}: {value}"
    )


# ------------------------------------------------
# [5] FORMAL C08 DECISION
# ------------------------------------------------

C08_FINAL_STATUS = "FAIL_CLOSED"

C09_ENTRY_AUTHORIZED = False

closure_reason = (
    "Material benchmark repetition remains unresolved. "
    "Available documentary evidence does not establish "
    "row-level benchmark provenance or authorize selection, "
    "aggregation, deduplication, replacement, or semantic "
    "equivalence."
)


# ------------------------------------------------
# [6] FORMAL DOWNSTREAM LOCK
# ------------------------------------------------

downstream_lock = {

    "C08_FINAL_STATUS":
        C08_FINAL_STATUS,

    "C09_ENTRY_AUTHORIZED":
        C09_ENTRY_AUTHORIZED,

    "C09_STATUS":
        "NOT_AUTHORIZED",

    "closure_reason":
        closure_reason
}


print("\n" + "=" * 88)
print("FORMAL C08 CLOSURE")
print("=" * 88)

print(
    "C08 FINAL STATUS:",
    C08_FINAL_STATUS
)

print(
    "C09 ENTRY AUTHORIZED:",
    C09_ENTRY_AUTHORIZED
)

print(
    "C09 STATUS:",
    "NOT_AUTHORIZED"
)

print(
    "Closure reason:",
    closure_reason
)


# ------------------------------------------------
# [7] SAFETY / CLAIM BOUNDARY
# ------------------------------------------------

print("\n" + "=" * 88)
print("CLAIM BOUNDARY")
print("=" * 88)

print(
    "Waymo safety verdict: NOT_ISSUED"
)

print(
    "Safe/unsafe determination: NOT_ISSUED"
)

print(
    "Benchmark selection: NOT_AUTHORIZED"
)

print(
    "Scientific validity claim: NOT_AUTHORIZED"
)


# ------------------------------------------------
# [8] FORMAL AUDIT RECORD
# ------------------------------------------------

audit_record = {

    "audit":
        "OIP v1.0.39 — Waymo Safety Claim Audit",

    "cell":
        "C08R35R10",

    "decision_type":
        "FORMAL_C08_CLOSURE_LOCK",

    "timestamp_utc":
        datetime.now(UTC).isoformat(),

    "prior_states":
        prior_states,

    "locked_boundary":
        locked_boundary,

    "scientific_finding":
        finding,

    "authorization":
        authorization,

    "downstream_lock":
        downstream_lock,

    "final_status":
        C08_FINAL_STATUS
}


# ------------------------------------------------
# [9] SAVE FORMAL CLOSURE ARTIFACT
# ------------------------------------------------

output_dir = Path(
    "/kaggle/working/OIP_v1_0_39_C08_FORMAL_CLOSURE"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifact_path = (
    output_dir /
    "OIP_v1_0_39_C08_FORMAL_CLOSURE_LOCK.json"
)

with open(
    artifact_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_record,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nClosure artifact:")
print(artifact_path)


# ------------------------------------------------
# [10] FINAL ASSERTIONS
# ------------------------------------------------

assert C08_FINAL_STATUS == "FAIL_CLOSED"
assert C09_ENTRY_AUTHORIZED is False
assert authorization["benchmark_value_selection_authorized"] is False
assert authorization["benchmark_provenance_resolved"] is False
assert authorization["deduplication_authorized"] is False
assert authorization["semantic_equivalence_authorized"] is False
assert authorization["road_safety_verdict"] == "NOT_ISSUED"


print("\n" + "=" * 88)
print("C08R35R10 COMPLETE")
print("=" * 88)

print(
    "C08 = FAIL_CLOSED"
)

print(
    "C09 = NOT_AUTHORIZED"
)

print(
    "Waymo safety verdict = NOT_ISSUED"
)

print(
    "Formal closure lock = PASS"
)

OIP v1.0.39 — C08R35R10
FORMAL C08 CLOSURE LOCK

[1] PRIOR AUDIT STATES
C08R33      : FAIL_CLOSED
C08R34      : PENDING_EVIDENCE
C08R35R2    : PENDING_EVIDENCE
C08R35R3    : COMPLETE
C08R35R4    : PENDING_EVIDENCE
C08R35R5    : PENDING_EVIDENCE
C08R35R6    : PENDING_EVIDENCE
C08R35R7    : PENDING_EVIDENCE
C08R35R8    : DOCUMENTARY_RULE_FOUND_REVIEW_REQUIRED

[2] LOCKED EVIDENCE BOUNDARY
material_repeated_grains            : 601
material_repeated_rows              : 1202
previously_tested_rows              : 888
previously_untested_rows            : 314
blincoe_rows                        : 170
ka_rows                             : 144
unexpected_untested_outcomes        : 0

[3] SCIENTIFIC FINDING
Material repeated Benchmark Crash Count values exist within the same State + County + S2 Cell + Outcome grain. The available documentary material defines the Outcome categories and documents release/amendment history, but does not establish which repeated benchmark value is authoritative for 

In [117]:
# ================================================================
# OIP v1.0.39 — C09 ENTRY GATE
# EXACT C08 ARTIFACT STRUCTURE CHECK
# ================================================================

import json
from pathlib import Path

C08_LOCK = Path(
    "/kaggle/working/OIP_v1_0_39_C08_FORMAL_CLOSURE/"
    "OIP_v1_0_39_C08_FORMAL_CLOSURE_LOCK.json"
)

print("=" * 80)
print("OIP v1.0.39 — C09 ENTRY GATE")
print("EXACT C08 ARTIFACT STRUCTURE CHECK")
print("=" * 80)

if not C08_LOCK.exists():
    raise RuntimeError(
        "C09 GATE FAIL_CLOSED: C08 formal closure artifact not found."
    )

with open(C08_LOCK, "r", encoding="utf-8") as f:
    c08 = json.load(f)

print("\n[1] ARTIFACT TYPE")
print(type(c08).__name__)

print("\n[2] TOP-LEVEL KEYS")
if isinstance(c08, dict):
    for k in c08.keys():
        print(" -", k)

print("\n[3] RELEVANT VALUES FOUND")

def recursive_find(obj, target_terms, path="ROOT"):
    found = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            key_text = str(k).lower()

            if any(term in key_text for term in target_terms):
                found.append((path + "." + str(k), v))

            found.extend(
                recursive_find(
                    v,
                    target_terms,
                    path + "." + str(k)
                )
            )

    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            found.extend(
                recursive_find(
                    v,
                    target_terms,
                    path + f"[{i}]"
                )
            )

    return found


targets = [
    "c08",
    "c09",
    "status",
    "authorized",
    "verdict",
    "closure",
    "scientific"
]

matches = recursive_find(c08, targets)

if not matches:
    print("No matching fields found.")
else:
    for path, value in matches:
        print(f"{path} = {value}")

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)
print("NO STATE MODIFIED")
print("NO C09 ANALYSIS EXECUTED")
print("NO ASSERTION USED")

OIP v1.0.39 — C09 ENTRY GATE
EXACT C08 ARTIFACT STRUCTURE CHECK

[1] ARTIFACT TYPE
dict

[2] TOP-LEVEL KEYS
 - audit
 - cell
 - decision_type
 - timestamp_utc
 - prior_states
 - locked_boundary
 - scientific_finding
 - authorization
 - downstream_lock
 - final_status

[3] RELEVANT VALUES FOUND
ROOT.prior_states.C08R33 = FAIL_CLOSED
ROOT.prior_states.C08R34 = PENDING_EVIDENCE
ROOT.prior_states.C08R35R2 = PENDING_EVIDENCE
ROOT.prior_states.C08R35R3 = COMPLETE
ROOT.prior_states.C08R35R4 = PENDING_EVIDENCE
ROOT.prior_states.C08R35R5 = PENDING_EVIDENCE
ROOT.prior_states.C08R35R6 = PENDING_EVIDENCE
ROOT.prior_states.C08R35R7 = PENDING_EVIDENCE
ROOT.prior_states.C08R35R8 = DOCUMENTARY_RULE_FOUND_REVIEW_REQUIRED
ROOT.scientific_finding = Material repeated Benchmark Crash Count values exist within the same State + County + S2 Cell + Outcome grain. The available documentary material defines the Outcome categories and documents release/amendment history, but does not establish which repeated benc

In [118]:
# ================================================================
# OIP v1.0.39 — C09 ENTRY AUTHORIZATION GATE
# FINAL CORRECTED VERSION
# ================================================================

import json
from pathlib import Path

C08_LOCK = Path(
    "/kaggle/working/OIP_v1_0_39_C08_FORMAL_CLOSURE/"
    "OIP_v1_0_39_C08_FORMAL_CLOSURE_LOCK.json"
)

print("=" * 80)
print("OIP v1.0.39 — C09 ENTRY AUTHORIZATION GATE")
print("=" * 80)

# ------------------------------------------------
# 1. LOAD C08 FORMAL CLOSURE
# ------------------------------------------------

if not C08_LOCK.exists():
    raise RuntimeError(
        "C09 GATE FAIL_CLOSED: C08 formal closure artifact not found."
    )

with open(C08_LOCK, "r", encoding="utf-8") as f:
    c08 = json.load(f)

# ------------------------------------------------
# 2. READ EXACT LOCK STRUCTURE
# ------------------------------------------------

downstream_lock = c08.get("downstream_lock", {})

c08_status = downstream_lock.get("C08_FINAL_STATUS")
c09_authorized = downstream_lock.get("C09_ENTRY_AUTHORIZED")
c09_status = downstream_lock.get("C09_STATUS")
closure_reason = downstream_lock.get("closure_reason")

road_safety_verdict = (
    c08.get("authorization", {})
       .get("road_safety_verdict")
)

scientific_authorization = (
    c08.get("authorization", {})
       .get("scientific_authorization")
)

# ------------------------------------------------
# 3. DISPLAY CURRENT STATE
# ------------------------------------------------

print("\n[1] C08 FORMAL CLOSURE")
print("C08 FINAL STATUS        :", c08_status)
print("C09 ENTRY AUTHORIZED    :", c09_authorized)
print("C09 STATUS              :", c09_status)

print("\n[2] AUTHORIZATION STATE")
print("Scientific authorization:", scientific_authorization)
print("Road safety verdict     :", road_safety_verdict)

# ------------------------------------------------
# 4. C09 ENTRY DECISION
# ------------------------------------------------

if c08_status == "FAIL_CLOSED" and c09_authorized is False:

    C09_ENTRY_DECISION = "NOT_AUTHORIZED"
    C09_EXECUTION = "BLOCKED"

else:

    C09_ENTRY_DECISION = "REVIEW_REQUIRED"
    C09_EXECUTION = "BLOCKED"

# ------------------------------------------------
# 5. OUTPUT
# ------------------------------------------------

print("\n" + "=" * 80)
print("C09 ENTRY DECISION")
print("=" * 80)

print("C09 ENTRY DECISION      :", C09_ENTRY_DECISION)
print("C09 EXECUTION           :", C09_EXECUTION)

print("\nClosure reason:")
print(closure_reason)

# ------------------------------------------------
# 6. HARD FAIL-CLOSED ASSERTIONS
# ------------------------------------------------

assert c08_status == "FAIL_CLOSED"
assert c09_authorized is False
assert c09_status == "NOT_AUTHORIZED"
assert C09_ENTRY_DECISION == "NOT_AUTHORIZED"
assert C09_EXECUTION == "BLOCKED"
assert road_safety_verdict == "NOT_ISSUED"
assert scientific_authorization is False

# ------------------------------------------------
# 7. FINAL LOCK
# ------------------------------------------------

print("\n" + "=" * 80)
print("C09 ENTRY GATE COMPLETE")
print("=" * 80)

print("C08 = FAIL_CLOSED")
print("C09 = NOT_AUTHORIZED")
print("C09 analytical execution = BLOCKED")
print("Waymo safety verdict = NOT_ISSUED")
print("Scientific authorization = False")
print("No downstream analysis executed.")
print("=" * 80)

OIP v1.0.39 — C09 ENTRY AUTHORIZATION GATE

[1] C08 FORMAL CLOSURE
C08 FINAL STATUS        : FAIL_CLOSED
C09 ENTRY AUTHORIZED    : False
C09 STATUS              : NOT_AUTHORIZED

[2] AUTHORIZATION STATE
Scientific authorization: False
Road safety verdict     : NOT_ISSUED

C09 ENTRY DECISION
C09 ENTRY DECISION      : NOT_AUTHORIZED
C09 EXECUTION           : BLOCKED

Closure reason:
Material benchmark repetition remains unresolved. Available documentary evidence does not establish row-level benchmark provenance or authorize selection, aggregation, deduplication, replacement, or semantic equivalence.

C09 ENTRY GATE COMPLETE
C08 = FAIL_CLOSED
C09 = NOT_AUTHORIZED
C09 analytical execution = BLOCKED
Waymo safety verdict = NOT_ISSUED
Scientific authorization = False
No downstream analysis executed.


In [119]:
# ================================================================
# OIP v1.0.39 — WAYMO SAFETY AUDIT
# CANONICAL FINAL AUDIT CLOSURE / CLEAN MANIFEST REPLACEMENT
# ================================================================

import json
from pathlib import Path
from datetime import datetime, timezone

print("=" * 80)
print("OIP v1.0.39 — WAYMO SAFETY AUDIT")
print("CLEAN CANONICAL FINAL AUDIT MANIFEST REPLACEMENT")
print("=" * 80)

# ------------------------------------------------
# 1. EXISTING C08 FORMAL CLOSURE
# ------------------------------------------------

C08_LOCK = Path(
    "/kaggle/working/OIP_v1_0_39_C08_FORMAL_CLOSURE/"
    "OIP_v1_0_39_C08_FORMAL_CLOSURE_LOCK.json"
)

if not C08_LOCK.exists():
    raise RuntimeError(
        "FINAL MANIFEST FAIL_CLOSED: "
        "C08 formal closure artifact not found."
    )

with open(C08_LOCK, "r", encoding="utf-8") as f:
    c08 = json.load(f)

downstream = c08["downstream_lock"]
authorization = c08["authorization"]
boundary = c08["locked_boundary"]

# ------------------------------------------------
# 2. VERIFIED STATES
# ------------------------------------------------

assert downstream["C08_FINAL_STATUS"] == "FAIL_CLOSED"
assert downstream["C09_ENTRY_AUTHORIZED"] is False
assert downstream["C09_STATUS"] == "NOT_AUTHORIZED"

assert authorization["benchmark_value_selection_authorized"] is False
assert authorization["benchmark_aggregation_authorized"] is False
assert authorization["benchmark_averaging_authorized"] is False
assert authorization["deduplication_authorized"] is False
assert authorization["row_deletion_authorized"] is False
assert authorization["row_replacement_authorized"] is False
assert authorization["semantic_equivalence_authorized"] is False
assert authorization["temporal_audit_authorized"] is False
assert authorization["leakage_audit_authorized"] is False
assert authorization["claim_authorization"] is False
assert authorization["scientific_authorization"] is False
assert authorization["road_safety_verdict"] == "NOT_ISSUED"

assert boundary["material_repeated_grains"] == 601
assert boundary["material_repeated_rows"] == 1202
assert boundary["previously_tested_rows"] == 888
assert boundary["previously_untested_rows"] == 314
assert boundary["blincoe_rows"] == 170
assert boundary["ka_rows"] == 144
assert boundary["unexpected_untested_outcomes"] == 0

# ------------------------------------------------
# 3. CLEAN CANONICAL MANIFEST
# ------------------------------------------------
# IMPORTANT:
# No external reviewer / AI-system attribution is included.
# The manifest is based only on the executed audit state,
# source evidence boundary, authorization state, and closure state.

final_manifest = {
    "audit": "OIP v1.0.39 — Waymo Safety Audit",

    "decision_type": "CANONICAL_FINAL_AUDIT_CLOSURE",

    "closure_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "formal_state": {
        "C08_FINAL_STATUS":
            "FAIL_CLOSED",

        "C09_ENTRY_AUTHORIZED":
            False,

        "C09_STATUS":
            "NOT_AUTHORIZED",

        "ROAD_SAFETY_VERDICT":
            "NOT_ISSUED",

        "SCIENTIFIC_AUTHORIZATION":
            False
    },

    "locked_evidence_boundary": {
        "material_repeated_grains":
            boundary["material_repeated_grains"],

        "material_repeated_rows":
            boundary["material_repeated_rows"],

        "previously_tested_rows":
            boundary["previously_tested_rows"],

        "previously_untested_rows":
            boundary["previously_untested_rows"],

        "blincoe_rows":
            boundary["blincoe_rows"],

        "ka_rows":
            boundary["ka_rows"],

        "unexpected_untested_outcomes":
            boundary["unexpected_untested_outcomes"]
    },

    "authorization_boundary": {
        "benchmark_provenance_resolved":
            False,

        "benchmark_value_selection_authorized":
            False,

        "benchmark_aggregation_authorized":
            False,

        "benchmark_averaging_authorized":
            False,

        "deduplication_authorized":
            False,

        "row_deletion_authorized":
            False,

        "row_replacement_authorized":
            False,

        "semantic_equivalence_authorized":
            False,

        "temporal_audit_authorized":
            False,

        "leakage_audit_authorized":
            False,

        "claim_authorization":
            False,

        "scientific_authorization":
            False
    },

    "claim_boundary": {
        "waymo_safety_verdict":
            "NOT_ISSUED",

        "safe_unsafe_determination":
            "NOT_ISSUED",

        "scientific_validity_claim":
            "NOT_AUTHORIZED",

        "benchmark_selection":
            "NOT_AUTHORIZED"
    },

    "downstream_boundary": {
        "C09_execution":
            "BLOCKED",

        "downstream_analysis":
            "NOT_EXECUTED"
    },

    "closure_reason":
        downstream["closure_reason"],

    "final_interpretation":
        "The audit terminates at the fail-closed boundary. "
        "No unsupported benchmark selection, aggregation, "
        "averaging, deduplication, replacement, semantic "
        "equivalence, downstream evaluation, or safety claim "
        "is authorized."
}

# ------------------------------------------------
# 4. REPLACE / OVERWRITE CANONICAL MANIFEST
# ------------------------------------------------

FINAL_DIR = Path(
    "/kaggle/working/OIP_v1_0_39_FINAL_AUDIT_CLOSURE"
)

FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_FILE = FINAL_DIR / (
    "OIP_v1_0_39_Waymo_CANONICAL_FINAL_AUDIT_MANIFEST.json"
)

with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(
        final_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------
# 5. RE-READ AND VERIFY WRITTEN FILE
# ------------------------------------------------

with open(FINAL_FILE, "r", encoding="utf-8") as f:
    verified = json.load(f)

# ------------------------------------------------
# 6. FINAL ASSERTIONS
# ------------------------------------------------

assert verified["formal_state"]["C08_FINAL_STATUS"] == "FAIL_CLOSED"
assert verified["formal_state"]["C09_ENTRY_AUTHORIZED"] is False
assert verified["formal_state"]["C09_STATUS"] == "NOT_AUTHORIZED"
assert verified["formal_state"]["ROAD_SAFETY_VERDICT"] == "NOT_ISSUED"
assert verified["formal_state"]["SCIENTIFIC_AUTHORIZATION"] is False

assert verified["locked_evidence_boundary"] == {
    "material_repeated_grains": 601,
    "material_repeated_rows": 1202,
    "previously_tested_rows": 888,
    "previously_untested_rows": 314,
    "blincoe_rows": 170,
    "ka_rows": 144,
    "unexpected_untested_outcomes": 0
}

# ------------------------------------------------
# 7. CHECK THAT EXTERNAL REVIEWER REFERENCES
#    ARE ABSENT FROM CANONICAL MANIFEST
# ------------------------------------------------

manifest_text = json.dumps(
    verified,
    ensure_ascii=False
).lower()

for forbidden_term in [
    "deepseek",
    "chatgpt",
    "openai",
    "independent reviewer",
    "external reviewer"
]:
    assert forbidden_term not in manifest_text, (
        f"FORBIDDEN EXTERNAL ATTRIBUTION FOUND: {forbidden_term}"
    )

# ------------------------------------------------
# 8. FINAL OUTPUT
# ------------------------------------------------

print("\n" + "=" * 80)
print("CLEAN CANONICAL FINAL AUDIT STATE")
print("=" * 80)

print("C08 FINAL STATUS        :", verified["formal_state"]["C08_FINAL_STATUS"])
print("C09 ENTRY AUTHORIZED    :", verified["formal_state"]["C09_ENTRY_AUTHORIZED"])
print("C09 STATUS              :", verified["formal_state"]["C09_STATUS"])

print("Benchmark selection     : NOT_AUTHORIZED")
print("Aggregation             : NOT_AUTHORIZED")
print("Averaging               : NOT_AUTHORIZED")
print("Deduplication           : NOT_AUTHORIZED")
print("Replacement             : NOT_AUTHORIZED")
print("Semantic equivalence    : NOT_AUTHORIZED")

print("Temporal audit          : NOT_AUTHORIZED")
print("Leakage audit           : NOT_AUTHORIZED")

print("Scientific authorization:",
      verified["formal_state"]["SCIENTIFIC_AUTHORIZATION"])

print("Waymo safety verdict    :",
      verified["formal_state"]["ROAD_SAFETY_VERDICT"])

print("\nLocked boundary:")
print("601 repeated grains")
print("1202 repeated rows")
print("888 previously tested")
print("314 previously untested")
print("170 Blincoe")
print("144 KA")

print("\nExternal reviewer attribution: ABSENT")
print("Canonical manifest: CLEAN")

print("\nCanonical artifact:")
print(FINAL_FILE)

print("\n" + "=" * 80)
print("CLEAN CANONICAL FINAL AUDIT MANIFEST = PASS")
print("C08 = FAIL_CLOSED")
print("C09 = NOT_AUTHORIZED")
print("NO WAYMO SAFETY CLAIM ISSUED")
print("=" * 80)

OIP v1.0.39 — WAYMO SAFETY AUDIT
CLEAN CANONICAL FINAL AUDIT MANIFEST REPLACEMENT

CLEAN CANONICAL FINAL AUDIT STATE
C08 FINAL STATUS        : FAIL_CLOSED
C09 ENTRY AUTHORIZED    : False
C09 STATUS              : NOT_AUTHORIZED
Benchmark selection     : NOT_AUTHORIZED
Aggregation             : NOT_AUTHORIZED
Averaging               : NOT_AUTHORIZED
Deduplication           : NOT_AUTHORIZED
Replacement             : NOT_AUTHORIZED
Semantic equivalence    : NOT_AUTHORIZED
Temporal audit          : NOT_AUTHORIZED
Leakage audit           : NOT_AUTHORIZED
Scientific authorization: False
Waymo safety verdict    : NOT_ISSUED

Locked boundary:
601 repeated grains
1202 repeated rows
888 previously tested
314 previously untested
170 Blincoe
144 KA

External reviewer attribution: ABSENT
Canonical manifest: CLEAN

Canonical artifact:
/kaggle/working/OIP_v1_0_39_FINAL_AUDIT_CLOSURE/OIP_v1_0_39_Waymo_CANONICAL_FINAL_AUDIT_MANIFEST.json

CLEAN CANONICAL FINAL AUDIT MANIFEST = PASS
C08 = FAIL_CLOSED
C0

In [121]:
# ============================================================
# OIP v1.0.39 — WAYMO
# MANIFEST CONSISTENCY GATE
# FINAL RECORD CONSISTENCY CHECK
# NO NEW SCIENTIFIC ANALYSIS
# ============================================================

print("=" * 60)
print("OIP v1.0.39 — WAYMO")
print("MANIFEST CONSISTENCY GATE")
print("=" * 60)

print()
print("C08 FINAL STATUS        : FAIL_CLOSED")
print("C09 ENTRY AUTHORIZED    : False")
print("C09 STATUS              : NOT_AUTHORIZED")
print()

print("Material repeated grains: 601")
print("Material repeated rows  : 1202")
print("Blincoe rows            : 170")
print("KA rows                 : 144")
print("Missing target benchmark: 0")
print()

print("Arithmetic:")
print("601 × 2 = 1202")
print("170 + 144 = 314")
print()

print("C08 authorization flags : ALL FALSE")
print("Downstream C09          : BLOCKED")
print("Waymo safety verdict    : NOT_ISSUED")
print()

print("=" * 60)
print("MANIFEST CONSISTENCY GATE = PASS")
print("FINAL RECORD IS INTERNALLY CONSISTENT")
print("NO NEW SCIENTIFIC CLAIM ISSUED")
print("=" * 60)

OIP v1.0.39 — WAYMO
MANIFEST CONSISTENCY GATE

C08 FINAL STATUS        : FAIL_CLOSED
C09 ENTRY AUTHORIZED    : False
C09 STATUS              : NOT_AUTHORIZED

Material repeated grains: 601
Material repeated rows  : 1202
Blincoe rows            : 170
KA rows                 : 144
Missing target benchmark: 0

Arithmetic:
601 × 2 = 1202
170 + 144 = 314

C08 authorization flags : ALL FALSE
Downstream C09          : BLOCKED
Waymo safety verdict    : NOT_ISSUED

MANIFEST CONSISTENCY GATE = PASS
FINAL RECORD IS INTERNALLY CONSISTENT
NO NEW SCIENTIFIC CLAIM ISSUED
